In [ ]:
# ======================================================================================================================
#  GeoBridge-Flow  -  one-cell notebook for Kaggle (GPU T4 x2) and Google Colab
# ======================================================================================================================
#  Any-to-any conversion between images, text and speech for languages with little or no written data.
#
#    text   --grapheme-byte codec-->  z_txt --reader--\                              /--flow writer--> z_txt --> text
#    speech --tone-gated codec---->   z_aud --reader---+--> hub code z* (barycentres +---flow writer--> z_aud --> speech
#    image  --frozen vision tower-->  z_img --ODE-----/      of frozen image patches) \--flow writer--> z_img --> image
#
#  HOW TO RUN
#    Kaggle : Settings -> Accelerator "GPU T4 x2", Internet "On"  (Persistence "Files only" keeps the checkpoints
#             between sessions).  Paste this cell, run it.  Both GPUs are used: one batch per GPU in every
#             training step, and the frozen vision / speech encoders are run on both for feature extraction.
#    Colab  : Runtime -> Change runtime type -> T4 GPU.  Paste this cell, run it.  USE_DRIVE = True keeps the run
#             folder on Google Drive, so that a new session continues where the last one stopped.
#    Interrupted?  Run the cell again.  Downloads, the split file, the feature caches, checkpoints and finished
#             ablation variants are found on disk; training resumes at the exact step it had reached.
#    Session limit: the cell stops by itself TIME_BUDGET_HOURS after the session started, with the step it reached
#             saved, instead of being cut off (Kaggle ends a session after 12 hours).  Run it again to continue.
#    A first look: PRESET = "synth_fast" runs the whole pipeline on the controlled benchmark in minutes, with no
#             download.
#    Out of GPU memory?  A batch is 64 caption groups of the benchmark plus 64 extra ones per GPU, 13 of them
#             spoken.  OVERRIDES = {"extra_per_batch": 32} halves the extra part; use a new EXP_NAME with it.
#    What to watch: every validation of the transport phase prints one "[text]" line per direction - content F1,
#             BLEU and chrF++, each next to its chance level (the same outputs scored against the caption of another
#             image) - and "[sample]" lines (reference | output).  A score is evidence only by its distance from
#             chance: on Flickr8k a caption that ignores its source still gets about 1 BLEU and 15 chrF++.
#             Whether the model speaks is one line of "infer": "[infer] written speech heard by ..." gives what an
#             independent recogniser understands of the vocoder alone, the renderer alone, the speech codec's
#             reconstruction, speech written from text, and Yoruba speech translated into English speech.
#
#  CONTENTS
#   00. SETTINGS AND ENVIRONMENT   (the only part meant to be edited)
#   01. CONFIG
#   02. UTILITIES
#   03. TEXT UNITS (grapheme-byte tokenisation, no vocabulary)
#   04. AUDIO DSP (pitch, mel, MFCC, Griffin-Lim)
#   05. DATA: download
#   06. DATA: SynthWorld (controlled benchmark)
#   07. DATA: Flickr8k-Tri (real benchmark)
#   08. DATA: frozen front-ends and the one-time feature cache
#   09. DATA: dataset, persistent splits, loaders
#   10. MODEL: layers
#   11. MODEL: entropic optimal transport
#   12. MODEL: grapheme-byte text codec
#   13. MODEL: tone-gated speech codec
#   14. MODEL: barycentric visual interlingua (hub), explainer, reader
#   15. MODEL: hub (+) private flow writers
#   16. MODEL: preview pixel decoder
#   17. MODEL: GeoBridge (assembled)
#   18. LOSSES
#   19. LATENT CACHE (transport phase)
#   20. METRICS
#   21. TRAINER
#   22. INFERENCER (figures, internal views, saved outputs)
#   23. ABLATION STUDY
#   24. PIPELINE STAGES
#   25. RUN
# ======================================================================================================================
from __future__ import annotations

# ======================================================================================================================
# 00. SETTINGS
# ======================================================================================================================
PRESET = "flickr8k_yo_t4"        # benchmark + model size.  Real data: "flickr8k_yo_t4" (2 x T4), "flickr8k_yo",
                                 # "flickr8k_yo_cpu".  Controlled benchmark: "synth", "synth_fast", "synth_tiny".
                                 # The full real presets add to the training split about 23,700 Flickr30k photos
                                 # with English captions and 5,000 SpokenCOCO photos with 25,031 spoken captions
                                 # (OVERRIDES = {"extra_images": 0, "extra_speech": 0} trains on Flickr8k alone).
STAGES = ("prepare", "train", "eval", "infer")   # any of: prepare, train, eval, infer, ablate  (run in this order)
OVERRIDES = {}                   # any Config field, e.g. {"phase_a_steps": 8000, "bridge_frac": 0.05, "seed": 1}
EXP_NAME = "flickr8k_yo_t4_v6"   # name of the run folder ("" = the preset name).  A run folder is only ever resumed
                                 # with the settings it was started with: new settings need a new name.
EVAL_SPLIT = "test"              # split scored by "eval" and drawn by "infer"

ABLATION_PRESET = PRESET         # benchmark of the ablation study, run after the stages above ("" = none).
                                 # PRESET ablates the run above: that run is the full model and provides the
                                 # representation, so only variants are trained, cheapest first.  On 2 x T4 a
                                 # variant that keeps the representation costs about 2 h (groups read, decoder, island,
                                 # private, lora), one that changes it about 4.5 h (hub, tone, speech, data,
                                 # text_unit, bridge).
                                 # "synth_fast" / "synth": the controlled benchmark, minutes per variant.
ABLATION_GROUPS = ("island", "decoder")   # () = all groups
ABLATION_ONLY = ("bridge pairs only", "decoder frozen")   # () = every variant of those groups; else parts of names.
                                 # These two fit in the session after the main run.  In the last run the first
                                 # showed no effect of back-translation and the second was cut by the session
                                 # limit before it had test scores; both are worth measuring again now that the
                                 # Yoruba recordings are cleaned and the speech latent keeps the order of time.
ABLATION_SEEDS = (0,)            # e.g. (0, 1, 2) for mean +- std in the tables
ABLATION_OVERRIDES = {}          # Config fields of the ablation runs (on top of OVERRIDES when the preset is PRESET)

TIME_BUDGET_HOURS = 11.0         # stop cleanly this long after the session started (0 = never).  Leaves room for
                                 # the evaluation and the figures before Kaggle's 12-hour limit.

WORK_DIR = ""                    # runs/ (checkpoints, curves, figures, tables).  "" = /kaggle/working, /content or .
DATA_DIR = ""                    # downloads + feature caches (about 37 GB for the real benchmark).  "" = automatic
USE_DRIVE = False                # Colab only: put WORK_DIR on Google Drive
SHOW_FIGURES = True              # display the curves and the inferencer figures below the cell when done

# ======================================================================================================================
# 00. ENVIRONMENT  (Kaggle / Colab / anything else; installs only what is missing)
# ======================================================================================================================
import importlib.util as _ilu
import os as _os
import subprocess as _sp
import sys as _sys
import time as _time

IN_KAGGLE = _os.path.isdir("/kaggle/working")
IN_COLAB = "google.colab" in _sys.modules and not IN_KAGGLE


def _session_start() -> float:
    """When the session began, as well as it can be known: the hosted time limit counts from there, not from the
    moment this cell is run.  On Kaggle / Colab that is the start of the container; elsewhere, of this kernel."""
    now = _time.time()
    try:
        import psutil

        kernel = psutil.Process().create_time()
        box = psutil.Process(1).create_time() if (IN_KAGGLE or IN_COLAB) else kernel
        start = box if 0 <= now - box < 12 * 3600 else kernel
        return start if 0 <= now - start < 12 * 3600 else now
    except Exception:
        return now


SESSION_START = _session_start()


def _first_writable(*candidates: str) -> str:
    for c in candidates:
        try:
            _os.makedirs(c, exist_ok=True)
            return c
        except OSError:
            continue
    raise OSError(f"none of {candidates} is writable")


if not WORK_DIR:
    if IN_KAGGLE:
        WORK_DIR = "/kaggle/working/geobridge"            # kept as notebook output (20 GB limit: runs only)
    elif IN_COLAB and USE_DRIVE:
        from google.colab import drive as _drive

        _drive.mount("/content/drive")
        WORK_DIR = "/content/drive/MyDrive/geobridge"
    else:
        WORK_DIR = "/content/geobridge" if IN_COLAB else _os.path.abspath("geobridge_runs")
if not DATA_DIR:
    if IN_KAGGLE:
        DATA_DIR = _first_writable("/kaggle/temp/geobridge_data", "/tmp/geobridge_data")   # scratch disk, not output
    else:
        DATA_DIR = "/content/geobridge_data" if IN_COLAB else _os.path.abspath("geobridge_data")
_os.makedirs(WORK_DIR, exist_ok=True)
_os.makedirs(DATA_DIR, exist_ok=True)
_os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
_os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

_NEEDS_HUB = not PRESET.startswith("synth") or not (ABLATION_PRESET or "synth").startswith("synth")
_NEEDED = {"regex": "regex", "sacrebleu": "sacrebleu", "tqdm": "tqdm", "matplotlib": "matplotlib", "sklearn": "scikit-learn"}
if _NEEDS_HUB:
    _NEEDED.update({"transformers": "transformers", "huggingface_hub": "huggingface_hub", "pyarrow": "pyarrow",
                    "soundfile": "soundfile", "PIL": "pillow"})
if "ablate" in STAGES or ABLATION_PRESET:
    _NEEDED["tokenizers"] = "tokenizers"
_missing = [pip for mod, pip in _NEEDED.items() if _ilu.find_spec(mod) is None]
if _missing:
    print(f"[setup] installing {_missing}")
    _sp.run([_sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)


# ======================================================================================================================
# IMPORTS
# ======================================================================================================================
import argparse
import copy
import csv
import gc
import glob
import hashlib
import io
import json
import logging
import math
import multiprocessing
import os
import random
import re
import shutil
import sys
import time
import unicodedata
import warnings
import wave
from collections import Counter
from dataclasses import asdict, dataclass, fields
from typing import Any, Dict, Iterable, Iterator, List, Optional, Sequence, Tuple

import numpy as np
import regex
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Sampler
from tqdm import tqdm


# ======================================================================================================================
# 01. CONFIG
#     (geobridge/config.py)
# ======================================================================================================================
# Single source of truth for every hyper-parameter of GeoBridge-Flow.
#
# Everything that changes behaviour lives in :class:`Config`.  A config is
#   * built from a preset (``Config.preset("flickr8k_yo")``),
#   * overridden from the CLI (``--set key=value``) or from an ablation variant,
#   * saved next to every checkpoint, so a run is always reproducible from its folder.
# ======================================================================================================================

@dataclass
class Config:
    # ------------------------------------------------------------------ experiment
    exp_name: str = "geobridge"
    out_root: str = "runs"
    seed: int = 0
    device: str = "auto"                 # auto | cpu | cuda | cuda:0 | mps
    n_gpu: int = 0                       # GPUs used for training and feature extraction (0 = every visible one)
    step_batches: int = 0                # batches per optimisation step (0 = one per GPU; >1 on one device = accumulation)
    amp: bool = True                     # mixed precision (cuda only); OT always runs in fp32
    num_workers: int = 2
    torch_threads: int = 0               # 0 = leave PyTorch default

    # ------------------------------------------------------------------ data
    dataset: str = "flickr8k_yo"         # flickr8k_yo | synthworld
    data_root: str = "data"
    max_images: int = 0                  # >0: deterministic subset of images (quick runs)
    extra_images: int = 0                # extra image + hub-language TEXT pairs, training only (no speech, no other
                                         # language): -1 = all there are, N = the first N, 0 = none.  flickr8k_yo: the
                                         # Flickr30k photos that are not Flickr8k photos; synthworld: N more scenes
    extra_speech: int = 0                # extra images with hub-language text AND speech, training only: -1 = all,
                                         # N = the first N, 0 = none.  flickr8k_yo: the SpokenCOCO test part (5,000
                                         # images, 25,031 captions read by 1,766 speakers); synthworld: N more scenes
    lang_balance: bool = False           # every language has the same say in the losses of its modality (grounding,
                                         # sufficiency, credit, reading), however many of its utterances a batch
                                         # holds.  Measured on SynthWorld: worse on every count (the few island
                                         # views become a few very heavy, noisy ones); kept as an ablation only
    extra_per_batch: int = 0             # extra caption groups added to every batch (0 = as many as batch_size)
    extra_speech_share: float = 0.2      # ... share of them that are spoken ones, when both kinds exist.  A fifth is
                                         # about what the real pools hold (25k spoken of 143k extra captions).  Measured
                                         # on SynthWorld (island speech -> text; 74 % with extra text alone): a fifth
                                         # 77 %, a third 63 % (without the reader's exposure and augmentation), all 33 %
    listen_only: bool = False            # True: extra speech only LISTENS - its views are read into the hub code that
                                         # the text of the same caption and the image define, and take no part in the
                                         # terms that shape the hub (grounding, sufficiency, credit, minimality,
                                         # agreement).  Measured on SynthWorld: it undoes most of the damage of extra
                                         # speech that fills every extra place (island speech -> text 33 % -> 58 %),
                                         # but at a fifth of the places grounded speech does no damage and listening
                                         # is the worse of the two (68 % against 77 %).  For large shares; an ablation
    audio_anchor: str = "off"            # tokens of the speech latent that are laid out in time (AudioCodec.span_means):
                                         # "off"   none: free queries, as for text
                                         # "extra" a speaking path beside the codec: the tokens beyond the n_tokens
                                         #         every latent has are laid out in time and read by a decoder of
                                         #         their own; codec, content tokens and everything the hub reads are
                                         #         untouched (no gradient comes back from the speaking path)
                                         # "all"   every token, and the hub reads them all (measured: the hub cannot
                                         #         find content in a latent ordered by time; an ablation only)
    audio_private_slots: int = 0         # private tokens of the SPEECH latent (0 = private_slots).  An utterance has
                                         # more that no picture determines than a caption has - how every sound is
                                         # timed and said - and has to carry it to be said again
    highpass_hz: float = 0.0             # high-pass every recording before trimming, pitch tracking and feature
                                         # extraction (0 = off; 70 in the real presets: below the lowest voice pitch)
    # Role of every language.  A language can appear in several roles.
    hub_langs: Tuple[str, ...] = ("en",)        # (image, text, audio) triplets
    island_langs: Tuple[str, ...] = ("yo",)     # speech only; text is never seen in training
    text_only_langs: Tuple[str, ...] = ("hi",)  # written, no speech (script-robustness branch)
    bridge_frac: float = 0.2             # fraction of island speech that keeps its image pairing
    island_unpaired: bool = True         # feed the remaining island speech as an unpaired stream
    use_island_text: bool = False        # True = "written" upper bound (island text used in training)
    strict_nonparallel: bool = True      # a caption is visible in ONE language only during training, so no
                                         # sentence of a non-hub language has its translation in the train set
    island_caps: Tuple[int, ...] = (0,)  # caption indices the island language may use in training
    text_only_caps: Tuple[int, ...] = (4,)   # caption indices given to the text-only language(s)
    split_fracs: Tuple[float, float, float] = (0.90, 0.05, 0.05)  # only for images w/o official split
    max_text_bytes: int = 160
    max_graphemes: int = 128
    max_audio_sec: float = 10.0
    trim_silence_db: float = 35.0

    # synthetic controlled benchmark (dataset == "synthworld")
    synth_n_images: int = 2400
    synth_caps_per_image: int = 4
    synth_n_colors: int = 8
    synth_n_shapes: int = 8
    synth_grid: int = 4
    synth_n_speakers: int = 6
    synth_two_obj_prob: float = 0.35
    synth_zipf: float = 0.7              # object types follow a Zipf law (0 = uniform).  Uniform, independent
                                         # attributes are the one case where no unpaired alignment can work
    synth_feat_noise: float = 0.15
    synth_vision_dim: int = 64

    # ------------------------------------------------------------------ frozen front-ends
    vision_encoder: str = "google/siglip-base-patch16-224"   # any HF vision tower | "synthetic"
    vision_grid: int = 7                 # patch tokens are pooled to grid x grid (+1 global token)
    speech_encoder: str = "utter-project/mHuBERT-147"        # any HF wav2vec2/HuBERT | "mfcc"
    speech_sem_layers: Tuple[int, ...] = (8, 9)   # hidden states averaged -> segmental/semantic stream
    speech_spk_layers: Tuple[int, ...] = (1, 2)   # early layers, time-averaged -> paralinguistic vector
    speech_pool: int = 2                 # 50 Hz -> 25 Hz
    sample_rate: int = 16000
    n_mels: int = 80
    mel_spec: str = "internal"           # internal (50 fps, Griffin-Lim) | speecht5 (62.5 fps, pretrained HiFi-GAN vocoder)
    mel_hop: int = 320                   # hop of the internal mel
    feat_batch_size: int = 16

    # ------------------------------------------------------------------ latent geometry
    latent_dim: int = 128                # d: token width shared by hub and all modality latents
    hub_slots: int = 8                   # K_h: tokens of the visual interlingua
    private_slots: int = 8               # K_p: modality-private tokens (Gaussian at t = 0)

    # ------------------------------------------------------------------ text codec (N4)
    text_unit: str = "grapheme"          # grapheme (ours) | byte | bpe
    bpe_vocab: int = 2000
    max_bytes_per_grapheme: int = 8
    text_width: int = 256
    text_enc_layers: int = 4
    text_dec_layers: int = 4
    text_heads: int = 4
    latent_noise_max: float = 0.5        # noise-robust latent training (decoder sees z + sigma*eps)
    dropout: float = 0.0                 # residual dropout in the codecs, grounder, explainer and reader (never the flows).
                                         # Measured on SynthWorld: 0.1 destroys the two-object coverage; keep 0 with w_seek > 0
    enc_dropout: float = 0.0             # residual dropout in the text and speech ENCODERS only (grounder and explainer
                                         # stay deterministic).  Measured on SynthWorld: 0.1 destroys the two-object
                                         # coverage as well (15 % against 97 %); kept as an ablation, not as a remedy

    # ------------------------------------------------------------------ audio codec (N5)
    audio_width: int = 256
    audio_enc_layers: int = 4
    audio_dec_layers: int = 3
    audio_heads: int = 4
    tone_mode: str = "gated"             # gated (ours) | on (ungated) | off (single stream)
    tone_gate_init: float = 2.0          # logit at release -> gamma = 0.88
    gate_price: str = "uniform"          # who pays for an open gate: every language in the batch (uniform) | every
                                         # grounded utterance, for the gate of its language (evidence)
    speech_mask: float = 0.0             # time masking of the speech encoder input: max share of an utterance per span
    gate_release_frac: float = 0.5       # share of phase A during which the tonality gates are held open and
                                         # the slots have no null option ("structure first, sparsity later")
    bypass_dim: int = 64                 # paralinguistic (speaker / register) bypass code
    use_bypass: bool = True
    mel_width: int = 192                 # mel renderer

    # ------------------------------------------------------------------ hub
    hub_mode: str = "bary"               # bary (ours) | global | free | text
    hub_width: int = 256
    hub_layers: int = 2
    hub_heads: int = 4
    hub_temp: float = 0.07               # attention temperature over patches
    hub_inhibit: float = 2.0             # inhibition of return, in units of the full cosine range (0 = independent slots)
    hub_temp_start: float = 6.0          # attention temperature multiplier at step 0 (annealed to 1 while dense)
    ground_temp: float = 0.07            # InfoNCE temperature
    agree_mode: str = "mutual"           # text and speech of a caption pull each other's code (mutual) | text is the target (teach)
    canonical_text: bool = True          # transport phase: speech with a same-language transcript takes the text-grounded code
    read_mode: str = "amortized"         # amortized (image-free reader, one pass) | ode (backward flow of the spoke)

    # ------------------------------------------------------------------ spokes (writers)
    flow_width: int = 384
    flow_layers: int = 6
    flow_heads: int = 6
    lora_rank: int = 8                   # 0 disables language-factorised LoRA
    lora_alpha: float = 16.0
    private_noise: bool = True           # False: K_p private tokens are zeros (deterministic spokes)
    ode_steps: int = 4                   # steps per hop at test time
    train_ode_steps: int = 2             # steps of the writes done inside the training losses
    ode_solver: str = "midpoint"         # euler | midpoint | heun
    sample_temp: float = 0.6             # std of the private noise at generation time
    eval_temp: float = 0.5               # same, for the scored (deterministic-seeded) evaluation
    asr_model: str = "openai/whisper-base.en"   # recogniser that listens to the written speech in `infer`
                                         # (English hub language and SpeechT5 mel space only; "" = skip)
    draft_t0: float = 0.0                # start writing from the explainer's draft at time t0 (0 = from the source)
    t_sampling: str = "uniform"          # uniform | logit_normal
    exposure: str = "reader"             # read hubs the writers are also trained on: reader | ode | none
    p_read: float = 0.5                  # share of paired views written from a read hub instead of the grounded one
    hub_jitter: float = 0.15             # std (upper bound) of the noise added to the hub part of a writer's source
    read_noise: float = 0.0              # std (upper bound) of the noise on the latent the reader is trained on (phase B)
    read_aug: int = 2                    # time-masked encodings of every spoken training utterance kept in the latent
                                         # cache; the reader reads a random one of them (or the clean one) and must
                                         # give the same hub code.  Transport phase only: nothing upstream moves
    read_aug_span: float = 0.15          # ... each hides two spans of up to this share of the utterance
    w_dec: float = 1.0                   # write-latent exposure: the byte decoder is trained on latents written by the
                                         # text writer (0 = decoder frozen after phase A)
    p_dec_written: float = 0.75          # ... share of those latents that are written (the rest are encoder latents)
    w_reread: float = 1.0                # write-latent exposure of the READER: real hub codes are written as speech in
                                         # the spoken hub language(s) and read back (back-translation does this for
                                         # the island language only).  0 = off
    image_spoke: bool = True
    thumb_size: int = 64                 # preview pixel decoder (image latent -> RGB)
    pix_width: int = 96
    pix_contrast: float = 4.0            # preview loss: extra weight of pixels far from the mean colour (0 = plain L1)

    # ------------------------------------------------------------------ unpaired island speech: hub-space back-translation
    bt_weight: float = 1.0               # 0 = use the bridge pairs only
    bt_relay: float = 1.0                # relay term: text writers learn from codes read off synthetic island speech
    bt_warmup_frac: float = 0.2          # share of phase B before back-translation starts
    bt_rectify: float = 0.5              # how far pseudo hub codes are moved along the OT map onto real ones (0 = off)
    bt_eps: float = 0.05
    bt_bank_mult: int = 4                # real hub codes drawn per unpaired utterance

    # ------------------------------------------------------------------ fused-GW docking of the reader (ablation; off)
    dock_mode: str = "none"              # none | g2w | gw | w
    dock_marginal: str = "balanced"      # balanced (log-Sinkhorn) | semi (semi-relaxed: collapses, see docs)
    dock_eps: float = 0.05
    dock_outer_iters: int = 10
    dock_sinkhorn_iters: int = 50
    dock_alpha_start: float = 0.0        # weight of the model-induced Wasserstein cost
    dock_alpha_end: float = 0.8
    dock_anneal_frac: float = 0.6        # fraction of phase B over which alpha is annealed
    dock_warmup_frac: float = 0.1        # fraction of phase B before docking is switched on
    naive_cycle_weight: float = 0.0      # roadmap's ODE cycle; kept to show it is vacuous

    # ------------------------------------------------------------------ loss weights
    w_text_rec: float = 1.0
    w_audio_rec: float = 1.0
    w_mel: float = 1.0
    w_orth: float = 0.1
    w_gate: float = 0.01                 # "pay for pitch" sparsity on the tonality gate
    w_ground: float = 1.0
    w_agree: float = 0.5
    w_expl: float = 1.0                  # sufficiency: the hub code must predict the utterance latent
    w_seek: float = 0.5                  # counterfactual patch credit: pull each slot to the patch that would explain more
    seek_gain: float = 0.02              # predicted drop of unexplained variance at which that pull is half on
    seek_verify: bool = False            # also carry each proposed move out and drop it if it verifiably explains less
    w_read: float = 1.0                  # the image-free reader must predict the hub code
    w_min: float = 0.5                   # minimality: the hub code must be predictable by the reader
    w_pix: float = 1.0                   # preview pixel decoder (0 removes it and its thumbnails)
    w_cfm: float = 1.0
    w_dock: float = 1.0
    w_free_hub_var: float = 1.0          # variance floor for the unconstrained ("free") hub baseline
    text_replay: int = 0                 # extra text-only captions per phase-A step (text is cheap; codec converges faster)

    # ------------------------------------------------------------------ optimisation
    batch_size: int = 64
    island_batch_size: int = 64
    phase_a_steps: int = 20000           # representation phase: codecs + grounder
    phase_a_select: bool = False         # start the transport phase from the phase-A state with the best validation
                                         # hub losses (grounding + sufficiency + read) instead of the last one
    phase_b_steps: int = 40000           # transport phase: writers, exposure, back-translation
    lr: float = 3e-4
    lr_flow: float = 3e-4
    lr_gate: float = 0.05                # the tonality gates are scalars: they need a much larger step
    bridge_per_batch: int = 8            # bridge pairs are few: guarantee this many in every paired batch
    weight_decay: float = 0.01
    warmup_steps: int = 500
    grad_clip: float = 1.0
    ema_decay: float = 0.999
    log_every: int = 50
    eval_every: int = 2000
    save_every: int = 1000               # step checkpoints -> resume from any step
    keep_last: int = 3                   # number of step checkpoints kept (milestones are kept too)
    milestone_every: int = 10000
    eval_max_items: int = 400            # cap for the in-training generation metrics
    train_only: str = ""                 # e.g. "lora:yo" -> language-incremental docking
    init_from: str = ""                  # checkpoint to initialise weights from (phase reuse)

    # ------------------------------------------------------------------ derived helpers
    @property
    def n_tokens(self) -> int:
        return self.hub_slots + self.private_slots

    def private_of(self, mod: str) -> int:
        """Private tokens of a modality's latent (speech may have more than the others)."""
        return (self.audio_private_slots or self.private_slots) if mod == "aud" else self.private_slots

    def tokens_of(self, mod: str) -> int:
        """Tokens of a modality's latent: the hub slots and its private ones."""
        return self.hub_slots + self.private_of(mod)

    @property
    def max_tokens(self) -> int:
        return max(self.n_tokens, self.tokens_of("aud"))

    def hub_tokens_of(self, mod: str) -> int:
        """Tokens of a modality's latent that the hub reads, grounds and explains (the rest is for its decoder)."""
        return self.n_tokens if mod == "aud" and self.audio_anchor == "extra" else self.tokens_of(mod)

    @property
    def mel_fps(self) -> float:
        return 62.5 if self.mel_spec == "speecht5" else self.sample_rate / self.mel_hop

    @property
    def total_steps(self) -> int:
        return self.phase_a_steps + self.phase_b_steps

    @property
    def all_langs(self) -> Tuple[str, ...]:
        out = []
        for l in (*self.hub_langs, *self.island_langs, *self.text_only_langs):
            if l not in out:
                out.append(l)
        return tuple(out)

    @property
    def run_dir(self) -> str:
        return os.path.join(self.out_root, self.exp_name)

    @property
    def cache_dir(self) -> str:
        return os.path.join(self.data_root, "cache", self.dataset_tag)

    @property
    def extra_cache_dir(self) -> str:
        """Caches of the extra image-text pairs.  Separate from the benchmark's own cache, which they never touch."""
        n = "all" if self.extra_images < 0 else f"n{self.extra_images}"
        if self.dataset == "synthworld":
            return os.path.join(self.cache_dir, f"extra_{n}")
        return os.path.join(self.data_root, "cache", f"flickr30k_extra_{n}_b{self.max_text_bytes}")

    @property
    def extra_speech_cache_dir(self) -> str:
        """Caches of the extra spoken data (images, text, speech features)."""
        n = "all" if self.extra_speech < 0 else f"n{self.extra_speech}"
        if self.dataset == "synthworld":
            return os.path.join(self.cache_dir, f"extra_speech_{n}")
        return os.path.join(self.data_root, "cache", f"spokencoco_extra_{n}_b{self.max_text_bytes}_s{int(self.max_audio_sec)}")

    @property
    def dataset_tag(self) -> str:
        if self.dataset == "synthworld":
            key = (self.synth_n_images, self.synth_caps_per_image, self.synth_n_colors, self.synth_n_shapes,
                   self.synth_grid, self.synth_n_speakers, self.synth_two_obj_prob, self.synth_feat_noise,
                   self.synth_vision_dim, self.synth_zipf)
            return "synthworld_" + hashlib.md5(repr(key).encode()).hexdigest()[:8]
        return self.dataset + (f"_n{self.max_images}" if self.max_images else "")

    # ------------------------------------------------------------------ (de)serialisation
    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

    def save(self, path: str) -> None:
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        with open(path, "w") as f:
            json.dump(self.to_dict(), f, indent=2)

    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> "Config":
        known = {f.name: f for f in fields(cls)}
        kw = {}
        for k, v in d.items():
            if k not in known:
                continue
            if isinstance(v, list):
                v = tuple(v)
            kw[k] = v
        return cls(**kw)

    @classmethod
    def load(cls, path: str) -> "Config":
        with open(path) as f:
            return cls.from_dict(json.load(f))

    def clone(self, **overrides: Any) -> "Config":
        c = copy.deepcopy(self)
        c.update(overrides)
        return c

    def update(self, overrides: Dict[str, Any]) -> "Config":
        known = {f.name: f for f in fields(self)}
        for k, v in overrides.items():
            if k not in known:
                raise KeyError(f"unknown config field '{k}'")
            if k == "audio_anchor" and str(v).lower() in ("true", "false"):   # (it was a switch at first)
                v = "all" if str(v).lower() == "true" else "off"
            if k == "audio_anchor" and v not in ("off", "extra", "all"):
                raise ValueError(f"audio_anchor must be off, extra or all, not {v!r}")
            setattr(self, k, _coerce(v, getattr(self, k)))
        return self

    def signature(self, keys: Tuple[str, ...]) -> str:
        """Stable hash of a subset of fields (used to share phase-A checkpoints across ablations)."""
        d = self.to_dict()
        blob = json.dumps({k: d[k] for k in sorted(keys)}, sort_keys=True, default=str)
        return hashlib.md5(blob.encode()).hexdigest()[:10]

    # ------------------------------------------------------------------ presets
    @classmethod
    def preset(cls, name: str) -> "Config":
        if name not in PRESETS:
            raise KeyError(f"unknown preset '{name}'. available: {sorted(PRESETS)}")
        return cls().update(PRESETS[name])


def _coerce(value: Any, ref: Any) -> Any:
    """Cast CLI strings to the type of the current field value."""
    if not isinstance(value, str) or isinstance(ref, str):
        if isinstance(ref, tuple) and isinstance(value, list):
            return tuple(value)
        return value
    if isinstance(ref, bool):
        return value.lower() in ("1", "true", "yes", "y", "on")
    if isinstance(ref, int):
        return int(float(value))
    if isinstance(ref, float):
        return float(value)
    if isinstance(ref, tuple):
        items = [v for v in value.replace("(", "").replace(")", "").split(",") if v.strip() != ""]
        if len(ref) and not isinstance(ref[0], str):
            cast = type(ref[0])
            return tuple(cast(float(v)) if cast is int else cast(v) for v in items)
        return tuple(v.strip() for v in items)
    return value


# Fields that determine phase A (codecs + grounder).  Two configs with the same signature over these
# fields can share one phase-A checkpoint, which makes phase-B ablations several times cheaper.
PHASE_A_KEYS: Tuple[str, ...] = (
    "seed", "dataset", "max_images", "extra_images", "extra_speech", "extra_per_batch", "extra_speech_share", "listen_only", "lang_balance", "audio_anchor", "audio_private_slots", "highpass_hz", "hub_langs", "island_langs", "text_only_langs", "bridge_frac",
    "use_island_text", "island_unpaired", "strict_nonparallel", "island_caps", "text_only_caps", "vision_encoder", "vision_grid", "speech_encoder", "speech_sem_layers",
    "speech_pool", "latent_dim", "hub_slots", "private_slots", "text_unit", "bpe_vocab", "text_width",
    "text_enc_layers", "text_dec_layers", "latent_noise_max", "audio_width", "audio_enc_layers",
    "audio_dec_layers", "tone_mode", "bypass_dim", "use_bypass", "hub_width", "hub_layers",
    "hub_temp", "hub_inhibit", "hub_temp_start", "ground_temp", "agree_mode", "w_text_rec", "w_audio_rec", "w_mel", "w_orth", "w_gate", "w_ground",
    "w_agree", "w_expl", "w_seek", "seek_gain", "seek_verify", "w_read", "w_min", "dropout", "enc_dropout", "gate_price", "speech_mask", "phase_a_select", "w_pix", "text_replay", "thumb_size", "pix_width", "pix_contrast", "batch_size", "island_batch_size", "phase_a_steps", "lr", "weight_decay",
    "lr_gate", "bridge_per_batch", "gate_release_frac", "tone_gate_init",
    "warmup_steps", "synth_n_images", "synth_caps_per_image", "synth_n_colors", "synth_n_shapes",
    "synth_grid", "synth_n_speakers", "synth_two_obj_prob", "synth_feat_noise", "synth_vision_dim", "synth_zipf",
    "max_text_bytes", "max_graphemes", "max_audio_sec", "mel_width", "n_mels", "mel_spec",
)


_SYNTH = dict(
    dataset="synthworld", vision_encoder="synthetic", speech_encoder="mfcc", vision_grid=4,
    hub_langs=("sa",), island_langs=("sb",), text_only_langs=("sc",), text_only_caps=(1,),
    max_text_bytes=96, max_graphemes=64, max_audio_sec=4.0, num_workers=0, amp=False,
)

# What the real benchmark changes relative to the defaults.  Each item answers something measured in a full
# Flickr8k run on 2 x T4, and each either was validated on SynthWorld or cannot disturb the representation phase:
#   phase_a_select  the validation grounding loss is best after about 4,000 steps and rises afterwards, whatever
#                   the schedule; this only selects a state, it does not change the training dynamics
#   read_noise      the reader's validation error rises throughout the transport phase (the cached latents never
#                   change); transport phase only
#   gate_price      with a uniform price the Yoruba gate closed to 0: it has few image pairs to defend it
# Schedule: 8,000 + 10,000 steps.  In the second run every validation curve of the transport phase had peaked
# within 9,000 of its steps (text read+write at 8,000-9,000, image -> text at 2,000) and declined afterwards.
# Deliberately NOT here: dropout, anywhere upstream of the hub.  Measured on SynthWorld it destroys the two-object
# coverage that the credit loss buys: 13 % against 93 % in the codecs, grounder and explainer; 15 % against 97 %
# confined to the encoders.
_REAL = dict(mel_spec="speecht5", phase_a_select=True, read_noise=0.2, gate_price="evidence", highpass_hz=70.0,
             audio_anchor="extra", audio_private_slots=24)
# More data for the hub language is the answer to the overfitting of the representation phase: with 6,847 training
# images every encoder, trained from scratch, memorises the captions within 15 epochs.  All Flickr30k photos that
# are not Flickr8k photos (about 23,700, five English captions each, text only) are added to the training split;
# as many of their caption groups as of the benchmark's own join every batch.
# Speech is the weak side (English speech finds its image three times less often than English text, and the
# speech codec rarely generalised), so the SpokenCOCO test part joins too: 5,000 more images whose 25,031 captions
# are text AND speech, from 1,766 speakers against Flickr8k's 183.  Spoken captions take a fifth of the extra
# places of a batch, not more: on SynthWorld extra speech next to extra text improves what speech is grounded on
# and leaves the rest about where it was, while extra speech alone, filling the batch, halves the island's scores.
_MORE = dict(extra_images=-1, extra_speech=-1)

PRESETS: Dict[str, Dict[str, Any]] = {
    # ---- real benchmark: Flickr8k images + English text/speech + Yoruba speech (+ Hindi text)
    "flickr8k_yo": dict(_REAL, **_MORE, text_replay=128, phase_a_steps=8000, phase_b_steps=10000, eval_every=1000),
    # same benchmark on two 16 GB GPUs (Kaggle "GPU T4 x2"); also runs on a single T4 (Colab), one batch per step.
    # Every step consumes one batch per GPU, so the schedule is counted in optimiser steps, not batches.
    "flickr8k_yo_t4": dict(
        _REAL, **_MORE, text_replay=128, batch_size=64, island_batch_size=64, feat_batch_size=32,
        phase_a_steps=8000, phase_b_steps=8000, eval_every=1000, save_every=1000, milestone_every=8000,
        num_workers=3, eval_max_items=600,
    ),
    # same benchmark, sized for a laptop / CPU sanity run
    "flickr8k_yo_cpu": dict(
        _REAL, text_replay=96, max_images=1200, latent_dim=96, hub_slots=6, private_slots=6, text_width=192, text_enc_layers=3,
        text_dec_layers=3, audio_width=192, audio_enc_layers=3, audio_dec_layers=2, hub_width=192,
        flow_width=256, flow_layers=4, flow_heads=4, lora_rank=4, batch_size=48, island_batch_size=48,
        phase_a_steps=2500, phase_b_steps=3500, eval_every=1000, save_every=500, milestone_every=2000,
        eval_max_items=200, num_workers=2, mel_width=128,
        bridge_frac=0.2, warmup_steps=200,
    ),
    # ---- controlled procedural benchmark (ground truth known for every factor)
    "synth": dict(
        _SYNTH, latent_dim=64, hub_slots=4, private_slots=4, text_width=128, text_enc_layers=2,
        text_dec_layers=2, audio_width=128, audio_enc_layers=2, audio_dec_layers=2, hub_width=128,
        flow_width=192, flow_layers=4, flow_heads=4, lora_rank=4, batch_size=64, island_batch_size=64,
        phase_a_steps=3000, phase_b_steps=4000, eval_every=1000, save_every=500, milestone_every=2000,
        eval_max_items=300, mel_width=96, warmup_steps=200,
        bridge_frac=0.2,
    ),
    # small world + small model: a full run takes minutes on a CPU; used for the reference ablation
    "synth_fast": dict(
        _SYNTH, synth_n_images=1200, latent_dim=48, hub_slots=4, private_slots=4, text_width=96,
        text_enc_layers=2, text_dec_layers=2, audio_width=96, audio_enc_layers=2, audio_dec_layers=2,
        hub_width=96, hub_layers=2, flow_width=128, flow_layers=3, flow_heads=4, lora_rank=4, batch_size=48,
        island_batch_size=48, phase_a_steps=1500, phase_b_steps=2500, eval_every=500, save_every=500,
        milestone_every=1000, eval_max_items=120, mel_width=64,
        warmup_steps=100, bridge_frac=0.2, bypass_dim=32, torch_threads=2,
    ),
    # seconds-long smoke test used by tests/ and CI
    "synth_tiny": dict(
        _SYNTH, synth_n_images=240, latent_dim=32, hub_slots=3, private_slots=3, text_width=64,
        text_enc_layers=1, text_dec_layers=1, text_heads=2, audio_width=64, audio_enc_layers=1,
        audio_dec_layers=1, audio_heads=2, hub_width=64, hub_layers=1, hub_heads=2, flow_width=64,
        flow_layers=2, flow_heads=2, lora_rank=2, batch_size=16, island_batch_size=16, phase_a_steps=30,
        phase_b_steps=30, eval_every=30, save_every=20, milestone_every=40, log_every=10,
        eval_max_items=24, mel_width=48, warmup_steps=5,
        bridge_frac=0.25, bypass_dim=16, bridge_per_batch=3,
    ),
}


# ======================================================================================================================
# 02. UTILITIES
#     (geobridge/utils.py)
# ======================================================================================================================
# Small shared helpers: seeding, devices, JSON io, logging, parameter accounting.
# ======================================================================================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed % (2 ** 32))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_device(name: str = "auto") -> torch.device:
    if name == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    return torch.device(name)


def gpu_ids(cfg, device: torch.device) -> list:
    """GPUs a run may use: all visible ones (``cfg.n_gpu = 0``) or the first ``cfg.n_gpu``; [] off CUDA."""
    if device.type != "cuda" or device.index not in (None, 0):
        return [device.index] if device.type == "cuda" else []
    n = torch.cuda.device_count()
    return list(range(min(n, cfg.n_gpu) if cfg.n_gpu > 0 else n))


def worker_kwargs(num_workers: int) -> Dict[str, Any]:
    """DataLoader worker settings.  Workers are forked where the platform allows it: the dataset (memory-mapped
    feature arrays, and classes that may have been defined in a notebook cell) is then shared, not pickled."""
    if num_workers <= 0:
        return {"num_workers": 0}
    fork = "fork" in multiprocessing.get_all_start_methods()
    return {"num_workers": num_workers, **({"multiprocessing_context": "fork"} if fork else {})}


def get_logger(name: str = "geobridge", log_file: Optional[str] = None) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.propagate = False
    fmt = logging.Formatter("%(asctime)s | %(message)s", "%H:%M:%S")
    if not any(isinstance(h, logging.StreamHandler) and not isinstance(h, logging.FileHandler)
               for h in logger.handlers):
        sh = logging.StreamHandler(sys.stdout)
        sh.setFormatter(fmt)
        logger.addHandler(sh)
    if log_file:
        os.makedirs(os.path.dirname(log_file) or ".", exist_ok=True)
        if not any(isinstance(h, logging.FileHandler) and h.baseFilename == os.path.abspath(log_file)
                   for h in logger.handlers):
            fh = logging.FileHandler(log_file)
            fh.setFormatter(fmt)
            logger.addHandler(fh)
    return logger


def save_json(obj: Any, path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_json_default)
    os.replace(tmp, path)


def load_json(path: str) -> Any:
    with open(path, encoding="utf8") as f:
        return json.load(f)


def _json_default(o: Any) -> Any:
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, torch.Tensor):
        return o.detach().cpu().tolist()
    return str(o)


def count_params(module: torch.nn.Module, trainable_only: bool = False) -> int:
    return sum(p.numel() for p in module.parameters() if (p.requires_grad or not trainable_only))


def count_named(module: torch.nn.Module, must_contain: Iterable[str]) -> int:
    keys = tuple(must_contain)
    return sum(p.numel() for n, p in module.named_parameters() if any(k in n for k in keys))


def to_device(batch: Any, device: torch.device) -> Any:
    if isinstance(batch, torch.Tensor):
        return batch.to(device, non_blocking=True)
    if isinstance(batch, dict):
        return {k: to_device(v, device) for k, v in batch.items()}
    if isinstance(batch, (list, tuple)) and len(batch) and isinstance(batch[0], torch.Tensor):
        return type(batch)(to_device(v, device) for v in batch)
    return batch


class Averager:
    """Running mean of scalar dictionaries (used for tqdm postfixes and epoch summaries)."""

    def __init__(self) -> None:
        self.sum: Dict[str, float] = {}
        self.n: Dict[str, int] = {}

    def add(self, d: Dict[str, float]) -> None:
        for k, v in d.items():
            if v is None:
                continue
            v = float(v)
            if v != v:  # NaN
                continue
            self.sum[k] = self.sum.get(k, 0.0) + v
            self.n[k] = self.n.get(k, 0) + 1

    def mean(self) -> Dict[str, float]:
        return {k: self.sum[k] / max(1, self.n[k]) for k in self.sum}

    def reset(self) -> None:
        self.sum.clear()
        self.n.clear()


def grid_factor(n: int) -> tuple:
    """Most square (rows, cols) factorisation of n (used to lay out image-latent tokens)."""
    r = int(np.floor(np.sqrt(n)))
    while r > 1 and n % r:
        r -= 1
    return r, n // r


class quiet_transformers:
    """Context manager: load a pretrained model without the library's informational notices (a vision tower whose
    config carries text-token ids, a vocoder whose hub config names an older class, ...)."""

    def __enter__(self):
        from transformers.utils import logging as hf_logging

        self._log, self._level = hf_logging, hf_logging.get_verbosity()
        hf_logging.set_verbosity_error()
        return self

    def __exit__(self, *exc) -> bool:
        self._log.set_verbosity(self._level)
        return False


class OutOfTime(RuntimeError):
    """The session's time budget is used up.  Raised by the trainer right after it has saved the exact step it
    reached, so that running the same command again continues from there."""


_BUDGET: Dict[str, float] = {}


def set_time_budget(hours: float, since: Optional[float] = None) -> None:
    """Training stops cleanly ``hours`` after ``since`` (a ``time.time()`` value; default: now).  0 = no limit.

    Hosted notebooks end a session after a fixed time and may keep nothing of a run that is cut off; a run that
    stops by itself keeps its checkpoints, curves and tables."""
    _BUDGET.clear()
    if hours and hours > 0:
        _BUDGET["deadline"] = (time.time() if since is None else since) + 3600.0 * hours


def time_left() -> float:
    """Seconds until the time budget is used up (infinite when there is none)."""
    return _BUDGET.get("deadline", float("inf")) - time.time()


# A font per script that the default matplotlib font lacks: (a character of the script, font file, source).
_SCRIPT_FONTS = (
    ("\u0915", "NotoSansDevanagari-Regular.ttf",
     "https://raw.githubusercontent.com/notofonts/notofonts.github.io/main/fonts/NotoSansDevanagari/hinted/ttf/"
     "NotoSansDevanagari-Regular.ttf"),
)


_FONTS: Dict[str, Any] = {}


def _font_chain(font_dir: str) -> tuple:
    """(font fallback chain, scripts still without a font).  Scans the installed fonts once per process."""
    from matplotlib import font_manager, ft2font

    def covering(char: str) -> list:
        names = []
        for f in font_manager.fontManager.ttflist:
            try:
                if ord(char) in ft2font.FT2Font(f.fname).get_charmap():
                    names.append(f.name)
            except Exception:
                continue
        return names

    chain, missing = ["DejaVu Sans"], []
    for char, fname, url in _SCRIPT_FONTS:
        have = covering(char)
        if not have and font_dir:
            path = os.path.join(font_dir, fname)
            try:
                if not os.path.exists(path):
                    import urllib.request

                    os.makedirs(font_dir, exist_ok=True)
                    urllib.request.urlretrieve(url, path + ".tmp")
                    os.replace(path + ".tmp", path)
                font_manager.fontManager.addfont(path)
                have = [font_manager.FontProperties(fname=path).get_name()]
            except Exception:   # offline, or the mirror moved: boxes instead of glyphs, nothing else changes
                have = []
        if not have:
            missing.append(fname)
        chain += [n for n in have[:1] if n not in chain]
    known = {f.name for f in font_manager.fontManager.ttflist}
    chain += [n for n in ("FreeSerif", "FreeSans", "Noto Sans", "Arial Unicode MS") if n in known and n not in chain]
    return [n for n in chain if n in known], missing


def setup_matplotlib(font_dir: str = "") -> None:
    """Headless backend (outside notebooks) and a font fallback chain that covers every script of the benchmark.

    Figures label graphemes of each language.  Where the system has no font for a script (Kaggle and Colab images
    ship none for Devanagari) and ``font_dir`` is given, one small font file is fetched there, once.  Without a
    font the labels are drawn as boxes; either way the figure is written, and matplotlib's per-glyph warnings are
    silenced."""
    import matplotlib

    if "ipykernel" not in sys.modules:  # in a notebook the inline backend stays; figures are saved and closed
        matplotlib.use("Agg")
    logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
    warnings.filterwarnings("ignore", message=r"Glyph \d+ .*missing from", category=UserWarning)
    warnings.filterwarnings("ignore", message=r"Matplotlib currently does not support .* natively", category=UserWarning)
    if "chain" not in _FONTS or (font_dir and _FONTS["missing"]):
        _FONTS["chain"], _FONTS["missing"] = _font_chain(font_dir)
    if _FONTS["chain"]:
        matplotlib.rcParams["font.family"] = _FONTS["chain"]


def legend_if_any(ax, **kwargs) -> None:
    """``ax.legend`` only when something on the axis has a label (an empty legend raises a warning per figure)."""
    if ax.get_legend_handles_labels()[0]:
        ax.legend(**kwargs)


# ======================================================================================================================
# 03. TEXT UNITS (grapheme-byte tokenisation, no vocabulary)
#     (geobridge/text_units.py)
# ======================================================================================================================
# Tokenizer-free text units.
#
# ``grapheme``  (ours)  one unit per extended grapheme cluster; the unit is embedded from its UTF-8 bytes.
# ``byte``              one unit per UTF-8 byte (ByT5-style baseline).
# ``bpe``               byte-level BPE learned on the hub (high-resource) language only, i.e. the situation
#                       real pretrained vocabularies are in when they meet a new script.
#
# All three share one tensor layout so the text encoder is identical across the ablation:
# ``units``  LongTensor [L, M]  byte ids of each unit (PAD = 256), or the BPE id in column 0
# ``target`` LongTensor [T]     BOS + UTF-8 bytes + EOS  (the decoder is byte-level in every variant)
# ======================================================================================================================

PAD, BOS, EOS = 256, 257, 258
BYTE_VOCAB = 259

try:  # the `regex` package implements UAX #29 extended grapheme clusters (\X)
    import regex as _regex

    _GRAPHEME = _regex.compile(r"\X")
except Exception:  # pragma: no cover - fallback keeps the package importable without `regex`
    _regex = None
    _GRAPHEME = None

_EXTEND_CATS = {"Mn", "Mc", "Me"}
_VIRAMAS = {0x094D, 0x09CD, 0x0A4D, 0x0ACD, 0x0B4D, 0x0BCD, 0x0C4D, 0x0CCD, 0x0D4D, 0x0DCA}


def normalize_text(s: str) -> str:
    return unicodedata.normalize("NFC", " ".join(s.strip().split()))


def graphemes(s: str) -> List[str]:
    """Extended grapheme clusters.  Indic conjuncts (C + virama + C) are kept together."""
    if _GRAPHEME is not None:
        parts = _GRAPHEME.findall(s)
    else:
        parts = []
        for ch in s:
            if parts and (unicodedata.category(ch) in _EXTEND_CATS or ch in "‍‌"):
                parts[-1] += ch
            else:
                parts.append(ch)
    out: List[str] = []
    for p in parts:  # merge "…virama" + next consonant (UAX #29 GB9c, absent from older tables)
        if out and ord(out[-1][-1]) in _VIRAMAS:
            out[-1] += p
        else:
            out.append(p)
    return out


def _chunk(b: bytes, m: int) -> List[bytes]:
    return [b[i:i + m] for i in range(0, len(b), m)] or [b]


class TextUnitizer:
    """Turns strings into unit tensors and reports script-fairness statistics."""

    def __init__(self, mode: str = "grapheme", max_units: int = 128, max_bytes_per_unit: int = 8,
                 max_text_bytes: int = 160, bpe_path: Optional[str] = None) -> None:
        assert mode in ("grapheme", "byte", "bpe")
        self.mode = mode
        self.max_units = max_units
        self.m = max_bytes_per_unit if mode == "grapheme" else 1
        self.max_text_bytes = max_text_bytes
        self.bpe = None
        if mode == "bpe":
            from tokenizers import Tokenizer

            if not (bpe_path and os.path.exists(bpe_path)):
                raise FileNotFoundError("BPE tokenizer not trained yet; call TextUnitizer.train_bpe first")
            self.bpe = Tokenizer.from_file(bpe_path)
        self._cache: Dict[str, Tuple[torch.Tensor, torch.Tensor]] = {}

    # ------------------------------------------------------------------ BPE baseline
    @staticmethod
    def train_bpe(texts: Iterable[str], path: str, vocab_size: int = 2000) -> None:
        from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

        tok = Tokenizer(models.BPE(unk_token=None))
        tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
        tok.decoder = decoders.ByteLevel()
        trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=[],
                                      initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False)
        tok.train_from_iterator((normalize_text(t) for t in texts), trainer)
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        tok.save(path)

    @property
    def unit_vocab(self) -> int:
        return self.bpe.get_vocab_size() + 1 if self.mode == "bpe" else BYTE_VOCAB

    @property
    def unit_pad(self) -> int:
        return self.bpe.get_vocab_size() if self.mode == "bpe" else PAD

    # ------------------------------------------------------------------ segmentation
    def segment(self, s: str) -> List[bytes]:
        s = normalize_text(s)
        if self.mode == "grapheme":
            out: List[bytes] = []
            for g in graphemes(s):
                out.extend(_chunk(g.encode("utf8"), self.m))  # over-long clusters are split, never truncated
            return out
        if self.mode == "byte":
            return [bytes([b]) for b in s.encode("utf8")]
        return [t.encode("utf8") for t in self.bpe.encode(s).tokens]  # only used for statistics

    def encode(self, s: str) -> Tuple[torch.Tensor, torch.Tensor]:
        hit = self._cache.get(s)
        if hit is not None:
            return hit
        s_n = normalize_text(s)
        raw = s_n.encode("utf8")[: self.max_text_bytes]
        s_n = raw.decode("utf8", errors="ignore")  # never cut inside a code point
        raw = s_n.encode("utf8")
        if self.mode == "bpe":
            ids = self.bpe.encode(s_n).ids[: self.max_units] or [self.unit_pad]
            units = torch.tensor(ids, dtype=torch.long).unsqueeze(1)
        else:
            segs = self.segment(s_n)[: self.max_units] or [b" "]
            units = torch.full((len(segs), self.m), PAD, dtype=torch.long)
            for i, seg in enumerate(segs):
                units[i, : len(seg)] = torch.tensor(list(seg), dtype=torch.long)
        target = torch.tensor([BOS] + list(raw) + [EOS], dtype=torch.long)
        out = (units, target)
        if len(self._cache) < 500_000:
            self._cache[s] = out
        return out

    def collate(self, texts: Sequence[str]) -> Dict[str, torch.Tensor]:
        enc = [self.encode(t) for t in texts]
        L = max(u.shape[0] for u, _ in enc)
        T = max(t.shape[0] for _, t in enc)
        units = torch.full((len(enc), L, self.m), self.unit_pad, dtype=torch.long)
        mask = torch.zeros(len(enc), L, dtype=torch.bool)
        target = torch.full((len(enc), T), PAD, dtype=torch.long)
        for i, (u, t) in enumerate(enc):
            units[i, : u.shape[0]] = u
            mask[i, : u.shape[0]] = True
            target[i, : t.shape[0]] = t
        return {"units": units, "unit_mask": mask, "bytes": target}

    # ------------------------------------------------------------------ statistics (fertility)
    def fertility(self, texts: Sequence[str]) -> Dict[str, float]:
        """Units per word / per grapheme / per byte: the script-fairness numbers of the tokenizer study."""
        n_units = n_words = n_graph = n_bytes = 0
        for t in texts:
            t = normalize_text(t)
            n_units += len(self.segment(t))
            n_words += max(1, len(t.split()))
            n_graph += max(1, len(graphemes(t)))
            n_bytes += max(1, len(t.encode("utf8")))
        return {"units_per_word": n_units / max(1, n_words), "units_per_grapheme": n_units / max(1, n_graph),
                "units_per_byte": n_units / max(1, n_bytes), "bytes_per_grapheme": n_bytes / max(1, n_graph)}


def bytes_to_text(ids: Sequence[int]) -> str:
    out = []
    for i in ids:
        i = int(i)
        if i == EOS:
            break
        if i < 256:
            out.append(i)
    return bytes(out).decode("utf8", errors="ignore")


# ======================================================================================================================
# 04. AUDIO DSP (pitch, mel, MFCC, Griffin-Lim)
#     (geobridge/audio_dsp.py)
# ======================================================================================================================
# Dependency-light audio DSP: mel / MFCC, YIN pitch tracking, tone features, Griffin-Lim.
#
# The *tone stream* of GeoBridge-Flow is built here.  It is deliberately tiny and interpretable:
#
#     tone[t] = ( z-scored log-F0 , voiced flag , delta log-F0 , z-scored log-energy )
#
# log-F0 is standardised **per utterance over voiced frames**, which removes the speaker's pitch register
# (non-lexical in every language) and keeps the relative contour (lexical in tonal languages).  The removed
# register (mean, std of log-F0) travels in the paralinguistic bypass instead.
# ======================================================================================================================

try:
    import torchaudio.functional as AF
except Exception:  # pragma: no cover - a plain filterbank and resampler are used instead
    AF = None

_FB_CACHE: Dict[Tuple, torch.Tensor] = {}


def _mel_fb(sr: int, n_fft: int, n_mels: int) -> torch.Tensor:
    key = (sr, n_fft, n_mels)
    if key not in _FB_CACHE:
        if AF is not None:
            fb = AF.melscale_fbanks(n_fft // 2 + 1, 0.0, sr / 2, n_mels, sr, norm="slaney", mel_scale="slaney")
        else:  # pragma: no cover - plain triangular HTK filterbank
            f = torch.linspace(0, sr / 2, n_fft // 2 + 1)
            m = torch.linspace(0, 2595 * math.log10(1 + sr / 2 / 700), n_mels + 2)
            hz = 700 * (10 ** (m / 2595) - 1)
            fb = torch.zeros(n_fft // 2 + 1, n_mels)
            for i in range(n_mels):
                lo, c, hi = hz[i], hz[i + 1], hz[i + 2]
                fb[:, i] = torch.clamp(torch.minimum((f - lo) / (c - lo), (hi - f) / (hi - c)), min=0)
        _FB_CACHE[key] = fb
    return _FB_CACHE[key]


def stft_mag(wav: torch.Tensor, n_fft: int, hop: int) -> torch.Tensor:
    """wav [T] -> magnitude [frames, n_fft/2+1] (centered, Hann)."""
    win = torch.hann_window(n_fft, device=wav.device)
    spec = torch.stft(wav, n_fft, hop_length=hop, win_length=n_fft, window=win, center=True,
                      pad_mode="reflect", return_complex=True)
    return spec.abs().transpose(0, 1)


def log_mel(wav: torch.Tensor, sr: int = 16000, n_fft: int = 1024, hop: int = 320, n_mels: int = 80) -> torch.Tensor:
    mag = stft_mag(wav, n_fft, hop)
    mel = mag @ _mel_fb(sr, n_fft, n_mels).to(mag.device)
    return torch.log(mel.clamp_min(1e-5))


def mfcc(wav: torch.Tensor, sr: int = 16000, n_mfcc: int = 13, hop: int = 160, n_fft: int = 400,
         n_mels: int = 40) -> torch.Tensor:
    """MFCC + deltas [frames, 2*n_mfcc].  Low-order cepstra are largely pitch-invariant, which makes them a
    faithful stand-in for the 'semantic but tone-poor' deep SSL layer in the synthetic benchmark."""
    lm = log_mel(wav, sr, n_fft, hop, n_mels)
    n = lm.shape[1]
    k = torch.arange(n_mfcc).unsqueeze(1).float()
    i = torch.arange(n).unsqueeze(0).float()
    dct = torch.cos(math.pi / n * (i + 0.5) * k) * math.sqrt(2.0 / n)
    c = lm @ dct.t()
    c = c - c.mean(0, keepdim=True)  # cepstral mean normalisation
    d = torch.zeros_like(c)
    if c.shape[0] > 2:
        d[1:-1] = (c[2:] - c[:-2]) / 2
    return torch.cat([c, d], dim=1)


# ---------------------------------------------------------------------------------------------- YIN
def yin_f0(wav: np.ndarray, sr: int = 16000, hop: int = 160, frame: int = 1024, fmin: float = 60.0,
           fmax: float = 500.0, thresh: float = 0.15) -> Tuple[np.ndarray, np.ndarray]:
    """Vectorised YIN (de Cheveigne & Kawahara, 2002).  Returns (f0 [frames] in Hz, 0 = unvoiced; aperiodicity)."""
    wav = np.asarray(wav, dtype=np.float64)
    if len(wav) < frame:
        wav = np.pad(wav, (0, frame - len(wav)))
    pad = frame // 2
    x = np.pad(wav, (pad, pad), mode="reflect")
    n_frames = 1 + (len(x) - frame) // hop
    idx = np.arange(frame)[None, :] + hop * np.arange(n_frames)[:, None]
    fr = x[idx]
    tau_min, tau_max = max(2, int(sr / fmax)), min(frame // 2 - 1, int(sr / fmin))
    w = frame - tau_max  # integration window
    # difference function d(tau) = sum_j (x_j - x_{j+tau})^2 over j < w, via FFT cross-correlation
    nfft = 1 << int(np.ceil(np.log2(frame + w)))
    a = np.fft.rfft(fr[:, :w], nfft)
    b = np.fft.rfft(fr, nfft)
    corr = np.fft.irfft(np.conj(a) * b, nfft)[:, : tau_max + 1]
    cs = np.cumsum(np.concatenate([np.zeros((n_frames, 1)), fr ** 2], axis=1), axis=1)
    taus = np.arange(tau_max + 1)
    e0 = cs[:, w:w + 1] - cs[:, 0:1]
    et = cs[:, taus + w] - cs[:, taus]
    d = np.maximum(e0 + et - 2 * corr, 0.0)
    cmnd = np.ones_like(d)
    csum = np.cumsum(d[:, 1:], axis=1)
    cmnd[:, 1:] = d[:, 1:] * taus[1:][None, :] / np.maximum(csum, 1e-12)
    seg = cmnd[:, tau_min: tau_max + 1]
    below = seg < thresh
    # first tau under the threshold, then walk to the local minimum; otherwise the global minimum
    first = np.where(below.any(1), below.argmax(1), seg.argmin(1))
    tau = first.copy()
    for _ in range(64):
        nxt = np.minimum(tau + 1, seg.shape[1] - 1)
        better = seg[np.arange(n_frames), nxt] < seg[np.arange(n_frames), tau]
        if not better.any():
            break
        tau = np.where(better, nxt, tau)
    ap = seg[np.arange(n_frames), tau]
    t = tau + tau_min
    tm, tp = np.clip(t - 1, 0, tau_max), np.clip(t + 1, 0, tau_max)
    y0, y1, y2 = cmnd[np.arange(n_frames), tm], cmnd[np.arange(n_frames), t], cmnd[np.arange(n_frames), tp]
    denom = y0 - 2 * y1 + y2
    shift = np.where(np.abs(denom) > 1e-9, 0.5 * (y0 - y2) / np.where(np.abs(denom) > 1e-9, denom, 1.0), 0.0)
    f0 = sr / np.maximum(t + np.clip(shift, -1, 1), 1e-6)
    energy = np.sqrt((fr ** 2).mean(1))
    voiced = (ap < max(thresh, 0.25)) & (energy > 0.02 * (energy.max() + 1e-9)) & (f0 >= fmin) & (f0 <= fmax)
    return np.where(voiced, f0, 0.0), ap


def _pool_frames(x: np.ndarray, n: int) -> np.ndarray:
    """Average-pool / interpolate a [T, ...] sequence to exactly n frames."""
    if len(x) == n:
        return x
    src = np.linspace(0, len(x) - 1, n)
    lo = np.floor(src).astype(int)
    hi = np.minimum(lo + 1, len(x) - 1)
    wgt = (src - lo).reshape(-1, *([1] * (x.ndim - 1)))
    return x[lo] * (1 - wgt) + x[hi] * wgt


def tone_features(wav: np.ndarray, n_frames: int, sr: int = 16000) -> Tuple[np.ndarray, np.ndarray]:
    """Returns (tone [n_frames, 4], f0stats [2] = mean/std of log-F0 over voiced frames)."""
    f0, _ = yin_f0(wav, sr=sr)
    voiced = f0 > 0
    hop = 160
    pad = 512
    x = np.pad(np.asarray(wav, dtype=np.float64), (pad, pad))
    n = len(f0)
    idx = np.arange(1024)[None, :] + hop * np.arange(n)[:, None]
    idx = np.minimum(idx, len(x) - 1)
    loge = np.log(np.sqrt((x[idx] ** 2).mean(1)) + 1e-5)
    if voiced.sum() >= 3:
        lf = np.log(f0[voiced])
        mu, sd = float(lf.mean()), float(lf.std() + 1e-3)
    else:
        mu, sd = math.log(150.0), 0.2
    z = np.zeros(n)
    z[voiced] = (np.log(f0[voiced]) - mu) / sd
    # fill unvoiced gaps by linear interpolation so the delta is meaningful, then mask again
    if voiced.sum() >= 2:
        zi = np.interp(np.arange(n), np.flatnonzero(voiced), z[voiced])
    else:
        zi = z
    dz = np.gradient(zi) if n > 1 else np.zeros(n)
    ez = (loge - loge.mean()) / (loge.std() + 1e-3)
    feats = np.stack([z, voiced.astype(np.float64), dz * voiced, ez], axis=1)
    out = _pool_frames(feats, n_frames)
    out[:, 1] = (out[:, 1] > 0.5).astype(np.float64)
    out[:, 0] *= out[:, 1]
    out[:, 2] *= out[:, 1]
    return out.astype(np.float32), np.array([mu, sd], dtype=np.float32)


def flatten_tone(tone: torch.Tensor) -> torch.Tensor:
    """Counterfactual used by the tone-reliance probes: remove the pitch contour, keep voicing and energy."""
    out = tone.clone()
    out[..., 0] = 0.0
    out[..., 2] = 0.0
    return out


def trim_silence(wav: np.ndarray, top_db: float = 35.0, sr: int = 16000, pad_ms: float = 60.0) -> np.ndarray:
    """Trim leading / trailing silence.  The threshold adapts to the noise floor of the clip (10th percentile
    + 10 dB, never above -12 dB), so padded noisy recordings are trimmed and continuous speech is left alone."""
    if top_db <= 0 or len(wav) < sr // 10:
        return wav
    hop, win = sr // 100, sr // 40
    n = 1 + max(0, len(wav) - win) // hop
    idx = np.arange(win)[None, :] + hop * np.arange(n)[:, None]
    rms = np.sqrt((wav[idx] ** 2).mean(1) + 1e-12)
    db = 20 * np.log10(rms / (np.percentile(rms, 99) + 1e-12) + 1e-12)
    thr = max(-top_db, min(float(np.percentile(db, 10)) + 10.0, -12.0))
    act = np.convolve((db > thr).astype(np.float64), np.ones(15) / 15, mode="same") > 0.5  # ignore clicks
    keep = np.flatnonzero(act)
    if len(keep) == 0:
        return wav
    pad = int(pad_ms / 1000 * sr)
    a, b = max(0, keep[0] * hop - pad), min(len(wav), keep[-1] * hop + win + pad)
    return wav[a:b]


def highpass(wav: np.ndarray, cutoff_hz: float, sr: int = 16000, ramp_hz: float = 20.0) -> np.ndarray:
    """Zero-phase high-pass: nothing below ``cutoff - ramp``, everything above ``cutoff``, a raised cosine between.
    Field recordings can carry most of their energy as rumble below the lowest voice pitch (YFACC Yoruba: half of
    it below 80 Hz).  That rumble hides the pauses from the silence trimmer and the periodicity from the pitch
    tracker, and it is what an utterance-normalising speech encoder then normalises by."""
    if cutoff_hz <= 0 or len(wav) < 32:
        return wav
    spec = np.fft.rfft(np.asarray(wav, dtype=np.float64))
    f = np.fft.rfftfreq(len(wav), 1.0 / sr)
    g = 0.5 - 0.5 * np.cos(np.pi * np.clip((f - (cutoff_hz - ramp_hz)) / ramp_hz, 0.0, 1.0))
    return np.fft.irfft(spec * g, len(wav)).astype(np.float32)


def resample(wav: np.ndarray, sr_in: int, sr_out: int) -> np.ndarray:
    if sr_in == sr_out:
        return wav.astype(np.float32)
    t = torch.from_numpy(np.ascontiguousarray(wav)).float()
    if AF is not None:
        return AF.resample(t, sr_in, sr_out).numpy()
    n = int(round(len(wav) * sr_out / sr_in))  # pragma: no cover
    return F.interpolate(t[None, None], size=n, mode="linear", align_corners=False)[0, 0].numpy()


def griffin_lim(logmel: torch.Tensor, sr: int = 16000, n_fft: int = 1024, hop: int = 320, n_iter: int = 48) -> torch.Tensor:
    """log-mel [frames, n_mels] -> waveform [T] (pseudo-inverse mel + Griffin-Lim; no learned vocoder)."""
    fb = _mel_fb(sr, n_fft, logmel.shape[1]).to(logmel.device)
    mag = torch.clamp(torch.exp(logmel) @ torch.linalg.pinv(fb), min=1e-5).transpose(0, 1)  # [F, frames]
    win = torch.hann_window(n_fft, device=logmel.device)
    angles = torch.exp(2j * math.pi * torch.rand(mag.shape, device=logmel.device))
    wav: Optional[torch.Tensor] = None
    for _ in range(n_iter):
        wav = torch.istft(mag * angles, n_fft, hop_length=hop, win_length=n_fft, window=win, center=True)
        spec = torch.stft(wav, n_fft, hop_length=hop, win_length=n_fft, window=win, center=True,
                          pad_mode="reflect", return_complex=True)
        m = min(spec.shape[1], mag.shape[1])
        angles = torch.ones_like(mag, dtype=spec.dtype)
        angles[:, :m] = spec[:, :m] / spec[:, :m].abs().clamp_min(1e-8)
    assert wav is not None
    return wav / wav.abs().max().clamp_min(1e-5) * 0.95


# ======================================================================================================================
# 05. DATA: download
#     (geobridge/data/download.py)
# ======================================================================================================================
# Automatic download of the exact public corpora the benchmark is assembled from.
#
# Every source is pinned to the revision that was verified to exist and to join correctly
# (image id + caption index) on 2026-11-21.  If a pinned revision disappears the loader falls back to
# ``main`` and says so.  Nothing here needs an account or a gated-dataset agreement.
#
# Benchmark "flickr8k_yo" = Flickr8k-Tri:
#     images ............. 8,091 Flickr8k photos
#     en text ............ 40,455 crowd captions (5 / image)                  [Hodosh et al., 2013]
#     en speech .......... 40,000 spoken captions, 183 speakers (FACC)        [Harwath & Glass, 2015]
#     yo speech (+text) .. 5,981 spoken Yoruba captions, 1 speaker (YFACC)    [Olaleye et al., 2023]
#     hi text ............ 40,455 machine-translated Hindi captions (silver)  -> script-robustness branch
#
# Extra hub-language data (``cfg.extra_images``), training only:
#     images + en text ... Flickr30k, 31,783 photos with 5 crowd captions each     [Young et al., 2014]
#                          8,091 of them ARE the Flickr8k photos, with the same captions.  They are dropped by
#                          photo id (flickr8k.py), all of them and whatever their split: otherwise test images and
#                          the captions held out for the other languages would come back in through this door.
#
# Extra hub-language SPEECH (``cfg.extra_speech``), training only:
#     images + en text + en speech ... SpokenCOCO, the 5,000-image test part: 25,031 captions read aloud by 1,766
#                          speakers (Flickr8k's speech has 183)                    [Hsu et al., 2021; CC BY-SA 4.0]
#                          The mirror holds audio and images; the caption text and the Flickr address of every
#                          image come from the COCO annotations, joined by caption id (in the audio file name)
#                          and image id.  A COCO photo that is a Flickr8k photo would be dropped by the same
#                          photo-id rule (none of the 5,000 is).
#
# Licensing note: the two Flickr8k speech mirrors and the Flickr30k mirror carry no licence metadata on the Hub; the
# photos are under Flickr terms of use.  Cite the original corpora and check their terms before redistributing
# derived data.
# ======================================================================================================================

@dataclass(frozen=True)
class HubSource:
    repo: str
    revision: str
    patterns: Tuple[str, ...]
    note: str


FLICKR8K_YO: Dict[str, HubSource] = {
    "captions": HubSource("shivangibithel/Flickr8k", "89173bc6ee039ac347ced51fb19ea1260d4c4b38",
                          ("captions.txt",), "English captions, 5 per image, in Flickr8k.token order"),
    "images_en_speech": HubSource("deep9539/flickr-audio-image", "eccef427e2398ade609d40c1dcc9b5c30dce8641",
                                  ("images/*.parquet", "audio/*.parquet", "qrels/*.parquet"),
                                  "Flickr8k images + Flickr Audio Caption Corpus (id = <image>_<caption idx>)"),
    "yoruba": HubSource("nolimitsxl/yfacc_yoruba", "bc5607b24083120b1c528f4c8126eca5d89a1485",
                        ("data/*.parquet",), "YFACC: Yoruba spoken captions + tone-marked transcripts"),
    "hindi": HubSource("ArefaMuzaffar/flickr8k_hindi_captions", "1a2a4352bc801562b115c3e5bb970de890f0caf4",
                       ("flickr8k_hindi.csv",), "machine-translated Hindi captions (Devanagari)"),
}

REGISTRY: Dict[str, Dict[str, HubSource]] = {"flickr8k_yo": FLICKR8K_YO}

# Flickr30k as nine parquet shards (image bytes, five captions, file name = Flickr photo id), sorted by photo id.
FLICKR30K = HubSource("lmms-lab/flickr30k", "765d117f3eec816f2bfdc2d73ebb50a6f77b86a4", ("data/test-*.parquet",),
                      "Flickr30k photos + 5 English captions each")
FLICKR30K_SHARDS = 9


def local_dir(data_root: str, repo: str) -> str:
    return os.path.join(data_root, "raw", repo.replace("/", "__"))


def download_sources(dataset: str, data_root: str, keys: List[str] = None, logger=None) -> Dict[str, str]:
    """Download (idempotent) and return {source key: local directory}."""
    from huggingface_hub import snapshot_download

    out: Dict[str, str] = {}
    for key, src in REGISTRY[dataset].items():
        if keys is not None and key not in keys:
            continue
        d = local_dir(data_root, src.repo)
        marker = os.path.join(d, f".complete_{src.revision[:12]}")
        if not os.path.exists(marker):
            if logger:
                logger.info(f"[download] {src.repo}@{src.revision[:8]}  ({src.note})")
            try:
                snapshot_download(src.repo, repo_type="dataset", revision=src.revision,
                                  allow_patterns=list(src.patterns), local_dir=d, max_workers=6)
            except Exception as e:  # pinned commit gone -> latest
                if logger:
                    logger.info(f"[download] pinned revision unavailable ({type(e).__name__}); using 'main'")
                snapshot_download(src.repo, repo_type="dataset", allow_patterns=list(src.patterns),
                                  local_dir=d, max_workers=6)
            with open(marker, "w") as f:
                f.write("ok\n")
        out[key] = d
    return out


# SpokenCOCO, test part (an audio-to-image retrieval set): "queries" = spoken captions, clips of one image together,
# file name <speaker>-<assignment>_<COCO image id>_<COCO caption id>; "corpus" = the images.
SPOKENCOCO = HubSource("whybe-choi/SpokenCOCOA2IRetrieval", "72731c49562b73ea7e1cbd296c40f3924a4e5fad",
                       ("queries/test-*.parquet", "corpus/test-*.parquet"), "SpokenCOCO test part: spoken captions + images")
SPOKENCOCO_AUDIO_SHARDS, SPOKENCOCO_IMAGE_SHARDS = 7, 2
# COCO annotations as two small tables: image id, Flickr address, caption ids, captions.
COCO_CAPTIONS = HubSource("phiyodr/coco2017", "036f3f8291db64d17faad9b09e59dd30bb65c4d7",
                          ("data/train-00000-of-00001-0084e041f1902997.parquet",
                           "data/validation-00000-of-00001-e3c37e369512a3aa.parquet"),
                          "COCO captions by caption id, and the Flickr address of every image")


def download_file(src: HubSource, name: str, data_root: str, logger=None) -> str:
    """Download one file of a pinned source (idempotent) and return its path."""
    from huggingface_hub import hf_hub_download

    d = local_dir(data_root, src.repo)
    path = os.path.join(d, name)
    if not os.path.exists(path):
        if logger:
            logger.info(f"[download] {src.repo}@{src.revision[:8]} {name}  ({src.note})")
        try:
            hf_hub_download(src.repo, name, repo_type="dataset", revision=src.revision, local_dir=d)
        except Exception as e:  # pinned commit gone -> latest
            if logger:
                logger.info(f"[download] pinned revision unavailable ({type(e).__name__}); using 'main'")
            hf_hub_download(src.repo, name, repo_type="dataset", local_dir=d)
    return path


def download_flickr30k(data_root: str, shard: int, logger=None) -> str:
    """Download one shard of the Flickr30k mirror (idempotent) and return its path.  Shards are fetched one by one,
    so a small ``extra_images`` needs one file of 0.5 GB and not all 4.4 GB."""
    from huggingface_hub import hf_hub_download

    src = FLICKR30K
    d = local_dir(data_root, src.repo)
    name = f"data/test-{shard:05d}-of-{FLICKR30K_SHARDS:05d}.parquet"
    path = os.path.join(d, name)
    if not os.path.exists(path):
        if logger:
            logger.info(f"[download] {src.repo}@{src.revision[:8]} {name}  ({src.note})")
        try:
            hf_hub_download(src.repo, name, repo_type="dataset", revision=src.revision, local_dir=d)
        except Exception as e:  # pinned commit gone -> latest
            if logger:
                logger.info(f"[download] pinned revision unavailable ({type(e).__name__}); using 'main'")
            hf_hub_download(src.repo, name, repo_type="dataset", local_dir=d)
    return path


# ======================================================================================================================
# 06. DATA: SynthWorld (controlled benchmark)
#     (geobridge/data/synthworld.py)
# ======================================================================================================================
# SynthWorld: a procedural tri-modal benchmark in which every factor of the problem is known exactly.
#
# It exists for two reasons.  (1) The whole pipeline can be exercised in minutes on a CPU, with no downloads.
# (2) It gives *controlled* evidence for claims real corpora can only support indirectly:
#
#   * tonal minimal pairs   - in language ``sb`` colour words come in pairs that share every segment and differ
#                             only in their tone pattern, so colour is unrecoverable without pitch;
#   * a disjoint island     - ``sb`` is speech-only (its orthography is held out for scoring);
#   * script fragmentation  - ``sc`` is written in Devanagari syllables (6 UTF-8 bytes per grapheme cluster);
#   * sentence-level meaning - a caption describes a *subset* of the objects in its image, so a scene-level
#                             visual pivot cannot translate it, and grounding can be scored by a pointing game.
#
# Languages
#   sa  hub language      Latin script, non-tonal, text + speech, several speakers, adjective-noun order
#   sb  island language   tonal, speech only (2 speakers), noun-adjective order
#   sc  written language  Devanagari, text only
#
# Pitch moves by a similar amount in ``sa`` and ``sb``; only in ``sb`` is the movement lexical.
# Object types follow a Zipf law (``synth_zipf``), as content words do in real captions.
# ======================================================================================================================

CONS = list("ptkbdgmnslfv")
VOWS = list("aeiou")
FORMANTS = {"a": (750, 1250), "e": (450, 2000), "i": (300, 2400), "o": (500, 900), "u": (330, 800)}
CONS_SPEC = {  # (noise centre Hz, noise bandwidth Hz, voiced)
    "p": (800, 400, False), "t": (3500, 700, False), "k": (2000, 500, False),
    "b": (700, 350, True), "d": (3000, 700, True), "g": (1800, 500, True),
    "m": (1000, 250, True), "n": (1500, 250, True), "s": (5200, 900, False),
    "l": (1200, 300, True), "f": (3000, 1500, False), "v": (2600, 1200, True),
}
DEVA_C = dict(zip(CONS, "पतकबदगमनसलफव"))
DEVA_V = {"a": "ा", "e": "े", "i": "ि", "o": "ो", "u": "ु"}
ACUTE = {"a": "á", "e": "é", "i": "í", "o": "ó", "u": "ú"}
GRAVE = {"a": "à", "e": "è", "i": "ì", "o": "ò", "u": "ù"}
TONE_ST = {"H": 3.0, "L": -3.0, "M": 0.0}  # semitones relative to the speaker's register
COLOR_RGB = [(220, 40, 40), (40, 170, 60), (50, 90, 220), (235, 200, 40), (200, 60, 200), (40, 200, 210),
             (240, 140, 30), (120, 80, 40), (250, 250, 250), (90, 90, 90)]
SHAPES = ["circle", "square", "triangle", "diamond", "cross", "bar", "ring", "star", "hex", "tee"]
COLOR_NAMES = ["red", "green", "blue", "yellow", "magenta", "cyan", "orange", "brown", "white", "grey"]

SYNTH_LANGS = {
    "sa": {"tonal": False, "script": "Latin", "written": True, "spoken": True},
    "sb": {"tonal": True, "script": "Latin+tone marks (held out)", "written": False, "spoken": True},
    "sc": {"tonal": False, "script": "Devanagari", "written": True, "spoken": False},
}

Word = List[Tuple[str, str, str]]  # syllables as (consonant, vowel, tone)


class SynthWorld:
    def __init__(self, cfg) -> None:
        self.cfg = cfg
        self.nc, self.ns, self.g = cfg.synth_n_colors, cfg.synth_n_shapes, cfg.synth_grid
        assert self.nc % 2 == 0 and self.nc <= len(COLOR_RGB) and self.ns <= len(SHAPES)
        rng = np.random.default_rng(20261121)  # the *world* is fixed; cfg.seed only affects training
        self.lex = self._make_lexicons(rng)
        d = cfg.synth_vision_dim
        self.e_color = rng.standard_normal((self.nc, d)) / math.sqrt(d) * 2.0
        self.e_shape = rng.standard_normal((self.ns, d)) / math.sqrt(d) * 2.0
        self.e_bg = rng.standard_normal(d) / math.sqrt(d) * 0.6
        n_sa = cfg.synth_n_speakers
        self.speakers = {
            "sa": [(float(rng.uniform(95, 240)), float(rng.uniform(0.9, 1.15))) for _ in range(n_sa)],
            "sb": [(float(rng.uniform(110, 210)), float(rng.uniform(0.92, 1.1))) for _ in range(2)],
        }
        self.scenes, self.caps = self._make_scenes(rng, cfg.synth_n_images)
        self.x_scenes: Dict[int, List] = {}   # extra scenes by image index (they follow the benchmark's own images)
        self.x_caps: List[Dict] = []          # captions of the extra scenes that are also spoken

    # ------------------------------------------------------------------ lexicons
    def _stems(self, rng, n: int, used: set) -> List[List[Tuple[str, str]]]:
        out = []
        while len(out) < n:
            st = tuple((str(rng.choice(CONS)), str(rng.choice(VOWS))) for _ in range(2))
            if st not in used:
                used.add(st)
                out.append(list(st))
        return out

    def _make_lexicons(self, rng) -> Dict[str, Dict]:
        lex: Dict[str, Dict] = {}
        for lang in ("sa", "sb", "sc"):
            used: set = set()
            if lang == "sb":  # tonal minimal pairs: colours 2i and 2i+1 share a stem
                stems = self._stems(rng, self.nc // 2, used)
                colors = []
                for st in stems:
                    colors.append([(c, v, t) for (c, v), t in zip(st, "HL")])
                    colors.append([(c, v, t) for (c, v), t in zip(st, "LH")])
                shapes = [[(c, v, str(rng.choice(["H", "L"]))) for c, v in st]
                          for st in self._stems(rng, self.ns, used)]
                conj = [(c, v, "M") for c, v in self._stems(rng, 1, used)[0][:1]]
            else:
                colors = [[(c, v, "M") for c, v in st] for st in self._stems(rng, self.nc, used)]
                shapes = [[(c, v, "M") for c, v in st] for st in self._stems(rng, self.ns, used)]
                conj = [(c, v, "M") for c, v in self._stems(rng, 1, used)[0][:1]]
            lex[lang] = {"color": colors, "shape": shapes, "conj": conj,
                         "order": "noun_adj" if lang == "sb" else "adj_noun"}
        return lex

    def spell(self, word: Word, lang: str) -> str:
        if lang == "sc":
            return "".join(DEVA_C[c] + DEVA_V[v] for c, v, _ in word)
        if lang == "sb":
            return "".join(c + {"H": ACUTE[v], "L": GRAVE[v], "M": v}[t] for c, v, t in word)
        return "".join(c + v for c, v, _ in word)

    def words(self, content: Sequence[Tuple[int, int]], lang: str) -> List[Word]:
        lx, out = self.lex[lang], []
        for i, (c, s) in enumerate(content):
            if i:
                out.append(lx["conj"])
            pair = [lx["color"][c], lx["shape"][s]]
            out.extend(pair if lx["order"] == "adj_noun" else pair[::-1])
        return out

    def text(self, content: Sequence[Tuple[int, int]], lang: str) -> str:
        return " ".join(self.spell(w, lang) for w in self.words(content, lang))

    def parse(self, text: str, lang: str) -> Optional[List[Tuple[int, int]]]:
        """Inverse of :meth:`text`; ``None`` when the string is not a well-formed caption."""
        lx = self.lex[lang]
        col = {self.spell(w, lang): i for i, w in enumerate(lx["color"])}
        shp = {self.spell(w, lang): i for i, w in enumerate(lx["shape"])}
        conj = self.spell(lx["conj"], lang)
        toks = [t for t in text.strip().split() if t != conj]
        if len(toks) == 0 or len(toks) % 2:
            return None
        out = []
        for a, b in zip(toks[0::2], toks[1::2]):
            if lx["order"] != "adj_noun":
                a, b = b, a
            if a not in col or b not in shp:
                return None
            out.append((col[a], shp[b]))
        return out

    # ------------------------------------------------------------------ scenes & captions
    def _make_scenes(self, rng, n: int):
        cfg, scenes, caps = self.cfg, [], []
        cells = self.g * self.g
        # Zipfian object types: like real captions, a few (colour, shape) combinations are very frequent and
        # most are rare, and colour is not independent of shape.  This is what gives unpaired data a shape.
        n_types = self.nc * self.ns
        rank = np.random.default_rng(4242).permutation(n_types)
        p_type = 1.0 / (1.0 + rank) ** cfg.synth_zipf
        self.type_prob = p_type / p_type.sum()
        for i in range(n):
            n_obj = int(rng.integers(2, 4))
            pos = rng.choice(cells, size=n_obj, replace=False)
            types = rng.choice(n_types, size=n_obj, p=self.type_prob)
            objs = [(int(p), int(t) // self.ns, int(t) % self.ns) for p, t in zip(pos, types)]
            scenes.append(objs)
            for k in range(cfg.synth_caps_per_image):
                two = rng.random() < cfg.synth_two_obj_prob
                pick = rng.choice(n_obj, size=2 if two else 1, replace=False)
                caps.append({"img": i, "cap": k, "objs": [objs[int(j)] for j in pick],
                             "spk_sa": int(rng.integers(len(self.speakers["sa"]))),
                             "spk_sb": int(rng.integers(len(self.speakers["sb"]))),
                             "seed": int(rng.integers(1 << 30))})
        return scenes, caps

    # ------------------------------------------------------------------ manifest
    def manifest(self) -> Dict:
        images = [{"id": f"im{i:05d}", "scene": sc} for i, sc in enumerate(self.scenes)]
        utts = []
        for ci, c in enumerate(self.caps):
            content = [(o[1], o[2]) for o in c["objs"]]
            cells = [o[0] for o in c["objs"]]
            for lang in ("sa", "sb", "sc"):
                info = SYNTH_LANGS[lang]
                utts.append({
                    "uid": f"{lang}/im{c['img']:05d}/{c['cap']}", "img": c["img"], "cap": c["cap"], "lang": lang,
                    "text": self.text(content, lang), "has_audio": info["spoken"],
                    "spk": f"{lang}{c.get('spk_' + lang, 0)}" if info["spoken"] else "",
                    "content": content, "cells": cells, "cap_row": ci,
                })
        return {"name": "synthworld", "langs": SYNTH_LANGS, "images": images, "utts": utts,
                "meta": {"grid": self.g, "colors": COLOR_NAMES[: self.nc], "shapes": SHAPES[: self.ns]}}

    def extra_manifest(self, n: int, speech: bool = False, first: Optional[int] = None) -> Dict:
        """``n`` more scenes with captions in the hub language only - as text, and with ``speech`` also spoken:
        what a high-resource language has plenty of.  Drawn from the same world with their own random streams, so
        the benchmark itself is untouched.  ``first``: index of the first of these images among all images."""
        n = (len(self.scenes) if speech else 4 * len(self.scenes)) if n < 0 else n
        H, tag = self.cfg.hub_langs[0], ("xsp" if speech else "xim")
        scenes, caps = self._make_scenes(np.random.default_rng(20261123 if speech else 20261122), n)
        first = len(self.scenes) + len(self.x_scenes) if first is None else first
        self.x_scenes.update({first + i: sc for i, sc in enumerate(scenes)})
        row0 = len(self.caps) + len(self.x_caps)
        if speech:
            self.x_caps += caps
        images = [{"id": f"{tag}{i:05d}", "scene": sc, "official_split": "train", "extra": True} for i, sc in enumerate(scenes)]
        utts = [{"uid": f"{H}/{tag}{c['img']:05d}/{c['cap']}", "img": c["img"], "cap": c["cap"], "lang": H,
                 "text": self.text([(o[1], o[2]) for o in c["objs"]], H), "has_audio": speech,
                 "spk": f"{H}{c['spk_' + H]}" if speech else "", "cap_row": row0 + j if speech else -1,
                 "content": [(o[1], o[2]) for o in c["objs"]], "cells": [o[0] for o in c["objs"]]} for j, c in enumerate(caps)]
        return {"name": "synthworld_extra" + ("_speech" if speech else ""), "images": images, "utts": utts, "meta": {"scenes": n}}

    def _scene(self, img: int) -> List:
        """Scene of an image index: the benchmark's own images first, then the extra ones."""
        return self.scenes[img] if img < len(self.scenes) else self.x_scenes[img]

    # ------------------------------------------------------------------ vision
    def vision_tokens(self, img: int) -> np.ndarray:
        """Analytic 'frozen encoder' output: one token per grid cell + a global token (last)."""
        rng = np.random.default_rng(7919 * (img + 1))
        d, cells = self.cfg.synth_vision_dim, self.g * self.g
        tok = np.tile(self.e_bg, (cells, 1))
        for p, c, s in self._scene(img):
            tok[p] = self.e_color[c] + self.e_shape[s]
        tok = tok + self.cfg.synth_feat_noise * rng.standard_normal((cells, d)) / math.sqrt(d) * 2.0
        return np.concatenate([tok, tok.mean(0, keepdims=True)], 0).astype(np.float32)

    def render(self, img: int, size: int = 128):
        from PIL import Image, ImageDraw

        im = Image.new("RGB", (size, size), (24, 24, 28))
        dr = ImageDraw.Draw(im)
        cell = size / self.g
        for p, c, s in self._scene(img):
            r, q = divmod(p, self.g)
            x0, y0, x1, y1 = q * cell + 3, r * cell + 3, (q + 1) * cell - 3, (r + 1) * cell - 3
            cx, cy, col = (x0 + x1) / 2, (y0 + y1) / 2, COLOR_RGB[c]
            name = SHAPES[s]
            if name == "circle":
                dr.ellipse([x0, y0, x1, y1], fill=col)
            elif name == "square":
                dr.rectangle([x0, y0, x1, y1], fill=col)
            elif name == "triangle":
                dr.polygon([(cx, y0), (x1, y1), (x0, y1)], fill=col)
            elif name == "diamond":
                dr.polygon([(cx, y0), (x1, cy), (cx, y1), (x0, cy)], fill=col)
            elif name == "cross":
                w = (x1 - x0) / 3
                dr.rectangle([cx - w / 2, y0, cx + w / 2, y1], fill=col)
                dr.rectangle([x0, cy - w / 2, x1, cy + w / 2], fill=col)
            elif name == "bar":
                dr.rectangle([x0, cy - (y1 - y0) / 6, x1, cy + (y1 - y0) / 6], fill=col)
            elif name == "ring":
                dr.ellipse([x0, y0, x1, y1], outline=col, width=max(2, int(cell / 7)))
            elif name == "star":
                pts = []
                for k in range(10):
                    rad = (x1 - x0) / 2 * (1.0 if k % 2 == 0 else 0.45)
                    a = -math.pi / 2 + k * math.pi / 5
                    pts.append((cx + rad * math.cos(a), cy + rad * math.sin(a)))
                dr.polygon(pts, fill=col)
            elif name == "hex":
                pts = [(cx + (x1 - x0) / 2 * math.cos(k * math.pi / 3), cy + (y1 - y0) / 2 * math.sin(k * math.pi / 3))
                       for k in range(6)]
                dr.polygon(pts, fill=col)
            else:  # tee
                w = (x1 - x0) / 3
                dr.rectangle([x0, y0, x1, y0 + w], fill=col)
                dr.rectangle([cx - w / 2, y0, cx + w / 2, y1], fill=col)
        return im

    # ------------------------------------------------------------------ speech synthesis
    def synth(self, cap_row: int, lang: str, sr: int = 16000) -> np.ndarray:
        c = self.caps[cap_row] if cap_row < len(self.caps) else self.x_caps[cap_row - len(self.caps)]
        rng = np.random.default_rng(c["seed"] + (0 if lang == "sa" else 77))
        f_base, fscale = self.speakers[lang][c["spk_" + lang]]
        words = self.words([(o[1], o[2]) for o in c["objs"]], lang)
        tonal = SYNTH_LANGS[lang]["tonal"]
        rate = float(rng.uniform(0.88, 1.15))
        n_syl = sum(len(w) for w in words)
        chunks, k = [np.zeros(int(0.06 * sr))], 0
        for w in words:
            for cons, vow, tone in w:
                # lexical tone (sb) vs. free, uninformative intonation of the same magnitude (sa)
                st = TONE_ST[tone] if tonal else float(rng.uniform(-3.0, 3.0))
                st += -1.0 * k / max(1, n_syl - 1) + float(rng.normal(0, 0.25))  # declination + jitter
                f0 = f_base * 2 ** (st / 12)
                chunks.append(_consonant(cons, f0, int(0.05 * sr / rate), sr, rng))
                chunks.append(_vowel(vow, f0, fscale, int(0.13 * sr / rate), sr, rng))
                k += 1
            chunks.append(np.zeros(int(0.04 * sr / rate)))
        chunks.append(np.zeros(int(0.06 * sr)))
        wav = np.concatenate(chunks)
        wav = wav / (np.abs(wav).max() + 1e-6) * 0.7
        return (wav + 0.004 * rng.standard_normal(len(wav))).astype(np.float32)

    def audio_groups(self, utts: List[Dict], chunk: int = 400) -> List[Dict]:
        rows = [u for u in utts if u["has_audio"]]
        groups = []
        for i in range(0, len(rows), chunk):
            part = rows[i:i + chunk]

            def loader(part=part) -> Iterator[Tuple[str, np.ndarray, int]]:
                for u in part:
                    yield u["uid"], self.synth(u["cap_row"], u["lang"]), 16000

            groups.append({"key": f"synth_{i // chunk:05d}", "uids": [u["uid"] for u in part], "loader": loader})
        return groups

    extra_audio_groups = audio_groups   # extra spoken captions are synthesised the same way


def _vowel(v: str, f0: float, fscale: float, n: int, sr: int, rng) -> np.ndarray:
    t = np.arange(n) / sr
    glide = 1.0 + 0.015 * np.linspace(-1, 1, n) * rng.choice([-1, 1])
    phase = 2 * np.pi * np.cumsum(f0 * glide) / sr
    out = np.zeros(n)
    f1, f2 = (f * fscale for f in FORMANTS[v])
    for h in range(1, int(3800 / f0) + 1):
        fh = h * f0
        amp = math.exp(-((fh - f1) ** 2) / (2 * 110 ** 2)) + 0.7 * math.exp(-((fh - f2) ** 2) / (2 * 150 ** 2)) + 0.02
        out += amp * np.sin(h * phase)
    env = np.minimum(1.0, np.minimum(t / 0.015, (t[-1] - t) / 0.02 + 1e-3))
    return out * env


def _consonant(c: str, f0: float, n: int, sr: int, rng) -> np.ndarray:
    centre, bw, voiced = CONS_SPEC[c]
    spec = np.fft.rfft(rng.standard_normal(n))
    freqs = np.fft.rfftfreq(n, 1 / sr)
    noise = np.fft.irfft(spec * np.exp(-((freqs - centre) ** 2) / (2 * bw ** 2)), n)
    noise = noise / (np.abs(noise).max() + 1e-6)
    env = np.concatenate([np.zeros(n // 3), np.hanning(2 * (n - n // 3))[: n - n // 3]])
    out = 0.5 * noise * env
    if voiced:
        t = np.arange(n) / sr
        out = out + 0.25 * np.sin(2 * np.pi * f0 * t) * np.hanning(n)
    return out


# ======================================================================================================================
# 07. DATA: Flickr8k-Tri (real benchmark)
#     (geobridge/data/flickr8k.py)
# ======================================================================================================================
# Flickr8k-Tri: the real benchmark, assembled from four public Hub repositories (see download.py).
#
# Join keys (verified on the pinned revisions):
#     image id            1000268201_693b08cb0e
#     English caption k   k-th line of that image in captions.txt            (k = 0..4)
#     English speech      audio id  <image id>_<k>                            (Flickr Audio Caption Corpus)
#     Yoruba speech/text  filename  S001_<image id>_<k>   (always k = 0)      (YFACC)
#     Hindi text          k-th row of that image in flickr8k_hindi.csv
#
# So the Yoruba utterance for an image is a spoken translation of English caption 0 of the same image, which
# gives an exact parallel reference for zero-pair Yoruba-speech -> English-text scoring.  The Yoruba transcript
# is carried in the manifest but, with the default config, never shown to the model: Yoruba is treated as an
# unwritten language and its orthography is used for analysis only.
#
# Extra hub-language speech (``cfg.extra_speech``): the test part of SpokenCOCO, 5,000 COCO photos whose captions
# were read aloud (25,031 clips, 1,766 speakers), for training only.  The clip name carries the speaker, the COCO
# image id and the COCO caption id; the caption text and the Flickr address of the photo come from the COCO
# annotations.  The Flickr8k photo-id rule below applies to these photos as well.
#
# Extra hub-language data (``cfg.extra_images``): Flickr30k photos with their English captions, as text, for
# training only.  Flickr30k CONTAINS the Flickr8k photos (file name = the Flickr photo id that Flickr8k ids start
# with) together with the very same captions, so every Flickr30k photo whose id is a Flickr8k photo id is dropped:
# whatever its Flickr8k split (test images must not come back as training data) and also when the Flickr8k image
# is not part of the current subset (its captions 0 and 4 belong to Yoruba and Hindi in the strict protocol).
# ======================================================================================================================

FLICKR_LANGS = {
    "en": {"tonal": False, "script": "Latin", "written": True, "spoken": True},
    "yo": {"tonal": True, "script": "Latin+tone marks (held out)", "written": False, "spoken": True},
    "hi": {"tonal": False, "script": "Devanagari (machine-translated, silver)", "written": True, "spoken": False},
}
YFACC_SPLIT = {"train": "train", "val": "valid", "test": "test"}


def _h(s: str) -> int:
    return int(hashlib.md5(s.encode()).hexdigest(), 16)


class Flickr8kTri:
    def __init__(self, cfg, logger=None) -> None:
        self.cfg, self.logger = cfg, logger
        keys = ["captions", "images_en_speech", "yoruba"] + (["hindi"] if "hi" in cfg.all_langs else [])
        self.dirs = download_sources("flickr8k_yo", cfg.data_root, keys, logger)

    # ------------------------------------------------------------------ manifest
    def manifest(self) -> Dict:
        import pyarrow.parquet as pq

        cfg = self.cfg
        caps: Dict[str, List[str]] = {}
        with open(os.path.join(self.dirs["captions"], "captions.txt"), encoding="utf8") as f:
            next(f)
            for line in f:
                name, cap = line.rstrip("\n").split(",", 1)
                caps.setdefault(name[:-4], []).append(cap.strip().strip('"'))

        # which parquet file holds which image / utterance (ids only; no media is decoded here)
        self.img_file: Dict[str, Tuple[str, int]] = {}
        for fp in sorted(glob.glob(os.path.join(self.dirs["images_en_speech"], "images", "*.parquet"))):
            for r, i in enumerate(pq.read_table(fp, columns=["id"]).column("id").to_pylist()):
                self.img_file[i] = (fp, r)
        self.aud_file: Dict[str, Tuple[str, int]] = {}
        for fp in sorted(glob.glob(os.path.join(self.dirs["images_en_speech"], "audio", "*.parquet"))):
            for r, i in enumerate(pq.read_table(fp, columns=["id"]).column("id").to_pylist()):
                self.aud_file["en/" + i] = (fp, r)
        yo: Dict[str, Dict] = {}
        for fp in sorted(glob.glob(os.path.join(self.dirs["yoruba"], "data", "*.parquet"))):
            split = YFACC_SPLIT[os.path.basename(fp).split("-")[0]]
            t = pq.read_table(fp, columns=["filename", "transcript"]).to_pydict()
            for r, (fn, tr) in enumerate(zip(t["filename"], t["transcript"])):
                img, k = fn.split("_", 1)[1].rsplit("_", 1)
                yo[img] = {"cap": int(k), "text": tr, "split": split}
                self.aud_file[f"yo/{img}_{k}"] = (fp, r)
        hi: Dict[str, List[str]] = {}
        if "hi" in cfg.all_langs:
            with open(os.path.join(self.dirs["hindi"], "flickr8k_hindi.csv"), encoding="utf8") as f:
                rd = csv.reader(f, delimiter=";")
                next(rd)
                for row in rd:
                    if len(row) >= 4:
                        hi.setdefault(row[1][:-4], []).append(row[3].strip())

        ids = sorted(i for i in self.img_file if i in caps)
        if cfg.max_images and cfg.max_images < len(ids):
            ids = sorted(sorted(ids, key=_h)[: cfg.max_images])
        images, utts = [], []
        for n, img in enumerate(ids):
            images.append({"id": img, "official_split": yo[img]["split"] if img in yo else ""})
            for k, cap in enumerate(caps[img][:5]):
                utts.append({"uid": f"en/{img}/{k}", "img": n, "cap": k, "lang": "en", "text": cap,
                             "has_audio": f"en/{img}_{k}" in self.aud_file, "spk": "", "src": f"en/{img}_{k}"})
                if img in hi and k < len(hi[img]) and "hi" in cfg.all_langs:
                    utts.append({"uid": f"hi/{img}/{k}", "img": n, "cap": k, "lang": "hi", "text": hi[img][k],
                                 "has_audio": False, "spk": "", "src": ""})
            if img in yo and "yo" in cfg.all_langs:
                k = yo[img]["cap"]
                utts.append({"uid": f"yo/{img}/{k}", "img": n, "cap": k, "lang": "yo", "text": yo[img]["text"],
                             "has_audio": True, "spk": "yo_S001", "src": f"yo/{img}_{k}"})
        langs = {l: FLICKR_LANGS[l] for l in cfg.all_langs if l in FLICKR_LANGS}
        return {"name": "flickr8k_yo", "langs": langs, "images": images, "utts": utts,
                "meta": {"n_images_total": len(self.img_file)}}

    # ------------------------------------------------------------------ extra hub-language image-text pairs
    def flickr8k_photo_ids(self) -> set:
        """Flickr photo id of EVERY Flickr8k image (the part of the image id before the underscore), read from the
        caption file itself so that it does not depend on the subset in use."""
        ids = set()
        with open(os.path.join(self.dirs["captions"], "captions.txt"), encoding="utf8") as f:
            next(f)
            for line in f:
                ids.add(line.split(",", 1)[0].rsplit(".", 1)[0].split("_")[0])
        return ids

    def extra_manifest(self, n: int, speech: bool = False, first: Optional[int] = None) -> Dict:
        """The first ``n`` (all if n < 0) Flickr30k photos that are not Flickr8k photos, with their captions;
        with ``speech``, SpokenCOCO photos with their captions and the recordings of them."""
        import pyarrow.parquet as pq

        cfg, H = self.cfg, self.cfg.hub_langs[0]
        if H != "en":
            raise ValueError("the extra data is English: it needs 'en' as the hub language")
        if speech:
            return self._spokencoco_manifest(n)
        own = self.flickr8k_photo_ids()
        images, utts, dropped, too_long = [], [], 0, 0
        for shard in range(FLICKR30K_SHARDS):
            fp = download_flickr30k(cfg.data_root, shard, self.logger)
            t = pq.read_table(fp, columns=["filename", "caption"]).to_pydict()
            for row, (fn, caps) in enumerate(zip(t["filename"], t["caption"])):
                pid = fn.rsplit(".", 1)[0]
                if pid in own:
                    dropped += 1
                    continue
                caps = [(k, " ".join(c.split())) for k, c in enumerate(caps[:5])]
                keep = [(k, c) for k, c in caps if c and len(c.encode("utf8")) <= cfg.max_text_bytes]
                too_long += len(caps) - len(keep)
                if not keep:
                    continue
                images.append({"id": f"f30k_{pid}", "official_split": "train", "extra": True,
                               "src": [os.path.basename(fp), row]})
                utts += [{"uid": f"{H}/f30k_{pid}/{k}", "img": len(images) - 1, "cap": k, "lang": H, "text": c,
                          "has_audio": False, "spk": "", "src": ""} for k, c in keep]
                if 0 < n <= len(images):
                    break
            if 0 < n <= len(images):
                break
        assert not {im["id"][5:] for im in images} & own
        return {"name": "flickr30k_extra", "images": images, "utts": utts,
                "meta": {"source": f"{FLICKR30K.repo}@{FLICKR30K.revision[:12]}", "shards_read": shard + 1,
                         "dropped_flickr8k_photos": dropped, "dropped_captions_over_max_text_bytes": too_long}}

    def _spokencoco_manifest(self, n: int) -> Dict:
        """SpokenCOCO test part: images, their captions as text, and one recording of every caption.

        Clips of one image lie together in the audio files, so a subset of ``n`` images needs only the first
        audio file(s).  Captions longer than ``max_text_bytes`` are left out with their recordings."""
        import pyarrow.parquet as pq

        cfg, H = self.cfg, self.cfg.hub_langs[0]
        text, flickr = {}, {}
        for name in COCO_CAPTIONS.patterns:   # caption id -> text, image id -> Flickr address
            t = pq.read_table(download_file(COCO_CAPTIONS, name, cfg.data_root, self.logger),
                              columns=["image_id", "flickr_url", "ids", "captions"]).to_pydict()
            for iid, url, ids, caps in zip(t["image_id"], t["flickr_url"], t["ids"], t["captions"]):
                flickr[int(iid)] = url or ""
                text.update({int(c): s for c, s in zip(ids, caps)})
        where: Dict[int, List] = {}           # image id -> [image file, row]
        for k in range(SPOKENCOCO_IMAGE_SHARDS):
            name = f"corpus/test-{k:05d}-of-{SPOKENCOCO_IMAGE_SHARDS:05d}.parquet"
            ids = pq.read_table(download_file(SPOKENCOCO, name, cfg.data_root, self.logger), columns=["id"]).column("id").to_pylist()
            where.update({int(i): [name, r] for r, i in enumerate(ids)})
        own = self.flickr8k_photo_ids()
        photo = lambda iid: flickr.get(iid, "").rsplit("/", 1)[-1].split("_")[0]
        clips: Dict[int, List] = {}           # image id -> [(caption id, speaker, audio file, row)], in file order
        dropped, no_image, too_long, no_text = set(), set(), 0, 0
        for k in range(SPOKENCOCO_AUDIO_SHARDS):
            name = f"queries/test-{k:05d}-of-{SPOKENCOCO_AUDIO_SHARDS:05d}.parquet"
            ids = pq.read_table(download_file(SPOKENCOCO, name, cfg.data_root, self.logger), columns=["id"]).column("id").to_pylist()
            for r, cid in enumerate(ids):
                head, iid, cap = cid.rsplit("_", 2)
                iid, cap = int(iid), int(cap)
                if photo(iid) in own or iid not in where:
                    (dropped if photo(iid) in own else no_image).add(iid)
                    continue
                s = " ".join(text.get(cap, "").split())
                if not s or len(s.encode("utf8")) > cfg.max_text_bytes:
                    too_long, no_text = too_long + bool(s), no_text + (not s)
                    continue
                clips.setdefault(iid, []).append((cap, head.split("-")[0], s, name, r))
            if 0 < n <= len(clips) - 1:       # the last image of a file may continue in the next one
                break
        images, utts = [], []
        for iid in list(clips)[: n if n > 0 else None]:
            images.append({"id": f"coco_{iid}", "official_split": "train", "extra": True, "src": where[iid]})
            for j, (cap, spk, s, name, r) in enumerate(sorted(clips[iid])):
                utts.append({"uid": f"{H}/coco_{iid}/{j}", "img": len(images) - 1, "cap": j, "lang": H, "text": s,
                             "has_audio": True, "spk": f"coco_{spk}", "src": [name, r]})
        assert not {photo(int(im["id"][5:])) for im in images} & own
        return {"name": "spokencoco_extra", "images": images, "utts": utts,
                "meta": {"source": f"{SPOKENCOCO.repo}@{SPOKENCOCO.revision[:12]} + {COCO_CAPTIONS.repo}@{COCO_CAPTIONS.revision[:12]}",
                         "speakers": len({u["spk"] for u in utts}), "dropped_flickr8k_photos": len(dropped),
                         "clips_without_image": len(no_image),
                         "dropped_captions_over_max_text_bytes": too_long, "captions_without_text": no_text}}

    def _row_group_parts(self, fp: str, rows: List[Tuple[object, int]], per_part: int) -> List[Tuple[List[int], Dict]]:
        """Split the wanted (key, row) pairs of a parquet file by row group: [(row groups of a part, {group: [(key, row in group)]})]."""
        import pyarrow.parquet as pq

        meta = pq.ParquetFile(fp).metadata
        starts = np.cumsum([0] + [meta.row_group(i).num_rows for i in range(meta.num_row_groups)])
        want: Dict[int, List[Tuple[object, int]]] = {}
        for key, r in rows:
            g = int(np.searchsorted(starts, r, side="right") - 1)
            want.setdefault(g, []).append((key, r - int(starts[g])))
        keys = sorted(want)
        return [(keys[a: a + per_part], want) for a in range(0, len(keys), per_part)]

    def _extra_file(self, name: str) -> str:
        """Local path of a file of the extra sources (downloaded if it is not there)."""
        if name.startswith(("queries/", "corpus/")):
            return download_file(SPOKENCOCO, name, self.cfg.data_root, self.logger)
        fp = os.path.join(local_dir(self.cfg.data_root, FLICKR30K.repo), "data", name)
        return fp if os.path.exists(fp) else download_flickr30k(self.cfg.data_root, int(name.split("-")[1]), self.logger)

    def extra_image_groups(self, images: List[Dict]) -> List[Dict]:
        """Loader groups of the extra photos: a few parquet row groups each, so that memory stays bounded and an
        interrupted feature extraction resumes from the last finished group."""
        by_file: Dict[str, List[Tuple[int, int]]] = {}
        for n, im in enumerate(images):
            by_file.setdefault(im["src"][0], []).append((n, int(im["src"][1])))
        groups = []
        for name, rows in sorted(by_file.items()):
            fp = self._extra_file(name)
            for part, want in self._row_group_parts(fp, rows, 8 if name.startswith("test-") else 2):

                def loader(fp=fp, part=part, want=want) -> Iterator[Tuple[int, object]]:
                    import pyarrow.parquet as pq
                    from PIL import Image

                    pf = pq.ParquetFile(fp)
                    for g in part:
                        col = pf.read_row_group(g, columns=["image"]).column("image")
                        for n, r in want[g]:
                            yield n, Image.open(io.BytesIO(col[r].as_py()["bytes"])).convert("RGB")

                groups.append({"key": f"{name}_rg{part[0]:04d}", "idx": [n for g in part for n, _ in want[g]],
                               "loader": loader})
        return groups

    def extra_audio_groups(self, utts: List[Dict]) -> List[Dict]:
        """Loader groups of the extra recordings: one parquet row group each (about 700 clips)."""
        by_file: Dict[str, List[Tuple[str, int]]] = {}
        for u in utts:
            if u["has_audio"]:
                by_file.setdefault(u["src"][0], []).append((u["uid"], int(u["src"][1])))
        groups = []
        for name, rows in sorted(by_file.items()):
            fp = self._extra_file(name)
            for part, want in self._row_group_parts(fp, rows, 1):

                def loader(fp=fp, part=part, want=want) -> Iterator[Tuple[str, np.ndarray, int]]:
                    import pyarrow.parquet as pq
                    import soundfile as sf

                    pf = pq.ParquetFile(fp)
                    for g in part:
                        col = pf.read_row_group(g, columns=["audio"]).column("audio")
                        for uid, r in want[g]:
                            wav, sr = sf.read(io.BytesIO(col[r].as_py()["bytes"]), dtype="float32", always_2d=True)
                            yield uid, wav.mean(1), sr

                groups.append({"key": f"coco_{os.path.basename(name)}_rg{part[0]:04d}",
                               "uids": [uid for g in part for uid, _ in want[g]], "loader": loader})
        return groups

    # ------------------------------------------------------------------ media access
    def image_groups(self, images: List[Dict]) -> List[Dict]:
        by_file: Dict[str, List[Tuple[int, int]]] = {}
        for n, im in enumerate(images):
            fp, r = self.img_file[im["id"]]
            by_file.setdefault(fp, []).append((n, r))
        groups = []
        for fp, rows in sorted(by_file.items()):
            def loader(fp=fp, rows=rows) -> Iterator[Tuple[int, object]]:
                import pyarrow.parquet as pq
                from PIL import Image

                col = pq.read_table(fp, columns=["image"]).column("image")
                for n, r in rows:
                    yield n, Image.open(io.BytesIO(col[r].as_py()["bytes"])).convert("RGB")

            groups.append({"key": os.path.basename(fp), "idx": [n for n, _ in rows], "loader": loader})
        return groups

    def load_image(self, image_id: str):
        import pyarrow.parquet as pq
        from PIL import Image

        fp, r = self.img_file[image_id]
        col = pq.read_table(fp, columns=["image"]).column("image")
        return Image.open(io.BytesIO(col[r].as_py()["bytes"])).convert("RGB")

    def audio_groups(self, utts: List[Dict]) -> List[Dict]:
        by_file: Dict[str, List[Tuple[str, int]]] = {}
        for u in utts:
            if u["has_audio"]:
                fp, r = self.aud_file[u["src"]]
                by_file.setdefault(fp, []).append((u["uid"], r))
        groups = []
        for fp, rows in sorted(by_file.items()):
            def loader(fp=fp, rows=rows) -> Iterator[Tuple[str, np.ndarray, int]]:
                import pyarrow.parquet as pq
                import soundfile as sf

                col = pq.read_table(fp, columns=["audio"]).column("audio")
                for uid, r in rows:
                    wav, sr = sf.read(io.BytesIO(col[r].as_py()["bytes"]), dtype="float32", always_2d=True)
                    yield uid, wav.mean(1), sr

            tag = "yo_" if "yfacc" in fp else "en_"
            groups.append({"key": tag + os.path.basename(fp), "uids": [u for u, _ in rows], "loader": loader})
        return groups


# ======================================================================================================================
# 08. DATA: frozen front-ends and the one-time feature cache
#     (geobridge/data/features.py)
# ======================================================================================================================
# Frozen front-ends and the one-time feature cache.
#
# Everything expensive (vision tower, speech SSL model, pitch tracking, mel) runs exactly once per dataset and
# is stored as memory-mapped arrays.  Extraction is chunked by source file, so an interrupted run resumes from
# the last finished chunk instead of starting again.
#
# Cache layout (``<data_root>/cache/<dataset_tag>/``)
#     manifest.json, splits.json
#     vision_<tag>/tokens.npy          [n_images, P+1, D_v]  float16   (last token = global)
#     audio_<tag>/sem.npy              [frames, D_s]         float16   deep-layer ("semantic") stream, 25 Hz
#     audio_<tag>/tone.npy             [frames, 4]           float32   tone stream (see audio_dsp.tone_features)
#     mel_<spec>_<tag>/mel.npy         [mel frames, n_mels]  float16   renderer targets in the vocoder's feature space
#     audio_<tag>/spk.npy, f0stats.npy [n_utts, D_spk], [n_utts, 2]    paralinguistic bypass inputs
#     audio_<tag>/index.json           uid -> [row, frame offset, n_frames]
#     audio_<tag>/stats.json           per-dimension mean / std of every stream (train split)
# ======================================================================================================================

def _tag(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "-", name).strip("-")


def on_all_gpus(module: nn.Module, cfg, device: torch.device) -> nn.Module:
    """Frozen encoder on ``device``; with several GPUs every batch is split across all of them."""
    module = module.to(device).eval().requires_grad_(False)
    ids = gpu_ids(cfg, device)
    return nn.DataParallel(module, device_ids=ids) if len(ids) > 1 else module


# ================================================================================== vision
class _VisionTokens(nn.Module):
    """pixels [B, 3, H, W] -> [B, grid * grid + 1, D]: pooled patch tokens, then the global token."""

    def __init__(self, tower: nn.Module, grid: int) -> None:
        super().__init__()
        self.tower, self.grid = tower, grid

    def forward(self, px: torch.Tensor) -> torch.Tensor:
        out = self.tower(pixel_values=px)
        h = out.last_hidden_state
        side = int(round(h.shape[1] ** 0.5))
        if side * side == h.shape[1]:  # SigLIP: no CLS; pooled head output is the global token
            patches = h
            glob = out.pooler_output if getattr(out, "pooler_output", None) is not None else h.mean(1)
        else:  # ViT / DINOv2 / CLIP: CLS (+ registers) first
            side = int(h.shape[1] ** 0.5)
            patches, glob = h[:, h.shape[1] - side * side:], h[:, 0]
        g = patches.transpose(1, 2).reshape(patches.shape[0], -1, side, side)
        g = F.adaptive_avg_pool2d(g.float(), self.grid).flatten(2).transpose(1, 2)
        return torch.cat([g, glob.float().unsqueeze(1)], 1)


class VisionFrontend:
    """Any Hugging Face vision tower -> (grid x grid pooled patch tokens, global token)."""

    def __init__(self, cfg, device: torch.device) -> None:
        from transformers import AutoImageProcessor, AutoModel

        self.device, self.amp = device, bool(cfg.amp and device.type == "cuda")
        with quiet_transformers():
            self.proc = AutoImageProcessor.from_pretrained(cfg.vision_encoder)
            model = AutoModel.from_pretrained(cfg.vision_encoder)
        tower = getattr(model, "vision_model", model)  # SigLIP / CLIP: keep the vision tower only
        self.net = on_all_gpus(_VisionTokens(tower, cfg.vision_grid), cfg, device)

    @torch.no_grad()
    def __call__(self, images: List) -> np.ndarray:
        px = self.proc(images=images, return_tensors="pt")["pixel_values"].to(self.device)
        with torch.autocast(device_type=self.device.type, enabled=self.amp):
            return self.net(px).float().cpu().numpy()


def cache_vision(cfg, builder, manifest: Dict, device: torch.device, logger, extra_from: Optional[int] = None,
                 root: Optional[str] = None) -> str:
    """Frozen vision tokens of every image of ``manifest``.  ``extra_from`` / ``root``: the manifest is that of
    extra data, cached under ``root``; its images follow the first ``extra_from`` images in the builder's numbering."""
    extra = extra_from is not None
    out = os.path.join(root or cfg.cache_dir, f"vision_{_tag(cfg.vision_encoder)}_g{cfg.vision_grid}")
    meta_p = os.path.join(out, "meta.json")
    if os.path.exists(meta_p):
        return out
    os.makedirs(out, exist_ok=True)
    n = len(manifest["images"])
    if cfg.vision_encoder == "synthetic":
        toks = np.stack([builder.vision_tokens((extra_from or 0) + i) for i in range(n)]).astype(np.float16)
        np.save(os.path.join(out, "tokens.npy"), toks)
    else:
        fe = VisionFrontend(cfg, device)
        arr = None
        groups = builder.extra_image_groups(manifest["images"]) if extra else builder.image_groups(manifest["images"])
        for grp in tqdm(groups, desc="vision features" + (" (extra images)" if extra else ""), dynamic_ncols=True):
            part_p = os.path.join(out, f"part_{_tag(grp['key'])}.npz")
            if os.path.exists(part_p):
                continue
            idx, feats, buf, buf_idx = [], [], [], []
            for i, img in grp["loader"]():
                buf.append(img)
                buf_idx.append(i)
                if len(buf) == cfg.feat_batch_size:
                    feats.append(fe(buf)); idx += buf_idx; buf, buf_idx = [], []
            if buf:
                feats.append(fe(buf)); idx += buf_idx
            np.savez(part_p, idx=np.array(idx), feats=np.concatenate(feats).astype(np.float16))
        for fp in sorted(p for p in os.listdir(out) if p.startswith("part_")):
            z = np.load(os.path.join(out, fp))
            if arr is None:
                arr = np.zeros((n, *z["feats"].shape[1:]), dtype=np.float16)
            arr[z["idx"]] = z["feats"]
        np.save(os.path.join(out, "tokens.npy"), arr)
        for fp in [p for p in os.listdir(out) if p.startswith("part_")]:
            os.remove(os.path.join(out, fp))
        toks = arr
    save_json({"n": n, "tokens": int(toks.shape[1]), "dim": int(toks.shape[2]), "encoder": cfg.vision_encoder}, meta_p)
    logger.info(f"[cache] vision tokens {tuple(toks.shape)} -> {out}")
    return out


def cache_thumbs(cfg, builder, manifest: Dict, vdir: str, logger, extra_from: Optional[int] = None) -> str:
    """Small RGB thumbnails of every image (uint8 [n, S, S, 3]): targets of the preview pixel decoder."""
    path = os.path.join(vdir, f"thumbs_{cfg.thumb_size}.npy")
    if os.path.exists(path):
        return path
    n, S = len(manifest["images"]), cfg.thumb_size
    out = np.zeros((n, S, S, 3), dtype=np.uint8)
    if hasattr(builder, "render"):
        for i in range(n):
            out[i] = np.asarray(builder.render((extra_from or 0) + i, S))
    else:
        groups = builder.extra_image_groups(manifest["images"]) if extra_from is not None else builder.image_groups(manifest["images"])
        for grp in tqdm(groups, desc="thumbnails" + (" (extra images)" if extra_from is not None else ""), dynamic_ncols=True):
            for i, img in grp["loader"]():
                out[i] = np.asarray(img.resize((S, S)))
    np.save(path + ".tmp.npy", out)
    os.replace(path + ".tmp.npy", path)
    logger.info(f"[cache] thumbnails {out.shape} -> {path}")
    return path


def fit_anchor(tokens: np.ndarray, train_idx: np.ndarray, d: int, path: str) -> Dict[str, torch.Tensor]:
    """Frozen anchor projection: PCA to d dims with ONE global scale, i.e. an isometry (up to scale) of the
    vision geometry restricted to its top-d subspace.  Fitted once in float64 via eigh of the covariance,
    never touched by training."""
    if os.path.exists(path):
        return torch.load(path)
    rng = np.random.default_rng(0)
    sel = train_idx if len(train_idx) <= 4000 else rng.choice(train_idx, 4000, replace=False)
    x = np.asarray(tokens[np.sort(sel)], dtype=np.float64).reshape(-1, tokens.shape[-1])
    mean = x.mean(0)
    cov = np.cov((x - mean).T)
    w, v = np.linalg.eigh(cov)
    order = np.argsort(w)[::-1]
    w, v = w[order], v[:, order]
    k = min(d, v.shape[1])
    basis = np.zeros((v.shape[1], d))
    basis[:, :k] = v[:, :k]
    scale = 1.0 / np.sqrt(max(w[:k].mean(), 1e-8))
    out = {"mean": torch.tensor(mean, dtype=torch.float32), "basis": torch.tensor(basis, dtype=torch.float32),
           "scale": torch.tensor(scale, dtype=torch.float32),
           "explained": torch.tensor(float(w[:k].sum() / max(w.sum(), 1e-12)))}
    torch.save(out, path)
    return out


# ================================================================================== speech
class MFCCFrontend:
    """No-download front-end (synthetic benchmark, unit tests)."""

    dim, spk_dim = 26, 40

    def __init__(self, cfg) -> None:
        self.pool = 4  # 100 fps -> 25 fps

    def __call__(self, wavs: List[np.ndarray]) -> List[Tuple[np.ndarray, np.ndarray]]:
        out = []
        for w in wavs:
            t = torch.from_numpy(w).float()
            m = mfcc(t)
            n = max(1, m.shape[0] // self.pool)
            sem = m[: n * self.pool].reshape(n, self.pool, -1).mean(1) if m.shape[0] >= self.pool else m.mean(0, keepdim=True)
            spk = log_mel(t, n_fft=400, hop=160, n_mels=40).mean(0)
            out.append((sem.numpy(), spk.numpy()))
        return out


class _SpeechStreams(nn.Module):
    """waveforms [B, T] -> (deep-layer mean [B, frames, D], early-layer mean [B, frames, D])."""

    def __init__(self, model: nn.Module, sem_layers: Tuple[int, ...], spk_layers: Tuple[int, ...], use_mask: bool) -> None:
        super().__init__()
        self.model, self.sem_layers, self.spk_layers, self.use_mask = model, tuple(sem_layers), tuple(spk_layers), use_mask

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        hs = self.model(x, attention_mask=mask if self.use_mask else None, output_hidden_states=True).hidden_states
        sem = torch.stack([hs[l] for l in self.sem_layers]).mean(0).float()
        early = torch.stack([hs[l] for l in self.spk_layers]).mean(0).float()
        return sem, early


class SSLFrontend:
    """HuBERT / wav2vec2-family encoder.  ``sem`` = mean of deep hidden states (pooled to 25 Hz),
    ``spk`` = time-average of early hidden states."""

    def __init__(self, cfg, device: torch.device) -> None:
        from transformers import AutoConfig, AutoModel

        self.cfg, self.device, self.pool = cfg, device, cfg.speech_pool
        self.amp = bool(cfg.amp and device.type == "cuda")
        model = AutoModel.from_pretrained(cfg.speech_encoder)
        conf = AutoConfig.from_pretrained(cfg.speech_encoder)
        self.dim = self.spk_dim = conf.hidden_size
        layer_norm = getattr(conf, "feat_extract_norm", "group") == "layer"
        self.normalize = layer_norm
        try:
            from transformers import AutoFeatureExtractor

            self.normalize = bool(AutoFeatureExtractor.from_pretrained(cfg.speech_encoder).do_normalize)
        except Exception:
            pass
        self.frames = model._get_feat_extract_output_lengths
        # group-norm models were trained on zero-padded batches without a mask; layer-norm models need one
        self.net = on_all_gpus(_SpeechStreams(model, cfg.speech_sem_layers, cfg.speech_spk_layers, layer_norm), cfg, device)

    @torch.no_grad()
    def __call__(self, wavs: List[np.ndarray]) -> List[Tuple[np.ndarray, np.ndarray]]:
        lens = [len(w) for w in wavs]
        x = torch.zeros(len(wavs), max(max(lens), 400))
        mask = torch.zeros(x.shape, dtype=torch.long)
        for i, w in enumerate(wavs):
            t = torch.from_numpy(w).float()
            if self.normalize:
                t = (t - t.mean()) / (t.std() + 1e-7)
            x[i, : len(w)] = t
            mask[i, : len(w)] = 1
        with torch.autocast(device_type=self.device.type, enabled=self.amp):
            sem, early = self.net(x.to(self.device), mask.to(self.device))
        sem, early = sem.float().cpu(), early.float().cpu()   # one transfer per batch, then cheap slicing
        out = []
        for i, n in enumerate(self.frames(torch.tensor(lens)).tolist()):
            n = max(1, min(int(n), sem.shape[1]))
            s = sem[i, :n]
            m = max(1, n // self.pool)
            s = s[: m * self.pool].reshape(m, self.pool, -1).mean(1) if n >= self.pool else s.mean(0, keepdim=True)
            out.append((s.numpy(), early[i, :n].mean(0).numpy()))
        return out


def prep_wave(cfg, wav: np.ndarray, sr: int) -> np.ndarray:
    """Resample, remove rumble, trim silence, truncate: the one waveform every stream of an utterance is computed
    from."""
    wav = highpass(resample(wav, sr, cfg.sample_rate), cfg.highpass_hz, cfg.sample_rate)
    wav = trim_silence(wav, cfg.trim_silence_db, cfg.sample_rate)[: int(cfg.max_audio_sec * cfg.sample_rate)]
    if len(wav) < cfg.sample_rate // 10:
        wav = np.pad(wav, (0, cfg.sample_rate // 10 - len(wav)))
    return np.ascontiguousarray(wav, dtype=np.float32)


_ST5 = {}


def compute_mel(cfg, wav: np.ndarray, n_frames: int) -> np.ndarray:
    """Log-mel of an utterance, resampled in time to exactly round(n_frames * mel_fps / 25) frames."""
    if cfg.mel_spec == "speecht5":  # the feature space of the pretrained SpeechT5 HiFi-GAN vocoder
        if "fe" not in _ST5:
            from transformers import SpeechT5FeatureExtractor

            _ST5["fe"] = SpeechT5FeatureExtractor()
        mel = _ST5["fe"](audio_target=wav, sampling_rate=cfg.sample_rate, return_tensors="np")["input_values"][0]
    else:
        mel = log_mel(torch.from_numpy(wav).float(), cfg.sample_rate, 1024, cfg.mel_hop, cfg.n_mels).numpy()
    return _pool_frames(mel, max(1, int(round(n_frames * cfg.mel_fps / 25.0)))).astype(np.float16)


_MEL_JOB: Dict = {}   # what the worker processes of cache_mel inherit when they are forked


def _mel_of_group(i: int) -> List[Tuple[str, np.ndarray]]:
    cfg, frames = _MEL_JOB["cfg"], _MEL_JOB["frames"]
    return [(uid, compute_mel(cfg, prep_wave(cfg, wav, sr), frames[uid])) for uid, wav, sr in _MEL_JOB["groups"][i]["loader"]()]


def cache_mel(cfg, builder, manifest: Dict, splits: Optional[Dict], adir: str, logger, root: Optional[str] = None) -> str:
    """Mel targets of the renderer, in the feature space of the configured vocoder.  A separate, cheap pass
    (no speech encoder), so switching vocoder never re-extracts the expensive streams.  ``root``: the manifest is
    that of extra spoken data (training only, cached under ``root``)."""
    out = os.path.join(root or cfg.cache_dir, f"mel_{cfg.mel_spec}_{os.path.basename(adir)[6:]}")
    if os.path.exists(os.path.join(out, "index.json")):
        return out
    os.makedirs(out, exist_ok=True)
    frames = {k: v[2] for k, v in load_json(os.path.join(adir, "index.json")).items()}
    utts = [u for u in manifest["utts"] if u["has_audio"] and u["uid"] in frames]
    mels, index, off = [], {}, 0
    groups = builder.extra_audio_groups(utts) if root else builder.audio_groups(utts)
    # Pure CPU work, one source file per task: spread over worker processes where the platform can fork (the tasks
    # are closures over open corpora).  Results come back in order, so the cache is the same as from one process.
    n_proc = min(len(groups), max(1, cfg.num_workers))
    pool = None
    if n_proc > 1 and "fork" in multiprocessing.get_all_start_methods():
        _MEL_JOB.update(cfg=cfg, groups=groups, frames=frames)
        pool = multiprocessing.get_context("fork").Pool(n_proc, initializer=torch.set_num_threads, initargs=(1,))
        parts = pool.imap(_mel_of_group, range(len(groups)))
    else:
        _MEL_JOB.update(cfg=cfg, groups=groups, frames=frames)
        parts = map(_mel_of_group, range(len(groups)))
    for part in tqdm(parts, total=len(groups), desc=f"mel ({cfg.mel_spec})", dynamic_ncols=True):
        for uid, m in part:
            index[uid] = [off, int(m.shape[0])]
            mels.append(m)
            off += m.shape[0]
    if pool is not None:
        pool.close()
        pool.join()
    _MEL_JOB.clear()
    arr = np.concatenate(mels)
    np.save(os.path.join(out, "mel.npy"), arr)
    img_split = splits["images"] if splits else {}
    train = [index[u["uid"]] for u in utts if img_split.get(manifest["images"][u["img"]]["id"], "train") == "train"][:3000] or list(index.values())[:3000]
    sample = np.concatenate([np.asarray(arr[o: o + n], np.float32) for o, n in train])
    save_json({"mel_mean": float(sample.mean()), "mel_std": float(sample.std() + 1e-4), "fps": cfg.mel_fps}, os.path.join(out, "stats.json"))
    save_json(index, os.path.join(out, "index.json"))
    logger.info(f"[cache] mel ({cfg.mel_spec}, {cfg.mel_fps} fps): {len(index)} utterances -> {out}")
    return out


def speech_tag(cfg) -> str:
    hp = f"_hp{cfg.highpass_hz:g}" if cfg.highpass_hz > 0 else ""   # another waveform, another cache
    if cfg.speech_encoder == "mfcc":
        return "mfcc" + hp
    return f"{_tag(cfg.speech_encoder)}_L{'-'.join(map(str, cfg.speech_sem_layers))}_p{cfg.speech_pool}{hp}"


def cache_audio(cfg, builder, manifest: Dict, splits: Optional[Dict], device: torch.device, logger,
                root: Optional[str] = None) -> str:
    """Speech streams of every spoken utterance of ``manifest``.  ``root``: the manifest is that of extra spoken
    data (training only, cached under ``root``; its own statistics are stored but the model normalises with the
    benchmark's)."""
    out = os.path.join(root or cfg.cache_dir, f"audio_{speech_tag(cfg)}")
    if os.path.exists(os.path.join(out, "index.json")):
        return out
    os.makedirs(os.path.join(out, "chunks"), exist_ok=True)
    utts = [u for u in manifest["utts"] if u["has_audio"]]
    groups = builder.extra_audio_groups(utts) if root else builder.audio_groups(utts)
    fe = None
    pbar = tqdm(total=sum(len(g["uids"]) for g in groups), desc="speech features" + (" (extra speech)" if root else ""),
                dynamic_ncols=True)
    for grp in groups:
        cp = os.path.join(out, "chunks", _tag(grp["key"]) + ".npz")
        if os.path.exists(cp):
            pbar.update(len(grp["uids"]))
            continue
        if fe is None:
            fe = MFCCFrontend(cfg) if cfg.speech_encoder == "mfcc" else SSLFrontend(cfg, device)
        uids, sems, tones, spks, f0s, lens = [], [], [], [], [], []
        buf: List[Tuple[str, np.ndarray]] = []

        def flush() -> None:
            if not buf:
                return
            order = np.argsort([len(w) for _, w in buf])
            feats = fe([buf[i][1] for i in order])
            for i, (sem, spk) in zip(order, feats):
                uid, wav = buf[i]
                n = sem.shape[0]
                tone, f0s_ = tone_features(wav, n, cfg.sample_rate)
                uids.append(uid); sems.append(sem.astype(np.float16)); tones.append(tone)
                spks.append(spk.astype(np.float32)); f0s.append(f0s_); lens.append(n)
            buf.clear()

        for uid, wav, sr in grp["loader"]():
            buf.append((uid, prep_wave(cfg, wav, sr)))
            if len(buf) >= cfg.feat_batch_size:
                flush()
            pbar.update(1)
        flush()
        np.savez(cp + ".tmp.npz", uids=np.array(uids), lens=np.array(lens), sem=np.concatenate(sems),
                 tone=np.concatenate(tones), spk=np.stack(spks), f0=np.stack(f0s))
        os.replace(cp + ".tmp.npz", cp)
    pbar.close()

    # ---- consolidate chunks into memory-mapped arrays (the chunks of THIS list of groups: a directory listing
    #      would also pick up what another process working on the same cache has left or not yet written)
    chunk_files = sorted(_tag(g["key"]) + ".npz" for g in groups)
    zs = [np.load(os.path.join(out, "chunks", c)) for c in chunk_files]
    total = int(sum(z["lens"].sum() for z in zs))
    n_utt = int(sum(len(z["lens"]) for z in zs))
    d_s, d_spk = zs[0]["sem"].shape[1], zs[0]["spk"].shape[1]
    sem = np.lib.format.open_memmap(os.path.join(out, "sem.npy"), "w+", np.float16, (total, d_s))
    tone = np.lib.format.open_memmap(os.path.join(out, "tone.npy"), "w+", np.float32, (total, 4))
    spk, f0 = np.zeros((n_utt, d_spk), np.float32), np.zeros((n_utt, 2), np.float32)
    index: Dict[str, List[int]] = {}
    off = row = 0
    for z in zs:
        m = int(z["lens"].sum())
        sem[off: off + m], tone[off: off + m] = z["sem"], z["tone"]
        o = off
        for uid, n, s, f in zip(z["uids"].tolist(), z["lens"].tolist(), z["spk"], z["f0"]):
            index[uid] = [row, o, int(n)]
            spk[row], f0[row] = s, f
            o += int(n); row += 1
        off += m
    sem.flush(); tone.flush()
    np.save(os.path.join(out, "spk.npy"), spk)
    np.save(os.path.join(out, "f0stats.npy"), f0)

    # ---- normalisation statistics from the training split only
    img_split = splits["images"] if splits else {}
    lang_of = {u["uid"]: (u["lang"], manifest["images"][u["img"]]["id"]) for u in utts}
    tr_rows = [v for k, v in index.items() if img_split.get(lang_of[k][1], "train") == "train"]
    rng = np.random.default_rng(0)
    pick = [tr_rows[i] for i in rng.permutation(len(tr_rows))[:3000]] or list(index.values())[:3000]
    xs = np.concatenate([np.asarray(sem[o: o + n], np.float32) for _, o, n in pick])
    stats = {"sem_mean": xs.mean(0), "sem_std": xs.std(0) + 1e-4, "dim": d_s, "spk_dim": d_spk,
             "spk_mean": spk.mean(0), "spk_std": spk.std(0) + 1e-4,
             "f0_mean": f0.mean(0), "f0_std": f0.std(0) + 1e-4, "frame_rate": 25.0}
    save_json(stats, os.path.join(out, "stats.json"))
    save_json(index, os.path.join(out, "index.json"))  # written last: marks the cache as complete
    for c in chunk_files:
        os.remove(os.path.join(out, "chunks", c))
    logger.info(f"[cache] speech: {n_utt} utterances, {total} frames ({total / 25 / 3600:.2f} h) -> {out}")
    return out


class AudioStore:
    """Read-only random access to the cached speech streams (and, if given, the mel targets)."""

    def __init__(self, path: str, mel_path: Optional[str] = None) -> None:
        self.index: Dict[str, List[int]] = load_json(os.path.join(path, "index.json"))
        self.stats = {k: (np.array(v, dtype=np.float32) if isinstance(v, list) else v)
                      for k, v in load_json(os.path.join(path, "stats.json")).items()}
        self.stats.update(mel_mean=0.0, mel_std=1.0)
        self.sem = np.load(os.path.join(path, "sem.npy"), mmap_mode="r")
        self.tone = np.load(os.path.join(path, "tone.npy"), mmap_mode="r")
        self.mel, self.mel_index = None, {}
        if mel_path:
            self.mel = np.load(os.path.join(mel_path, "mel.npy"), mmap_mode="r")
            self.mel_index = load_json(os.path.join(mel_path, "index.json"))
            ms = load_json(os.path.join(mel_path, "stats.json"))
            self.stats.update(mel_mean=ms["mel_mean"], mel_std=ms["mel_std"])
        self.spk = np.load(os.path.join(path, "spk.npy"))
        self.f0 = np.load(os.path.join(path, "f0stats.npy"))

    def get(self, uid: str, max_frames: int, mel_fps: float = 50.0) -> Dict[str, np.ndarray]:
        row, off, n = self.index[uid]
        n_use = min(n, max_frames)
        out = {"sem": np.array(self.sem[off: off + n_use], dtype=np.float32),
               "tone": np.array(self.tone[off: off + n_use], dtype=np.float32),
               "spk": self.spk[row], "f0": self.f0[row]}
        if self.mel is not None:
            mo, mn = self.mel_index[uid]
            out["mel"] = np.array(self.mel[mo: mo + min(mn, int(round(n_use * mel_fps / 25.0)))], dtype=np.float32)
        return out


class AudioStores:
    """The benchmark's own speech cache and those of extra spoken data, read as one.  Normalisation statistics are
    the benchmark's: the model is built from them and extra data must not move them."""

    def __init__(self, stores: List[AudioStore]) -> None:
        self.stores, self.stats = list(stores), stores[0].stats
        self.index: Dict[str, int] = {uid: k for k, st in enumerate(self.stores) for uid in st.index}

    def get(self, uid: str, max_frames: int, mel_fps: float = 50.0) -> Dict[str, np.ndarray]:
        return self.stores[self.index[uid]].get(uid, max_frames, mel_fps)


# ======================================================================================================================
# 09. DATA: dataset, persistent splits, loaders
#     (geobridge/data/dataset.py)
# ======================================================================================================================
# Dataset assembly, persistent splits, and the three data streams used by the trainer.
#
# Streams
#   paired  one item = one caption group (image, caption index) with every *visible* view of it:
#           hub-language text + speech, text-only-language text, and island speech only if that utterance is
#           one of the bridge pairs.
#   island  unpaired speech of the island language(s): no image, no text, no link to anything.
#   eval    caption groups of the valid / test split with all views (held-out text is exposed as reference).
#
# Splits are created once and stored in ``splits.json``; later runs load the file, they never re-draw it.
# Bridge membership is a stored per-utterance rank in [0, 1): an utterance is a bridge pair iff
# rank < bridge_frac, so the 1 % / 5 % / 20 % / 100 % budgets are nested subsets of one another.
#
# Extra image-text pairs (``cfg.extra_images``) are images with captions in the hub language only, as text.  They
# are appended after the benchmark's own images, belong to the training split by construction, never enter
# ``splits.json`` or any evaluation, and have their own feature caches.
# ======================================================================================================================

class _GroupDataset(Dataset):
    def __init__(self, data: "GeoBridgeData", groups: List[Dict], train: bool) -> None:
        self.data, self.groups, self.train = data, groups, train

    def __len__(self) -> int:
        return len(self.groups)

    def __getitem__(self, i: int) -> Dict:
        g = self.groups[i]
        return {"img": g["img"], "cap": g["cap"], "views": g["train_views"] if self.train else g["valid_views"],
                "train": self.train, "listen": self.train and self.data.cfg.listen_only and bool(g.get("extra"))}


class _Stacked:
    """Several row-indexed arrays read as one: the benchmark's own images first, then the extra ones."""

    def __init__(self, parts: Sequence) -> None:
        self.parts = list(parts)
        self.offsets = np.cumsum([0] + [len(p) for p in self.parts])
        self.shape = (int(self.offsets[-1]), *self.parts[0].shape[1:])

    def __len__(self) -> int:
        return self.shape[0]

    def __getitem__(self, i: int):
        k = int(np.searchsorted(self.offsets, int(i), side="right") - 1)
        return self.parts[k][int(i) - int(self.offsets[k])]


class _IslandDataset(Dataset):
    def __init__(self, utt_ids: List[int]) -> None:
        self.ids = utt_ids

    def __len__(self) -> int:
        return len(self.ids)

    def __getitem__(self, i: int) -> int:
        return self.ids[i]


class GeoBridgeData:
    def __init__(self, cfg, logger=None, device: Optional[torch.device] = None, need_mel: bool = True) -> None:
        self.cfg = cfg
        self.log = logger or get_logger()
        device = device or torch.device("cpu")
        os.makedirs(cfg.cache_dir, exist_ok=True)

        if cfg.dataset == "synthworld":

            self.builder = SynthWorld(cfg)
        elif cfg.dataset == "flickr8k_yo":

            self.builder = Flickr8kTri(cfg, self.log)
        else:
            raise ValueError(f"unknown dataset {cfg.dataset}")

        # ---- manifest (built once)
        mp = os.path.join(cfg.cache_dir, "manifest.json")
        if os.path.exists(mp):
            self.manifest = load_json(mp)
            if hasattr(self.builder, "img_file") is False and cfg.dataset == "flickr8k_yo":
                self.builder.manifest()  # rebuild the media locators (cheap: ids only)
        else:
            self.manifest = self.builder.manifest()
            save_json(self.manifest, mp)
        self.images, self.utts = self.manifest["images"], self.manifest["utts"]
        self.lang_info = self.manifest["langs"]
        self.langs = [l for l in cfg.all_langs if l in self.lang_info]
        self.lang2id = {l: i for i, l in enumerate(cfg.all_langs)}

        # ---- splits (created once, then always loaded)
        self.splits = self._load_or_make_splits()
        self.img_split = [self.splits["images"][im["id"]] for im in self.images]

        # ---- frozen-feature caches (computed once)
        vdir = cache_vision(cfg, self.builder, self.manifest, device, self.log)
        self.vision = np.load(os.path.join(vdir, "tokens.npy"), mmap_mode="r")
        self.thumbs = None
        if cfg.w_pix > 0 and cfg.image_spoke:
            self.thumbs = np.load(cache_thumbs(cfg, self.builder, self.manifest, vdir, self.log), mmap_mode="r")
        self.vision_dim, self.n_vis_tokens = int(self.vision.shape[2]), int(self.vision.shape[1])
        adir = cache_audio(cfg, self.builder, self.manifest, self.splits, device, self.log)
        mdir = cache_mel(cfg, self.builder, self.manifest, self.splits, adir, self.log) if need_mel and cfg.w_mel > 0 else None
        self.audio = AudioStore(adir, mdir)
        self.sem_dim, self.spk_dim = int(self.audio.stats["dim"]), int(self.audio.stats["spk_dim"])
        train_imgs = np.array([i for i, s in enumerate(self.img_split) if s == "train"])
        self.anchor = fit_anchor(self.vision, train_imgs, cfg.latent_dim,
                                       os.path.join(vdir, f"anchor_d{cfg.latent_dim}.pt"))
        self.max_frames = int(cfg.max_audio_sec * 25)
        self.n_base_images, self.extra_meta = len(self.images), {}
        if cfg.extra_images or cfg.extra_speech:
            self._add_extras(device, need_mel)

        # ---- text units
        bpe_path = None
        if cfg.text_unit == "bpe":
            bpe_path = os.path.join(cfg.cache_dir, f"bpe_{cfg.bpe_vocab}_{'-'.join(cfg.hub_langs)}.json")
            if not os.path.exists(bpe_path):
                texts = [u["text"] for u in self.utts
                         if u["lang"] in cfg.hub_langs and self.img_split[u["img"]] == "train" and u["text"]]
                TextUnitizer.train_bpe(texts, bpe_path, cfg.bpe_vocab)
        self.unitizer = TextUnitizer(cfg.text_unit, cfg.max_graphemes, cfg.max_bytes_per_grapheme,
                                     cfg.max_text_bytes, bpe_path)
        self._build_groups()

    # ------------------------------------------------------------------ extra hub-language data
    def _add_extras(self, device: torch.device, need_mel: bool) -> None:
        """Append extra images with hub-language captions to the training split: as text (``extra_images``), and
        as text + speech (``extra_speech``).  Each source has its own caches; none touches the benchmark's."""
        cfg = self.cfg
        if not hasattr(self.builder, "extra_manifest"):
            raise ValueError(f"dataset '{cfg.dataset}' has no extra data")
        own = {im["id"] for im in self.images}
        stores = [self.audio]
        for speech, n, root in ((False, cfg.extra_images, cfg.extra_cache_dir), (True, cfg.extra_speech, cfg.extra_speech_cache_dir)):
            if not n:
                continue
            os.makedirs(root, exist_ok=True)
            mp, n0 = os.path.join(root, "manifest.json"), len(self.images)
            if os.path.exists(mp) and cfg.dataset != "synthworld":
                xm = load_json(mp)
            else:  # (the procedural benchmark rebuilds its scenes: they are also what its images are drawn from)
                xm = self.builder.extra_manifest(n, speech=speech, first=n0)
                save_json(xm, mp)
            assert not any(im["id"] in own for im in xm["images"]), "an extra image is one of the benchmark's own"
            own |= {im["id"] for im in xm["images"]}
            vdir = cache_vision(cfg, self.builder, xm, device, self.log, extra_from=n0, root=root)
            self.vision = _Stacked([*getattr(self.vision, "parts", [self.vision]), np.load(os.path.join(vdir, "tokens.npy"), mmap_mode="r")])
            if self.thumbs is not None:
                tp = cache_thumbs(cfg, self.builder, xm, vdir, self.log, extra_from=n0)
                self.thumbs = _Stacked([*getattr(self.thumbs, "parts", [self.thumbs]), np.load(tp, mmap_mode="r")])
            if speech:
                adir = cache_audio(cfg, self.builder, xm, None, device, self.log, root=root)
                mdir = cache_mel(cfg, self.builder, xm, None, adir, self.log, root=root) if need_mel and cfg.w_mel > 0 else None
                stores.append(AudioStore(adir, mdir))
            self.images = self.images + xm["images"]
            self.utts = self.utts + [dict(u, img=u["img"] + n0) for u in xm["utts"]]
            self.img_split = self.img_split + ["train"] * len(xm["images"])
            meta = dict(xm.get("meta", {}), images=len(xm["images"]), captions=len(xm["utts"]),
                        spoken=sum(u["has_audio"] for u in xm["utts"]))
            self.extra_meta["speech" if speech else "text"] = meta
            self.log.info(f"[extra] {meta['images']} extra images with {meta['captions']} {cfg.hub_langs[0]} captions ("
                          + (f"text + speech, {meta['speakers']} speakers" if speech and "speakers" in meta
                             else "text + speech" if speech else "text only") + ", training only)"
                          + (f"; {meta['dropped_flickr8k_photos']} Flickr8k photos left out" if "dropped_flickr8k_photos" in meta else ""))
        if len(stores) > 1:
            self.audio = AudioStores(stores)

    # ------------------------------------------------------------------ splits
    def _load_or_make_splits(self) -> Dict:
        cfg, path = self.cfg, os.path.join(self.cfg.cache_dir, "splits.json")
        if os.path.exists(path):
            sp = load_json(path)
            if all(im["id"] in sp["images"] for im in self.images):
                return sp
            self.log.info("[splits] existing splits.json does not cover the manifest; extending it")
        else:
            sp = {"images": {}, "bridge_rank": {}}
        rng = np.random.default_rng(0)  # fixed: the split is a property of the dataset, not of the run
        free = [im["id"] for im in self.images if im["id"] not in sp["images"] and not im.get("official_split")]
        for im in self.images:
            if im["id"] not in sp["images"] and im.get("official_split"):
                sp["images"][im["id"]] = im["official_split"]
        perm = rng.permutation(len(free))
        a, b, _ = cfg.split_fracs
        n_tr, n_va = int(round(a * len(free))), int(round(b * len(free)))
        for rank, j in enumerate(perm):
            sp["images"][free[j]] = "train" if rank < n_tr else ("valid" if rank < n_tr + n_va else "test")
        island = sorted(u["uid"] for u in self.utts if u["uid"] not in sp["bridge_rank"])
        ranks = rng.permutation(len(island)) / max(1, len(island))
        for uid, r in zip(island, ranks):
            sp["bridge_rank"][uid] = float(r)
        counts: Dict[str, int] = {}
        for s in sp["images"].values():
            counts[s] = counts.get(s, 0) + 1
        sp["counts"], sp["created"] = counts, time.strftime("%Y-%m-%d %H:%M:%S")
        sp["note"] = ("image-level split; official corpus split where one exists, otherwise a fixed random draw. "
                      "bridge_rank < bridge_frac marks the island utterances that keep their image pairing.")
        save_json(sp, path)
        self.log.info(f"[splits] created {path}: {counts}")
        return sp

    # ------------------------------------------------------------------ views
    def _build_groups(self) -> None:
        """Decide, once, which views of every caption the model may see in training.

        strict_nonparallel (default): in the training split a caption group is visible in exactly one
        language - the island language if it owns that caption index, else a text-only language if it owns
        it, else the hub language(s).  Captions are translations of each other across languages, so without
        this rule the corpus would be implicitly parallel through the shared image."""
        cfg = self.cfg
        by_key: Dict[Tuple[int, int], Dict] = {}
        for ui, u in enumerate(self.utts):
            if u["lang"] not in self.lang2id:
                continue
            g = by_key.setdefault((u["img"], u["cap"]), {"img": u["img"], "cap": u["cap"], "utts": {},
                                                         "train_views": [], "all_views": [],
                                                         "extra": u["img"] >= self.n_base_images})
            g["utts"][u["lang"]] = ui
            if u.get("text"):
                g["all_views"].append(("txt", u["lang"], ui))
            if u["has_audio"] and u["uid"] in self.audio.index:
                g["all_views"].append(("aud", u["lang"], ui))

        self.island_ids: List[int] = []
        self.n_bridge = 0
        self.groups: Dict[str, List[Dict]] = {"train": [], "valid": [], "test": []}
        self.pivots: Dict[int, List[Tuple[int, int]]] = {}
        for key in sorted(by_key):
            g = by_key[key]
            split, cap = self.img_split[g["img"]], g["cap"]
            has = lambda m, l: any(v[0] == m and v[1] == l for v in g["all_views"])
            owner = None  # the single language that may see this caption in training (strict mode)
            for l in cfg.island_langs:
                if cap in cfg.island_caps and has("aud", l):
                    owner = l
            if owner is None:
                for l in cfg.text_only_langs:
                    if l not in cfg.hub_langs and cap in cfg.text_only_caps and has("txt", l):
                        owner = l
            for mod, lang, ui in g["all_views"]:
                u = self.utts[ui]
                if cfg.strict_nonparallel and owner is not None and lang != owner:
                    continue
                if lang in cfg.island_langs:
                    if cap not in cfg.island_caps or not has("aud", lang):
                        continue
                    bridge = self.splits["bridge_rank"].get(u["uid"], 1.0) < cfg.bridge_frac
                    if mod == "aud" and bridge:
                        g["train_views"].append((mod, lang, ui))
                        self.n_bridge += int(split == "train")
                    elif mod == "aud" and split == "train" and cfg.island_unpaired:
                        self.island_ids.append(ui)
                    elif mod == "txt" and bridge and cfg.use_island_text:
                        g["train_views"].append((mod, lang, ui))
                elif lang in cfg.hub_langs:
                    g["train_views"].append((mod, lang, ui))
                elif lang in cfg.text_only_langs and mod == "txt" and (cap in cfg.text_only_caps or not cfg.strict_nonparallel):
                    g["train_views"].append((mod, lang, ui))
            # validation-loss views: everything the model could be trained on, never the held-out island text
            g["valid_views"] = [v for v in g["all_views"]
                                if not (v[0] == "txt" and v[1] in cfg.island_langs and not cfg.use_island_text)]
            if split == "train":
                for mod, lang, ui in g["train_views"]:
                    if mod == "txt" and lang in cfg.hub_langs:
                        self.pivots.setdefault(g["img"], []).append((cap, ui))
                if not g["train_views"]:
                    continue
            self.groups[split].append(g)
        # extra groups sort after the benchmark's own (their image indices are larger): train = [own ..., extra ...]
        self.n_own_groups = sum(not g["extra"] for g in self.groups["train"])
        assert all(g["extra"] for g in self.groups["train"][self.n_own_groups:])
        assert not any(g["extra"] for s in ("valid", "test") for g in self.groups[s])
        self.replay_pool = sorted({ui for g in self.groups["train"] for m, _, ui in g["train_views"] if m == "txt"})
        self.caps_of_img: Dict[int, List[Dict]] = {}
        for g in by_key.values():
            self.caps_of_img.setdefault(g["img"], []).append(g)
        self._rng = np.random.default_rng(0)

    def describe(self) -> Dict:
        own_split = self.img_split[: self.n_base_images]
        out = {"images": {s: own_split.count(s) for s in ("train", "valid", "test")},
               "caption_groups": {"train": self.n_own_groups, "valid": len(self.groups["valid"]), "test": len(self.groups["test"])},
               "extra": {"images": len(self.images) - self.n_base_images,
                         "caption_groups": len(self.groups["train"]) - self.n_own_groups,
                         "spoken": sum(any(v[0] == "aud" for v in g["train_views"]) for g in self.groups["train"][self.n_own_groups:]),
                         **self.extra_meta},
               "bridge_pairs_train": self.n_bridge, "island_unpaired_train": len(self.island_ids),
               "vision_tokens": [self.n_vis_tokens, self.vision_dim], "speech_dim": self.sem_dim,
               "anchor_explained_var": float(self.anchor["explained"])}
        for l in self.langs:
            us = [u for u in self.utts if u["lang"] == l]
            out[f"lang_{l}"] = {"utterances": len(us), "with_audio": sum(u["uid"] in self.audio.index for u in us),
                                **self.lang_info[l]}
        return out

    # ------------------------------------------------------------------ collation
    def _collate_text(self, ids: Sequence[int], items: Sequence[int]) -> Optional[Dict]:
        if not ids:
            return None
        out = self.unitizer.collate([self.utts[i]["text"] for i in ids])
        out["lang"] = torch.tensor([self.lang2id[self.utts[i]["lang"]] for i in ids])
        out["item"] = torch.tensor(list(items))
        out["utt"] = torch.tensor(list(ids))
        return out

    def _collate_audio(self, ids: Sequence[int], items: Sequence[int]) -> Optional[Dict]:
        if not ids:
            return None
        recs = [self.audio.get(self.utts[i]["uid"], self.max_frames, self.cfg.mel_fps) for i in ids]
        T, n = max(r["sem"].shape[0] for r in recs), len(recs)
        sem = torch.zeros(n, T, self.sem_dim)
        tone = torch.zeros(n, T, 4)
        mask = torch.zeros(n, T, dtype=torch.bool)
        for j, r in enumerate(recs):
            t = r["sem"].shape[0]
            sem[j, :t], tone[j, :t], mask[j, :t] = torch.from_numpy(r["sem"]), torch.from_numpy(r["tone"]), True
        out = {"sem": sem, "tone": tone, "mask": mask,
               "spk": torch.from_numpy(np.stack([r["spk"] for r in recs])),
               "f0": torch.from_numpy(np.stack([r["f0"] for r in recs])),
               "lang": torch.tensor([self.lang2id[self.utts[i]["lang"]] for i in ids]),
               "item": torch.tensor(list(items)), "utt": torch.tensor(list(ids))}
        if "mel" in recs[0]:
            lens = [r["mel"].shape[0] for r in recs]
            mel = torch.zeros(n, max(lens), self.cfg.n_mels)
            for j, r in enumerate(recs):
                mel[j, : lens[j]] = torch.from_numpy(r["mel"])
            out["mel"], out["mel_len"] = mel, torch.tensor(lens)
        return out

    def collate_groups(self, batch: Sequence[Dict]) -> Dict:
        img = torch.from_numpy(np.stack([np.asarray(self.vision[b["img"]], np.float32) for b in batch]))
        t_ids, t_items, a_ids, a_items, a_listen = [], [], [], [], []
        for j, b in enumerate(batch):
            for mod, _, ui in b["views"]:
                (t_ids if mod == "txt" else a_ids).append(ui)
                (t_items if mod == "txt" else a_items).append(j)
                if mod == "aud":
                    a_listen.append(bool(b.get("listen")))
        if any(a_listen):   # speech views of extra groups (cfg.listen_only) come last: the grounded ones stay in front
            order = sorted(range(len(a_ids)), key=lambda k: a_listen[k])
            a_ids, a_items, a_listen = ([x[k] for k in order] for x in (a_ids, a_items, a_listen))
        out = {"B": len(batch), "img": img, "img_idx": torch.tensor([b["img"] for b in batch]),
               "cap": torch.tensor([b["cap"] for b in batch]),
               "txt": self._collate_text(t_ids, t_items), "aud": self._collate_audio(a_ids, a_items)}
        if any(a_listen):
            out["aud"]["listen"] = torch.tensor(a_listen)
        train = bool(batch[0].get("train"))
        if self.thumbs is not None and train:
            out["thumb"] = torch.from_numpy(np.stack([np.asarray(self.thumbs[b["img"]]) for b in batch]))
        if self.cfg.text_replay > 0 and train and self.replay_pool:
            # text is cheap: the text codec also sees extra captions that take no part in grounding
            pick = self._rng.integers(len(self.replay_pool), size=self.cfg.text_replay)
            out["replay"] = self._collate_text([self.replay_pool[i] for i in pick], [-1] * len(pick))
        if self.cfg.hub_mode == "text":  # text-anchor baseline: one hub-language caption of the SAME IMAGE,
            piv = []                      # never the same caption index (that would be a parallel sentence)
            for b in batch:
                cands = [ui for cap, ui in self.pivots.get(b["img"], []) if cap != b["cap"]]
                piv.append(cands[int(self._rng.integers(len(cands)))] if cands else -1)
            keep = [j for j, ui in enumerate(piv) if ui >= 0]
            out["pivot"] = self._collate_text([piv[j] for j in keep], keep)
        return out

    def collate_island(self, ids: Sequence[int]) -> Dict:
        return {"B": len(ids), "img": None, "txt": None, "aud": self._collate_audio(list(ids), list(range(len(ids))))}

    # ------------------------------------------------------------------ loaders
    def paired_loader(self, split: str = "train", shuffle: bool = True, batch_size: Optional[int] = None) -> DataLoader:
        ds = _GroupDataset(self, self.groups[split], train=(split == "train"))
        return DataLoader(ds, batch_size=batch_size or self.cfg.batch_size, shuffle=shuffle, drop_last=shuffle,
                          collate_fn=self.collate_groups, **worker_kwargs(self.cfg.num_workers if shuffle else 0))

    def eval_groups(self, split: str, max_items: int = 0, require: Sequence[Tuple[str, str]] = (),
                    favour: Sequence[str] = ()) -> List[Dict]:
        """Deterministic subset of caption groups that contain all required (modality, language) views.

        ``favour``: languages whose captions fill up to half of the subset.  An island language owns one caption in
        five, so a plain random subset of 400 groups holds about 70 of its utterances - too few to steer anything by."""
        gs = [g for g in self.groups[split]
              if all(any(v[0] == m and v[1] == l for v in g["all_views"]) for m, l in require)]
        if max_items and len(gs) > max_items:
            order = np.random.default_rng(0).permutation(len(gs)).tolist()
            first = [i for i in order if any(v[1] in favour for v in gs[i]["all_views"])][:max_items // 2] if favour else []
            taken = set(first)
            idx = first + [i for i in order if i not in taken][:max_items - len(first)]
            gs = [gs[i] for i in sorted(idx)]
        return gs

    def eval_batches(self, groups: List[Dict], batch_size: int = 64):
        for i in range(0, len(groups), batch_size):
            part = groups[i:i + batch_size]
            yield part, self.collate_groups([{"img": g["img"], "cap": g["cap"], "views": g["all_views"]} for g in part])

    def references(self, img: int, lang: str) -> List[str]:
        """All captions of an image in a language (multi-reference scoring)."""
        return [self.utts[g["utts"][lang]]["text"] for g in self.caps_of_img.get(img, [])
                if lang in g["utts"] and self.utts[g["utts"][lang]].get("text")]


# ======================================================================================================================
# 10. MODEL: layers
#     (geobridge/models/layers.py)
# ======================================================================================================================
# Building blocks: language-factorised LoRA linear, attention, pre-LN transformer block, Perceiver resampler.
# ======================================================================================================================

def sincos(pos: torch.Tensor, dim: int, max_period: float = 10000.0) -> torch.Tensor:
    """Sinusoidal embedding of arbitrary (possibly fractional) positions.  pos [...] -> [..., dim]."""
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(half, device=pos.device, dtype=torch.float32) / half)
    args = pos.float().unsqueeze(-1) * freqs
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
    if dim % 2:
        emb = F.pad(emb, (0, 1))
    return emb


class LoRALinear(nn.Module):
    """y = W x + b + (alpha / r) * B_l A_l x, with one low-rank pair (A_l, B_l) per language l.

    The dense weight is the shared, language-agnostic drift; the low-rank pair is everything a language owns.
    B is zero-initialised, so a new language starts exactly at the shared field.
    """

    def __init__(self, in_f: int, out_f: int, n_lang: int = 0, rank: int = 0, alpha: float = 16.0, bias: bool = True):
        super().__init__()
        self.lin = nn.Linear(in_f, out_f, bias=bias)
        self.rank = rank if n_lang > 0 else 0
        if self.rank > 0:
            self.lora_A = nn.Parameter(torch.randn(n_lang, rank, in_f) / math.sqrt(in_f))
            self.lora_B = nn.Parameter(torch.zeros(n_lang, out_f, rank))
            self.scale = alpha / rank

    def forward(self, x: torch.Tensor, lang: Optional[torch.Tensor] = None) -> torch.Tensor:
        y = self.lin(x)
        if self.rank > 0 and lang is not None:
            a, b = self.lora_A[lang].to(x.dtype), self.lora_B[lang].to(x.dtype)  # [B, r, in], [B, out, r]
            y = y + self.scale * torch.einsum("bnr,bor->bno", torch.einsum("bni,bri->bnr", x, a), b)
        return y


class Attention(nn.Module):
    def __init__(self, dim: int, heads: int, ctx_dim: Optional[int] = None, n_lang: int = 0, rank: int = 0,
                 alpha: float = 16.0):
        super().__init__()
        self.heads, self.cross = heads, ctx_dim is not None
        self.q = LoRALinear(dim, dim, n_lang, rank, alpha)
        self.kv = LoRALinear(ctx_dim or dim, 2 * dim, n_lang, rank, alpha)
        self.proj = LoRALinear(dim, dim, n_lang, rank, alpha)

    def forward(self, x, ctx=None, key_mask=None, causal: bool = False, lang=None):
        B, T, D = x.shape
        src = ctx if ctx is not None else x
        S, H = src.shape[1], self.heads
        q = self.q(x, lang).view(B, T, H, D // H).transpose(1, 2)
        k, v = self.kv(src, lang).view(B, S, 2, H, D // H).permute(2, 0, 3, 1, 4)
        mask = None
        if key_mask is not None:
            mask = key_mask[:, None, None, :].expand(B, 1, T, S)
        if causal:
            tri = torch.ones(T, S, dtype=torch.bool, device=x.device).tril()[None, None]
            mask = tri if mask is None else (mask & tri)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.proj(out.transpose(1, 2).reshape(B, T, D), lang)


class Block(nn.Module):
    """Pre-LN transformer block: self-attention [+ cross-attention] + MLP.

    ``cond_dim > 0`` turns the norms into AdaLN (scale/shift predicted from a conditioning vector,
    zero-initialised so the block starts as a plain pre-LN block).  ``n_lang, rank`` enable language LoRA.
    """

    def __init__(self, dim: int, heads: int, mlp_ratio: float = 4.0, ctx_dim: Optional[int] = None, n_lang: int = 0,
                 rank: int = 0, alpha: float = 16.0, cond_dim: int = 0, self_attn: bool = True, dropout: float = 0.0):
        super().__init__()
        self.has_self, self.has_cross, self.cond = self_attn, ctx_dim is not None, cond_dim > 0
        self.drop = nn.Dropout(dropout)   # on every residual branch (identity when dropout = 0)
        affine = not self.cond
        if self_attn:
            self.n1 = nn.LayerNorm(dim, elementwise_affine=affine)
            self.attn = Attention(dim, heads, None, n_lang, rank, alpha)
        if self.has_cross:
            self.nc = nn.LayerNorm(dim, elementwise_affine=affine)
            self.xattn = Attention(dim, heads, ctx_dim, n_lang, rank, alpha)
        self.n2 = nn.LayerNorm(dim, elementwise_affine=affine)
        hid = int(dim * mlp_ratio)
        self.fc1 = LoRALinear(dim, hid, n_lang, rank, alpha)
        self.fc2 = LoRALinear(hid, dim, n_lang, rank, alpha)
        if self.cond:
            n_norm = int(self_attn) + int(self.has_cross) + 1
            self.ada = nn.Linear(cond_dim, 2 * dim * n_norm)
            nn.init.zeros_(self.ada.weight)
            nn.init.zeros_(self.ada.bias)

    def forward(self, x, mask=None, ctx=None, ctx_mask=None, causal: bool = False, lang=None, cond=None):
        mods = None
        if self.cond:
            mods = list(self.ada(cond).unsqueeze(1).chunk(2 * (int(self.has_self) + int(self.has_cross) + 1), dim=-1))

        def norm(layer, h):
            h = layer(h)
            if mods is not None:
                s, b = mods.pop(0), mods.pop(0)
                h = h * (1 + s) + b
            return h

        if self.has_self:
            x = x + self.drop(self.attn(norm(self.n1, x), key_mask=mask, causal=causal, lang=lang))
        if self.has_cross:
            x = x + self.drop(self.xattn(norm(self.nc, x), ctx=ctx, key_mask=ctx_mask, lang=lang))
        return x + self.drop(self.fc2(F.gelu(self.fc1(norm(self.n2, x), lang)), lang))


class Resampler(nn.Module):
    """Perceiver-style resampler: n learned queries read a variable-length sequence -> fixed [n, out_dim]."""

    def __init__(self, dim: int, n_queries: int, out_dim: int, layers: int = 2, heads: int = 4, dropout: float = 0.0):
        super().__init__()
        self.queries = nn.Parameter(torch.randn(n_queries, dim) * 0.02)
        self.blocks = nn.ModuleList([Block(dim, heads, ctx_dim=dim, dropout=dropout) for _ in range(layers)])
        self.norm = nn.LayerNorm(dim)
        self.out = nn.Linear(dim, out_dim)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None, anchor: Optional[torch.Tensor] = None) -> torch.Tensor:
        """``anchor`` [B, n, dim]: what every query starts from in addition to its learned vector (e.g. the mean of
        its own span of the sequence, which makes the n outputs an ordered summary instead of a free set)."""
        q = self.queries.unsqueeze(0).expand(x.shape[0], -1, -1)
        if anchor is not None:
            q = q + anchor
        for blk in self.blocks:
            q = blk(q, ctx=x, ctx_mask=mask)
        return self.out(self.norm(q))


def standardize_tokens(z: torch.Tensor) -> torch.Tensor:
    """Token-wise standardisation (parameter-free LayerNorm).  Every latent token has zero mean and unit
    variance across channels, so all modality latents live at the same scale without tracked statistics."""
    return F.layer_norm(z, (z.shape[-1],))


class MLP(nn.Module):
    def __init__(self, d_in: int, d_hid: int, d_out: int, zero_init: bool = False):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d_hid), nn.GELU(), nn.Linear(d_hid, d_out))
        if zero_init:
            nn.init.zeros_(self.net[-1].weight)
            nn.init.zeros_(self.net[-1].bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ======================================================================================================================
# 11. MODEL: entropic optimal transport
#     (geobridge/models/ot.py)
# ======================================================================================================================
# Entropic optimal transport in the log domain, and the fused / semi-relaxed Gromov-Wasserstein solver used
# for docking.  Everything runs in float32 outside autocast and uses only matrix products and logsumexp - no
# SVD, no inversion - so it is safe under mixed-precision training and with small batches.
#
# Fused objective (alpha in [0, 1], optional pinning cost R):
#
#     FGW_alpha(Pi) = (1 - alpha) * sum_{ijkl} (C1_ik - C2_jl)^2 Pi_ij Pi_kl  +  alpha * <M, Pi>  +  <R, Pi>
#
#     alpha = 0  pure Gromov-Wasserstein: compares only *relations inside* each set (robust to a displaced or
#                rotated island, blind to where it sits)
#     alpha = 1  pure Wasserstein with a cross-set cost M (sharp once the two sets share coordinates)
#
# Marginals
#     balanced       Pi 1 = a,  Pi^T 1 = b           (log-domain Sinkhorn inner loop)        <- default
#     semi-relaxed   Pi 1 = a,  column marginal free (closed-form row-softmax inner step)
#
# The column constraint is what keeps docking from degenerating.  With the bank side free, the plan may send
# every island sample to the same few hub codes; since the Wasserstein part of the cost is induced by the model
# being trained, that choice is self-reinforcing.  Measured on SynthWorld: the share of bank codes in use fell
# from 88 % to 6 % during training and island translation dropped to chance, whereas the balanced plan kept
# 90 % of the codes in use.  Semi-relaxed is kept as an ablation.
# ======================================================================================================================

def log_sinkhorn(cost: torch.Tensor, a: torch.Tensor, b: torch.Tensor, eps: float, iters: int = 50) -> torch.Tensor:
    """Balanced entropic OT plan for a cost matrix [n, m]; fully in the log domain."""
    cost = cost.float()
    la, lb = torch.log(a.float().clamp_min(1e-30)), torch.log(b.float().clamp_min(1e-30))
    f, g = torch.zeros_like(a, dtype=torch.float32), torch.zeros_like(b, dtype=torch.float32)
    k = -cost / eps
    for _ in range(iters):
        f = eps * (la - torch.logsumexp(k + g.unsqueeze(0) / eps, dim=1))
        g = eps * (lb - torch.logsumexp(k + f.unsqueeze(1) / eps, dim=0))
    return torch.exp(k + f.unsqueeze(1) / eps + g.unsqueeze(0) / eps)


def gw_cost(C1: torch.Tensor, C2: torch.Tensor, Pi: torch.Tensor) -> torch.Tensor:
    """Square-loss GW 'tensor product' L(C1, C2) (x) Pi  ->  [n, m]  in O(n^2 m + n m^2)."""
    a, b = Pi.sum(1), Pi.sum(0)
    const = (C1 ** 2 @ a).unsqueeze(1) + (C2 ** 2 @ b).unsqueeze(0)
    return const - 2.0 * C1 @ Pi @ C2.t()


@torch.no_grad()
def solve_fgw(C1: torch.Tensor, C2: torch.Tensor, M: Optional[torch.Tensor] = None, alpha: float = 0.0,
              eps: float = 0.05, outer: int = 10, inner: int = 50, marginal: str = "semi",
              pin: Optional[torch.Tensor] = None, a: Optional[torch.Tensor] = None,
              b: Optional[torch.Tensor] = None, init: Optional[torch.Tensor] = None) -> torch.Tensor:
    """Entropic (fused) GW by mirror descent.  Returns the coupling Pi [n, m] (no gradient)."""
    with torch.autocast(device_type=C1.device.type, enabled=False):
        C1, C2 = C1.float(), C2.float()
        n, m = C1.shape[0], C2.shape[0]
        a = torch.full((n,), 1.0 / n, device=C1.device) if a is None else a.float()
        b = torch.full((m,), 1.0 / m, device=C1.device) if b is None else b.float()
        Pi = a.unsqueeze(1) * b.unsqueeze(0) if init is None else init.float()
        for _ in range(outer):
            cost = torch.zeros(n, m, device=C1.device)
            if alpha < 1.0:
                cost = cost + (1.0 - alpha) * gw_cost(C1, C2, Pi)
            if M is not None and alpha > 0.0:
                cost = cost + alpha * M.float()
            if pin is not None:
                cost = cost + pin.float()
            if marginal == "semi":
                Pi = a.unsqueeze(1) * torch.softmax(-cost / eps, dim=1)
            else:
                Pi = log_sinkhorn(cost, a, b, eps, inner)
        return Pi


def fgw_value(C1: torch.Tensor, C2: torch.Tensor, Pi: torch.Tensor, M: Optional[torch.Tensor], alpha: float) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Objective at a *fixed* coupling.  Differentiable w.r.t. C1 / M (envelope theorem: at the optimum the
    gradient through Pi vanishes, so Pi is treated as a constant).  Returns (total, gw term, w term)."""
    with torch.autocast(device_type=C1.device.type, enabled=False):
        C1, C2, Pi = C1.float(), C2.float(), Pi.float()
        gw = (gw_cost(C1, C2, Pi) * Pi).sum()
        w = (M.float() * Pi).sum() if M is not None else torch.zeros((), device=C1.device)
        return (1.0 - alpha) * gw + alpha * w, gw, w


def pairwise_cos_dist(x: torch.Tensor, y: Optional[torch.Tensor] = None) -> torch.Tensor:
    x = torch.nn.functional.normalize(x.float(), dim=-1)
    y = x if y is None else torch.nn.functional.normalize(y.float(), dim=-1)
    return (1.0 - x @ y.t()).clamp_min(0.0)


def scale_free(C: torch.Tensor) -> torch.Tensor:
    """Divide a cost / structure matrix by its mean, so that eps has the same meaning in every space."""
    return C / C.mean().clamp_min(1e-8)


def coupling_stats(Pi: torch.Tensor) -> Dict[str, float]:
    rows = Pi / Pi.sum(1, keepdim=True).clamp_min(1e-30)
    ent = -(rows * torch.log(rows.clamp_min(1e-30))).sum(1)
    return {"row_entropy": float(ent.mean() / math.log(Pi.shape[1])), "max_prob": float(rows.max(1).values.mean()),
            "used_cols": float((Pi.sum(0) > 0.5 / Pi.shape[1] / Pi.shape[0] * Pi.shape[0]).float().mean())}


# ======================================================================================================================
# 12. MODEL: grapheme-byte text codec
#     (geobridge/models/text_codec.py)
# ======================================================================================================================
# Grapheme-Byte Compressor (GBC): tokenizer-free text <-> fixed-size continuous latent.
#
# Encoder   UTF-8 bytes --(pooled per grapheme cluster)--> units --transformer--> resampler --> N x d latent
# Decoder   byte-level autoregressive transformer cross-attending to the (noise-perturbed) latent
#
# The encoder's sequence length is the number of *grapheme clusters*, so a Devanagari or Ethiopic sentence costs
# the same number of positions as a Latin sentence with the same number of written symbols, and the latent has
# the same size for every script.  There is no vocabulary to extend when a new script arrives.
# ======================================================================================================================

class TextCodec(nn.Module):
    def __init__(self, cfg, unit_vocab: int, unit_pad: int, unit_bytes: int, n_lang: int):
        super().__init__()
        self.cfg, W, d, N = cfg, cfg.text_width, cfg.latent_dim, cfg.n_tokens
        self.unit_pad = unit_pad
        self.unit_emb = nn.Embedding(unit_vocab, W, padding_idx=unit_pad)
        self.intra_pos = nn.Parameter(torch.zeros(unit_bytes, W))
        self.unit_proj = nn.Sequential(nn.Linear(2 * W, W), nn.GELU(), nn.Linear(W, W))
        p_enc = max(cfg.dropout, cfg.enc_dropout)
        self.enc = nn.ModuleList([Block(W, cfg.text_heads, dropout=p_enc) for _ in range(cfg.text_enc_layers)])
        self.resampler = Resampler(W, N, d, layers=2, heads=cfg.text_heads, dropout=p_enc)
        # decoder
        self.byte_emb = nn.Embedding(BYTE_VOCAB, W)
        self.lang_emb = nn.Embedding(n_lang, W)
        self.lat_in = nn.Linear(d, W)
        self.lat_pos = nn.Parameter(torch.randn(N, W) * 0.02)
        self.dec = nn.ModuleList([Block(W, cfg.text_heads, ctx_dim=W, dropout=cfg.dropout) for _ in range(cfg.text_dec_layers)])
        self.out_norm = nn.LayerNorm(W)
        self.out = nn.Linear(W, BYTE_VOCAB)

    # ------------------------------------------------------------------ encoder
    def embed_units(self, units: torch.Tensor) -> torch.Tensor:
        """units [B, L, M] -> unit embeddings [B, L, W] (mean + max pooling over the bytes of each unit)."""
        valid = (units != self.unit_pad).unsqueeze(-1)
        e = (self.unit_emb(units) + self.intra_pos[: units.shape[2]]) * valid
        mean = e.sum(2) / valid.sum(2).clamp_min(1)
        mx = e.masked_fill(~valid, -1e4).max(2).values
        mx = torch.where(valid.any(2), mx, torch.zeros_like(mx))
        return self.unit_proj(torch.cat([mean, mx], -1))

    def encode(self, units: torch.Tensor, unit_mask: torch.Tensor, return_units: bool = False):
        h0 = self.embed_units(units)
        h = h0 + sincos(torch.arange(h0.shape[1], device=h0.device), h0.shape[-1])
        for blk in self.enc:
            h = blk(h, mask=unit_mask)
        z = standardize_tokens(self.resampler(h, unit_mask))
        return (z, h0) if return_units else z

    def encode_from_embeddings(self, h0: torch.Tensor, unit_mask: torch.Tensor) -> torch.Tensor:
        """Same as :meth:`encode` but starting from unit embeddings (used for grapheme-level attribution)."""
        h = h0 + sincos(torch.arange(h0.shape[1], device=h0.device), h0.shape[-1])
        for blk in self.enc:
            h = blk(h, mask=unit_mask)
        return standardize_tokens(self.resampler(h, unit_mask))

    # ------------------------------------------------------------------ decoder
    def decoder(self) -> nn.Module:
        """The byte decoder as one module: a *view* on the same sub-modules and parameters (nothing is copied or
        renamed), so that the transport phase can train the decoder while the encoder stays frozen."""
        view = nn.Module()
        for name in ("byte_emb", "lang_emb", "lat_in", "dec", "out_norm", "out"):
            view.add_module(name, getattr(self, name))
        view.register_parameter("lat_pos", self.lat_pos)
        return view

    def decode_loss(self, z: torch.Tensor, tokens: torch.Tensor, lang: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Teacher-forced byte cross-entropy of ``tokens`` [B, T] (BOS ... EOS, PAD) given the latent z, and the
        byte accuracy."""
        logits = self._decode_states(z, tokens[:, :-1], lang)
        tgt = tokens[:, 1:]
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]).float(), tgt.reshape(-1), ignore_index=PAD)
        with torch.no_grad():
            m = tgt != PAD
            acc = ((logits.argmax(-1) == tgt) & m).sum().float() / m.sum().clamp_min(1)
        return loss, acc

    def _decode_states(self, z: torch.Tensor, tokens: torch.Tensor, lang: torch.Tensor) -> torch.Tensor:
        ctx = self.lat_in(z) + self.lat_pos
        x = self.byte_emb(tokens) + sincos(torch.arange(tokens.shape[1], device=z.device), ctx.shape[-1])
        x = x + self.lang_emb(lang).unsqueeze(1)
        for blk in self.dec:
            x = blk(x, ctx=ctx, causal=True)
        return self.out(self.out_norm(x))

    def loss(self, txt: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        z = self.encode(txt["units"], txt["unit_mask"])
        zin = z
        if self.training and self.cfg.latent_noise_max > 0:  # noise-robust latent: flows never hit z exactly
            sigma = torch.rand(z.shape[0], 1, 1, device=z.device) * self.cfg.latent_noise_max
            sigma = sigma * (torch.rand_like(sigma) < 0.7)
            zin = z + sigma * torch.randn_like(z)
        loss, acc = self.decode_loss(zin, txt["bytes"], txt["lang"])
        return {"loss": loss, "z": z, "byte_acc": acc}

    @torch.no_grad()
    def generate(self, z: torch.Tensor, lang: torch.Tensor, max_len: Optional[int] = None) -> List[str]:
        max_len = max_len or self.cfg.max_text_bytes + 2
        B = z.shape[0]
        tokens = torch.full((B, 1), BOS, dtype=torch.long, device=z.device)
        done = torch.zeros(B, dtype=torch.bool, device=z.device)
        for _ in range(max_len):
            nxt = self._decode_states(z, tokens, lang)[:, -1].argmax(-1)
            nxt = torch.where(done, torch.full_like(nxt, PAD), nxt)
            tokens = torch.cat([tokens, nxt.unsqueeze(1)], 1)
            done |= nxt == EOS
            if done.all():
                break
        return [bytes_to_text(t[1:].tolist()) for t in tokens]


# ======================================================================================================================
# 13. MODEL: tone-gated speech codec
#     (geobridge/models/audio_codec.py)
# ======================================================================================================================
# Tone-gated speech codec.
#
# Three factors of a spoken utterance are kept apart on purpose:
#
#   segmental content   deep SSL features                           -> always enters the latent
#   lexical tone        speaker-normalised F0 contour (z log-F0, delta) -> enters the latent through a learned
#                       per-language gate  gamma_l in [0, 1]  that is charged a sparsity price ("pay for pitch");
#                       voicing and energy are not gated (they are segmental cues in every language)
#   paralinguistics     pitch register + timbre (speaker vector)    -> never enters the latent; it travels in a
#                       bypass code that only the decoder / renderer sees, and is decorrelated from the latent
#
# The latent is trained to reconstruct the *segmental* stream only, so nothing in the reconstruction loss asks
# for pitch.  The gate therefore stays open only if pitch helps the objectives that depend on meaning (visual
# grounding, and through it the flows).  In a tonal language it does; in a non-tonal one the price wins and the
# gate closes.  The gate values are thus a learned, per-language estimate of how lexical pitch is.
#
# Schedule ("train dense, then sparsify"): for the first ``gate_release_frac`` of phase A all gates are held at 1
# and no price is charged, so the model first learns *whether* pitch is useful; then the gates are released and
# priced.  Releasing them closed, or pricing them from step 0, kills them before grounding has learned anything
# (a closed sigmoid gate has no gradient left to reopen).
#
# The F0 contour needed for synthesis is predicted by a prosody head that reads the latent with a
# stop-gradient: for a tonal language the latent contains tone (the gate is open) and the head can recover the
# lexical contour; for a non-tonal language it falls back to a neutral contour, which is the right behaviour.
# ======================================================================================================================

class MelRenderer(nn.Module):
    """(segmental features, tone contour, bypass code) at 25 Hz -> log-mel at the vocoder's frame rate.
    Frame-synchronous, so no duration model is needed; the waveform comes from the vocoder that owns the mel
    feature space (pretrained HiFi-GAN for ``mel_spec="speecht5"``, Griffin-Lim for the internal mel)."""

    def __init__(self, sem_dim: int, bypass_dim: int, width: int, n_mels: int):
        super().__init__()
        self.inp = nn.Linear(sem_dim + 3 + bypass_dim, width)
        self.convs = nn.ModuleList([nn.Conv1d(width, width, 5, padding=2) for _ in range(4)])
        self.out = nn.Linear(width, n_mels)

    def forward(self, sem: torch.Tensor, tone3: torch.Tensor, bypass: torch.Tensor, out_len: int) -> torch.Tensor:
        x = torch.cat([sem, tone3, bypass.unsqueeze(1).expand(-1, sem.shape[1], -1)], -1)
        h = F.interpolate(self.inp(x).transpose(1, 2), size=out_len, mode="linear", align_corners=False)
        for conv in self.convs:
            h = h + F.gelu(conv(h))
        return self.out(h.transpose(1, 2))


class AudioCodec(nn.Module):
    def __init__(self, cfg, sem_dim: int, spk_dim: int, n_lang: int, stats: Dict):
        super().__init__()
        self.cfg, W, d, N = cfg, cfg.audio_width, cfg.latent_dim, cfg.tokens_of("aud")
        self.n_tokens = N
        mode = {True: "all", False: "off"}.get(cfg.audio_anchor, cfg.audio_anchor)   # (a run saved when it was a switch)
        assert mode in ("off", "extra", "all"), cfg.audio_anchor
        self.mode = mode
        # tokens laid out in time: none, those beyond the n_tokens every latent has (they come last), or all
        self.n_ordered = {"off": 0, "extra": N - cfg.n_tokens, "all": N}[mode]
        self.n_content = N - self.n_ordered if mode == "extra" else N   # tokens of the codec the hub reads
        assert mode != "extra" or self.n_ordered > 0, "audio_anchor = extra needs audio_private_slots > private_slots"
        self.sem_dim = sem_dim
        t = lambda k: torch.as_tensor(stats[k], dtype=torch.float32)
        for k in ("sem_mean", "sem_std", "spk_mean", "spk_std", "f0_mean", "f0_std"):
            self.register_buffer(k, t(k))
        self.register_buffer("mel_mean", torch.tensor(float(stats.get("mel_mean", 0.0))))
        self.register_buffer("mel_std", torch.tensor(float(stats.get("mel_std", 1.0))))
        # ---- encoder
        self.sem_in = nn.Linear(sem_dim, W)
        # The gated path sees ONLY the pitch contour (z-scored log-F0 and its delta).  Voicing and energy are
        # useful in every language (they mark syllable structure), so they enter ungated; otherwise the gate
        # would stay open everywhere and say nothing about lexical tone.  All maps of the gated path are
        # bias-free, so a flat contour (zeros) is exactly equivalent to a closed gate.
        self.aux_in = nn.Linear(2, W)
        self.tone_net = nn.Sequential(nn.Conv1d(2, W, 5, padding=2, bias=False), nn.GELU(),
                                      nn.Conv1d(W, W, 5, padding=2, bias=False))
        self.film_scale = nn.Linear(W, W, bias=False)
        self.film_shift = nn.Linear(W, W, bias=False)
        self.gate_logit = nn.Parameter(torch.full((n_lang,), float(cfg.tone_gate_init)))
        self.gate_forced_open = False  # set by the trainer during the first part of phase A (not a parameter)
        self.lang_emb = nn.Embedding(n_lang, W)
        p_enc = max(cfg.dropout, cfg.enc_dropout)
        self.enc = nn.ModuleList([Block(W, cfg.audio_heads, dropout=p_enc) for _ in range(cfg.audio_enc_layers)])
        self.resampler = Resampler(W, self.n_content, d, layers=2, heads=cfg.audio_heads, dropout=cfg.enc_dropout)
        # ---- paralinguistic bypass
        self.bypass = MLP(spk_dim + 2, 2 * cfg.bypass_dim, cfg.bypass_dim)
        self.byp_in = nn.Linear(cfg.bypass_dim, W)
        self.byp_gate = nn.Parameter(torch.zeros(1))
        # ---- decoder (latent -> frame-level segmental features) and heads
        self.lat_in = nn.Linear(d, W)
        self.lat_pos = nn.Parameter(torch.randn(self.n_content, W) * 0.02)
        self.q_proj = nn.Linear(2 * W, W)
        self.dec = nn.ModuleList([Block(W, cfg.audio_heads, ctx_dim=W, dropout=cfg.dropout) for _ in range(cfg.audio_dec_layers)])
        self.dec_norm = nn.LayerNorm(W)
        self.sem_out = nn.Linear(W, sem_dim)
        self.dur_head = MLP(d, W, 1)
        self.pros_dec = nn.ModuleList([Block(W, cfg.audio_heads, ctx_dim=W) for _ in range(1)])
        self.pros_out = nn.Linear(W, 3)  # z-scored log-F0, voicing logit, z-scored log-energy
        self.renderer = MelRenderer(sem_dim, cfg.bypass_dim, cfg.mel_width, cfg.n_mels)
        # ---- the speaking path (audio_anchor = extra): time-ordered tokens with an encoder and a decoder of their
        #      own.  It reads the recording's streams itself (the pitch contour too, ungated: the hub never sees
        #      these tokens) and the content tokens with a stop-gradient, so everything above trains exactly as it
        #      would without it: what the hub reads cannot be changed by what is added for speaking.
        if mode == "extra":
            self.speak_feat = nn.Linear(sem_dim + 4, W)
            self.speak_lang = nn.Embedding(n_lang, W)
            self.speak_enc = nn.ModuleList([Block(W, cfg.audio_heads, dropout=cfg.dropout) for _ in range(2)])
            self.time_resampler = Resampler(W, self.n_ordered, d, layers=2, heads=cfg.audio_heads)
            self.speak_in = nn.Linear(d, W)
            self.speak_pos = nn.Parameter(torch.randn(N, W) * 0.02)
            self.speak_q = nn.Linear(2 * W, W)
            self.speak_byp = nn.Linear(cfg.bypass_dim, W)
            self.speak_dec = nn.ModuleList([Block(W, cfg.audio_heads, ctx_dim=W, dropout=cfg.dropout) for _ in range(cfg.audio_dec_layers)])
            self.speak_norm = nn.LayerNorm(W)
            self.speak_out = nn.Linear(W, sem_dim)
            self.speak_pros_dec = Block(W, cfg.audio_heads, ctx_dim=W)
            self.speak_pros = nn.Linear(W, 3)

    # ------------------------------------------------------------------ gate
    def gate(self, lang: Optional[torch.Tensor] = None) -> torch.Tensor:
        mode = self.cfg.tone_mode
        if mode == "off":
            g = torch.zeros_like(self.gate_logit)
        elif mode == "on" or self.gate_forced_open:
            g = torch.ones_like(self.gate_logit)
        else:
            g = torch.sigmoid(self.gate_logit)
        return g if lang is None else g[lang]

    def norm_sem(self, sem: torch.Tensor) -> torch.Tensor:
        return (sem - self.sem_mean) / self.sem_std

    def bypass_code(self, spk: torch.Tensor, f0: torch.Tensor) -> torch.Tensor:
        if not self.cfg.use_bypass:
            return torch.zeros(spk.shape[0], self.cfg.bypass_dim, device=spk.device, dtype=spk.dtype)
        x = torch.cat([(spk - self.spk_mean) / self.spk_std, (f0 - self.f0_mean) / self.f0_std], -1)
        return self.bypass(x)

    # ------------------------------------------------------------------ encoder
    def time_mask(self, mask: torch.Tensor, spans: int = 2) -> torch.Tensor:
        """SpecAugment-style time masking of the segmental stream: per utterance, ``spans`` random spans of up
        to ``speech_mask`` of its length are hidden from the encoder (the codec still has to reconstruct them).
        mask [B, T] valid frames -> keep [B, T] in {0, 1}."""
        B, T = mask.shape
        n = mask.sum(1, keepdim=True).float()
        pos = torch.arange(T, device=mask.device).unsqueeze(0)
        keep = torch.ones(B, T, device=mask.device)
        for _ in range(spans):
            width = torch.rand(B, 1, device=mask.device) * self.cfg.speech_mask * n
            start = torch.rand(B, 1, device=mask.device) * (n - width).clamp_min(0)
            keep = keep * ((pos < start) | (pos >= start + width)).float()
        return keep

    def encode(self, sem: torch.Tensor, tone: torch.Tensor, mask: torch.Tensor, lang: torch.Tensor,
               sem_is_normed: bool = False, gate_override: Optional[float] = None,
               keep: Optional[torch.Tensor] = None) -> torch.Tensor:
        """``keep`` [B, T] in {0, 1}: frames of the segmental stream the encoder may see (the others are replaced by
        the feature mean); used to encode time-masked versions of an utterance."""
        x = sem if sem_is_normed else self.norm_sem(sem)
        if keep is not None:
            x = x * keep.unsqueeze(-1).to(x.dtype)
        elif self.training and self.cfg.speech_mask > 0:
            x = x * self.time_mask(mask).unsqueeze(-1).to(x.dtype)   # masked frames = the feature mean
        h = self.sem_in(x) + self.aux_in(tone[..., [1, 3]])
        g = self.gate(lang) if gate_override is None else torch.full((x.shape[0],), gate_override, device=x.device)
        if self.cfg.tone_mode != "off" or gate_override is not None:
            tau = self.tone_net(tone[..., [0, 2]].transpose(1, 2)).transpose(1, 2)
            g = g.view(-1, 1, 1).to(h.dtype)
            # both FiLM terms are bounded by tanh, so gamma is an upper bound on what pitch can contribute;
            # with an unbounded shift the network could cancel a shrinking gate by growing its weights
            h = h * (1 + g * torch.tanh(self.film_scale(tau))) + g * torch.tanh(self.film_shift(tau))
        h = h + sincos(torch.arange(h.shape[1], device=h.device), h.shape[-1]) + self.lang_emb(lang).unsqueeze(1)
        for blk in self.enc:
            h = blk(h, mask=mask)
        if self.mode == "all":
            return standardize_tokens(self.resampler(h, mask, self.span_means(h, mask, self.n_tokens)))
        z = standardize_tokens(self.resampler(h, mask))                       # content tokens: free queries
        if self.mode == "extra":                                               # + how it is said, laid out in time
            e = self.speak_feat(torch.cat([x, tone.to(x.dtype)], -1))
            e = e + sincos(torch.arange(e.shape[1], device=e.device), e.shape[-1]) + self.speak_lang(lang).unsqueeze(1)
            for blk in self.speak_enc:
                e = blk(e, mask=mask)
            z = torch.cat([z, standardize_tokens(self.time_resampler(e, mask, self.span_means(e, mask, self.n_ordered)))], 1)
        return z

    # ------------------------------------------------------------------ tokens that keep the order of time
    # Speech, unlike a caption's bytes, is reconstructed by regression, and the regression has a comfortable
    # resting place: every latent token the utterance mean, every frame decoded to that mean.  On real speech
    # that is where a set of free queries stays (measured on Flickr8k: reconstructions constant in time, written
    # speech a hum).  A token laid out in time starts from the mean of its own span of the utterance, and each
    # frame of the decoder starts from the ordered tokens around its own position, so the order of time is in
    # the latent by construction and attention only has to add detail.
    #
    # Which tokens?  Laying out every token ("all") gives a codec that reconstructs and a hub that can no longer
    # find what the utterance is about (SynthWorld: island speech -> text 5 % against 77 %): free queries had
    # been putting each thing that is said into the same token wherever it is said.  Adding ordered tokens to
    # the free ones and training them together loses the same thing more slowly, because the ordered tokens then
    # do the reconstruction and nothing puts the content into the free ones.  So ("extra") the free content
    # tokens and their decoder are left exactly as they are - they are all the hub reads, grounds and explains -
    # and a SPEAKING PATH is added beside them: time-ordered tokens from a small encoder of their own (the content
    # encoder, trained for the gist, keeps too little of the frames: measured), and a second decoder that reads
    # content and ordered tokens with a stop-gradient on the content.  What is said is one objective, how it is
    # said is another, and the second cannot disturb the first.
    @staticmethod
    def span_means(h: torch.Tensor, mask: torch.Tensor, n: int) -> torch.Tensor:
        """Masked mean of the frames over n equal spans of each utterance's own length: [B, T, W] -> [B, n, W]."""
        T = h.shape[1]
        length = mask.sum(1, keepdim=True).clamp_min(1)                                    # [B, 1]
        span = (torch.arange(T, device=h.device).unsqueeze(0) * n // length).clamp_max(n - 1)   # [B, T]
        w = F.one_hot(span, n).to(h.dtype) * mask.unsqueeze(-1).to(h.dtype)                 # [B, T, n]
        return torch.einsum("btn,btw->bnw", w, h) / w.sum(1).clamp_min(1).unsqueeze(-1)

    @staticmethod
    def span_upsample(ctx: torch.Tensor, n_frames: torch.Tensor, T: int) -> torch.Tensor:
        """The inverse layout: every frame gets the linear interpolation of the tokens around its own position.
        [B, n, W] -> [B, T, W]"""
        n, W = ctx.shape[1], ctx.shape[2]
        pos = torch.arange(T, device=ctx.device).unsqueeze(0).float()
        u = ((pos + 0.5) / n_frames.clamp_min(1).unsqueeze(1).float() * n - 0.5).clamp(0, n - 1)   # [B, T]
        lo = u.floor().long()
        hi = (lo + 1).clamp_max(n - 1)
        a = (u - lo.float()).unsqueeze(-1).to(ctx.dtype)
        take = lambda i: ctx.gather(1, i.unsqueeze(-1).expand(-1, -1, W))
        return (1 - a) * take(lo) + a * take(hi)

    # ------------------------------------------------------------------ decoder
    def _frame_queries(self, n_frames: torch.Tensor, T: int, W: int, device) -> Tuple[torch.Tensor, torch.Tensor]:
        pos = torch.arange(T, device=device).unsqueeze(0)
        mask = pos < n_frames.unsqueeze(1)
        rel = pos.float() / n_frames.clamp_min(1).unsqueeze(1).float() * 1000.0
        q = self.q_proj(torch.cat([sincos(pos.expand(len(n_frames), -1), W), sincos(rel, W)], -1))
        return q, mask

    def decode(self, z: torch.Tensor, lang: torch.Tensor, n_frames: torch.Tensor, bypass: torch.Tensor,
               T: Optional[int] = None, content_only: bool = False) -> Dict[str, torch.Tensor]:
        """latent -> normalised segmental features [B, T, D_s] and prosody [B, T, 3].
        With a speaking path (audio_anchor = extra) this is the speaking decoder, which reads all tokens;
        ``content_only`` gives the content decoder instead, which reads the content tokens alone."""
        T = T or int(n_frames.max())
        if self.mode == "extra" and not content_only:
            return self._speak(z, n_frames, bypass, T)
        z = z[:, : self.n_content]
        W = self.lat_in.out_features
        byp = torch.tanh(self.byp_gate) * self.byp_in(bypass).unsqueeze(1)
        q, mask = self._frame_queries(n_frames, T, W, z.device)
        ctx = self.lat_in(z) + self.lat_pos
        h = q + byp
        up = self.span_upsample(ctx, n_frames, T) if self.mode == "all" else None
        if up is not None:
            h = h + up
        for blk in self.dec:
            h = blk(h, mask=mask, ctx=ctx)
        sem = self.sem_out(self.dec_norm(h))
        # prosody head: stop-gradient on the latent (it must not force pitch into the latent)
        ctx_p = self.lat_in(z.detach()).detach() + self.lat_pos.detach()
        hp = q.detach() + self.byp_in(bypass).unsqueeze(1)
        if up is not None:
            hp = hp + up.detach()
        for blk in self.pros_dec:
            hp = blk(hp, mask=mask, ctx=ctx_p)
        return {"sem": sem, "pros": self.pros_out(hp), "mask": mask}

    def _speak(self, z: torch.Tensor, n_frames: torch.Tensor, bypass: torch.Tensor, T: int) -> Dict[str, torch.Tensor]:
        """The speaking decoder: every frame starts from the time-ordered tokens around it and attends to all
        tokens.  The content tokens and the bypass code come in with a stop-gradient."""
        W = self.speak_in.out_features
        z = torch.cat([z[:, : self.n_content].detach(), z[:, self.n_content:]], 1)
        pos = torch.arange(T, device=z.device).unsqueeze(0)
        mask = pos < n_frames.unsqueeze(1)
        rel = pos.float() / n_frames.clamp_min(1).unsqueeze(1).float() * 1000.0
        q = self.speak_q(torch.cat([sincos(pos.expand(len(n_frames), -1), W), sincos(rel, W)], -1))
        ctx = self.speak_in(z) + self.speak_pos
        h = q + self.speak_byp(bypass.detach()).unsqueeze(1) + self.span_upsample(ctx[:, self.n_content:], n_frames, T)
        for blk in self.speak_dec:
            h = blk(h, mask=mask, ctx=ctx)
        h = self.speak_norm(h)
        # prosody: a head of its own on a stop-gradient.  Trained on the decoder's own states, the three prosody
        # objectives slow the reconstruction of the segmental stream down (real speech, 300 steps: 5 % of the
        # frame variance beyond the mean with them, 14 % without)
        hp = self.speak_pros_dec(h.detach(), mask=mask, ctx=ctx.detach())
        return {"sem": self.speak_out(h), "pros": self.speak_pros(hp), "mask": mask}

    def predict_frames(self, z: torch.Tensor) -> torch.Tensor:
        n = torch.exp(self.dur_head(z.mean(1)).squeeze(-1) + math.log(50.0))
        return n.round().clamp(4, int(self.cfg.max_audio_sec * 25)).long()

    # ------------------------------------------------------------------ training loss
    def loss(self, aud: Dict[str, torch.Tensor], n_grounded: Optional[int] = None) -> Dict[str, torch.Tensor]:
        """``n_grounded``: the first n utterances of the batch are image-paired (the rest is unpaired speech);
        used by the evidence-proportional gate price."""
        cfg = self.cfg
        sem, tone, mask, lang = self.norm_sem(aud["sem"]), aud["tone"], aud["mask"], aud["lang"]
        z = self.encode(sem, tone, mask, lang, sem_is_normed=True)
        zin = z
        if self.training and cfg.latent_noise_max > 0:
            sigma = torch.rand(z.shape[0], 1, 1, device=z.device) * cfg.latent_noise_max
            sigma = sigma * (torch.rand_like(sigma) < 0.7)
            zin = z + sigma * torch.randn_like(z)
        n_frames = mask.sum(1)
        p = self.bypass_code(aud["spk"], aud["f0"])
        m = mask.unsqueeze(-1).float()
        denom = m.sum().clamp_min(1)
        voiced = tone[..., 1:2]

        def stream_losses(out: Dict[str, torch.Tensor]) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
            """Segmental MSE, contour L1 (voiced frames), voicing BCE, energy L1 of decoded streams."""
            sem_ = (((out["sem"] - sem) ** 2) * m).sum() / denom / sem.shape[-1]
            f0_ = ((out["pros"][..., 0:1] - tone[..., 0:1]).abs() * voiced * m).sum() / (voiced * m).sum().clamp_min(1)
            voi_ = (F.binary_cross_entropy_with_logits(out["pros"][..., 1:2].float(), voiced, reduction="none") * m).sum() / denom
            en_ = ((out["pros"][..., 2:3] - tone[..., 3:4]).abs() * m).sum() / denom
            return sem_, f0_, voi_, en_

        l_sem, l_f0, l_voi, l_en = stream_losses(self.decode(zin, lang, n_frames, p, T=sem.shape[1], content_only=True))
        l_dur = (self.dur_head(z.detach().mean(1)).squeeze(-1) - (torch.log(n_frames.float()) - math.log(50.0))).abs().mean()
        res = {"z": z, "sem_mse": l_sem, "f0_l1": l_f0, "dur_l1": l_dur,
               "loss": l_sem + 0.5 * (l_f0 + l_voi + l_en) + 0.5 * l_dur}
        if self.mode == "extra":   # the speaking path: its own objective, no gradient into anything above
            s_sem, s_f0, s_voi, s_en = stream_losses(self._speak(zin, n_frames, p, sem.shape[1]))
            res["speak_mse"] = s_sem
            res["loss"] = res["loss"] + s_sem + 0.5 * (s_f0 + s_voi + s_en)
        # mel renderer (teacher-forced on real streams; independent of the latent)
        if "mel" in aud and cfg.w_mel > 0:
            tone3 = torch.stack([tone[..., 0], tone[..., 1], tone[..., 3]], -1)
            mel_hat = self.renderer(sem, tone3, p.detach(), aud["mel"].shape[1])
            mel = (aud["mel"] - self.mel_mean) / self.mel_std
            mm = (torch.arange(mel.shape[1], device=mel.device).unsqueeze(0) < aud["mel_len"].unsqueeze(1)).unsqueeze(-1).float()
            res["mel_l1"] = ((mel_hat - mel).abs() * mm).sum() / mm.sum().clamp_min(1) / mel.shape[-1]
            res["loss"] = res["loss"] + cfg.w_mel * res["mel_l1"]
        # decorrelate what goes to the hub from what bypasses it
        if cfg.use_bypass and cfg.w_orth > 0 and z.shape[0] > 3:
            res["orth"] = cross_correlation(z[:, : self.n_content].mean(1), p)
            res["loss"] = res["loss"] + cfg.w_orth * res["orth"]
        # pay-for-pitch.  "uniform": every language present in the batch pays for its gate.  "evidence": every
        # *grounded* utterance pays for the gate of its language, so price and benefit (which only grounded
        # utterances can provide) scale with the same count - otherwise a language with few image pairs is
        # priced out whatever its pitch is worth (measured on Flickr8k: the Yoruba gate closed to 0)
        if cfg.tone_mode == "gated" and not self.gate_forced_open:
            if cfg.gate_price == "evidence":
                n = lang.shape[0] if n_grounded is None else n_grounded
                res["gate_pen"] = self.gate(lang[:n]).mean() if n > 0 else z.new_zeros(())
            else:
                res["gate_pen"] = self.gate()[lang.unique()].mean()
            res["loss"] = res["loss"] + cfg.w_gate * res["gate_pen"]
        return res

    # ------------------------------------------------------------------ synthesis
    @torch.no_grad()
    def synthesize_mel(self, z: torch.Tensor, lang: torch.Tensor, spk: torch.Tensor, f0: torch.Tensor,
                       n_frames: Optional[torch.Tensor] = None, content_only: bool = False) -> Tuple[torch.Tensor, torch.Tensor, Dict]:
        """latent (+ a reference speaker for the bypass) -> log-mel [B, T_mel, n_mels], frame counts, streams.
        ``content_only``: rendered by the content decoder (what the model says without its speaking path)."""
        n_frames = n_frames if n_frames is not None else self.predict_frames(z)
        p = self.bypass_code(spk, f0)
        out = self.decode(z, lang, n_frames, p, content_only=content_only)
        pros = out["pros"]
        voiced = (torch.sigmoid(pros[..., 1]) > 0.5).float()
        tone3 = torch.stack([pros[..., 0] * voiced, voiced, pros[..., 2]], -1)
        ratio = self.cfg.mel_fps / 25.0
        mel = self.renderer(out["sem"], tone3, p, int(round(out["sem"].shape[1] * ratio))) * self.mel_std + self.mel_mean
        return mel, n_frames, {"sem": out["sem"], "tone3": tone3, "mel_len": torch.round(n_frames.float() * ratio).long()}


    @torch.no_grad()
    def render_real(self, aud: Dict[str, torch.Tensor]) -> torch.Tensor:
        """Log-mel rendered from the REAL streams of an utterance (segmental features, contour, bypass code): what
        the renderer does on its own, with no latent in the way.  -> [B, T_mel, n_mels]"""
        tone = aud["tone"]
        tone3 = torch.stack([tone[..., 0], tone[..., 1], tone[..., 3]], -1)
        n_mel = int(round(aud["sem"].shape[1] * self.cfg.mel_fps / 25.0))
        mel = self.renderer(self.norm_sem(aud["sem"]), tone3, self.bypass_code(aud["spk"], aud["f0"]), n_mel)
        return mel * self.mel_std + self.mel_mean


def cross_correlation(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Mean squared entry of the batch cross-correlation matrix between two sets of features (the
    orthogonality penalty L_orth: zero iff no linear dependence between hub-bound and bypass codes)."""
    a = (a.float() - a.float().mean(0)) / (a.float().std(0) + 1e-4)
    b = (b.float() - b.float().mean(0)) / (b.float().std(0) + 1e-4)
    c = a.t() @ b / a.shape[0]
    return (c ** 2).mean()


# ======================================================================================================================
# 14. MODEL: barycentric visual interlingua (hub), explainer, reader
#     (geobridge/models/hub.py)
# ======================================================================================================================
# The hub: a Barycentric Visual Interlingua (BVI), plus the anchors it is ablated against.
#
# An utterance (text or speech, any language) is turned into K_h *slot queries*.  The slots attend, one after
# the other, over the frozen patch tokens of the image the utterance describes; the hub token of a slot is the
# attention-weighted **barycentre** of those frozen tokens:
#
#         z*_k = sum_p  alpha_kp * v_p ,     alpha_k = softmax_p( <q_k, key(v_p)> / tau  -  inhibition_k )
#
# (each slot is inhibited on what earlier slots took and may choose "null").  A hub code is, by construction,
# a point of conv(V_image + origin)^{K_h}: it lives in the frozen vision geometry, contains no text, and is
# sentence-level.  It can be drawn on the image, which makes the interlingua directly inspectable.
#
# Mutual sufficiency.  The grounder is trained so that the hub code is the information the sentence and the
# image have in common:
#
#     L_ground  contrastive, under the same sequential attention - the code is found in the right image
#     L_expl    Explainer(z*) must reconstruct the utterance latent - the code says everything the sentence says
#     L_min     an image-free Reader(utterance) must predict z*     - the code says nothing the sentence does not
#     L_agree   text and speech of one caption (same language) get the same code
#
# L_min is what removes slots that look at objects the sentence never mentions: such a token cannot be
# predicted from the sentence, a mentioned object or a null slot can.  The Reader is not thrown away: it is the
# test-time read path (utterance -> hub in one pass, no image, no ODE), and the Explainer is the writer's draft.
# No loss ever pairs two languages.
#
# Anchors available for the ablation (``cfg.hub_mode``):
#     bary    (ours) utterance-conditioned barycentres of frozen patch tokens
#     global  scene-level visual pivot: a fixed pooling of the image, identical for all its captions
#     free    unconstrained learned anchor in the spirit of FlowBind: hub = f(modality latent), no vision geometry
#     text    text anchor: the first K_h tokens of a hub-language text latent (image-pivoted for the island)
# ======================================================================================================================

class VisualInterlingua(nn.Module):
    MODS = {"txt": 0, "aud": 1}

    def __init__(self, cfg, vision_dim: int, n_vis_tokens: int, anchor: Dict[str, torch.Tensor], n_lang: int):
        super().__init__()
        self.cfg, W, d = cfg, cfg.hub_width, cfg.latent_dim
        self.K, self.P = cfg.hub_slots, n_vis_tokens  # P counts the global token (last)
        self.register_buffer("a_mean", anchor["mean"].clone())
        self.register_buffer("a_basis", anchor["basis"].clone())
        self.register_buffer("a_scale", anchor["scale"].clone())
        self.lat_in = nn.Linear(d, W)
        self.lat_pos = nn.Parameter(torch.randn(cfg.max_tokens, W) * 0.02)   # (latents can differ in length)
        self.visible = {m: cfg.hub_tokens_of(m) for m in self.MODS}   # tokens of each latent that the hub reads
        self.mod_emb = nn.Embedding(2, W)
        self.lang_emb = nn.Embedding(n_lang, W)
        self.slots = nn.Parameter(torch.randn(self.K, W))  # unit scale: slots must start out *different*
        self.blocks = nn.ModuleList([Block(W, cfg.hub_heads, ctx_dim=W, dropout=cfg.dropout) for _ in range(cfg.hub_layers)])
        self.norm = nn.LayerNorm(W)
        self.q_out = nn.Linear(W, W)
        self.key = nn.Linear(d, W)
        self.null_key = nn.Parameter(torch.randn(W) * 0.1)
        self.null_bias = nn.Parameter(torch.zeros(()))
        self.null_enabled = True   # the trainer switches it off for the first part of phase A (not a parameter)
        self.temp_scale = 1.0      # > 1 early in phase A: soft attention, annealed to the sharp final value
        self.explainer = Explainer(cfg, n_lang)   # hub -> utterance latent  (sufficiency; later the writer's draft)
        self.reader = HubReader(cfg, n_lang)      # utterance latent -> hub  (minimality; later the read path)
        if cfg.hub_mode == "free":                # the unconstrained anchor exists only where it is used
            self.free_out = nn.Linear(W, d)
            self.free_img = nn.Linear(d, d)
        side = int(round((self.P - 1) ** 0.5))
        self.grid = (side, side) if side * side == self.P - 1 else (1, self.P - 1)

    # ------------------------------------------------------------------ frozen anchor geometry
    def project(self, img: torch.Tensor) -> torch.Tensor:
        """Raw vision tokens [B, P, D_v] -> anchored tokens [B, P, d] (frozen PCA isometry + one global scale)."""
        return ((img.float() - self.a_mean) @ self.a_basis) * self.a_scale

    def image_latent(self, V: torch.Tensor, n_tokens: int) -> torch.Tensor:
        """Image-side modality latent: the anchored patch grid pooled to n_tokens tokens."""
        B, _, d = V.shape
        r, c = grid_factor(n_tokens)
        g = V[:, :-1].transpose(1, 2).reshape(B, d, *self.grid)
        return F.adaptive_avg_pool2d(g, (r, c)).flatten(2).transpose(1, 2)

    def scene_code(self, V: torch.Tensor) -> torch.Tensor:
        """Scene-level pivot (hub_mode == "global"): global token + pooled regions, same for every caption."""
        if self.K == 1:
            return V[:, -1:]
        return torch.cat([V[:, -1:], self.image_latent(V, self.K - 1)], 1)

    # ------------------------------------------------------------------ utterance -> slot queries
    def slot_features(self, z: torch.Tensor, mod: str, lang: torch.Tensor) -> torch.Tensor:
        z = z[:, : self.visible[mod]]
        ctx = self.lat_in(z) + self.lat_pos[: z.shape[1]] + self.mod_emb.weight[self.MODS[mod]] + self.lang_emb(lang).unsqueeze(1)
        h = self.slots.unsqueeze(0).expand(z.shape[0], -1, -1)
        for blk in self.blocks:
            h = blk(h, ctx=ctx)
        return self.norm(h)

    def queries(self, z: torch.Tensor, mod: str, lang: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.q_out(self.slot_features(z, mod, lang)), dim=-1)

    def keys(self, V: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.key(V), dim=-1)

    @property
    def tau(self) -> float:
        return self.cfg.hub_temp * self.temp_scale

    def _sequential(self, logits: torch.Tensor, null: torch.Tensor) -> torch.Tensor:
        """Sequential attention with inhibition of return.  logits [..., K, P], null [..., K] ->
        log-probabilities [..., K, P + 1]; the last column is the null option, so every row sums to one."""
        taken = torch.zeros_like(logits[..., 0, :])
        rows = []
        for k in range(logits.shape[-2]):
            la = torch.log_softmax(torch.cat([logits[..., k, :] - self.cfg.hub_inhibit / self.tau * taken,
                                              null[..., k:k + 1]], dim=-1), dim=-1)
            rows.append(la)
            taken = taken + la[..., :-1].exp()
        return torch.stack(rows, -2)

    def _null_logits(self, q: torch.Tensor) -> torch.Tensor:
        if not self.null_enabled:  # first part of phase A: every slot has to look at something
            return torch.full(q.shape[:-1], -1e4, device=q.device, dtype=torch.float32)
        return (q.float() @ F.normalize(self.null_key, dim=-1).float()) / self.tau + self.null_bias

    def ground(self, q: torch.Tensor, keys: torch.Tensor, item: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """One pass for everything the grounder needs.

        q [n, K, W] slot queries, keys [B, P, W] patch keys of every image of the batch, item [n] the image of
        each utterance.  Returns
          score [n, B]         how well every image supports the slot sequence (contrastive logits),
          attn  [n, K, P-1]    the attention on the utterance's own image (its hub code is attn @ patches),
          logp  [n, K, P]      its logarithm, with the null option as last column (never clamped, so a loss on
                               it reaches a patch that currently has no attention at all),
        all read off the same sequential, inhibited attention.  Three properties were each needed in practice:
        * patches only - the global token is the image mean; a slot that settles on it makes the hub scene-level;
        * sequential with inhibition of return, and a score computed under that same attention - with the usual
          "best patch per slot, independently" score every slot is fully credited for re-finding one object;
        * a null option (hub token = origin), offered only after the slots have learned their roles, and soft
          attention annealed to sharp.
        """
        cos = torch.einsum("nkw,bpw->nbkp", q, keys[:, :-1]).float()
        null = self._null_logits(q).unsqueeze(1).expand(-1, keys.shape[0], -1)
        logp = self._sequential(cos / self.tau, null)
        attn = logp[..., :-1].exp()
        score = (attn * cos).sum(-1).mean(-1)
        rows = torch.arange(q.shape[0], device=q.device)
        return score, attn[rows, item], logp[rows, item]

    def attend(self, q: torch.Tensor, V: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Own-image path only (cache building, figures): q [n, K, W], V [n, P, d] -> hub [n, K, d], attn."""
        logits = torch.einsum("nkw,npw->nkp", q, self.keys(V)[:, :-1]).float() / self.tau
        attn = self._sequential(logits, self._null_logits(q))[..., :-1].exp()
        return torch.einsum("nkp,npd->nkd", attn.to(V.dtype), V[:, :-1]), attn

    def free_code(self, z: torch.Tensor, mod: str, lang: torch.Tensor, scene: torch.Tensor) -> torch.Tensor:
        """Unconstrained learned anchor (baseline): average of an utterance code and an image code."""
        return 0.5 * (self.free_out(self.slot_features(z, mod, lang)) + self.free_img(scene))

    # ------------------------------------------------------------------ retrieval in hub space
    @staticmethod
    def hub_to_image_scores(hub: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
        """How well each hub code [n, K, d] is explained by the patches of each gallery image [G, P, d].
        Grounder-free: used to score hub estimates that came out of a backward ODE.  -> [n, G]"""
        h, v = F.normalize(hub.float(), dim=-1), F.normalize(V[:, :-1].float(), dim=-1)
        best = torch.einsum("nkd,gpd->ngkp", h, v).amax(-1)                 # [n, G, K]
        w = hub.float().norm(dim=-1)                                        # empty (null) slots weigh ~0
        return (best * w.unsqueeze(1)).sum(-1) / w.sum(-1, keepdim=True).clamp_min(1e-6)


class _SlotTransformer(nn.Module):
    """n_out learned queries read a set of n_in tokens (shared shape of the Explainer and the Reader)."""

    def __init__(self, cfg, n_lang: int, n_in: int, n_out: int, layers: int):
        super().__init__()
        W, d = cfg.hub_width, cfg.latent_dim
        self.inp = nn.Linear(d, W)
        self.in_pos = nn.Parameter(torch.randn(n_in, W) * 0.02)
        self.mod_emb = nn.Embedding(2, W)
        self.lang_emb = nn.Embedding(n_lang, W)
        self.queries = nn.Parameter(torch.randn(n_out, W) * 0.02)
        self.blocks = nn.ModuleList([Block(W, cfg.hub_heads, ctx_dim=W, dropout=cfg.dropout) for _ in range(layers)])
        self.norm = nn.LayerNorm(W)
        self.out = nn.Linear(W, d)

    def forward(self, x: torch.Tensor, mod: int, lang: torch.Tensor, n_out: Optional[int] = None) -> torch.Tensor:
        """``n_out``: how many of the output queries to use (the latents of the modalities can differ in length)."""
        ctx = self.inp(x) + self.in_pos[: x.shape[1]] + self.mod_emb.weight[mod] + self.lang_emb(lang).unsqueeze(1)
        q = self.queries[: n_out].unsqueeze(0).expand(x.shape[0], -1, -1)
        for blk in self.blocks:
            q = blk(q, ctx=ctx)
        return self.out(self.norm(q))


class Explainer(_SlotTransformer):
    """hub code [n, K_h, d] -> expected utterance latent [n, N, d].

    Phase A: sufficiency - the hub code alone must reconstruct the utterance (the t = 0 view of flow matching:
    what the hub does not determine is unexplained variance of the latent).  Afterwards it is the writer's
    *draft*: generation may start from the interpolation between the source and this prediction."""

    def __init__(self, cfg, n_lang: int):
        super().__init__(cfg, n_lang, cfg.hub_slots, cfg.max_tokens, 2)
        # (it explains the tokens the hub reads; what a decoder alone reads is not the hub's to explain)
        self.n_out = {m: cfg.hub_tokens_of(name) for name, m in VisualInterlingua.MODS.items()}

    def forward(self, code: torch.Tensor, mod: int, lang: torch.Tensor) -> torch.Tensor:  # type: ignore[override]
        return super().forward(code, mod, lang, self.n_out[mod])


class HubReader(_SlotTransformer):
    """utterance latent [n, N, d] -> hub code [n, K_h, d], with no image.

    Phase A: minimality - the grounded hub code must be predictable from the utterance alone.  Phase B and test
    time: the read path.  Reading is a function (one code per sentence), so it is amortised in one pass;
    writing is a distribution (many realisations per code), so it is a flow.  The reader regresses onto codes
    that do not depend on the speaker, so the hub it returns carries no speaker identity."""

    def __init__(self, cfg, n_lang: int):
        super().__init__(cfg, n_lang, cfg.max_tokens, cfg.hub_slots, cfg.hub_layers)
        self.visible = {m: cfg.hub_tokens_of(m) for m in VisualInterlingua.MODS}

    def forward(self, z: torch.Tensor, mod: str, lang: torch.Tensor) -> torch.Tensor:  # type: ignore[override]
        return super().forward(z[:, : self.visible[mod]], VisualInterlingua.MODS[mod], lang)


# ======================================================================================================================
# 15. MODEL: hub (+) private flow writers
#     (geobridge/models/flow.py)
# ======================================================================================================================
# Hub (+) private bidirectional flow spokes.
#
# One spoke per modality.  Its state is N = K_h + K_p tokens of width d:
#
#     t = 0   x_0 = [ z* (K_h hub tokens) ; xi (K_p private tokens, xi ~ N(0, I)) ]
#     t = 1   x_1 = z_m  (the modality latent)
#     path    x_t = (1 - t) x_0 + t x_1 ,   target velocity  u = x_1 - x_0
#
# Forward integration (0 -> 1) *generates* the modality from a hub code; the private tokens absorb everything
# the hub does not determine (wording, speaker, phrasing), so one hub code maps to many valid outputs.
# Backward integration (1 -> 0) *reads* a modality sample and returns both its hub code (first K_h tokens) and
# its private code.  The velocity field is a shared transformer per modality whose linear maps carry a
# rank-r LoRA pair per language.
# ======================================================================================================================

class VelocityNet(nn.Module):
    def __init__(self, cfg, n_lang: int, n_private: Optional[int] = None):
        super().__init__()
        n_private = cfg.private_slots if n_private is None else n_private
        W, d, N = cfg.flow_width, cfg.latent_dim, cfg.hub_slots + n_private
        self.W = W
        self.inp = nn.Linear(d, W)
        self.pos = nn.Parameter(torch.randn(N, W) * 0.02)
        self.kind = nn.Parameter(torch.randn(2, W) * 0.02)  # hub slot vs private slot
        self.register_buffer("kind_idx", torch.cat([torch.zeros(cfg.hub_slots), torch.ones(n_private)]).long())
        self.time = nn.Sequential(nn.Linear(W, W), nn.SiLU(), nn.Linear(W, W))
        self.lang_emb = nn.Embedding(max(1, n_lang), W)
        rank = cfg.lora_rank if n_lang > 0 else 0
        self.blocks = nn.ModuleList([Block(W, cfg.flow_heads, n_lang=n_lang, rank=rank, alpha=cfg.lora_alpha, cond_dim=W)
                                     for _ in range(cfg.flow_layers)])
        self.out_norm = nn.LayerNorm(W)
        self.out = nn.Linear(W, d)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, x: torch.Tensor, t: torch.Tensor, lang: torch.Tensor) -> torch.Tensor:
        cond = self.time(sincos(t * 1000.0, self.W)) + self.lang_emb(lang)
        h = self.inp(x) + self.pos + self.kind[self.kind_idx]
        for blk in self.blocks:
            h = blk(h, lang=lang, cond=cond)
        return self.out(self.out_norm(h))


class Spoke(nn.Module):
    def __init__(self, cfg, n_lang: int, n_private: Optional[int] = None):
        """``n_private``: private tokens of this spoke's latent (default cfg.private_slots)."""
        super().__init__()
        self.cfg, self.K = cfg, cfg.hub_slots
        self.n_private = cfg.private_slots if n_private is None else n_private
        self.vel = VelocityNet(cfg, n_lang, self.n_private)

    # ------------------------------------------------------------------ source construction
    def source(self, hub: torch.Tensor, temp: float = 1.0, private: Optional[torch.Tensor] = None) -> torch.Tensor:
        """[hub ; private].  ``private`` given -> reuse it; otherwise draw N(0, temp^2) (zeros if disabled)."""
        B, _, d = hub.shape
        if private is None:
            private = torch.randn(B, self.n_private, d, device=hub.device, dtype=hub.dtype) * temp
            if not self.cfg.private_noise:
                private = torch.zeros_like(private)
        return torch.cat([hub, private], 1)

    # ------------------------------------------------------------------ conditional flow matching
    def cfm_loss(self, x0: torch.Tensor, x1: torch.Tensor, lang: torch.Tensor,
                 weight: Optional[torch.Tensor] = None) -> torch.Tensor:
        B = x0.shape[0]
        if self.cfg.t_sampling == "logit_normal":
            t = torch.sigmoid(torch.randn(B, device=x0.device))
        else:
            t = torch.rand(B, device=x0.device)
        tt = t.view(B, 1, 1).to(x0.dtype)
        v = self.vel((1 - tt) * x0 + tt * x1, t, lang)
        per = ((v.float() - (x1 - x0).float()) ** 2).mean(dim=(1, 2))
        if weight is not None:
            return (per * weight).sum() / weight.sum().clamp_min(1e-6)
        return per.mean()

    # ------------------------------------------------------------------ ODE integration
    def solve(self, x: torch.Tensor, lang: torch.Tensor, direction: int, steps: Optional[int] = None,
              solver: Optional[str] = None, grad: bool = False, return_path: bool = False,
              t_start: Optional[float] = None):
        """direction = +1: t t_start -> 1 (hub -> modality; t_start defaults to 0);
        direction = -1: t t_start -> 0 (modality -> hub; t_start defaults to 1)."""
        steps, solver = steps or self.cfg.ode_steps, solver or self.cfg.ode_solver
        t0 = (0.0 if direction > 0 else 1.0) if t_start is None else float(t_start)
        h = direction * ((1.0 - t0) if direction > 0 else t0) / steps
        t = torch.full((x.shape[0],), t0, device=x.device)
        path: List[torch.Tensor] = [x]
        with torch.set_grad_enabled(grad):
            for _ in range(steps):
                if solver == "euler":
                    x = x + h * self.vel(x, t, lang)
                elif solver == "heun":
                    k1 = self.vel(x, t, lang)
                    k2 = self.vel(x + h * k1, t + h, lang)
                    x = x + h * 0.5 * (k1 + k2)
                else:  # midpoint
                    k1 = self.vel(x, t, lang)
                    x = x + h * self.vel(x + 0.5 * h * k1, t + 0.5 * h, lang)
                t = t + h
                if return_path:
                    path.append(x)
        return (x, path) if return_path else x

    def read(self, z: torch.Tensor, lang: torch.Tensor, grad: bool = False, steps: Optional[int] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """modality latent -> (hub estimate [B, K_h, d], private code [B, K_p, d]) by the backward ODE."""
        x0 = self.solve(z, lang, -1, steps=steps, grad=grad)
        return x0[:, : self.K], x0[:, self.K:]

    def write(self, hub: torch.Tensor, lang: torch.Tensor, temp: Optional[float] = None,
              private: Optional[torch.Tensor] = None, grad: bool = False, steps: Optional[int] = None,
              draft: Optional[torch.Tensor] = None, t0: float = 0.0) -> torch.Tensor:
        """hub code -> modality latent.

        ``draft`` (the Explainer's one-pass prediction of the latent) with ``t0`` in (0, 1] starts the ODE on
        the straight path between the source and the draft, at time t0, instead of at the source: the flow
        only has to finish what the draft began (t0 = 1 returns the draft itself, with no ODE step)."""
        temp = self.cfg.sample_temp if temp is None else temp
        x = self.source(hub, temp, private)
        if draft is not None and draft.shape[1] < x.shape[1]:   # tokens the draft does not cover start from the source
            draft = torch.cat([draft, x[:, draft.shape[1]:]], 1)
        if draft is None or t0 <= 0.0:
            return self.solve(x, lang, +1, steps=steps, grad=grad)
        if t0 >= 1.0:
            return draft
        steps = max(1, int(round((steps or self.cfg.ode_steps) * (1.0 - t0))))
        return self.solve((1.0 - t0) * x + t0 * draft, lang, +1, steps=steps, grad=grad, t_start=t0)


# ======================================================================================================================
# 16. MODEL: preview pixel decoder
#     (geobridge/models/pixel.py)
# ======================================================================================================================
# Preview pixel decoder: image latent (the anchored patch grid pooled to N tokens) -> a small RGB image.
#
# It exists so that the image-writing direction produces something that can be looked at.  It is deliberately
# tiny and trained with a contrast-weighted L1 loss on thumbnails (see ``GeoBridgeLoss.phase_a``), so its output
# is a low-resolution preview, not a photograph; a pretrained unCLIP-style decoder on the same vision space is
# the route to photorealism.  Its input is frozen (the anchored patch grid), so it shares no gradient with the
# rest of the model.
# ======================================================================================================================

class PreviewDecoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, C, self.size = cfg.latent_dim, cfg.pix_width, cfg.thumb_size
        self.grid = grid_factor(cfg.n_tokens)
        self.inp = nn.Linear(d, C)
        self.pos = nn.Parameter(torch.randn(cfg.n_tokens, C) * 0.02)
        self.mix = Block(C, 4)
        chans = [C, C // 2, C // 4, C // 4]
        self.ups = nn.ModuleList([nn.Conv2d(chans[i], chans[i + 1], 3, padding=1) for i in range(3)])
        self.out = nn.Conv2d(chans[-1], 3, 3, padding=1)

    def forward(self, z_img: torch.Tensor) -> torch.Tensor:
        """z_img [B, N, d] -> RGB [B, 3, S, S] in [0, 1]."""
        h = self.mix(self.inp(z_img) + self.pos)
        h = h.transpose(1, 2).reshape(z_img.shape[0], -1, *self.grid)
        h = F.interpolate(h, size=(self.size // 8, self.size // 8), mode="bilinear", align_corners=False)
        for conv in self.ups:
            h = F.gelu(conv(F.interpolate(h, scale_factor=2, mode="nearest")))
        return torch.sigmoid(self.out(h))


# ======================================================================================================================
# 17. MODEL: GeoBridge (assembled)
#     (geobridge/models/geobridge.py)
# ======================================================================================================================
# GeoBridge-Flow: the assembled model.
#
#     text  --GBC codec-->  z_txt  --reader-->  \                           /--text writer (flow)-->  z_txt
#     speech -tone-gated->  z_aud  --reader-->   }  hub z*  (barycentres   {---speech writer (flow)->  z_aud
#     image --frozen ViT->  z_img  --ODE------>  /   of frozen patches)     \--image writer (flow)-->  z_img
#
# Reading is a function: one hub code per sentence.  It is done by the image-free **reader** in a single pass
# (or, optionally, by the backward ODE of the modality's flow).  Writing is a distribution: one hub code has
# many valid realisations, so every writer is a flow whose source is ``[hub tokens ; private Gaussian tokens]``.
# Training-time hub codes come from the grounder and need the paired image; at test time nothing but the
# source is needed.
# ======================================================================================================================

class GeoBridge(nn.Module):
    def __init__(self, cfg, spec: Dict):
        super().__init__()
        self.cfg, self.spec = cfg, spec
        self.langs: List[str] = list(spec["langs"])
        self.lang2id = {l: i for i, l in enumerate(self.langs)}
        n_lang = len(self.langs)
        anchor = spec.get("anchor") or {
            "mean": torch.zeros(spec["vision_dim"]), "basis": torch.zeros(spec["vision_dim"], cfg.latent_dim),
            "scale": torch.ones(())}
        stats = spec.get("audio_stats") or {
            "sem_mean": torch.zeros(spec["sem_dim"]), "sem_std": torch.ones(spec["sem_dim"]),
            "spk_mean": torch.zeros(spec["spk_dim"]), "spk_std": torch.ones(spec["spk_dim"]),
            "f0_mean": torch.zeros(2), "f0_std": torch.ones(2), "mel_mean": 0.0, "mel_std": 1.0}
        self.text = TextCodec(cfg, spec["unit_vocab"], spec["unit_pad"], spec["unit_bytes"], n_lang)
        self.audio = AudioCodec(cfg, spec["sem_dim"], spec["spk_dim"], n_lang, stats)
        self.hub = VisualInterlingua(cfg, spec["vision_dim"], spec["n_vis_tokens"], anchor, n_lang)
        spokes = {"txt": Spoke(cfg, n_lang), "aud": Spoke(cfg, n_lang, cfg.private_of("aud"))}
        if cfg.image_spoke:
            spokes["img"] = Spoke(cfg, 0)
            if cfg.w_pix > 0:
                self.pixel = PreviewDecoder(cfg)
        self.spokes = nn.ModuleDict(spokes)

    # ------------------------------------------------------------------ parameter groups
    def phase_a_modules(self) -> List[nn.Module]:
        mods: List[nn.Module] = [self.text, self.audio, self.hub]
        return mods + ([self.pixel] if hasattr(self, "pixel") else [])

    def phase_b_modules(self) -> List[nn.Module]:
        """Writers and the reader, which keeps training on the cached (now fixed) codes; the explainer too
        when writing starts from its draft, and the text decoder with ``w_dec`` (the text *encoder* stays
        frozen, so the cached latents stay valid).  The 'free' anchor is learned jointly with the flows."""
        mods: List[nn.Module] = [self.spokes, self.hub if self.cfg.hub_mode == "free" else self.hub.reader]
        if self.cfg.hub_mode != "free" and self.cfg.draft_t0 > 0:
            mods.append(self.hub.explainer)
        if self.cfg.w_dec > 0:   # the byte decoder learns to read what the text writer actually writes
            mods.append(self.text.decoder())
        return mods

    # ------------------------------------------------------------------ modality latents
    def encode(self, mod: str, sub: Dict[str, torch.Tensor], gate_override: Optional[float] = None) -> torch.Tensor:
        if mod == "txt":
            return self.text.encode(sub["units"], sub["unit_mask"])
        return self.audio.encode(sub["sem"], sub["tone"], sub["mask"], sub["lang"], gate_override=gate_override)

    def image_latent(self, V: torch.Tensor) -> torch.Tensor:
        return self.hub.image_latent(V, self.cfg.n_tokens)

    # ------------------------------------------------------------------ training-time hub code of a paired view
    def paired_hub(self, mod: str, z: torch.Tensor, lang: torch.Tensor, V_item: torch.Tensor,
                   anchor_z: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Hub code of image-paired utterances under the configured anchor (V_item [n, P, d] = their images).
        Used when no contrastive score is needed (latent cache, figures); phase A uses ``hub.ground``."""
        mode = self.cfg.hub_mode
        if mode == "bary":
            return self.hub.attend(self.hub.queries(z, mod, lang), V_item)[0]
        if mode == "global":
            return self.hub.scene_code(V_item)
        if mode == "free":
            return self.hub.free_code(z, mod, lang, self.hub.scene_code(V_item))
        if mode == "text":
            return (anchor_z if anchor_z is not None else z)[:, : self.cfg.hub_slots]
        raise ValueError(mode)

    # ------------------------------------------------------------------ read: modality -> hub
    def read(self, mod: str, z: torch.Tensor, lang: Optional[torch.Tensor] = None, mode: Optional[str] = None,
             steps: Optional[int] = None) -> torch.Tensor:
        """Hub code of a modality latent with nothing but the latent.  ``amortized`` = the reader, one pass;
        ``ode`` = backward flow.  Images have no reader (an image does not say one thing): always the ODE."""
        mode = mode or self.cfg.read_mode
        if mod == "img" or mode == "ode":
            lang = torch.zeros(z.shape[0], dtype=torch.long, device=z.device) if lang is None or mod == "img" else lang
            return self.spokes[mod].read(z, lang, steps=steps)[0]
        return self.hub.reader(z, mod, lang)

    # ------------------------------------------------------------------ write: hub -> modality
    def write(self, mod: str, hub: torch.Tensor, lang: Optional[torch.Tensor] = None, temp: Optional[float] = None,
              steps: Optional[int] = None, private: Optional[torch.Tensor] = None, t0: Optional[float] = None) -> torch.Tensor:
        """Modality latent written from a hub code by the flow.  With ``t0 > 0`` the ODE starts from the
        explainer's one-pass draft (text and speech only)."""
        if lang is None or mod == "img":
            lang = torch.zeros(hub.shape[0], dtype=torch.long, device=hub.device)
        t0 = self.cfg.draft_t0 if t0 is None else t0
        draft = None
        if t0 > 0 and mod != "img":
            draft = self.hub.explainer(hub, self.hub.MODS[mod], lang)
        return self.spokes[mod].write(hub, lang, temp=temp, private=private, steps=steps, draft=draft, t0=t0)

    @torch.no_grad()
    def hub_to_text(self, hub: torch.Tensor, lang_name: str, temp: Optional[float] = None) -> List[str]:
        lang = torch.full((hub.shape[0],), self.lang2id[lang_name], dtype=torch.long, device=hub.device)
        return self.text.generate(self.write("txt", hub, lang, temp=temp), lang)

    @torch.no_grad()
    def hub_to_speech(self, hub: torch.Tensor, lang_name: str, spk: torch.Tensor, f0: torch.Tensor,
                      temp: Optional[float] = None):
        """hub -> (log-mel, frame counts, speech latent, streams).  ``spk, f0`` give the voice (bypass)."""
        lang = torch.full((hub.shape[0],), self.lang2id[lang_name], dtype=torch.long, device=hub.device)
        z = self.write("aud", hub, lang, temp=temp)
        mel, n_frames, streams = self.audio.synthesize_mel(z, lang, spk, f0)
        return mel, n_frames, z, streams

    @torch.no_grad()
    def hub_to_image(self, hub: torch.Tensor, temp: Optional[float] = None) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """hub -> (image latent [B, N, d], preview RGB [B, 3, S, S] or None)."""
        z = self.write("img", hub, temp=temp)
        return z, (self.pixel(z) if hasattr(self, "pixel") else None)


def build_spec(data) -> Dict:
    """Everything the model needs to know about the data; stored in every checkpoint."""
    u = data.unitizer
    return {"langs": list(data.cfg.all_langs), "lang_info": data.lang_info, "vision_dim": data.vision_dim,
            "n_vis_tokens": data.n_vis_tokens, "sem_dim": data.sem_dim, "spk_dim": data.spk_dim,
            "unit_vocab": u.unit_vocab, "unit_pad": u.unit_pad, "unit_bytes": u.m,
            "anchor": {k: v for k, v in data.anchor.items()}, "audio_stats": dict(data.audio.stats)}


def spec_for_checkpoint(spec: Dict) -> Dict:
    """Shapes only (the tensors themselves are model buffers and travel in the state dict)."""
    return {k: v for k, v in spec.items() if k not in ("anchor", "audio_stats")}


# ======================================================================================================================
# 18. LOSSES
#     (geobridge/losses.py)
# ======================================================================================================================
# All training objectives of GeoBridge-Flow in one class.
#
# Phase A - representation (codecs, grounder, explainer, reader, preview decoder)
#     L_A = w_text_rec * L_text + w_audio_rec * L_audio(segmental, prosody, duration, mel, orth, gate price)
#         + w_ground * L_ground        the hub code is found in the right image          (contrastive)
#         + w_expl   * L_expl          the hub code explains the sentence                (sufficiency)
#         + w_seek   * L_seek          ... and every slot is pulled to the patch that would explain more
#                                      (counterfactual patch credit)
#         + w_read   * L_read          an image-free reader predicts the hub code
#         + w_min    * L_min           ... and the hub code must be predictable by it    (minimality)
#         + w_agree  * L_agree  + w_pix * L_pix
#
# Phase B - transport (writers = flows, plus the reader), computed on *cached* latents: the frozen codecs and the
# grounder are not run again.
#     L_B = w_cfm * (L_cfm^txt + L_cfm^aud + L_cfm^img)     hub (+) private flow matching; a share of the views
#                                                           is written from a *read* hub (exposure)
#         + w_read * L_read                                  keeps the reader on the grounded codes
#         + w_dec * L_dec                                    the byte decoder learns to read written latents
#         + bt_weight * (L_bt_write + L_bt_read + bt_relay * L_bt_relay)   hub-space back-translation of unpaired
#                                                           island speech, relayed to the text writers
#         [+ w_dock * L_dock + naive_cycle_weight * L_cycle] ablation / diagnostic terms, off by default
#
# No term ever pairs two different languages: cross-lingual transfer can only pass through the hub.
# ======================================================================================================================

def _pad_time(x: torch.Tensor, L: int) -> torch.Tensor:
    return F.pad(x, (0, 0, 0, L - x.shape[1])) if x.dim() == 3 else F.pad(x, (0, L - x.shape[1]))


def cat_audio(a: Optional[Dict], b: Optional[Dict]) -> Optional[Dict]:
    """Concatenate two collated speech sub-batches (padding the time axis)."""
    if a is None or b is None:
        return a if b is None else b
    T = max(a["sem"].shape[1], b["sem"].shape[1])
    Tm = max(a["mel"].shape[1], b["mel"].shape[1]) if "mel" in a and "mel" in b else 0
    out = {}
    for k in a:
        if k not in b:
            continue
        x, y = a[k], b[k]
        if k in ("sem", "tone", "mask", "mel"):  # time axes; "mel_len" travels along untouched
            L = Tm if k == "mel" else T
            x, y = _pad_time(x, L), _pad_time(y, L)
        out[k] = torch.cat([x, y], 0)
    return out


def cat_text(a: Optional[Dict], b: Optional[Dict], pad_id: int) -> Optional[Dict]:
    """Concatenate two collated text sub-batches (padding units and bytes with their pad ids)."""
    if a is None or b is None:
        return a if b is None else b
    L, T = max(a["units"].shape[1], b["units"].shape[1]), max(a["bytes"].shape[1], b["bytes"].shape[1])

    def pad(d: Dict) -> Dict:
        return {"units": F.pad(d["units"], (0, 0, 0, L - d["units"].shape[1]), value=pad_id),
                "unit_mask": F.pad(d["unit_mask"], (0, L - d["unit_mask"].shape[1])),
                "bytes": F.pad(d["bytes"], (0, T - d["bytes"].shape[1]), value=256), "lang": d["lang"]}

    pa, pb = pad(a), pad(b)
    return {k: torch.cat([pa[k], pb[k]], 0) for k in pa}


def same_lang_pairs(txt: Optional[Dict], aud: Optional[Dict]) -> Tuple[torch.Tensor, torch.Tensor]:
    """Indices (i_txt, i_aud) of text / speech views of the same caption in the same language."""
    if txt is None or aud is None:
        e = torch.zeros(0, dtype=torch.long)
        return e, e
    key = {(int(i), int(l)): n for n, (i, l) in enumerate(zip(txt["item"].tolist(), txt["lang"].tolist()))}
    it, ia = [], []
    for n, (i, l) in enumerate(zip(aud["item"].tolist(), aud["lang"].tolist())):
        if (i, l) in key:
            it.append(key[(i, l)])
            ia.append(n)
    dev = txt["item"].device
    return torch.tensor(it, dtype=torch.long, device=dev), torch.tensor(ia, dtype=torch.long, device=dev)


def preview_loss(pred: torch.Tensor, target: torch.Tensor, contrast: float) -> Tuple[torch.Tensor, torch.Tensor]:
    """Contrast-weighted L1 between RGB images [B, 3, S, S] in [0, 1]; also returns the plain L1.

    A pixel counts in proportion to how far it is from the batch's mean colour.  Plain L1 is minimised by the
    median image, which is the bare background wherever the content covers a small part of the picture; a
    decoder trained that way never starts to use its input (measured: an all-background preview)."""
    err = (pred - target).abs()
    away = (target - target.mean(dim=(0, 2, 3), keepdim=True)).abs().mean(1, keepdim=True)
    w = 1.0 + contrast * away / away.mean().clamp_min(1e-6)
    return (w * err).sum() / (w.sum() * err.shape[1]), err.mean()


def lang_weights(lang: torch.Tensor, balance: bool) -> torch.Tensor:
    """Per-view weights of the views of one modality.  ``balance``: every language present gets the same total
    weight (the weights still sum to the number of views), so the loss is the mean over languages of the
    per-language means.  Otherwise all ones."""
    if not balance or lang.numel() == 0:
        return torch.ones(lang.shape[0], device=lang.device)
    ids, inv, counts = torch.unique(lang, return_inverse=True, return_counts=True)
    return (lang.shape[0] / ids.numel()) / counts[inv].float()


def wmean(x: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    """Mean of the per-view values ``x`` under the per-view weights ``w`` (the plain mean when they are all ones)."""
    return (x * w).sum() / w.sum().clamp_min(1e-6)


class GeoBridgeLoss:
    def __init__(self, cfg, lang2id: Dict[str, int]):
        self.cfg = cfg
        self.hub_ids = [lang2id[l] for l in cfg.hub_langs if l in lang2id]

    # ============================================================================ phase A
    def phase_a(self, model, batch: Dict, island: Optional[Dict] = None) -> Tuple[torch.Tensor, Dict[str, float]]:
        cfg, hub, logs = self.cfg, model.hub, {}
        V = hub.project(batch["img"])
        keys = hub.keys(V)
        total = V.new_zeros(())
        txt, aud = batch["txt"], batch["aud"]
        z: Dict[str, torch.Tensor] = {}

        # ---- codecs (one forward per modality: paired views + replay text / unpaired island speech)
        txt_all = cat_text(txt, batch.get("replay"), model.text.unit_pad)
        if txt_all is not None:
            r = model.text.loss(txt_all)
            total = total + cfg.w_text_rec * r["loss"]
            if txt is not None:
                z["txt"] = r["z"][: txt["units"].shape[0]]
            logs.update(txt_ce=r["loss"].item(), txt_acc=r["byte_acc"].item())
        aud_all = cat_audio(aud, island["aud"] if island is not None else None)
        if aud_all is not None:
            n_paired = 0 if aud is None else aud["sem"].shape[0]
            # (listening speech is the last of the paired views and is not grounded: it pays for no gate)
            r = model.audio.loss(aud_all, n_grounded=n_paired - (int(aud["listen"].sum()) if aud is not None and "listen" in aud else 0))
            total = total + cfg.w_audio_rec * r["loss"]
            if aud is not None:
                z["aud"] = r["z"][:n_paired]
            logs.update({k: r[k].item() for k in ("sem_mse", "speak_mse", "f0_l1", "dur_l1", "mel_l1", "orth") if k in r})
            if "gate_pen" in r:
                logs["tone_price"] = r["gate_pen"].item()   # what the batch paid for open gates (not a gate value)

        # ---- listening speech (extra spoken captions, cfg.listen_only): it has been through the codec like all
        #      speech; from here on it is only READ - into the code that the text of the same caption and the image
        #      define - and is left out of every term that shapes the hub.  A hub that also has to make room for a
        #      flood of speech in the high-resource language does so at the expense of the island language.
        heard = None
        if aud is not None and "listen" in aud and "aud" in z and "txt" in z:
            hl = aud["listen"]
            it_h, ia_h = same_lang_pairs(txt, {"item": aud["item"][hl], "lang": aud["lang"][hl]})
            if len(ia_h):
                heard = {"z": z["aud"][hl][ia_h].detach(), "lang": aud["lang"][hl][ia_h], "txt": it_h}
            aud = {"item": aud["item"][~hl], "lang": aud["lang"][~hl]}   # the speech views that are grounded
            z["aud"] = z["aud"][~hl]
            if not len(aud["item"]):
                del z["aud"]

        # ---- grounding: one attention pass gives the contrastive scores and the hub code
        scores, items, codes, attns, logps, vw = [], [], {}, {}, {}, {}
        released = hub.null_enabled  # minimality starts when the dense stage ends
        for mod, sub in (("txt", txt), ("aud", aud)):
            if mod not in z:
                continue
            q = hub.queries(z[mod], mod, sub["lang"])
            score, attns[mod], logps[mod] = hub.ground(q, keys, sub["item"])
            codes[mod] = torch.einsum("nkp,npd->nkd", attns[mod].to(V.dtype), V[sub["item"]][:, :-1])
            scores.append(score)
            items.append(sub["item"])
            vw[mod] = lang_weights(sub["lang"], cfg.lang_balance)   # per-view weights (ones unless lang_balance)
        if scores:
            logits = torch.cat(scores) / cfg.ground_temp
            item = torch.cat(items)
            same = batch["img_idx"][item].unsqueeze(1) == batch["img_idx"].unsqueeze(0)  # [n, B]
            pos = F.one_hot(item, batch["B"]).bool()
            masked = logits.masked_fill(same & ~pos, -1e4)
            l_u2i = wmean(F.cross_entropy(masked, item, reduction="none"), torch.cat([vw[m] for m in codes]))
            logp = logits.t().log_softmax(dim=1)                                          # [B, n]
            l_i2u = -torch.logsumexp(logp.masked_fill(~same.t(), -1e4), dim=1)[same.t().any(1)].mean()
            ground = 0.5 * (l_u2i + l_i2u)
            total = total + cfg.w_ground * ground
            logs.update(ground=ground.item(), ground_acc=(masked.argmax(1) == item).float().mean().item())

            # the canonical code of a caption: where a transcript exists, the text-grounded code teaches speech
            # (same language only); text is the cleaner view, and speech then inherits everything it covers
            teacher = {m: c.detach() for m, c in codes.items()}
            it, ia = same_lang_pairs(txt, aud)
            if len(it) and len(codes) == 2:
                if cfg.agree_mode == "teach":
                    teacher["aud"] = teacher["aud"].clone()
                    teacher["aud"][ia] = teacher["txt"][it]
                if cfg.w_agree > 0:
                    agree = F.mse_loss(codes["aud"][ia], teacher["txt"][it] if cfg.agree_mode == "teach" else codes["txt"][it])
                    total = total + cfg.w_agree * agree
                    logs["agree"] = agree.item()
            for mod, sub in (("txt", txt), ("aud", aud)):
                if mod not in codes:
                    continue
                mid = hub.MODS[mod]
                if cfg.w_expl > 0:  # sufficiency: the code alone reconstructs the utterance latent
                    target, lang = z[mod][:, : cfg.hub_tokens_of(mod)].detach().float(), sub["lang"]   # what the hub reads

                    def unexplained(code: torch.Tensor, rows: Optional[torch.Tensor] = None) -> torch.Tensor:
                        """Per-utterance unexplained variance of code [m, K, d] (for the utterances ``rows``)."""
                        t, l = (target, lang) if rows is None else (target[rows], lang[rows])
                        return ((hub.explainer(code, mid, l).float() - t) ** 2).mean(dim=(1, 2))

                    err = unexplained(codes[mod])
                    unexp = err.mean()
                    total = total + cfg.w_expl * wmean(err, vw[mod])
                    logs[f"unexplained_{mod}"] = unexp.item()
                    if cfg.w_seek > 0 and codes[mod].requires_grad:  # a training signal only (needs a gradient)
                        seek, moved = self.seek(unexplained, err, codes[mod], logps[mod], V[sub["item"]][:, :-1], released,
                                                vw[mod])
                        total = total + cfg.w_seek * seek
                        logs[f"seek_{mod}"] = moved
                n, z_in, lang_in, goal, w = len(sub["lang"]), z[mod].detach(), sub["lang"], teacher[mod], vw[mod]
                if mod == "aud" and heard is not None:   # listening speech joins the reader's batch, one view one vote
                    z_in, lang_in = torch.cat([z_in, heard["z"]]), torch.cat([lang_in, heard["lang"]])
                    goal, w = torch.cat([goal, teacher["txt"][heard["txt"]]]), torch.cat([w, torch.ones_like(heard["lang"], dtype=w.dtype)])
                pred = hub.reader(z_in, mod, lang_in).float()
                per_view = ((pred - goal.float()) ** 2).mean(dim=(1, 2))
                total = total + cfg.w_read * wmean(per_view, w)
                logs[f"read_{mod}"] = per_view[:n].mean().item()
                if len(per_view) > n:
                    logs["heard"] = per_view[n:].mean().item()   # reader error on the listening speech
                pred = pred[:n]
                if cfg.w_min > 0 and released:  # minimality: nothing in the code that the sentence does not say
                    total = total + cfg.w_min * wmean(((codes[mod].float() - pred.detach()) ** 2).mean(dim=(1, 2)), vw[mod])
                logs[f"slot_use_{mod}"] = attns[mod].sum(-1).mean().item()
            if cfg.w_expl > 0:
                logs["unexplained"] = sum(v for k, v in logs.items() if k.startswith("unexplained_")) / len(codes)
            logs["read"] = sum(v for k, v in logs.items() if k.startswith("read_")) / len(codes)

        # ---- preview pixel decoder (frozen image latent -> thumbnail)
        if "thumb" in batch and hasattr(model, "pixel"):
            target = batch["thumb"].permute(0, 3, 1, 2).float() / 255.0
            pix, l1 = preview_loss(model.pixel(model.image_latent(V)).float(), target, cfg.pix_contrast)
            total = total + cfg.w_pix * pix
            logs["pix_l1"] = l1.item()
        gates = model.audio.gate().detach()
        for name, i in model.lang2id.items():
            if name not in cfg.text_only_langs:   # a language without speech has a gate parameter nothing ever trains
                logs[f"gate_{name}"] = gates[i].item()
        logs["loss"] = total.item()
        return total, logs

    def seek(self, unexplained, err: torch.Tensor, code: torch.Tensor, logp: torch.Tensor, patches: torch.Tensor,
             allow_null: bool, view_weight: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, float]:
        """Counterfactual patch credit ("look where it would explain more").

        A slot's hub token is a barycentre, so sufficiency reaches the attention only through the softmax
        Jacobian  d code_k / d logit_p = a_kp (v_p - code_k): a patch the slot does not look at yet (a_kp ~ 0)
        gets no credit, however much it would explain.  Once the contrastive loss (which is content with one
        object) has made the slots sharp, the second thing a sentence says can stay unseen for ever.

        propose  the counterfactual "slot k sits on patch p" is cheap to rank for all (k, p) at once:
                     gain_kp = - < d err / d code_k ,  v_p - code_k >        (v = 0 for the null option)
                 is the first-order drop of the utterance's unexplained variance.  Patches already taken by
                 earlier slots are not candidates.  Every slot keeps its best candidate.
        pull     cross-entropy on the slot's log-attention towards its candidate, weighted by
                 gain^2 / (gain^2 + seek_gain^2): a code that already explains its sentence is left alone.
                 The log-attention is not clamped, so the gradient (a - onehot) reaches unattended patches.
                 The threshold is absolute, in units of latent variance.  Judging the gain relative to the
                 utterance's unexplained variance was tried and halves the coverage it buys (measured): a
                 nearly explained sentence then gets pulled on noise.
        veto     (``seek_verify``, off by default) each proposed move is carried out and the explainer is run
                 on it; a move that verifiably explains *less* is dropped.  Measured: fewer slots on objects
                 the sentence does not mention, but also fewer second objects found, because the explainer has
                 to co-adapt before a newly covered object pays off.  Requiring a verified *gain* up front
                 switches the mechanism off altogether.

        unexplained(code, rows) -> per-utterance error;  err = unexplained(code) [n];  code [n, K, d];
        logp [n, K, P + 1] log-attention with the null option last;  patches [n, P, d].
        Returns the loss and the mean weight (share of slots being moved)."""
        cfg = self.cfg
        n, K, _ = code.shape
        rows = torch.arange(n, device=code.device)
        scale = float(cfg.max_tokens * cfg.latent_dim)  # differentiate the sum of squares: O(1) values, safe in fp16
        g = torch.autograd.grad(err.sum() * scale, code, retain_graph=True)[0].float() / scale   # [n, K, d]
        with torch.no_grad():
            attn = logp[..., :-1].exp()
            here = (g * code.float()).sum(-1, keepdim=True)                                   # [n, K, 1]
            gain = here - torch.einsum("nkd,npd->nkp", g, patches.float())                    # [n, K, P]
            gain = gain.masked_fill(attn.cumsum(1) - attn > 0.5, -1e4)                        # taken by earlier slots
            gain = torch.cat([gain, here if allow_null else torch.full_like(here, -1e4)], -1)  # the origin
            best, idx = gain.max(-1)                                                          # [n, K]
            w = best.clamp_min(0) ** 2
            w = w / (w + cfg.seek_gain ** 2)
            if cfg.seek_verify:
                value = torch.cat([patches, torch.zeros_like(patches[:, :1])], 1)[rows.unsqueeze(1), idx]   # [n, K, d]
                trial = code.detach().unsqueeze(1).repeat(1, K, 1, 1)                         # [n, move, slot, d]
                k = torch.arange(K, device=code.device)
                trial[:, k, k] = value.to(trial.dtype)                                        # move k replaces slot k
                after = unexplained(trial.flatten(0, 1), rows.repeat_interleave(K)).view(n, K)
                w = w * (after <= err.detach().unsqueeze(1) + cfg.seek_gain)
        pull = w if view_weight is None else w * view_weight.unsqueeze(1)   # (the reported share of slots moved is unweighted)
        return -(pull * logp.gather(-1, idx.unsqueeze(-1)).squeeze(-1).float()).mean(), float(w.mean())

    # ============================================================================ phase B
    def text_anchor(self, z: Dict[str, torch.Tensor], mod: str, sub: Dict, pairs, pivot: Optional[Dict],
                    n_items: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Text-anchor baseline: which text latent serves as the hub of each view, and whether one exists."""
        n, dev = z[mod].shape[0], z[mod].device
        hub_lang = torch.tensor([int(l) in self.hub_ids for l in sub["lang"].tolist()], device=dev)
        anchor = z[mod].clone() if mod == "txt" else torch.zeros_like(z[mod])
        ok = hub_lang.clone() if mod == "txt" else torch.zeros(n, dtype=torch.bool, device=dev)
        if mod == "aud" and "txt" in z and len(pairs[0]):  # hub-language speech: its own transcript
            anchor[pairs[1]] = z["txt"][pairs[0]]
            ok[pairs[1]] = True
        if pivot is not None and "pivot" in z:  # everything else: a caption of the same image (image pivot)
            row = torch.full((n_items,), -1, dtype=torch.long, device=dev)
            row[pivot["item"]] = torch.arange(len(pivot["item"]), device=dev)
            r = row[sub["item"]]
            use = (~ok) & (r >= 0)
            anchor[use] = z["pivot"][r[use]]
            ok = ok | use
        return anchor, ok

    def phase_b(self, model, batch: Dict, island: Optional[Dict], progress: float,
                bank=None) -> Tuple[torch.Tensor, Dict[str, float]]:
        """``batch`` comes from the latent cache: per modality the latents ``z``, the grounded hub codes
        ``hub``, ``lang``, ``item`` and ``ok``; plus the image latents ``img_lat`` [B, N, d].  ``bank`` is the
        latent cache itself (random real hub codes / views for the unpaired-speech terms)."""
        cfg, logs = self.cfg, {}
        sp, reader = model.spokes, model.hub.reader
        views = {m: batch[m] for m in ("txt", "aud") if batch.get(m) is not None}
        dev = batch["img_lat"].device
        total = batch["img_lat"].new_zeros(())

        # ---- grounded hub codes (cached; the unconstrained 'free' anchor is computed live and trained)
        hubs: Dict[str, torch.Tensor] = {}
        for mod, sub in views.items():
            if cfg.hub_mode == "free":
                hubs[mod] = model.hub.free_code(sub["z"], mod, sub["lang"], batch["scene"][sub["item"]])
            else:
                hubs[mod] = sub["hub"]
        if cfg.hub_mode == "free" and hubs:
            var = F.relu(1.0 - torch.cat(list(hubs.values())).float().flatten(1).std(0)).mean()
            total = total + cfg.w_free_hub_var * var
            logs["free_var"] = var.item()

        # ---- the reader keeps training, now on codes that no longer move; its (detached) outputs are the read
        #      hubs for exposure
        reads: Dict[str, torch.Tensor] = {}
        for mod, sub in views.items():
            ok, target = sub["ok"], hubs[mod].detach()
            z_in = sub.get("z_read", sub["z"])   # speech: possibly a time-masked encoding of the same utterance
            if cfg.read_noise > 0 and reader.training:   # the cached latents never change: without noise the
                sigma = torch.rand(z_in.shape[0], 1, 1, device=dev) * cfg.read_noise   # reader memorises them
                z_in = z_in + sigma.to(z_in.dtype) * torch.randn_like(z_in)
            r = reader(z_in, mod, sub["lang"])
            per_view = ((r[ok].float() - target[ok].float()) ** 2).mean(dim=(1, 2))
            total = total + cfg.w_read * (per_view * lang_weights(sub["lang"][ok], cfg.lang_balance)).mean()
            logs[f"read_{mod}"] = per_view.mean().item()
            reads[mod] = r.detach()
            if cfg.draft_t0 > 0 and cfg.hub_mode == "bary":  # the explainer is refined only if writing starts from its draft
                l_expl = F.mse_loss(model.hub.explainer(target[ok], model.hub.MODS[mod], sub["lang"][ok]).float(),
                                    sub["z"][ok][:, : cfg.hub_tokens_of(mod)].float())
                total = total + cfg.w_expl * l_expl
                logs[f"unexplained_{mod}"] = l_expl.item()

        # ---- exposure: a share of the views is written from a hub that was *read*, as it will be at test time
        pairs = same_lang_pairs(views.get("txt"), views.get("aud"))
        src = {m: hubs[m] for m in views}
        if cfg.exposure != "none" and cfg.p_read > 0:
            for mod, sub in views.items():
                other = "aud" if mod == "txt" else "txt"
                u = torch.rand(sub["z"].shape[0], device=dev)
                mine, theirs = (pairs[0], pairs[1]) if mod == "txt" else (pairs[1], pairs[0])
                new = src[mod].clone()
                if cfg.exposure == "reader":
                    self_read = u < cfg.p_read / 2
                    new[self_read] = reads[mod][self_read].to(new.dtype)
                    if len(mine):  # the same caption read from the other modality
                        cross = (u[mine] >= cfg.p_read / 2) & (u[mine] < cfg.p_read)
                        new[mine[cross]] = reads[other][theirs[cross]].to(new.dtype)
                elif len(mine):  # "ode": hub read by the other spoke's backward flow (costly; ablation)
                    cross = u[mine] < cfg.p_read
                    if cross.any():
                        o = views[other]
                        h, _ = sp[other].read(o["z"][theirs[cross]], o["lang"][theirs[cross]], steps=cfg.train_ode_steps)
                        new[mine[cross]] = h.to(new.dtype)
                src[mod] = new

        # ---- hub (+) private conditional flow matching.  The hub part of the source is jittered: a writer has
        #      to tolerate a code that was read with some error (the island reader's most of all).
        if cfg.hub_jitter > 0:
            for mod in src:
                sigma = torch.rand(src[mod].shape[0], 1, 1, device=dev) * cfg.hub_jitter
                src[mod] = src[mod] + sigma.to(src[mod].dtype) * torch.randn_like(src[mod])
        for mod, sub in views.items():
            l = sp[mod].cfm_loss(sp[mod].source(src[mod]), sub["z"], sub["lang"], weight=sub["ok"].float())
            total = total + cfg.w_cfm * l
            logs[f"cfm_{mod}"] = l.item()
        if "img" in sp and views:
            h_all = torch.cat([src[m][views[m]["ok"]] for m in views])
            it_all = torch.cat([views[m]["item"][views[m]["ok"]] for m in views])
            if len(h_all) > batch["B"]:
                pick = torch.randperm(len(h_all), device=dev)[: batch["B"]]
                h_all, it_all = h_all[pick], it_all[pick]
            l = sp["img"].cfm_loss(sp["img"].source(h_all), batch["img_lat"][it_all],
                                   torch.zeros(len(h_all), dtype=torch.long, device=dev))
            total = total + cfg.w_cfm * l
            logs["cfm_img"] = l.item()

        # ---- write-latent exposure of the byte decoder.  The codec was trained on encoder latents; at test
        #      time it only ever sees latents that a writer produced from a hub code, which are blurred wherever
        #      the hub leaves the wording open.  Here the decoder learns to turn those into text (the encoder is
        #      frozen, so the cached latents stay valid).
        if cfg.w_dec > 0 and "txt" in views and "bytes" in views["txt"]:
            t = views["txt"]
            temp = cfg.eval_temp + (1.0 - cfg.eval_temp) * float(torch.rand(()))
            with torch.no_grad():
                z_w = sp["txt"].write(src["txt"], t["lang"], temp=temp, steps=cfg.train_ode_steps)
                written = (torch.rand(z_w.shape[0], device=dev) < cfg.p_dec_written) & t["ok"]
                z_d = torch.where(written.view(-1, 1, 1), z_w.to(t["z"].dtype), t["z"])
            l_dec, acc = model.text.decode_loss(z_d, t["bytes"], t["lang"])
            total = total + cfg.w_dec * l_dec
            logs.update(dec_ce=l_dec.item(), dec_acc=acc.item())

        # ---- write-latent exposure of the reader.  Back-translation teaches the reader to read island speech that
        #      the speech writer wrote; nothing taught it to read written speech of a hub language, and on Flickr8k
        #      it could not (written English speech, read back, carried no content while written Yoruba did).
        #      Real hub codes are written as speech in a spoken hub language and read back.
        if cfg.w_reread > 0 and "aud" in sp and views:
            h_real = torch.cat([hubs[m][views[m]["ok"]] for m in views]).detach()
            h_real = h_real[torch.randperm(len(h_real), device=dev)[: cfg.batch_size]]
            spoken = torch.tensor([model.lang2id[l] for l in cfg.hub_langs], device=dev)
            lang_s = spoken[torch.randint(len(spoken), (len(h_real),), device=dev)]
            with torch.no_grad():
                synth = sp["aud"].write(h_real, lang_s, temp=1.0, steps=cfg.train_ode_steps)
            l_rr = F.mse_loss(reader(synth, "aud", lang_s).float(), h_real.float())
            total = total + cfg.w_reread * l_rr
            logs["reread"] = l_rr.item()

        # ---- unpaired island speech
        if island is not None and bank is not None:
            if cfg.bt_weight > 0 and progress >= cfg.bt_warmup_frac:
                l_bt, bl = self.back_translate(model, island, bank)
                total = total + cfg.bt_weight * l_bt
                logs.update(bl)
            if cfg.dock_mode != "none" and progress >= cfg.dock_warmup_frac:
                l_dock, dl = self.dock(model, island, bank, progress)
                total = total + cfg.w_dock * l_dock
                logs.update(dl)
        if cfg.naive_cycle_weight > 0 and len(pairs[0]) > 1:
            t, a = views["txt"], views["aud"]
            lc = self.naive_cycle(model, "txt", t["z"][pairs[0]], t["lang"][pairs[0]], "aud", a["lang"][pairs[1]],
                                  steps=cfg.train_ode_steps, grad=True)
            total = total + cfg.naive_cycle_weight * lc
            logs["naive_cycle"] = lc.item()
        logs["loss"] = total.item()
        return total, logs

    # ============================================================================ hub-space back-translation
    def back_translate(self, model, island: Dict, bank) -> Tuple[torch.Tensor, Dict[str, float]]:
        """Back-translation of unpaired island speech, with the hub as the pivot language.

        write   real unpaired speech z_u, pseudo code h = Reader(z_u)          -> speech writer learns  h -> z_u
        read    real hub code h, synthetic island speech s = Write(h)          -> reader learns         s -> h
        relay   the same s, read back: h' = Reader(s); real text z_t of h      -> text writer learns    h' -> z_t

        The target side of every direction is real.  ``relay`` is what connects the island to the writers of
        the other languages: no caption exists in both, so without it a text writer only ever sees hub codes
        read from its own language, never ones carrying the island reader's errors.

        Pseudo codes drift towards frequent content when they are fed back on themselves, so before use they
        are moved part of the way along the entropic optimal transport map onto a batch of real hub codes
        (balanced marginals: the batch of pseudo codes has to cover the real distribution; ``bt_rectify`` is
        the displacement, 0 = off, 1 = full transport).  ``bank`` is the latent cache."""
        cfg = self.cfg
        reader, spoke = model.hub.reader, model.spokes["aud"]
        z_u, lang_u = island["z"], island["lang"]
        n, dev = z_u.shape[0], z_u.device
        logs: Dict[str, float] = {}
        text = bank.sample_views("txt", n, own_only=True) if cfg.bt_relay > 0 else None
        real = (text["hub"] if text is not None else bank.sample_hubs(n)).to(dev, z_u.dtype)
        with torch.no_grad():
            h = reader(z_u, "aud", lang_u)
            if cfg.bt_rectify > 0:
                ref = bank.sample_hubs(cfg.bt_bank_mult * n).to(dev)
                with torch.autocast(device_type=dev.type, enabled=False):
                    cost = scale_free(torch.cdist(h.float().flatten(1), ref.float().flatten(1)) ** 2)
                    a = torch.full((n,), 1.0 / n, device=dev)
                    b = torch.full((ref.shape[0],), 1.0 / ref.shape[0], device=dev)
                    Pi = log_sinkhorn(cost, a, b, cfg.bt_eps, cfg.dock_sinkhorn_iters)
                    rows = Pi / Pi.sum(1, keepdim=True).clamp_min(1e-30)
                    moved = (rows @ ref.float().flatten(1)).view_as(h)
                    logs["bt_shift"] = ((moved - h.float()) ** 2).mean().item()
                    logs["bt_plan_entropy"] = coupling_stats(Pi)["row_entropy"]
                h = (h.float() + cfg.bt_rectify * (moved - h.float())).to(z_u.dtype)
            synth = spoke.write(real, lang_u, temp=1.0, steps=cfg.train_ode_steps)
        l_write = spoke.cfm_loss(spoke.source(h), z_u, lang_u)
        back = reader(synth, "aud", lang_u)
        l_read = F.mse_loss(back.float(), real.float())
        total = l_write + l_read
        logs.update(bt_write=l_write.item(), bt_read=l_read.item())
        if text is not None:
            writer = model.spokes["txt"]
            l_relay = writer.cfm_loss(writer.source(back.detach()), text["z"].to(dev, z_u.dtype), text["lang"].to(dev))
            total = total + cfg.bt_relay * l_relay
            logs["bt_relay"] = l_relay.item()
        return total, logs

    # ============================================================================ ablation: fused-GW docking
    def alpha(self, progress: float) -> float:
        cfg = self.cfg
        if cfg.dock_mode in ("gw", "w"):
            return 0.0 if cfg.dock_mode == "gw" else 1.0
        s = min(1.0, max(0.0, (progress - cfg.dock_warmup_frac) / max(1e-6, cfg.dock_anneal_frac)))
        return cfg.dock_alpha_start + (cfg.dock_alpha_end - cfg.dock_alpha_start) * s * s * (3 - 2 * s)

    def dock(self, model, island: Dict, bank, progress: float) -> Tuple[torch.Tensor, Dict[str, float]]:
        """Entropic fused Gromov-Wasserstein between the reader's codes for unpaired island speech and real hub
        codes, as a loss on the reader (envelope theorem).  Kept as the ablation of the original proposal:
        pure GW is blind to where the island sits, and neither variant has beaten the bridge pairs alone."""
        cfg = self.cfg
        h = model.hub.reader(island["z"], "aud", island["lang"])
        Y = bank.sample_hubs(cfg.bt_bank_mult * h.shape[0]).to(h.device)
        with torch.autocast(device_type=h.device.type, enabled=False):
            mu = Y.float().mean(0, keepdim=True)
            fs, ft = (h.float() - mu).flatten(1), (Y.float() - mu).flatten(1)
            C1, C2 = scale_free(pairwise_cos_dist(fs)), scale_free(pairwise_cos_dist(ft))
            M = scale_free(pairwise_cos_dist(fs, ft))
            alpha = self.alpha(progress)
            a = torch.full((fs.shape[0],), 1.0 / fs.shape[0], device=fs.device)
            init = a.unsqueeze(1) * torch.softmax(-M.detach() / cfg.dock_eps, dim=1)  # uniform is a fixed point
            Pi = solve_fgw(C1.detach(), C2, M.detach(), alpha, cfg.dock_eps, cfg.dock_outer_iters,
                              cfg.dock_sinkhorn_iters, cfg.dock_marginal, init=init)
            value, gw, w = fgw_value(C1, C2, Pi, M, alpha)
        stats = coupling_stats(Pi)
        return value, {"dock": value.item(), "dock_alpha": alpha, "pi_row_entropy": stats["row_entropy"],
                       "pi_used_cols": stats["used_cols"]}

    # ============================================================================ diagnostic
    def naive_cycle(self, model, m1: str, z1, lang1, m2: str, lang2, steps: Optional[int] = None,
                    grad: bool = False) -> torch.Tensor:
        """The cycle of the original roadmap:  z_1 -> hub -> z_2 -> hub -> z_1  through ODE maps only.
        Every map is invertible, so the loss is zero up to solver error for *any* velocity field
        (cf. AlignFlow, DDIB).  Kept to demonstrate exactly that."""
        s1, s2 = model.spokes[m1], model.spokes[m2]
        x0 = s1.solve(z1, lang1, -1, steps=steps, grad=grad)
        same = s1.n_private == s2.n_private   # latents of different length share the hub slots only
        src = x0 if same else torch.cat([x0[:, : s1.K], x0.new_zeros(x0.shape[0], s2.n_private, x0.shape[2])], 1)
        z2 = s2.solve(src, lang2, +1, steps=steps, grad=grad)
        x0b = s2.solve(z2, lang2, -1, steps=steps, grad=grad)
        back = x0b if same else torch.cat([x0b[:, : s1.K], x0[:, s1.K:]], 1)
        z1b = s1.solve(back, lang1, +1, steps=steps, grad=grad)
        return F.mse_loss(x0b, src) + F.mse_loss(z1b, z1)


# ======================================================================================================================
# 19. LATENT CACHE (transport phase)
#     (geobridge/cache.py)
# ======================================================================================================================
# Latent cache for the transport phase.
#
# In phase B the codecs and the grounder are frozen, so the latent of every utterance and its grounded hub code
# are constants.  They are computed once here and phase B then runs on tensors only: no feature loading, no
# codec, no grounder, no image.  Rebuilding after a resume is deterministic and takes a fraction of an epoch.
# ======================================================================================================================

def time_keep(mask: torch.Tensor, share: float, gen: torch.Generator, spans: int = 2) -> torch.Tensor:
    """SpecAugment-style time masking: mask [B, T] of valid frames -> keep [B, T] in {0, 1}, with ``spans`` random
    spans of up to ``share`` of each utterance hidden.  Drawn from ``gen`` so that a rebuilt cache is the same."""
    B, T = mask.shape
    n = mask.sum(1, keepdim=True).float().cpu()
    pos = torch.arange(T).unsqueeze(0)
    keep = torch.ones(B, T)
    for _ in range(spans):
        width = torch.rand(B, 1, generator=gen) * share * n
        start = torch.rand(B, 1, generator=gen) * (n - width).clamp_min(0)
        keep = keep * ((pos < start) | (pos >= start + width)).float()
    return keep.to(mask.device)


class LatentCache:
    def __init__(self, model, data, split: str, device: torch.device, loss: GeoBridgeLoss, chunk: int = 96,
                 island: bool = True):
        cfg = model.cfg
        n_aug = cfg.read_aug if split == "train" else 0   # augmented encodings of speech, for the reader only
        z_aug: List[torch.Tensor] = []
        groups = data.groups[split]
        key = "train_views" if split == "train" else "valid_views"
        z = {"txt": [], "aud": []}
        hub = {"txt": [], "aud": []}
        ok = {"txt": [], "aud": []}
        lang = {"txt": [], "aud": []}
        count = {"txt": [0] * len(groups), "aud": [0] * len(groups)}
        byte_rows: List[torch.Tensor] = []   # byte targets of the text views (for the decoder's write-latent exposure)
        img_lat: Dict[int, torch.Tensor] = {}
        scene: Dict[int, torch.Tensor] = {}
        was_training = model.training
        model.eval()
        with torch.no_grad():
            for s in range(0, len(groups), chunk):
                part = groups[s: s + chunk]
                batch = to_device(data.collate_groups([{"img": g["img"], "cap": g["cap"], "views": g[key]} for g in part]), device)
                V = model.hub.project(batch["img"])
                lat, sc = model.image_latent(V), model.hub.scene_code(V)
                for j, g in enumerate(part):
                    if g["img"] not in img_lat:
                        img_lat[g["img"]], scene[g["img"]] = lat[j].cpu(), sc[j].cpu()
                zz = {m: model.encode(m, batch[m]) for m in ("txt", "aud") if batch[m] is not None}
                if n_aug and batch["aud"] is not None:
                    a, gen = batch["aud"], torch.Generator().manual_seed(1_000_003 * cfg.seed + s)
                    z_aug.append(torch.stack([model.audio.encode(a["sem"], a["tone"], a["mask"], a["lang"],
                                                                 keep=time_keep(a["mask"], cfg.read_aug_span, gen))
                                              for _ in range(n_aug)], 1).to(torch.float16).cpu())
                if cfg.hub_mode == "text" and batch.get("pivot") is not None:
                    zz["pivot"] = model.encode("txt", batch["pivot"])
                pairs = same_lang_pairs(batch["txt"], batch["aud"])
                hh, vv = {}, {}
                for m in ("txt", "aud"):
                    if m not in zz:
                        continue
                    sub = batch[m]
                    anchor, vv[m] = (None, torch.ones(len(sub["item"]), dtype=torch.bool, device=device))
                    if cfg.hub_mode == "text":
                        anchor, vv[m] = loss.text_anchor(zz, m, sub, pairs, batch.get("pivot"), batch["B"])
                    hh[m] = zz[m][:, : cfg.hub_slots] if cfg.hub_mode == "free" else \
                        model.paired_hub(m, zz[m], sub["lang"], V[sub["item"]], anchor)
                if cfg.hub_mode == "bary" and cfg.canonical_text and len(pairs[0]) and len(hh) == 2:
                    # canonical code of a caption: where a transcript exists (same language), speech takes the
                    # text-grounded code - the cleaner of the two views - as reader target and writer source
                    hh["aud"][pairs[1]] = hh["txt"][pairs[0]]
                for m in hh:
                    sub = batch[m]
                    z[m].append(zz[m].float().cpu()); hub[m].append(hh[m].float().cpu())
                    ok[m].append(vv[m].cpu()); lang[m].append(sub["lang"].cpu())
                    if m == "txt":
                        byte_rows.extend(sub["bytes"].cpu().to(torch.int16))
                    for j in sub["item"].tolist():  # views arrive grouped by caption, in order
                        count[m][s + j] += 1
            self.isl_z: Optional[torch.Tensor] = None
            if island and split == "train" and len(data.island_ids) >= 2:
                zs, ls = [], []
                for s in range(0, len(data.island_ids), chunk):
                    b = to_device(data.collate_island(data.island_ids[s: s + chunk]), device)
                    zs.append(model.encode("aud", b["aud"]).float().cpu()); ls.append(b["aud"]["lang"].cpu())
                self.isl_z, self.isl_lang = torch.cat(zs), torch.cat(ls)
        model.train(was_training)
        self.mods = [m for m in ("txt", "aud") if z[m]]
        self.z = {m: torch.cat(z[m]) for m in self.mods}
        self.z_aug = torch.cat(z_aug) if z_aug else None   # [speech views, n_aug, N, d]
        self.hub = {m: torch.cat(hub[m]) for m in self.mods}
        self.ok = {m: torch.cat(ok[m]) for m in self.mods}
        self.lang = {m: torch.cat(lang[m]) for m in self.mods}
        self.start = {m: torch.tensor([0] + count[m]).cumsum(0) for m in self.mods}   # rows of group g: [start[g], start[g+1])
        self.bytes = torch.nn.utils.rnn.pad_sequence(byte_rows, batch_first=True, padding_value=PAD) if byte_rows else None
        self.img_of_group = torch.tensor([g["img"] for g in groups])
        imgs = sorted(img_lat)
        self.img_row = {i: n for n, i in enumerate(imgs)}
        self.img_lat = torch.stack([img_lat[i] for i in imgs])
        self.scene = torch.stack([scene[i] for i in imgs])
        self._all_hubs = torch.cat([self.hub[m][self.ok[m]] for m in self.mods]) if self.mods else torch.zeros(0)
        self._rows = {m: self.ok[m].nonzero().squeeze(1) for m in self.mods if self.ok[m].any()}
        n_own = data.n_own_groups if split == "train" else len(groups)   # extra groups come after the own ones
        self.own_rows = {m: int(self.start[m][n_own]) for m in self.mods}
        self.n_groups = len(groups)

    def batch(self, group_idx: List[int]) -> Dict:
        """Phase-B batch for a list of caption-group indices (same layout as the collated raw batch).
        Tensors are returned on the CPU; the training replica that consumes the batch moves it to its GPU."""
        rows_img = torch.tensor([self.img_row[int(self.img_of_group[g])] for g in group_idx])
        out: Dict = {"B": len(group_idx), "img_lat": self.img_lat[rows_img], "scene": self.scene[rows_img],
                     "txt": None, "aud": None}
        for m in self.mods:
            rows, items = [], []
            for j, g in enumerate(group_idx):
                a, b = int(self.start[m][g]), int(self.start[m][g + 1])
                rows.extend(range(a, b)); items.extend([j] * (b - a))
            if rows:
                r = torch.tensor(rows)
                out[m] = {"z": self.z[m][r], "hub": self.hub[m][r], "ok": self.ok[m][r], "lang": self.lang[m][r],
                          "item": torch.tensor(items)}
                if m == "aud" and self.z_aug is not None:   # what the reader reads: the clean latent or a masked one
                    k = torch.randint(self.z_aug.shape[1] + 1, (len(r),))
                    aug = self.z_aug[r, (k - 1).clamp_min(0)].to(out[m]["z"].dtype)
                    out[m]["z_read"] = torch.where((k > 0).view(-1, 1, 1), aug, out[m]["z"])
                if m == "txt" and self.bytes is not None:
                    b = self.bytes[r].long()
                    out[m]["bytes"] = b[:, : int((b != PAD).sum(1).max())]   # trim the padding to this batch
        return out

    def island_batch(self, idx: List[int]) -> Dict:
        r = torch.tensor(idx)
        return {"z": self.isl_z[r], "lang": self.isl_lang[r]}

    def sample_hubs(self, n: int) -> torch.Tensor:
        """Real hub codes of image-paired utterances (any language): the empirical hub distribution."""
        return self._all_hubs[torch.randint(len(self._all_hubs), (n,))]

    def sample_views(self, mod: str, n: int, own_only: bool = False) -> Optional[Dict[str, torch.Tensor]]:
        """n random image-paired views of one modality: latents, their hub codes and languages.

        ``own_only``: draw from the benchmark's own captions, not from the extra hub-language ones.  A uniform
        draw over everything would be almost all hub language, and the text writer of another written language
        would hardly ever be relayed to."""
        if mod not in self._rows:
            return None
        rows = self._rows[mod]
        if own_only:
            rows = rows[rows < self.own_rows[mod]]
        r = rows[torch.randint(len(rows), (n,))]
        return {"z": self.z[mod][r], "hub": self.hub[mod][r], "lang": self.lang[mod][r]}


# ======================================================================================================================
# 20. METRICS
#     (geobridge/metrics.py)
# ======================================================================================================================
# Evaluation of every conversion direction, always through the hub and never with the paired image.
#
# Text out      BLEU / chrF++ (sacrebleu) against the parallel reference and against all captions of the image;
#               content F1 (word overlap weighted by inverse document frequency, so that "a", "in", "the" count
#               for nothing); WER / CER for same-language transcription; exact content accuracy on SynthWorld.
#               Every score comes with its CHANCE LEVEL: the same outputs scored against the reference of another
#               image.  A caption model that ignores its source still writes fluent captions and collects n-gram
#               matches; on Flickr8k that is about 1 BLEU and 15 chrF++, so a BLEU of 3 is not "almost nothing".
# Hub space     recall@k of utterance -> image (hub code explained by gallery patches), cross-lingual
#               speech -> text retrieval.
# Image out     the image latent written from a hub code, retrieved among the true image latents (recall@k).
# Speech out    feature-level round trip: hub -> speech latent -> decoded frame features and predicted pitch ->
#               re-encoded -> read -> hub -> text, scored against the reference ("self-transcribed", an intrinsic
#               proxy for ASR-BLEU that needs no external recogniser).
# Tone          learned gate per language, tone-reliance index (how far the hub code moves when the pitch
#               contour is removed), and the translation score with the contour removed.
# ======================================================================================================================

# ------------------------------------------------------------------------------------------ string metrics
def edit_distance(a: Sequence, b: Sequence) -> int:
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]


def error_rate(hyps: Sequence[str], refs: Sequence[str], unit: str = "word") -> float:
    num = den = 0
    for h, r in zip(hyps, refs):
        h, r = normalize_text(h).lower(), normalize_text(r).lower()
        hs, rs = (h.split(), r.split()) if unit == "word" else (list(h), list(r))
        num += edit_distance(hs, rs)
        den += max(1, len(rs))
    return 100.0 * num / max(1, den)


def bleu_chrf(hyps: Sequence[str], refs: Sequence[Sequence[str]]) -> Dict[str, float]:
    """refs: one list of references per hypothesis (ragged allowed)."""
    import sacrebleu

    hyps = [normalize_text(h) or "." for h in hyps]
    n_ref = max(len(r) for r in refs)
    streams = [[normalize_text(r[min(k, len(r) - 1)]) for r in refs] for k in range(n_ref)]
    bleu = sacrebleu.corpus_bleu(hyps, streams, tokenize="13a", force=True)  # captions are distributed tokenised
    chrf = sacrebleu.corpus_chrf(hyps, streams, word_order=2)
    return {"bleu": float(bleu.score), "chrf": float(chrf.score)}


def words(s: str) -> List[str]:
    """Lower-cased words of any script: letters with their combining marks (Devanagari vowel signs, Yoruba tone
    marks) and digits; punctuation is dropped."""
    return regex.findall(r"[\p{L}\p{M}\p{N}]+", normalize_text(s).lower())


def content_f1(hyps: Sequence[str], refs: Sequence[str], pool: Optional[Sequence[str]] = None) -> float:
    """Corpus-level word F1 (%) with every word weighted by its inverse document frequency in ``pool`` (default:
    the references).  Function words occur in most sentences and weigh almost nothing; a word that occurs in few
    sentences is what tells one caption from another.  Needs no stop-word list, so it is the same measure in
    every language.  A word of the output that no reference contains gets the largest weight (it is an invention)."""
    docs = [set(words(r)) for r in (pool if pool is not None else refs)]
    n = len(docs)
    df = Counter(w for d in docs for w in d)
    idf = lambda w: math.log((n + 1.0) / (df.get(w, 0) + 1.0))
    match = w_hyp = w_ref = 0.0
    for h, r in zip(hyps, refs):
        ch, cr = Counter(words(h)), Counter(words(r))
        match += sum(idf(w) * min(c, cr[w]) for w, c in ch.items() if w in cr)
        w_hyp += sum(idf(w) * c for w, c in ch.items())
        w_ref += sum(idf(w) * c for w, c in cr.items())
    if match <= 0.0:
        return 0.0
    prec, rec = match / w_hyp, match / w_ref
    return 100.0 * 2.0 * prec * rec / (prec + rec)


def other_image(imgs: Sequence[int]) -> List[int]:
    """For every item, the index of an item that belongs to another image (a fixed rotation by half the list, moved
    on while the image is the same).  Scoring outputs against these references gives the chance level."""
    n = len(imgs)
    out = []
    for i in range(n):
        j, tries = (i + n // 2) % n, 0
        while imgs[j] == imgs[i] and tries < n:
            j, tries = (j + 1) % n, tries + 1
        out.append(j)
    return out


def recall_at_k(scores: torch.Tensor, target: torch.Tensor, ks=(1, 5, 10)) -> Dict[str, float]:
    """scores [n, G]; target [n] index of the correct gallery item (ties with identical galleries allowed)."""
    rank = (scores > scores.gather(1, target.view(-1, 1))).sum(1)
    return {f"r{k}": 100.0 * float((rank < k).float().mean()) for k in ks}


def eval_settings(cfg) -> Dict:
    """The settings that change what an evaluation measures without changing the trained model."""
    return {k: getattr(cfg, k) for k in ("ode_steps", "ode_solver", "eval_temp", "read_mode", "draft_t0")}


# ------------------------------------------------------------------------------------------ evaluator
class Evaluator:
    def __init__(self, cfg, data, device: torch.device):
        self.cfg, self.data, self.device = cfg, data, device
        self.synth = cfg.dataset == "synthworld"
        self.samples: Dict[str, List[Dict]] = {}   # reference / output pairs of the last evaluation, per task

    # ---- one pass over the split: read every view into hub coordinates
    @torch.no_grad()
    def collect(self, model, split: str, max_items: int = 0, flat_tone: bool = True, steps: Optional[int] = None,
                read_mode: Optional[str] = None) -> Dict:
        data, dev = self.data, self.device
        groups = data.eval_groups(split, max_items, favour=self.cfg.island_langs)
        store: Dict = {"groups": groups, "views": {}, "img_of_group": [], "V": {}, "Z": {}, "img_hub": [], "ground": {}}
        check_ground = self.synth and self.cfg.hub_mode == "bary"
        g0 = 0
        for part, batch in data.eval_batches(groups, 64):
            batch = _to(batch, dev)
            V = model.hub.project(batch["img"])
            lat = model.image_latent(V)
            for j, g in enumerate(part):
                store["img_of_group"].append(g["img"])
                if g["img"] not in store["V"]:
                    store["V"][g["img"]], store["Z"][g["img"]] = V[j].float().cpu(), lat[j].float().cpu()
            if "img" in model.spokes:
                store["img_hub"].append(model.read("img", lat, steps=steps).float().cpu())
            for mod in ("txt", "aud"):
                sub = batch[mod]
                if sub is None:
                    continue
                z = model.encode(mod, sub)
                hub = model.read(mod, z, sub["lang"], mode=read_mode, steps=steps)
                if check_ground:  # where the grounder looks, with the image in hand (ground truth: SynthWorld)
                    attn = model.hub.attend(model.hub.queries(z, mod, sub["lang"]), V[sub["item"]])[1].float().cpu()
                    self._slot_stats(store["ground"], attn, mod, [model.langs[i] for i in sub["lang"].tolist()],
                                     sub["utt"].tolist(), [part[j]["img"] for j in sub["item"].tolist()])
                hub_flat = None
                if mod == "aud" and flat_tone:
                    sub_f = dict(sub, tone=flatten_tone(sub["tone"]))
                    hub_flat = model.read("aud", model.encode("aud", sub_f), sub["lang"], mode=read_mode, steps=steps)
                for li in sub["lang"].unique().tolist():
                    sel = (sub["lang"] == li).nonzero().squeeze(1)
                    rec = store["views"].setdefault((mod, model.langs[li]), {"g": [], "hub": [], "z": [], "utt": [],
                                                                            "hub_flat": [], "spk": [], "f0": []})
                    rec["g"].append(sub["item"][sel].cpu() + g0)
                    rec["hub"].append(hub[sel].float().cpu())
                    rec["z"].append(z[sel].float().cpu())
                    rec["utt"].append(sub["utt"][sel].cpu())
                    if hub_flat is not None:
                        rec["hub_flat"].append(hub_flat[sel].float().cpu())
                    if mod == "aud":
                        rec["spk"].append(sub["spk"][sel].cpu())
                        rec["f0"].append(sub["f0"][sel].cpu())
            g0 += len(part)
        for rec in store["views"].values():
            for k in list(rec):
                rec[k] = torch.cat(rec[k]) if len(rec[k]) else None
        store["img_hub"] = torch.cat(store["img_hub"]) if store["img_hub"] else None
        store["img_ids"] = sorted(store["V"])
        store["gallery"] = torch.stack([store["V"][i] for i in store["img_ids"]])
        store["gallery_lat"] = torch.stack([store["Z"][i] for i in store["img_ids"]])
        return store

    # ---- helpers
    def _slot_stats(self, acc: Dict, attn: torch.Tensor, mod: str, langs: List[str], utts: List[int], imgs: List[int]) -> None:
        """Count, per view type, what the slots of each caption sit on: a mentioned object, an object the
        sentence does not mention, background, or nothing (null); and whether a two-object caption has both
        of its objects covered.  attn [n, K, P]."""
        for a, lang, u, img in zip(attn, langs, utts, imgs):
            c = acc.setdefault((mod, lang), {"slots": 0, "mentioned": 0, "unmentioned": 0, "background": 0, "null": 0,
                                             "two": 0, "both": 0})
            cells = set(self.data.utts[u]["cells"])
            objects = {cell for cell, _, _ in self.data.images[img]["scene"]}
            for k in range(a.shape[0]):
                top = int(a[k].argmax())
                kind = "null" if float(a[k].sum()) < 0.5 else "mentioned" if top in cells else \
                    "unmentioned" if top in objects else "background"
                c[kind] += 1
            c["slots"] += a.shape[0]
            if len(cells) == 2:
                c["two"] += 1
                c["both"] += int(all(float(a[:, cell].max()) > 0.3 for cell in cells))

    def _ref(self, g: Dict, lang: str) -> Optional[str]:
        ui = g["utts"].get(lang)
        return self.data.utts[ui]["text"] if ui is not None and self.data.utts[ui].get("text") else None

    @torch.no_grad()
    def _gen_text(self, model, hub: torch.Tensor, lang: str, bs: int = 64, seed: int = 0) -> List[str]:
        out: List[str] = []
        gen = torch.Generator(device="cpu").manual_seed(seed)
        d, kp = self.cfg.latent_dim, self.cfg.private_slots
        for i in range(0, len(hub), bs):
            h = hub[i:i + bs].to(self.device)
            priv = torch.randn(h.shape[0], kp, d, generator=gen).to(self.device) * self.cfg.eval_temp
            if not self.cfg.private_noise:
                priv = torch.zeros_like(priv)
            lang_id = torch.full((h.shape[0],), model.lang2id[lang], dtype=torch.long, device=self.device)
            z = model.write("txt", h, lang_id, private=priv)
            out += model.text.generate(z, lang_id)
        return out

    def _score_text(self, hyps: List[str], groups: List[Dict], tgt: str, same_lang: bool) -> Tuple[Dict[str, float], List[Dict]]:
        keep = [i for i, g in enumerate(groups) if self._ref(g, tgt) is not None]
        if not keep:
            return {}, []
        hy = [hyps[i] for i in keep]
        refs1 = [[self._ref(groups[i], tgt)] for i in keep]
        flat = [r[0] for r in refs1]
        m = bleu_chrf(hy, refs1)
        out = {"bleu": m["bleu"], "chrf": m["chrf"], "cf1": content_f1(hy, flat)}
        multi = [self.data.references(groups[i]["img"], tgt) or refs1[n] for n, i in enumerate(keep)]
        out["bleu_multi"] = bleu_chrf(hy, multi)["bleu"]
        if len({groups[i]["img"] for i in keep}) > 1:   # chance level: the reference of another image
            wrong = [flat[j] for j in other_image([groups[i]["img"] for i in keep])]
            c = bleu_chrf(hy, [[r] for r in wrong])
            out.update(chance_bleu=c["bleu"], chance_chrf=c["chrf"], chance_cf1=content_f1(hy, wrong, pool=flat))
        if same_lang:
            out["wer"] = error_rate(hy, [r[0] for r in refs1], "word")
            out["cer"] = error_rate(hy, [r[0] for r in refs1], "char")
        if self.synth:
            out.update(self._content_acc(hy, [groups[i] for i in keep], tgt))
        samples = [{"ref": refs1[n][0], "hyp": hy[n], "img": groups[i]["img"], "cap": groups[i]["cap"]}
                   for n, i in enumerate(keep[:60])]
        return out, samples

    def _content_acc(self, hyps: List[str], groups: List[Dict], lang: str) -> Dict[str, float]:
        """SynthWorld only: parse the output and compare with the true content of the source utterance."""
        w = self.data.builder
        exact = col = shp = twin = n_col_err = 0
        by_len = {1: [0, 0], 2: [0, 0]}  # objects in the sentence -> [exact, count]
        for h, g in zip(hyps, groups):
            ui = next(iter(g["utts"].values()))
            truth = [tuple(c) for c in self.data.utts[ui]["content"]]
            pred = w.parse(h, lang)
            exact += int(pred == truth)
            if len(truth) in by_len:
                by_len[len(truth)][0] += int(pred == truth)
                by_len[len(truth)][1] += 1
            if pred:
                col += int(pred[0][0] == truth[0][0])
                shp += int(pred[0][1] == truth[0][1])
                if pred[0][0] != truth[0][0]:
                    n_col_err += 1
                    twin += int(pred[0][0] // 2 == truth[0][0] // 2)  # confused with its tonal twin
        n = max(1, len(hyps))
        out = {"content_acc": 100.0 * exact / n, "color_acc": 100.0 * col / n, "shape_acc": 100.0 * shp / n,
               "twin_confusion": 100.0 * twin / max(1, n_col_err)}
        out.update({f"content{k}_acc": 100.0 * e / c for k, (e, c) in by_len.items() if c})
        return out

    # ---- main entry
    @torch.no_grad()
    def evaluate(self, model, split: str = "valid", max_items: int = 0, quick: bool = False,
                 save_path: Optional[str] = None, steps: Optional[int] = None, read_mode: Optional[str] = None,
                 step: Optional[int] = None) -> Dict[str, float]:
        """``step``: training step of the evaluated state; saved with the settings that shape an evaluation, so
        that a stored file says what it measured."""
        cfg, data = self.cfg, self.data
        was_training = model.training
        model.eval()
        store = self.collect(model, split, max_items, flat_tone=True, steps=steps, read_mode=read_mode)
        groups, views = store["groups"], store["views"]
        M: Dict[str, float] = {}
        samples: Dict[str, List[Dict]] = {}
        hub_langs = [l for l in cfg.hub_langs if ("txt", l) in views]
        island = [l for l in cfg.island_langs if ("aud", l) in views]
        written = list(hub_langs) + [l for l in cfg.text_only_langs if ("txt", l) in views and l not in hub_langs]
        if cfg.use_island_text:
            written += [l for l in island if ("txt", l) in views]

        # ---------------- text generation
        tasks: List[Tuple[str, str, str]] = []
        for H in hub_langs:
            tasks += [("aud", L, H) for L in island]
            if ("aud", H) in views:
                tasks.append(("aud", H, H))
            if store["img_hub"] is not None:
                tasks.append(("img", "-", H))
            tasks.append(("txt", H, H))  # read a sentence and write it again: what the hub code keeps
            if not quick:
                for T in written:
                    if T != H:
                        tasks += [("txt", H, T), ("txt", T, H)]
                        tasks += [("aud", L, T) for L in island if not cfg.use_island_text or T != L]
        seen = set()
        for mod, src, tgt in tasks:
            if (mod, src, tgt) in seen:
                continue
            seen.add((mod, src, tgt))
            if mod == "img":
                hub, gi = store["img_hub"], torch.arange(len(groups))
            else:
                hub, gi = views[(mod, src)]["hub"], views[(mod, src)]["g"]
            hyps = self._gen_text(model, hub, tgt)
            name = f"{mod}_{src}>txt_{tgt}".replace("_->", ">")
            m, s = self._score_text(hyps, [groups[i] for i in gi.tolist()], tgt, same_lang=(src == tgt))
            M.update({f"{name}/{k}": v for k, v in m.items()})
            samples[name] = s
            if mod == "aud" and views[(mod, src)].get("hub_flat") is not None and (not quick or src in island):
                hy_f = self._gen_text(model, views[(mod, src)]["hub_flat"], tgt)
                mf, _ = self._score_text(hy_f, [groups[i] for i in gi.tolist()], tgt, same_lang=(src == tgt))
                M.update({f"{name}/flat_{k}": v for k, v in mf.items() if k in ("bleu", "cf1", "content_acc", "color_acc")})

        # ---------------- hub-space retrieval
        img_index = {i: n for n, i in enumerate(store["img_ids"])}
        gal = store["gallery"].to(self.device)
        for (mod, lang), rec in views.items():
            if mod == "txt" and lang in cfg.island_langs and not cfg.use_island_text:
                continue  # the island orthography is held out: the model was never shown it
            if quick and not (lang in island or (mod == "txt" and lang in hub_langs)):
                continue
            target = torch.tensor([img_index[store["img_of_group"][g]] for g in rec["g"].tolist()])
            sc = torch.cat([model.hub.hub_to_image_scores(rec["hub"][i:i + 32].to(self.device), gal).cpu()
                            for i in range(0, len(rec["hub"]), 32)])
            M.update({f"{mod}_{lang}>img/{k}": v for k, v in recall_at_k(sc, target).items()})
        if "img" in model.spokes:  # the image-writing direction: generated image latent vs the true ones
            lat = F.normalize(store["gallery_lat"].flatten(1), dim=-1)
            gen = torch.Generator().manual_seed(0)
            for (mod, lang), rec in views.items():
                if (mod == "txt" and lang in cfg.island_langs and not cfg.use_island_text) or \
                        (quick and not (lang in island or (mod == "txt" and lang in hub_langs))):
                    continue
                target = torch.tensor([img_index[store["img_of_group"][g]] for g in rec["g"].tolist()])
                outs = []
                for i in range(0, len(rec["hub"]), 64):
                    h = rec["hub"][i:i + 64].to(self.device)
                    priv = torch.randn(h.shape[0], cfg.private_slots, cfg.latent_dim, generator=gen).to(self.device) * cfg.eval_temp
                    outs.append(model.write("img", h, private=priv if cfg.private_noise else torch.zeros_like(priv)).float().cpu())
                sc = F.normalize(torch.cat(outs).flatten(1), dim=-1) @ lat.t()
                M.update({f"{mod}_{lang}>img_gen/{k}": v for k, v in recall_at_k(sc, target).items()})
        for L in island:
            for H in hub_langs:
                a, t = views[("aud", L)], views[("txt", H)]
                pos = {g: n for n, g in enumerate(t["g"].tolist())}
                keep = [n for n, g in enumerate(a["g"].tolist()) if g in pos]
                if len(keep) > 1:
                    sim = F.normalize(a["hub"][keep].flatten(1), dim=-1) @ F.normalize(t["hub"].flatten(1), dim=-1).t()
                    tgt_idx = torch.tensor([pos[a["g"][n].item()] for n in keep])
                    M.update({f"aud_{L}>txt_{H}/hub_{k}": v for k, v in recall_at_k(sim, tgt_idx, (1, 5)).items()})

        # ---------------- speech generation (feature-level round trip, scored as text in the hub language)
        if not quick and hub_langs:
            H = hub_langs[0]
            for S in [l for l in list(hub_langs) + island if ("aud", l) in views]:
                for mod, src in ([("txt", H)] + ([("aud", L) for L in island if L != S] if S == H else [("aud", H)])):
                    if (mod, src) not in views:
                        continue
                    name = f"{mod}_{src}>aud_{S}"
                    hy, gi = self._speech_roundtrip(model, views[(mod, src)], views[("aud", S)], S, H)
                    m, s = self._score_text(hy, [groups[i] for i in gi], H, same_lang=False)
                    M.update({f"{name}/rt_{k}": v for k, v in m.items()
                              if k in ("bleu", "chrf", "cf1", "chance_cf1", "content_acc", "color_acc")})
                    samples[name + " (round trip)"] = s

        # ---------------- codecs: text encoded and decoded with no hub in between.  Scored during training too:
        # the decoder keeps training in the transport phase (write-latent exposure) and must not lose the exact
        # latents over it, and whatever passes through the hub cannot be more exact than this.
        for lang in (written if not quick else [l for l in written if l in hub_langs]):
            rec = views[("txt", lang)]
            z = rec["z"][: 100 if quick else 200].to(self.device)
            lid = torch.full((z.shape[0],), model.lang2id[lang], dtype=torch.long, device=self.device)
            hy = model.text.generate(z, lid)
            refs = [data.utts[u]["text"] for u in rec["utt"][: len(z)].tolist()]
            M[f"codec_txt_{lang}/cer"] = error_rate(hy, refs, "char")

        # ---------------- grounding (controlled benchmark): what the hub slots sit on
        for (mod, lang), c in store["ground"].items():
            if mod == "txt" and lang in cfg.island_langs and not cfg.use_island_text:
                continue  # the island's orthography is never trained on
            key = f"ground/{mod}_{lang}"
            M[f"{key}/slots_mentioned"] = 100.0 * c["mentioned"] / max(1, c["slots"])
            M[f"{key}/slots_unmentioned"] = 100.0 * c["unmentioned"] / max(1, c["slots"])
            M[f"{key}/slots_null"] = 100.0 * c["null"] / max(1, c["slots"])
            if c["two"]:
                M[f"{key}/both_covered"] = 100.0 * c["both"] / c["two"]

        # ---------------- tone
        gates = model.audio.gate().detach().cpu()
        for name, i in model.lang2id.items():
            if ("aud", name) in views:
                M[f"tone/gate_{name}"] = float(gates[i])
                rec = views[("aud", name)]
                if rec.get("hub_flat") is not None and len(rec["hub"]) > 2:
                    d_flat = ((rec["hub"] - rec["hub_flat"]) ** 2).mean()
                    d_ref = ((rec["hub"] - rec["hub"].roll(1, 0)) ** 2).mean()
                    M[f"tone/reliance_{name}"] = float(d_flat / d_ref.clamp_min(1e-8))

        # ---------------- selection score: content above chance, averaged over the text directions
        # (The island direction alone was too noisy to select on: with 67 of its utterances in the validation
        # subset its BLEU moved by more than a point from one validation to the next, and the first one won.)
        lifts = [M[k] - M.get(k[:-len("cf1")] + "chance_cf1", 0.0) for k in M
                 if k.endswith("/cf1") and ">txt_" in k and "/flat_" not in k and "/rt_" not in k]
        M["score"] = float(np.mean(lifts)) if lifts else 0.0
        self.samples = samples
        if save_path:
            save_json({"meta": {"step": step, "split": split, "items": max_items, "quick": quick, **eval_settings(cfg)},
                       "metrics": M, "samples": samples}, save_path)
        model.train(was_training)
        return M

    @torch.no_grad()
    def _speech_roundtrip(self, model, src: Dict, tgt_ref: Dict, S: str, H: str, max_n: int = 256):
        """hub(src) -> speech latent in language S -> frame features + predicted pitch -> codec encoder ->
        speech spoke -> hub -> text in H.  Returns hypotheses and the group index of each source item."""
        cfg, dev = self.cfg, self.device
        n = min(len(src["hub"]), max_n)
        hyps: List[str] = []
        gen = torch.Generator().manual_seed(0)
        for i in range(0, n, 64):
            hub = src["hub"][i:i + 64].to(dev)
            b = hub.shape[0]
            lid = torch.full((b,), model.lang2id[S], dtype=torch.long, device=dev)
            pick = torch.randint(len(tgt_ref["spk"]), (b,), generator=gen)  # a real voice of language S
            spk, f0 = tgt_ref["spk"][pick].to(dev), tgt_ref["f0"][pick].to(dev)
            priv = torch.randn(b, cfg.private_of("aud"), cfg.latent_dim, generator=gen).to(dev) * cfg.eval_temp
            z = model.write("aud", hub, lid, private=priv if cfg.private_noise else torch.zeros_like(priv))
            n_frames = model.audio.predict_frames(z)
            p = model.audio.bypass_code(spk, f0)
            out = model.audio.decode(z, lid, n_frames, p)
            voiced = (torch.sigmoid(out["pros"][..., 1]) > 0.5).float()
            zf0 = out["pros"][..., 0] * voiced
            dz = torch.zeros_like(zf0)
            dz[:, 1:-1] = (zf0[:, 2:] - zf0[:, :-2]) / 2
            tone = torch.stack([zf0, voiced, dz * voiced, out["pros"][..., 2]], -1) * out["mask"].unsqueeze(-1)
            z2 = model.audio.encode(out["sem"], tone, out["mask"], lid, sem_is_normed=True)
            hub2 = model.read("aud", z2, lid)
            hyps += self._gen_text(model, hub2.cpu(), H)
        return hyps, src["g"][:n].tolist()


def _to(batch, device):
    if isinstance(batch, torch.Tensor):
        return batch.to(device)
    if isinstance(batch, dict):
        return {k: _to(v, device) for k, v in batch.items()}
    return batch


# ======================================================================================================================
# 21. TRAINER
#     (geobridge/trainer.py)
# ======================================================================================================================
# Trainer: two-phase schedule in one resumable run.
#
#     phase A  (steps [0, phase_a_steps))      codecs, grounder, explainer, reader   -> representation
#     phase B  (steps [phase_a_steps, total))  writers (flows), reader, back-translation, on cached latents
#
# Checkpointing
#     checkpoints/step_XXXXXXXX.pt   every ``save_every`` steps (last ``keep_last`` kept, milestones kept forever)
#     checkpoints/last.pt            most recent state; training resumes from it automatically
#     checkpoints/phase_a.pt         end of phase A (shared by phase-B ablations)
#     checkpoints/best.pt            best validation score in phase B
# A checkpoint stores model, EMA, both optimisers, AMP scaler, RNG states, the full history and the exact
# position inside the epoch (epoch index + batch offset of both data streams).  The epoch order is a pure
# function of (seed, epoch), so resuming from any step of any epoch continues the same data stream.
#
# Several GPUs
#     One optimisation step consumes one batch per GPU (``cfg.step_batches``): every GPU runs the complete
#     forward pass of its own batch (codecs, grounder, losses) and the gradients are averaged, i.e. the step sees
#     ``n_gpu x batch_size`` caption groups.  See :class:`StepReplica`.
# ======================================================================================================================

TQDM_KEYS_A = ("loss", "txt_ce", "txt_acc", "sem_mse", "speak_mse", "mel_l1", "ground", "ground_acc", "unexplained", "read", "seek_txt", "pix_l1", "agree")
TQDM_KEYS_B = ("loss", "cfm_txt", "cfm_aud", "cfm_img", "read_txt", "read_aud", "dec_ce", "dec_acc", "reread", "bt_write", "bt_read", "bt_relay", "bt_shift", "dock")
# Settings that may differ between a checkpoint and the run that resumes from it (everything else must match).
RESUME_FREE = {"exp_name", "out_root", "data_root", "device", "amp", "num_workers", "torch_threads", "n_gpu", "step_batches",
               "feat_batch_size", "log_every", "eval_every", "save_every", "keep_last", "milestone_every", "eval_max_items",
               "phase_b_steps", "init_from", "train_only", "ode_steps", "ode_solver", "sample_temp", "eval_temp", "read_mode",
               "draft_t0", "asr_model"}


class ResumableBatchSampler(Sampler):
    """Batches of a per-epoch permutation that depends only on (seed, epoch); can start mid-epoch.

    ``extra_pool`` (indices of the bridge pairs) contributes ``n_extra`` items to every batch, drawn with a
    generator seeded by (seed, epoch, batch), so the stream is still a pure function of its position.

    ``aux``: pools of extra hub-language caption groups, each as (indices, items per batch).  A pool contributes
    its items to every batch ON TOP of the main ones.  It is walked through in its own shuffled order, without
    replacement, wrapping around on its own schedule: batch number g = epoch * len(self) + b takes positions
    [g * n, (g + 1) * n) of that pool's endless stream.  Still a pure function of (seed, epoch, batch).
    (``aux_pool`` / ``n_aux``: one such pool.)"""

    def __init__(self, n: int, batch_size: int, seed: int, extra_pool: Optional[List[int]] = None, n_extra: int = 0,
                 aux_pool: Optional[Sequence[int]] = None, n_aux: int = 0,
                 aux: Sequence[Tuple[Sequence[int], int]] = ()):
        self.n, self.seed = n, seed
        self.pool = list(extra_pool or [])
        self.k = min(n_extra, batch_size // 2) if self.pool else 0
        self.bs = batch_size - self.k
        pools = list(aux) + ([(aux_pool, n_aux)] if aux_pool is not None else [])
        self.aux = [(np.asarray(p, dtype=np.int64), min(int(k), len(p))) for p, k in pools if len(p) and k > 0]
        self.n_aux = sum(k for _, k in self.aux)
        self._aux_perm: Dict[int, Tuple[int, np.ndarray]] = {}
        self.epoch, self.offset = 0, 0

    def set_state(self, epoch: int, offset: int) -> None:
        self.epoch, self.offset = epoch, offset

    def __len__(self) -> int:
        return self.n // self.bs

    def _aux_items(self, j: int, g: int) -> List[int]:
        pool, k = self.aux[j]
        out: List[int] = []
        pos, need, m = g * k, k, len(pool)
        while need:
            lap, at = divmod(pos, m)
            if self._aux_perm.get(j, (-1, None))[0] != lap:   # one permutation per lap of the pool, kept until it is over
                self._aux_perm[j] = (lap, np.random.default_rng([self.seed, 104729 + j, lap]).permutation(m))
            take = min(need, m - at)
            out += pool[self._aux_perm[j][1][at: at + take]].tolist()
            pos, need = pos + take, need - take
        return out

    def __iter__(self) -> Iterator[List[int]]:
        perm = np.random.default_rng(self.seed * 100003 + self.epoch).permutation(self.n)
        for b in range(self.offset, len(self)):
            batch = perm[b * self.bs: (b + 1) * self.bs].tolist()
            if self.k:
                rng = np.random.default_rng([self.seed, self.epoch, b])
                batch += [self.pool[i] for i in rng.integers(len(self.pool), size=self.k)]
            for j in range(len(self.aux)):
                batch += self._aux_items(j, self.epoch * len(self) + b)
            yield batch


class EMA:
    def __init__(self, params: Dict[str, torch.nn.Parameter], decay: float):
        self.decay = decay
        self.shadow = {k: p.detach().clone() for k, p in params.items()}
        self.backup: Dict[str, torch.Tensor] = {}

    @torch.no_grad()
    def update(self, params: Dict[str, torch.nn.Parameter], step: int) -> None:
        d = min(self.decay, (1 + step) / (10 + step))
        for k, p in params.items():
            self.shadow[k].mul_(d).add_(p.detach(), alpha=1 - d)

    @torch.no_grad()
    def swap_in(self, params: Dict[str, torch.nn.Parameter]) -> None:
        self.backup = {k: p.detach().clone() for k, p in params.items()}
        for k, p in params.items():
            p.copy_(self.shadow[k])

    @torch.no_grad()
    def swap_out(self, params: Dict[str, torch.nn.Parameter]) -> None:
        for k, p in params.items():
            p.copy_(self.backup[k])
        self.backup = {}


def _plain(v):
    """Tuples and lists compare equal (a config read back from JSON holds lists)."""
    return list(v) if isinstance(v, (tuple, list)) else v


def settings_diff(old: Dict, new: Dict, free: Iterable[str] = RESUME_FREE) -> Dict[str, Tuple]:
    """Fields in which two configurations (as dicts) describe different runs: {field: (old, new)}.
    Fields absent from ``old`` (written by an earlier version of the code) count with their default."""
    old = {**Config().to_dict(), **old}
    return {k: (old.get(k), new[k]) for k in new if k not in free and _plain(old.get(k)) != _plain(new[k])}


class StepReplica(nn.Module):
    """The forward half of one optimisation step, in a form ``nn.DataParallel`` can run on every GPU.

    DataParallel scatters tensors, but a batch here is a nested dict of ragged sub-batches (and, in phase B,
    slices of the latent cache).  So one batch is prepared per GPU and the only thing that is scattered is an
    index: the replica that receives ``[i]`` takes the i-th prepared batch, moves it to its own device and
    computes the whole loss there, under its own autocast context.  Losses are gathered into a vector; logs are
    left in a slot of ``job``, which all replicas share by reference."""

    def __init__(self, model: GeoBridge, loss: GeoBridgeLoss, amp: bool):
        super().__init__()
        self.model = model
        self.loss, self.amp = loss, amp
        self.job: Dict = {}

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        i, dev, job = int(idx[0]), idx.device, self.job
        batch, island = to_device(job["batches"][i], dev), to_device(job["islands"][i], dev)
        with torch.autocast(device_type=dev.type, enabled=self.amp):
            if job["phase"] == "A":
                loss, logs = self.loss.phase_a(self.model, batch, island)
            else:
                loss, logs = self.loss.phase_b(self.model, batch, island, job["progress"], job["bank"])
        job["logs"][i] = logs
        return loss.reshape(1)


def chunked(it: Iterable, n: int) -> Iterator[List]:
    """Consecutive lists of up to n items."""
    buf: List = []
    for x in it:
        buf.append(x)
        if len(buf) == n:
            yield buf
            buf = []
    if buf:
        yield buf


class Trainer:
    def __init__(self, cfg: Config, data: Optional[GeoBridgeData] = None, resume: bool = True):
        self.cfg = cfg
        self.run_dir = cfg.run_dir
        self.ckpt_dir = os.path.join(self.run_dir, "checkpoints")
        os.makedirs(self.ckpt_dir, exist_ok=True)
        os.makedirs(os.path.join(self.run_dir, "curves"), exist_ok=True)
        self.log = get_logger(f"gb.{cfg.exp_name}", os.path.join(self.run_dir, "train.log"))
        if cfg.torch_threads:
            torch.set_num_threads(cfg.torch_threads)
        seed_everything(cfg.seed)
        self.device = resolve_device(cfg.device)
        self.data = data or GeoBridgeData(cfg, self.log, self.device)
        self.spec = build_spec(self.data)
        self.model = GeoBridge(cfg, self.spec).to(self.device)
        self.loss = GeoBridgeLoss(cfg, self.model.lang2id)
        self.evaluator = Evaluator(cfg, self.data, self.device)
        self.amp = bool(cfg.amp and self.device.type == "cuda")
        self.scaler = torch.amp.GradScaler("cuda", enabled=self.amp)
        self.gpus = gpu_ids(cfg, self.device)
        self.replica = StepReplica(self.model, self.loss, self.amp)
        self.parallel = nn.DataParallel(self.replica, device_ids=self.gpus) if len(self.gpus) > 1 else None
        self.step_batches = cfg.step_batches or max(1, len(self.gpus))

        a_params = [p for m in self.model.phase_a_modules() for p in m.parameters()]
        b_named = {f"{i}.{n}": p for i, m in enumerate(self.model.phase_b_modules()) for n, p in m.named_parameters()}
        self.b_named = b_named
        no_decay = lambda n, p: p.dim() < 2 or "lora_" in n or "emb" in n
        gate = [self.model.audio.gate_logit]
        self.opt_a = torch.optim.AdamW(
            [{"params": [p for p in a_params if p is not gate[0]], "weight_decay": cfg.weight_decay},
             {"params": gate, "weight_decay": 0.0, "fixed_lr": cfg.lr_gate}], lr=cfg.lr, betas=(0.9, 0.98))
        self.opt_b = torch.optim.AdamW(
            [{"params": [p for n, p in b_named.items() if not no_decay(n, p)], "weight_decay": cfg.weight_decay},
             {"params": [p for n, p in b_named.items() if no_decay(n, p)], "weight_decay": 0.0}],
            lr=cfg.lr_flow, betas=(0.9, 0.98))
        self.ema = EMA(b_named, cfg.ema_decay)

        self.train_ds = _GroupDataset(self.data, self.data.groups["train"], train=True)
        bridge_idx = [i for i, g in enumerate(self.data.groups["train"])
                      if any(l in cfg.island_langs for _, l, _ in g["train_views"])]
        # the benchmark's own caption groups make the epoch; extra image-text groups are added to every batch
        # extra spoken captions are what speech learns from: they get their own share of the extra places
        n_own, groups = self.data.n_own_groups, self.data.groups["train"]
        spoken = [i for i in range(n_own, len(groups)) if any(v[0] == "aud" for v in groups[i]["train_views"])]
        written = sorted(set(range(n_own, len(groups))) - set(spoken))
        n_x = cfg.extra_per_batch or cfg.batch_size
        n_sp = n_x if not written else int(round(cfg.extra_speech_share * n_x)) if spoken else 0
        self.sampler = ResumableBatchSampler(n_own, cfg.batch_size, cfg.seed, bridge_idx, cfg.bridge_per_batch,
                                             aux=[(written, n_x - n_sp), (spoken, n_sp)])
        self.extra_spoken_per_batch = n_sp if spoken and written else 0
        self.isl_sampler = None
        if len(self.data.island_ids) >= 2:
            bs = min(cfg.island_batch_size, len(self.data.island_ids))
            self.isl_sampler = ResumableBatchSampler(len(self.data.island_ids), bs, cfg.seed + 7)
        self.step, self.epoch, self.offset = 0, 0, 0
        self.isl_epoch, self.isl_offset = 0, 0
        self.cache: Dict[str, LatentCache] = {}
        self.history: Dict[str, List[Dict]] = {"train": [], "valid": []}
        self.best = {"score": -1e9, "step": -1}
        self.best_a = {"value": float("inf"), "step": -1}   # phase-A state with the best validation hub losses
        self._describe()
        if resume and not self._resume() and cfg.init_from:   # a resume under other settings raises here
            self._init_from(cfg.init_from)
        cfg.save(os.path.join(self.run_dir, "config.json"))
        save_json(self.data.describe(), os.path.join(self.run_dir, "data.json"))
        self._apply_train_only()

    # ------------------------------------------------------------------ bookkeeping
    def _describe(self) -> None:
        m, cfg = self.model, self.cfg
        total = count_params(m)
        lora = count_named(m.spokes, ["lora_"])
        n_lang = max(1, len(m.langs))
        per_lang = lora / n_lang + sum(count_params(s.vel.lang_emb) for s in m.spokes.values()) / n_lang
        spokes = count_params(m.spokes)
        self.param_report = {"total": total, "text_codec": count_params(m.text), "audio_codec": count_params(m.audio),
                             "hub": count_params(m.hub), "reader": count_params(m.hub.reader),
                             "explainer": count_params(m.hub.explainer), "spokes": spokes,
                             "per_language_flow_params": per_lang,
                             "per_language_pct_of_spokes": 100.0 * per_lang / max(1, spokes - lora)}
        self.log.info(f"[model] {total / 1e6:.2f}M params | text {self.param_report['text_codec'] / 1e6:.2f}M | "
                      f"audio {self.param_report['audio_codec'] / 1e6:.2f}M | hub {self.param_report['hub'] / 1e6:.2f}M | "
                      f"spokes {spokes / 1e6:.2f}M | per-language adapter {per_lang / 1e3:.1f}k "
                      f"({self.param_report['per_language_pct_of_spokes']:.2f}% of the shared flow backbone)")
        d = self.data.describe()
        self.log.info(f"[data] groups {d['caption_groups']} | bridge pairs {d['bridge_pairs_train']} | "
                      f"unpaired island {d['island_unpaired_train']} | device {self.device}"
                      + (f" x{len(self.gpus)} (DataParallel)" if len(self.gpus) > 1 else "")
                      + f" | {self.step_batches} batch(es) of {cfg.batch_size} per step | amp {self.amp}")
        if d["extra"].get("images"):
            self.log.info(f"[data] extra (training only): {d['extra']['images']} images, {d['extra']['caption_groups']} "
                          f"{cfg.hub_langs[0]} captions, {d['extra']['spoken']} of them also spoken; "
                          f"{self.sampler.n_aux} extra captions join every batch"
                          + (f" ({self.extra_spoken_per_batch} spoken)" if self.extra_spoken_per_batch else "")
                          + ("; extra speech listens only (it is read into the hub and does not shape it)"
                             if d["extra"]["spoken"] and cfg.listen_only else ""))
        save_json(self.param_report, os.path.join(self.run_dir, "params.json"))

    @property
    def phase(self) -> str:
        return "A" if self.step < self.cfg.phase_a_steps else "B"

    def _lr(self, step_in_phase: int, phase_len: int, base: float) -> float:
        cfg = self.cfg
        warm = min(1.0, (step_in_phase + 1) / max(1, cfg.warmup_steps))
        prog = min(1.0, step_in_phase / max(1, phase_len))
        return base * warm * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * prog)))

    def _set_phase_mode(self) -> None:
        m = self.model
        if self.phase == "A":
            m.train()
            for mod in m.phase_a_modules():
                mod.requires_grad_(True)
        else:
            for mod in m.phase_a_modules():
                mod.eval()
                mod.requires_grad_(False)
            self._ensure_cache()
            for mod in m.phase_b_modules():
                mod.train()
                mod.requires_grad_(True)
        self._apply_train_only()

    def _ensure_cache(self) -> None:
        """Phase B runs on cached latents and hub codes: the frozen codecs and the grounder are run exactly
        once more, here.  Deterministic, so it is simply rebuilt after a resume."""
        if self.cache:
            return
        t0 = time.time()
        self._set_schedules()
        self.cache = {"train": LatentCache(self.model, self.data, "train", self.device, self.loss),
                      "valid": LatentCache(self.model, self.data, "valid", self.device, self.loss, island=False)}
        c = self.cache["train"]
        self.log.info(f"[cache] latents and hub codes of {sum(len(v) for v in c.z.values())} paired views"
                      f"{'' if c.isl_z is None else f' and {len(c.isl_z)} unpaired island utterances'} "
                      f"cached in {time.time() - t0:.1f}s; phase B no longer runs the codecs or the grounder")

    def _apply_train_only(self) -> None:
        """``train_only="lora:<lang>"``: adding a language later - only that language's LoRA rows and
        language embeddings receive updates (the shared field is frozen, so nothing can be forgotten)."""
        spec = self.cfg.train_only
        if not spec.startswith("lora:"):
            return
        lang = self.model.lang2id[spec.split(":")[1]]
        for n, p in self.model.named_parameters():
            keep = "spokes" in n and ("lora_" in n or "lang_emb" in n)
            p.requires_grad_(keep and self.phase == "B")
            if keep and p.requires_grad and not hasattr(p, "_gb_hook"):
                mask = torch.zeros(p.shape[0], *([1] * (p.dim() - 1)), device=p.device)
                mask[lang] = 1.0
                p.register_hook(lambda g, mask=mask: g * mask)
                p._gb_hook = True

    # ------------------------------------------------------------------ checkpoints
    def _state(self) -> Dict:
        return {"model": self.model.state_dict(), "ema": self.ema.shadow, "opt_a": self.opt_a.state_dict(),
                "opt_b": self.opt_b.state_dict(), "scaler": self.scaler.state_dict(), "step": self.step,
                "epoch": self.epoch, "offset": self.offset, "isl_epoch": self.isl_epoch,
                "isl_offset": self.isl_offset, "history": self.history, "best": self.best, "best_a": self.best_a,
                "cfg": self.cfg.to_dict(), "spec": spec_for_checkpoint(self.spec),
                "rng": {"torch": torch.get_rng_state(), "numpy": np.random.get_state(), "python": random.getstate()}}

    def save_checkpoint(self, name: Optional[str] = None) -> str:
        path = os.path.join(self.ckpt_dir, name or f"step_{self.step:08d}.pt")
        tmp = path + ".tmp"
        torch.save(self._state(), tmp)
        os.replace(tmp, path)
        if name is None:
            shutil.copyfile(path, os.path.join(self.ckpt_dir, "last.pt"))
            steps = sorted(glob.glob(os.path.join(self.ckpt_dir, "step_*.pt")))
            removable = [p for p in steps[:-self.cfg.keep_last]
                         if int(os.path.basename(p)[5:13]) % max(1, self.cfg.milestone_every)]
            for p in removable:
                os.remove(p)
        return path

    def _check_same_run(self, ck: Dict, path: str) -> None:
        """Resuming continues a run; it must not silently continue it under different settings."""
        diff = settings_diff(ck.get("cfg", {}), self.cfg.to_dict())
        if diff:
            shown = ", ".join(f"{k}: {a} -> {b}" for k, (a, b) in list(diff.items())[:8])
            raise RuntimeError(
                f"{path} was written by a run with different settings ({shown}{', ...' if len(diff) > 8 else ''}). "
                f"Resuming would mix the two.  Use another exp_name for the new settings, or delete {self.run_dir}.")

    def load_checkpoint(self, path: str, weights_only: bool = False) -> Dict:
        ck = torch.load(path, map_location=self.device, weights_only=False)
        if not weights_only:
            self._check_same_run(ck, path)
        own = self.model.state_dict()
        state = {k: v for k, v in ck["model"].items() if k not in own or own[k].shape == v.shape}
        skipped = [k for k in ck["model"] if k not in state]  # e.g. a different LoRA rank in an ablation
        missing, unexpected = self.model.load_state_dict(state, strict=False)
        if missing or unexpected or skipped:
            self.log.info(f"[ckpt] non-strict load: {len(missing)} missing, {len(unexpected)} unexpected, "
                          f"{len(skipped)} shape-mismatched keys")
        if weights_only:
            return ck
        for k, v in ck["ema"].items():
            if k in self.ema.shadow and self.ema.shadow[k].shape == v.shape:
                self.ema.shadow[k] = v.to(self.device)
        self.opt_a.load_state_dict(ck["opt_a"])
        self.opt_b.load_state_dict(ck["opt_b"])
        self.scaler.load_state_dict(ck["scaler"])
        self.step, self.epoch, self.offset = ck["step"], ck["epoch"], ck["offset"]
        self.isl_epoch, self.isl_offset = ck.get("isl_epoch", 0), ck.get("isl_offset", 0)
        self.history, self.best = ck["history"], ck["best"]
        self.best_a = ck.get("best_a", self.best_a)
        try:
            torch.set_rng_state(ck["rng"]["torch"])
            np.random.set_state(ck["rng"]["numpy"])
            random.setstate(ck["rng"]["python"])
        except Exception:
            pass
        return ck

    def _resume(self) -> bool:
        last = os.path.join(self.ckpt_dir, "last.pt")
        cands = [last] if os.path.exists(last) else sorted(glob.glob(os.path.join(self.ckpt_dir, "step_*.pt")))[-1:]
        if not cands:
            return False
        self.load_checkpoint(cands[0])
        self.log.info(f"[resume] {cands[0]} -> step {self.step} (epoch {self.epoch}, batch {self.offset}, phase {self.phase})")
        return True

    def resume_from(self, path: str) -> None:
        """Resume from an explicit checkpoint (any step of any epoch)."""
        self.load_checkpoint(path)
        self.log.info(f"[resume] {path} -> step {self.step} (epoch {self.epoch}, batch {self.offset})")

    def _init_from(self, path: str) -> None:
        ck = self.load_checkpoint(path, weights_only=True)
        if ck["step"] >= ck["cfg"].get("phase_a_steps", 0) and not self.cfg.train_only:
            self.step = self.cfg.phase_a_steps  # representation phase already done: go straight to phase B
            self.history["train"] = [h for h in ck["history"]["train"] if h["step"] <= self.step]
            self.history["valid"] = [h for h in ck["history"]["valid"] if h["step"] <= self.step]
        elif self.cfg.train_only:
            self.step = self.cfg.phase_a_steps
            for k, v in ck["ema"].items():
                if k in self.ema.shadow and self.ema.shadow[k].shape == v.shape:
                    self.ema.shadow[k] = v.to(self.device)
        self.log.info(f"[init] weights from {path}; starting at step {self.step} (phase {self.phase})")

    # ------------------------------------------------------------------ data streams
    def _paired_loader(self):
        self.sampler.set_state(self.epoch, self.offset)
        if self.phase == "B":  # cached latents: a batch is assembled by indexing tensors, no loader needed
            return (self.cache["train"].batch(idx) for idx in self.sampler)
        return DataLoader(self.train_ds, batch_sampler=self.sampler, collate_fn=self.data.collate_groups,
                          pin_memory=self.device.type == "cuda", **worker_kwargs(self.cfg.num_workers))

    def _island_stream(self) -> Optional[Iterator[Dict]]:
        if self.isl_sampler is None:
            return None

        def gen():
            ds = _IslandDataset(self.data.island_ids)
            while True:
                phase = self.phase
                self.isl_sampler.set_state(self.isl_epoch, self.isl_offset)
                if phase == "B":
                    it = (self.cache["train"].island_batch(idx) for idx in self.isl_sampler)
                else:
                    it = iter(DataLoader(ds, batch_sampler=self.isl_sampler, num_workers=0,
                                         collate_fn=self.data.collate_island))
                finished = True
                for b in it:
                    self.isl_offset += 1
                    yield b
                    if self.phase != phase:  # the representation phase ended: switch to cached batches
                        finished = False
                        break
                if finished:
                    self.isl_epoch, self.isl_offset = self.isl_epoch + 1, 0

        return gen()

    def _set_schedules(self) -> None:
        """Step-dependent switches of phase A ("structure first, sparsity later").  Pure functions of the step,
        so they need no state in the checkpoint: dense stage = gates held open, no null slot, soft attention
        annealed to sharp; afterwards gates and slots are released and priced."""
        cfg, m = self.cfg, self.model
        release = cfg.gate_release_frac * cfg.phase_a_steps
        dense = self.step < release
        m.audio.gate_forced_open, m.hub.null_enabled = dense, not dense
        frac = min(1.0, self.step / max(1.0, release))
        m.hub.temp_scale = cfg.hub_temp_start ** (1.0 - frac)  # geometric anneal: start -> 1

    # ------------------------------------------------------------------ one optimisation step
    def train_step(self, batches: List[Dict], islands: List[Optional[Dict]]) -> Dict[str, float]:
        """One optimiser update from ``len(batches)`` batches: one per GPU, run in parallel when there are
        several GPUs (otherwise one after the other, which is plain gradient accumulation)."""
        cfg = self.cfg
        if self.phase == "A":
            opt, lr = self.opt_a, self._lr(self.step, cfg.phase_a_steps, cfg.lr)
        else:
            opt, lr = self.opt_b, self._lr(self.step - cfg.phase_a_steps, cfg.phase_b_steps, cfg.lr_flow)
        for g in opt.param_groups:
            g["lr"] = g.get("fixed_lr", lr)   # the scalar tonality gates keep a constant, much larger step
        self._set_schedules()
        opt.zero_grad(set_to_none=True)
        n = len(batches)
        self.replica.job = {"batches": batches, "islands": islands, "logs": [None] * n, "phase": self.phase,
                            "progress": (self.step - cfg.phase_a_steps) / max(1, cfg.phase_b_steps),
                            "bank": self.cache.get("train")}
        if self.parallel is not None and n > 1:
            losses = self.parallel(torch.arange(n, device=self.device))
        else:
            losses = torch.cat([self.replica(torch.tensor([i], device=self.device)) for i in range(n)])
        loss = losses.mean()
        logs = Averager()
        for l in self.replica.job["logs"]:
            logs.add(l)
        logs = logs.mean()
        self.replica.job = {}
        if not torch.isfinite(loss):
            self.log.info(f"[warn] non-finite loss at step {self.step}; step skipped")
            return logs
        self.scaler.scale(loss).backward()
        self.scaler.unscale_(opt)
        params = [p for g in opt.param_groups for p in g["params"] if p.grad is not None]
        logs["grad_norm"] = float(torch.nn.utils.clip_grad_norm_(params, cfg.grad_clip))
        self.scaler.step(opt)
        self.scaler.update()
        if self.phase == "B":
            self.ema.update(self.b_named, self.step - cfg.phase_a_steps)
        logs["lr"] = lr
        return logs

    # ------------------------------------------------------------------ validation
    @torch.no_grad()
    def validate(self, split: str = "valid", max_batches: int = 20, metrics: bool = True) -> Dict[str, float]:
        cfg, m = self.cfg, self.model
        was_training = m.training
        m.eval()
        self._set_schedules()
        use_ema = self.phase == "B" and self.step > cfg.phase_a_steps
        if use_ema:
            self.ema.swap_in(self.b_named)
        avg = Averager()
        if self.phase == "A":
            loader = self.data.paired_loader(split, shuffle=False, batch_size=cfg.batch_size)
            n_batches = len(loader)
        else:
            c = self.cache[split]
            loader = (c.batch(list(range(i, min(i + cfg.batch_size, c.n_groups)))) for i in range(0, c.n_groups, cfg.batch_size))
            n_batches = math.ceil(c.n_groups / cfg.batch_size)
        gen_state = torch.get_rng_state()
        torch.manual_seed(1234)  # same noise / time draws at every validation -> comparable curves
        n_val = min(n_batches, max_batches)
        # a nested bar that refreshes prints one line per refresh in notebook logs: draw it at the start and at the end
        pbar = tqdm(loader, total=n_val, desc=f"valid [{self.phase}]", leave=False, dynamic_ncols=True, mininterval=3600)
        for i, batch in enumerate(pbar):
            if i >= max_batches:
                break
            batch = to_device(batch, self.device)
            with torch.autocast(device_type=self.device.type, enabled=self.amp):
                if self.phase == "A":
                    _, logs = self.loss.phase_a(m, batch, None)
                else:
                    _, logs = self.loss.phase_b(m, batch, None, 1.0)
            avg.add(logs)
            if i == n_val - 1:
                pbar.set_postfix({k: f"{v:.3f}" for k, v in avg.mean().items() if k in TQDM_KEYS_A + TQDM_KEYS_B}, refresh=False)
        out = avg.mean()
        if metrics and self.phase == "B":
            ms = self.evaluator.evaluate(m, split, cfg.eval_max_items, quick=True)
            out.update(ms)
        torch.set_rng_state(gen_state)
        if use_ema:
            self.ema.swap_out(self.b_named)
        m.train(was_training)
        self._set_phase_mode()
        return out

    # ------------------------------------------------------------------ main loop
    def fit(self) -> Dict[str, float]:
        cfg = self.cfg
        total = cfg.total_steps
        if self.step >= total:
            self.log.info(f"[fit] run already complete ({self.step}/{total} steps)")
            return self.history["valid"][-1] if self.history["valid"] else {}
        island = self._island_stream()
        avg, t0 = Averager(), time.time()
        last_val: Dict[str, float] = {}
        while self.step < total:
            self._set_phase_mode()
            loader = self._paired_loader()
            n_ep, k = len(self.sampler), self.step_batches
            pbar = tqdm(chunked(loader, k), initial=self.offset // k, total=math.ceil(n_ep / k), dynamic_ncols=True,
                        mininterval=0.5, desc=f"epoch {self.epoch} | phase {self.phase} | step {self.step}/{total}")
            waited = time.time()
            for batches in pbar:
                phase_before = self.phase
                got = time.time()
                try:
                    logs = self.train_step(batches, [next(island) if island is not None else None for _ in batches])
                except RuntimeError as e:   # out of GPU memory: say what to change instead of a bare CUDA error
                    if "out of memory" not in str(e).lower():
                        raise
                    raise RuntimeError(
                        f"out of GPU memory at step {self.step} (phase {self.phase}): a batch is {cfg.batch_size} caption "
                        f"groups" + (f" + {self.sampler.n_aux} extra caption groups" if self.sampler.n_aux else "")
                        + " per GPU.  Lower 'extra_per_batch' (e.g. half) or 'batch_size', and use a new exp_name: "
                        "the feature caches are kept, only training restarts.") from e
                logs.update(t_data=got - waited, t_step=time.time() - got)   # seconds waiting for batches / computing
                self.step += 1
                self.offset += len(batches)
                avg.add(logs)
                keys = TQDM_KEYS_A if phase_before == "A" else TQDM_KEYS_B
                mean = avg.mean()
                pbar.set_postfix({k: f"{mean[k]:.3f}" for k in keys if k in mean}, refresh=False)
                pbar.set_description(f"epoch {self.epoch} | phase {phase_before} | step {self.step}/{total}", refresh=False)
                boundary = self.step == cfg.phase_a_steps
                if self.step % cfg.log_every == 0 or boundary or self.step == total:
                    self.history["train"].append({"step": self.step, "epoch": self.epoch, "phase": phase_before,
                                                  "time": time.time() - t0, **avg.mean()})
                    avg.reset()
                if self.step % cfg.eval_every == 0 or boundary or self.step == total:
                    if boundary:  # evaluate the end of phase A with phase-A losses
                        self.step -= 1
                        last_val = self.validate()
                        self.step += 1
                    else:
                        last_val = self.validate()
                    self.history["valid"].append({"step": self.step, "epoch": self.epoch, "phase": phase_before, **last_val})
                    self._log_validation(last_val, phase_before)
                    self.plot_curves()
                    save_json(self.history, os.path.join(self.run_dir, "history.json"))
                    if phase_before == "B" and last_val.get("score", -1e9) > self.best["score"]:
                        self.best = {"score": last_val["score"], "step": self.step}
                        self.save_checkpoint("best.pt")
                    if phase_before == "A":
                        self._track_phase_a(last_val)
                if boundary:
                    self._restore_best_phase_a()
                    self.save_checkpoint("phase_a.pt")
                    self.log.info(f"[phase] representation phase finished at step {self.step}; codecs and grounder "
                                  f"are now frozen, training the writers and the reader")
                saved = self.step % cfg.save_every == 0 or boundary or self.step == total
                if saved:
                    self.save_checkpoint()
                if time_left() <= 0 and self.step < total:   # out of session time: keep this exact step, stop
                    pbar.close()
                    if not saved:
                        self.save_checkpoint()
                    save_json(self.history, os.path.join(self.run_dir, "history.json"))
                    raise OutOfTime(f"time budget used up at step {self.step}/{total} of '{cfg.exp_name}' "
                                    f"(phase {self.phase}); the step is saved, run again to continue from it")
                if boundary or self.step >= total:
                    break
                waited = time.time()
            pbar.close()
            if self.offset >= n_ep:
                self.epoch, self.offset = self.epoch + 1, 0
        if not os.path.exists(os.path.join(self.ckpt_dir, "best.pt")):
            self.save_checkpoint("best.pt")
        self.save_checkpoint()
        save_json(self.history, os.path.join(self.run_dir, "history.json"))
        self.plot_curves()
        self.log.info(f"[fit] done: {self.step} steps, " + (
            f"best score {self.best['score']:.3f} at step {self.best['step']}" if self.best["step"] >= 0
            else "representation phase only (no transport phase, hence no score)"))
        return last_val

    # ------------------------------------------------------------------ phase-A model selection
    def _phase_a_best_path(self) -> str:
        return os.path.join(self.ckpt_dir, "phase_a_best.pt")

    def _track_phase_a(self, val: Dict[str, float]) -> None:
        """Remember the phase-A state with the best validation hub losses.  Only states after the release are
        comparable (and usable: the null option and minimality exist from then on)."""
        cfg = self.cfg
        keys = ("ground", "unexplained", "read")
        if not cfg.phase_a_select or self.step < cfg.gate_release_frac * cfg.phase_a_steps or any(k not in val for k in keys):
            return
        value = float(sum(val[k] for k in keys))
        if value < self.best_a["value"]:
            self.best_a = {"value": value, "step": self.step}
            tmp = self._phase_a_best_path() + ".tmp"
            torch.save({"model": self.model.state_dict(), "step": self.step, "value": value}, tmp)
            os.replace(tmp, self._phase_a_best_path())

    def _restore_best_phase_a(self) -> None:
        """At the end of phase A: continue from the best validated state rather than from the last one."""
        if not self.cfg.phase_a_select or self.best_a["step"] in (-1, self.step) or not os.path.exists(self._phase_a_best_path()):
            return
        spoken = [l for l in self.cfg.all_langs if l not in self.cfg.text_only_langs]
        gates = lambda: {l: float(self.model.audio.gate()[self.model.lang2id[l]]) for l in spoken}
        at_end = gates()
        ck = torch.load(self._phase_a_best_path(), map_location=self.device, weights_only=False)
        self.model.load_state_dict(ck["model"])
        self.log.info(f"[phase] the transport phase starts from the phase-A state of step {ck['step']} "
                      f"(validation grounding + sufficiency + read = {ck['value']:.3f}, the best after the release); "
                      f"later states fit the training set better and the validation set worse")
        now = gates()
        moving = [l for l in spoken if abs(now[l] - at_end[l]) > 0.05]
        self.log.info("[phase] tonality gates at that state: " + ", ".join(f"{l} {now[l]:.2f}" for l in spoken)
                      + "; at the end of the phase: " + ", ".join(f"{l} {at_end[l]:.2f}" for l in spoken)
                      + (f".  Still moving ({', '.join(moving)}): the gate values of this run are a snapshot, not a result"
                         if moving else ""))

    def _log_validation(self, val: Dict[str, float], phase: str) -> None:
        keys = [k for k in val if k in TQDM_KEYS_A + TQDM_KEYS_B or k.endswith(("/r1", "content_acc"))
                or k.startswith(("tone/", "codec_")) or k == "score"]
        msg = " | ".join(f"{k} {val[k]:.3f}" for k in keys[:40])
        self.log.info(f"[valid] step {self.step} phase {phase} | {msg}")
        for k in [k for k in val if k.endswith("/cf1") and "/flat_" not in k and "/rt_" not in k]:
            d = k[:-len("/cf1")]   # one line per text direction: every score next to its chance level
            parts = [f"{lab} {val[f'{d}/{m}']:.1f} (chance {val[f'{d}/chance_{m}']:.1f})"
                     for m, lab in (("cf1", "content F1"), ("bleu", "BLEU"), ("chrf", "chrF++")) if f"{d}/chance_{m}" in val]
            parts += [f"{lab} {val[f'{d}/{m}']:.1f}" for m, lab in (("bleu_multi", "BLEU multi-ref"), ("wer", "WER")) if f"{d}/{m}" in val]
            self.log.info(f"[text] {d} | " + " | ".join(parts))
        recent = [h for h in self.history["train"][-max(1, self.cfg.eval_every // max(1, self.cfg.log_every)):] if "t_step" in h]
        if recent:
            data, step = np.mean([h["t_data"] for h in recent]), np.mean([h["t_step"] for h in recent])
            self.log.info(f"[speed] {1.0 / max(1e-9, data + step):.2f} steps/s | waiting for batches {data:.3f}s, "
                          f"computing {step:.3f}s per step" + (" (the data loader is the bottleneck)" if data > step else ""))
        if phase == "B":   # what the outputs actually look like: two examples per direction
            H = self.cfg.hub_langs[0]
            for name in [f"aud_{L}>txt_{H}" for L in self.cfg.island_langs] + [f"aud_{H}>txt_{H}", f"txt_{H}>txt_{H}"]:
                for s in self.evaluator.samples.get(name, [])[:2]:
                    self.log.info(f"[sample] {name} | ref: {s['ref'][:90]} | out: {s['hyp'][:90]}")

    # ------------------------------------------------------------------ curves
    def plot_curves(self) -> None:

        setup_matplotlib()
        import matplotlib.pyplot as plt

        tr, va = self.history["train"], self.history["valid"]
        if not tr:
            return
        out = os.path.join(self.run_dir, "curves")
        pa = self.cfg.phase_a_steps

        def series(hist, key, phase=None):
            pts = [(h["step"], h[key]) for h in hist if key in h and (phase is None or h.get("phase") == phase)]
            return ([p[0] for p in pts], [p[1] for p in pts]) if pts else ([], [])

        panels = [("loss", "A", "phase A total loss"), ("txt_ce", "A", "text codec CE"), ("sem_mse", "A", "speech codec MSE"),
                  ("ground", "A", "grounding InfoNCE"), ("unexplained", "A", "latent variance NOT explained by the hub"),
                  ("read", "A", "reader error on the hub code (no image)"),
                  ("loss", "B", "phase B total loss"), ("cfm_txt", "B", "CFM text writer"), ("cfm_aud", "B", "CFM speech writer"),
                  ("cfm_img", "B", "CFM image writer"), ("read_aud", "B", "reader error, speech"),
                  ("bt_read", "B", "back-translation: read synthetic speech (train only)")]
        fig, axes = plt.subplots(3, 4, figsize=(19, 11))
        for ax, (key, ph, title) in zip(axes.ravel(), panels):
            xs, ys = series(tr, key, ph)
            xv, yv = series(va, key, ph)
            if xs:
                ax.plot(xs, ys, color="#1f77b4", lw=1.4, label="train")
            if xv:
                ax.plot(xv, yv, color="#d62728", lw=1.6, marker="o", ms=3.5, label="valid")
            if key == "sem_mse":   # the speaking path, when there is one: what the decoder that speaks reconstructs
                xs2, ys2 = series(tr, "speak_mse", ph)
                xv2, yv2 = series(va, "speak_mse", ph)
                if xs2:
                    ax.plot(xs2, ys2, color="#1f77b4", lw=1.2, ls="--", label="train, speaking path")
                    title = "speech codec MSE (solid: content tokens alone)"
                if xv2:
                    ax.plot(xv2, yv2, color="#d62728", lw=1.4, ls="--", marker="s", ms=3, label="valid, speaking path")
            ax.set_title(title, fontsize=10)
            ax.set_xlabel("step")
            ax.grid(alpha=0.25)
            if xs or xv:
                ax.legend(fontsize=8)
        fig.suptitle(f"{self.cfg.exp_name}: train vs valid losses (phase A = steps < {pa}, phase B after)", fontsize=12)
        fig.tight_layout()
        fig.savefig(os.path.join(out, "losses_train_vs_valid.png"), dpi=130)
        plt.close(fig)

        mkeys = sorted({k for h in va for k in h if (k.endswith(("/cf1", "/chance_cf1", "/r1", "content_acc")) or k == "score")
                        and "/flat_" not in k and "/rt_" not in k})
        tone = sorted({k for h in tr for k in h if k.startswith("gate_")})
        fig, axes = plt.subplots(1, 3, figsize=(19, 5))
        for k in mkeys[:16]:
            xs, ys = series(va, k)
            chance = k.endswith("/chance_cf1")
            axes[0].plot(xs, ys, marker=None if chance else "o", ms=3, ls=":" if chance else "-", lw=1.0 if chance else 1.5,
                         color="0.55" if chance else None, label=None if chance else k)
        axes[0].set_title("validation metrics through the hub (dotted grey: chance level of content F1)")
        if axes[0].get_legend_handles_labels()[0]:
            axes[0].legend(fontsize=6.5, ncol=2)
        for k in tone:
            xs, ys = series(tr, k)
            axes[1].plot(xs, ys, label=k.replace("gate_", "gate "))
        axes[1].axvline(pa, color="k", ls=":", lw=1)
        axes[1].set_title("lexical-tonality gate per language (training)")
        axes[1].set_ylim(-0.02, 1.02)
        if tone:
            axes[1].legend(fontsize=8)
        for k, lab in (("bt_write", "write: real speech, pseudo code"), ("bt_read", "read: synthetic speech, real code"),
                       ("bt_shift", "transport shift of the pseudo codes"), ("slot_use_txt", "slot use (text)")):
            xs, ys = series(tr, k)
            if xs:
                axes[2].plot(xs, ys, label=lab)
        axes[2].set_title("hub-space back-translation and slot use")
        if axes[2].lines:
            axes[2].legend(fontsize=8)
        for ax in axes:
            ax.set_xlabel("step")
            ax.grid(alpha=0.25)
        fig.tight_layout()
        fig.savefig(os.path.join(out, "metrics_gates_backtranslation.png"), dpi=130)
        plt.close(fig)


# ======================================================================================================================
# 22. INFERENCER (figures, internal views, saved outputs)
#     (geobridge/inferencer.py)
# ======================================================================================================================
# Inferencer: any-to-any conversion with a trained model, plus everything needed to look inside it.
#
# ``Inferencer(cfg).run_all()`` writes to ``<run_dir>/inference/``:
#
#   outputs/translations.md|json     every text-producing direction, with references
#   outputs/speech/*.wav + *.png     synthesised speech (pretrained HiFi-GAN or Griffin-Lim) with mel and pitch contour
#   outputs/private_diversity.json   one hub code, several private codes; hub interpolation
#   01_slot_attention.png            where each hub slot looks, for text / speech / island speech of one caption
#   02_flow_trajectories.png         ODE paths hub <-> modality (PCA), straightness, speed profile
#   03_hub_manifold.png              all languages and modalities in hub coordinates (PCA + t-SNE)
#   04_hub_alignment.png             island speech vs hub-language text in hub space: transport plans against the true partners
#   05_tone.png                      learned tonality gates, tone-reliance, pitch-scaling response
#   06_speech_attribution.png        which frames (segmental vs tone stream) drive which hub slot
#   07_text_attribution.png          which graphemes drive which hub slot
#   08_retrieval.png                 island-speech -> image similarity matrix and top-k galleries
#   09_ode_steps.png                 quality vs number of ODE steps; the plain ODE cycle is solver error
#   10_fertility.png                 units per word / per grapheme for each script and text unit
#   11_hub_agreement.png             same-caption vs different-caption hub distance across modalities
#   12_language_lora.png             how far each language moves each layer of the shared velocity fields
#   13_hub_vs_private.png            what the hub tokens and the private tokens of a speech read-out encode
#   14_codec_reconstruction.png      speech codec: natural vs reconstructed streams and rendered mel
#   15_generated_images.png          preview images written from text, speech and island speech, next to the true image
#   16_read_write_modes.png          reader vs backward-ODE reading; writing from the explainer's draft (t0 sweep)
#   report.json                      every number behind the figures
# ======================================================================================================================

LANG_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


def load_model(cfg: Config, data: GeoBridgeData, device: torch.device, checkpoint: Optional[str] = None,
               use_ema: bool = True) -> Tuple[GeoBridge, Dict]:
    path = checkpoint or os.path.join(cfg.run_dir, "checkpoints", "best.pt")
    if not os.path.exists(path):
        path = os.path.join(cfg.run_dir, "checkpoints", "last.pt")
    ck = torch.load(path, map_location=device, weights_only=False)
    model = GeoBridge(cfg, build_spec(data)).to(device)
    own = model.state_dict()
    model.load_state_dict({k: v for k, v in ck["model"].items() if k in own and own[k].shape == v.shape}, strict=False)
    if use_ema and ck.get("ema") and ck["step"] > cfg.phase_a_steps:
        named = {f"{i}.{n}": p for i, m in enumerate(model.phase_b_modules()) for n, p in m.named_parameters()}
        with torch.no_grad():
            for k, p in named.items():
                if k in ck["ema"] and ck["ema"][k].shape == p.shape:
                    p.copy_(ck["ema"][k].to(device))
    model.eval()
    return model, ck


class Inferencer:
    def __init__(self, cfg: Config, checkpoint: Optional[str] = None, data: Optional[GeoBridgeData] = None):
        self.cfg = cfg
        self.device = resolve_device(cfg.device)
        self.log = get_logger("gb.infer")
        self.data = data or GeoBridgeData(cfg, self.log, self.device)
        self.model, self.ck = load_model(cfg, self.data, self.device, checkpoint)
        self.evaluator = Evaluator(cfg, self.data, self.device)
        self.out = os.path.join(cfg.run_dir, "inference")
        os.makedirs(os.path.join(self.out, "outputs", "speech"), exist_ok=True)
        self.report: Dict = {"checkpoint_step": int(self.ck["step"])}
        self.hub_lang = cfg.hub_langs[0]
        self.island = [l for l in cfg.island_langs]
        seed_everything(0)

    # ================================================================== public conversion API
    @torch.no_grad()
    def text_to_hub(self, texts: List[str], lang: str) -> torch.Tensor:
        sub = self.data.unitizer.collate(texts)
        sub["lang"] = torch.full((len(texts),), self.model.lang2id[lang], dtype=torch.long)
        sub = _to(sub, self.device)
        return self.model.read("txt", self.model.encode("txt", sub), sub["lang"])

    @torch.no_grad()
    def translate_text(self, texts: List[str], src: str, tgt: str, temp: Optional[float] = None) -> List[str]:
        return self.model.hub_to_text(self.text_to_hub(texts, src), tgt, temp=temp)

    @torch.no_grad()
    def utterances_to_hub(self, utt_ids: List[int]) -> Tuple[torch.Tensor, Dict]:
        sub = _to(self.data._collate_audio(utt_ids, list(range(len(utt_ids)))), self.device)
        return self.model.read("aud", self.model.encode("aud", sub), sub["lang"]), sub

    @torch.no_grad()
    def speech_to_text(self, utt_ids: List[int], tgt: str) -> List[str]:
        return self.model.hub_to_text(self.utterances_to_hub(utt_ids)[0], tgt)

    @torch.no_grad()
    def hub_to_wave(self, hub: torch.Tensor, lang: str, ref_utt: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """hub [1, K, d] -> (waveform, log-mel, z-scored F0 contour) in the voice of ``ref_utt``."""
        ref = _to(self.data._collate_audio([ref_utt], [0]), self.device)
        mel, n_frames, _, streams = self.model.hub_to_speech(hub, lang, ref["spk"], ref["f0"])
        m = mel[0, : int(streams["mel_len"][0])].float().cpu()
        return self.vocode(m), m.numpy(), streams["tone3"][0, : int(n_frames[0]), 0].float().cpu().numpy()

    @torch.no_grad()
    def vocode(self, mel: torch.Tensor) -> np.ndarray:
        """log-mel [T, n_mels] -> waveform, with the vocoder that owns the mel feature space."""
        if self.cfg.mel_spec == "speecht5":
            if not hasattr(self, "_vocoder"):
                from transformers import SpeechT5HifiGan

                with quiet_transformers():   # the hub config says "hifigan", the class "speecht5_hifigan"
                    self._vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").eval()
            return self._vocoder(mel.float()).cpu().numpy()
        return griffin_lim(mel, self.cfg.sample_rate, 1024, self.cfg.mel_hop).numpy()

    def evaluate(self, split: str = "test", max_items: int = 0) -> Dict[str, float]:
        path = os.path.join(self.cfg.run_dir, f"eval_{split}.json")
        m = self.evaluator.evaluate(self.model, split, max_items, quick=False, save_path=path, step=int(self.ck["step"]))
        self.log.info(f"[eval] {split}, state of step {self.ck['step']}: " + " | ".join(
            f"{k} {v:.2f}" for k, v in m.items() if k.endswith(("/r1", "content_acc")) or k == "score"))
        for k in [k for k in m if k.endswith("/cf1") and "/flat_" not in k and "/rt_" not in k]:
            d = k[:-len("/cf1")]
            self.log.info(f"[eval] {d}: content F1 {m[k]:.1f} (chance {m.get(d + '/chance_cf1', float('nan')):.1f}) | BLEU "
                          f"{m[d + '/bleu']:.2f} (chance {m.get(d + '/chance_bleu', float('nan')):.2f}) | chrF++ {m[d + '/chrf']:.1f} "
                          f"(chance {m.get(d + '/chance_chrf', float('nan')):.1f})")
        return m

    # ================================================================== everything
    def run_all(self, split: str = "test") -> Dict:

        setup_matplotlib(os.path.join(self.cfg.data_root, "fonts"))
        self.split = split
        self.groups = self.data.eval_groups(split, 400, favour=self.cfg.island_langs)
        self.store = self.evaluator.collect(self.model, split, 400)
        steps = [self.save_outputs, self.speech_asr, self.fig_slot_attention, self.fig_flow_trajectories, self.fig_hub_manifold,
                 self.fig_hub_alignment, self.fig_tone, self.fig_speech_attribution, self.fig_text_attribution,
                 self.fig_retrieval, self.fig_ode_steps, self.fig_fertility, self.fig_hub_agreement,
                 self.fig_language_lora, self.fig_hub_vs_private, self.fig_codec_reconstruction,
                 self.fig_generated_images, self.fig_read_write_modes]
        for fn in steps:
            try:
                fn()
                self.log.info(f"[infer] {fn.__name__} done")
            except Exception as e:  # one failed figure must not lose the others
                self.log.info(f"[infer] {fn.__name__} FAILED: {type(e).__name__}: {e}")
                self.report.setdefault("failed", []).append(f"{fn.__name__}: {e}")
        save_json(self.report, os.path.join(self.out, "report.json"))
        self.log.info(f"[infer] everything written to {self.out}")
        return self.report

    # ------------------------------------------------------------------ small helpers
    def _view_utt(self, g: Dict, mod: str, lang: str) -> Optional[int]:
        for m, l, ui in g["all_views"]:
            if m == mod and l == lang:
                return ui
        return None

    def _image(self, img_idx: int, size: int = 224):
        b = self.data.builder
        if hasattr(b, "render"):
            return b.render(img_idx, size)
        return b.load_image(self.data.images[img_idx]["id"]).resize((size, size))

    def _attn_grid(self, attn: np.ndarray) -> np.ndarray:
        """attention over the patch tokens -> [grid, grid] map."""
        return attn.reshape(self.model.hub.grid)

    def _examples(self, n: int, need: List[Tuple[str, str]]) -> List[Dict]:
        """Up to n caption groups that have all the needed views, from n different images (and, where the
        benchmark has ground truth, the longest captions first, so multi-object sentences are shown)."""
        gs = [g for g in self.groups if all(self._view_utt(g, m, l) is not None for m, l in need)]
        gs = sorted(gs, key=lambda g: -len(self.evaluator._ref(g, self.hub_lang) or ""))
        out, seen = [], set()
        for g in gs[::max(1, len(gs) // (4 * n) or 1)] + gs:
            if g["img"] not in seen:
                out.append(g)
                seen.add(g["img"])
            if len(out) == n:
                break
        return out

    @torch.no_grad()
    def _ground(self, g: Dict, mod: str, lang: str):
        """Grounder attention (needs the image) and the reader's hub code (does not) for one view."""
        ui = self._view_utt(g, mod, lang)
        batch = _to(self.data.collate_groups([{"img": g["img"], "cap": g["cap"], "views": [(mod, lang, ui)]}]), self.device)
        sub = batch[mod]
        V = self.model.hub.project(batch["img"])
        z = self.model.encode(mod, sub)
        _, attn = self.model.hub.attend(self.model.hub.queries(z, mod, sub["lang"]), V)
        hub_read = self.model.read(mod, z, sub["lang"])
        implied = torch.softmax(torch.einsum("nkd,npd->nkp", F.normalize(hub_read.float(), dim=-1),
                                             F.normalize(V[:, :-1].float(), dim=-1)) / 0.05, dim=-1)
        weight = hub_read.float().norm(dim=-1)[0]  # an empty (null) slot has a code near the origin
        implied = (implied[0] * (weight / weight.sum().clamp_min(1e-6)).unsqueeze(-1)).sum(0)
        return attn[0].float().cpu().numpy(), implied.cpu().numpy(), hub_read, V

    # ================================================================== saved outputs
    @torch.no_grad()
    def save_outputs(self) -> None:
        cfg, data, model, H = self.cfg, self.data, self.model, self.hub_lang
        rows, md = [], ["# GeoBridge-Flow: sample conversions (all through the hub; no image at test time)\n"]
        views = self.store["views"]
        directions = [("aud", L, H) for L in self.island if ("aud", L) in views] + [("aud", H, H), ("img", "-", H)]
        for T in cfg.text_only_langs:
            if ("txt", T) in views:
                directions += [("txt", H, T), ("txt", T, H)]
        for mod, src, tgt in directions:
            if mod == "img":
                hub, gi = self.store["img_hub"][:12], list(range(12))
            elif (mod, src) in views:
                hub, gi = views[(mod, src)]["hub"][:12], views[(mod, src)]["g"][:12].tolist()
            else:
                continue
            hyps = self.evaluator._gen_text(model, hub, tgt)
            md.append(f"\n## {mod}:{src} -> text:{tgt}\n\n| source | reference | output |\n|---|---|---|")
            for h, g_i in zip(hyps, gi):
                g = self.store["groups"][g_i]
                src_txt = self.evaluator._ref(g, src) if src != "-" else f"image {data.images[g['img']]['id']}"
                ref = self.evaluator._ref(g, tgt) or ""
                rows.append({"direction": f"{mod}:{src}>{tgt}", "source": src_txt, "reference": ref, "output": h})
                md.append(f"| {src_txt} | {ref} | {h} |")
        save_json(rows, os.path.join(self.out, "outputs", "translations.json"))
        with open(os.path.join(self.out, "outputs", "translations.md"), "w", encoding="utf8") as f:
            f.write("\n".join(md) + "\n")

        # ---- one hub code, many private codes; and a walk between two hub codes
        div = []
        if ("txt", H) in views:
            for i in range(min(4, len(views[("txt", H)]["hub"]))):
                hub = views[("txt", H)]["hub"][i:i + 1].to(self.device)
                g = self.store["groups"][views[("txt", H)]["g"][i].item()]
                outs = {f"temp={t}": [model.hub_to_text(hub, H, temp=t)[0] for _ in range(3)] for t in (0.0, 0.5, 1.0)}
                div.append({"source": self.evaluator._ref(g, H), "samples": outs})
            a, b = views[("txt", H)]["hub"][0:1].to(self.device), views[("txt", H)]["hub"][1:2].to(self.device)
            walk = [model.hub_to_text((1 - w) * a + w * b, H, temp=0.0)[0] for w in np.linspace(0, 1, 6)]
            div.append({"hub_interpolation": walk})
        save_json(div, os.path.join(self.out, "outputs", "private_diversity.json"))

        # ---- speech synthesis: text -> speech (hub and island language), island speech -> hub-language speech
        import matplotlib.pyplot as plt

        synth = []
        ex = self._examples(3, [("txt", H), ("aud", H)])
        for n, g in enumerate(ex):
            hub = self.text_to_hub([self.evaluator._ref(g, H)], H)
            jobs = [(H, self._view_utt(g, "aud", H), f"tts_{H}_{n}")]
            for L in self.island:
                ref_u = next((self._view_utt(x, "aud", L) for x in self.groups if self._view_utt(x, "aud", L) is not None), None)
                if ref_u is not None:
                    jobs.append((L, ref_u, f"tts_{L}_{n}"))
                    if self._view_utt(g, "aud", L) is not None:
                        hub_l, _ = self.utterances_to_hub([self._view_utt(g, "aud", L)])
                        jobs.append((H, self._view_utt(g, "aud", H), f"s2st_{L}2{H}_{n}", hub_l))
            for job in jobs:
                lang, ref_u, name = job[0], job[1], job[2]
                wav, mel, f0 = self.hub_to_wave(job[3] if len(job) > 3 else hub, lang, ref_u)
                _write_wav(os.path.join(self.out, "outputs", "speech", name + ".wav"), wav, cfg.sample_rate)
                fig, ax = plt.subplots(2, 1, figsize=(8, 4.2), sharex=False)
                ax[0].imshow(mel.T, origin="lower", aspect="auto", cmap="magma")
                ax[0].set_title(f"{name}: generated log-mel  |  text: {self.evaluator._ref(g, H)}", fontsize=8)
                ax[1].plot(f0, color="#d62728")
                ax[1].set_title("predicted speaker-normalised log-F0 (0 = unvoiced)", fontsize=8)
                fig.tight_layout()
                fig.savefig(os.path.join(self.out, "outputs", "speech", name + ".png"), dpi=110)
                plt.close(fig)
                synth.append({"file": name + ".wav", "lang": lang, "text": self.evaluator._ref(g, H), "seconds": len(wav) / cfg.sample_rate})
        self.report["synthesised_speech"] = synth

    # ================================================================== written speech, heard by a recogniser
    @torch.no_grad()
    def speech_asr(self, n: int = 60) -> None:
        """Is the speech this model writes intelligible to a recogniser that was not trained with it?

        Every rung is vocoded, transcribed by an off-the-shelf English recogniser and scored against the caption:
            vocoder   the real utterance's own log-mel                               (ceiling of mel + vocoder)
            renderer  the real utterance's own streams -> rendered log-mel           (the renderer, no latent)
            codec     real speech -> speech latent -> decoded streams -> log-mel     (what the speech codec keeps)
            content   the same, decoded from the content tokens alone                (only with a speaking path)
            written   caption text -> hub -> written speech latent -> log-mel        (the model's text-to-speech)
            island    island speech -> hub -> written hub-language speech            (speech-to-speech translation)
        The evaluator's round-trip score reads written speech with the model's own reader, which says how well the
        model agrees with itself.  This does not depend on the model."""
        cfg, H = self.cfg, self.hub_lang
        if not cfg.asr_model or cfg.mel_spec != "speecht5" or H != "en":
            self.report["speech_asr"] = "skipped: needs asr_model, the SpeechT5 mel space and English as hub language"
            return
        from transformers import pipeline


        with quiet_transformers():
            asr = pipeline("automatic-speech-recognition", model=cfg.asr_model, device=0 if self.device.type == "cuda" else -1)
        hear = lambda wavs: [o["text"].strip() for o in asr([{"raw": np.asarray(w, np.float32), "sampling_rate": cfg.sample_rate}
                                                              for w in wavs], batch_size=16)]
        ex = self._examples(n, [("txt", H), ("aud", H)])
        refs = [self.evaluator._ref(g, H) for g in ex]
        utts = [self._view_utt(g, "aud", H) for g in ex]
        wavs: Dict[str, List[np.ndarray]] = {"vocoder": [], "renderer": [], "codec": [], "written": []}
        for g, ref, ui in zip(ex, refs, utts):
            sub = _to(self.data._collate_audio([ui], [0]), self.device)
            wavs["vocoder"].append(self.vocode(sub["mel"][0, : int(sub["mel_len"][0])].float().cpu()))
            wavs["renderer"].append(self.vocode(self.model.audio.render_real(sub)[0, : int(sub["mel_len"][0])].float().cpu()))
            z = self.model.encode("aud", sub)
            mel, _, streams = self.model.audio.synthesize_mel(z, sub["lang"], sub["spk"], sub["f0"])
            wavs["codec"].append(self.vocode(mel[0, : int(streams["mel_len"][0])].float().cpu()))
            if self.model.audio.mode == "extra":
                mel, _, streams = self.model.audio.synthesize_mel(z, sub["lang"], sub["spk"], sub["f0"], content_only=True)
                wavs.setdefault("content", []).append(self.vocode(mel[0, : int(streams["mel_len"][0])].float().cpu()))
            wavs["written"].append(self.hub_to_wave(self.text_to_hub([ref], H), H, ui)[0])
        rungs = {k: (hear(v), refs, [g["img"] for g in ex]) for k, v in wavs.items()}
        for L in self.island:   # island speech -> hub-language speech, in the voice of the caption's own hub-language utterance
            isl = [(g, self._view_utt(g, "aud", L), self._view_utt(g, "aud", H)) for g in self._examples(n, [("aud", L), ("aud", H)])]
            if isl:
                w = [self.hub_to_wave(self.utterances_to_hub([ul])[0], H, uh)[0] for _, ul, uh in isl]
                rungs[f"island_{L}"] = (hear(w), [self.evaluator._ref(g, H) for g, _, _ in isl], [g["img"] for g, _, _ in isl])
        out: Dict = {"recogniser": cfg.asr_model, "rungs": {}, "examples": {}}
        for k, (hyp, ref, img) in rungs.items():
            wer = 100.0 * sum(edit_distance(words(h), words(r)) for h, r in zip(hyp, ref)) / max(1, sum(len(words(r)) for r in ref))
            wrong = [ref[j] for j in other_image(img)] if len(set(img)) > 1 else ref
            out["rungs"][k] = {"n": len(ref), "wer": wer, "content_f1": content_f1(hyp, ref),
                               "chance_content_f1": content_f1(hyp, wrong, pool=ref)}
            out["examples"][k] = [{"reference": r, "heard": h} for h, r in list(zip(hyp, ref))[:8]]
        self.report["speech_asr"] = out["rungs"]
        save_json(out, os.path.join(self.out, "outputs", "speech_asr.json"))
        self.log.info(f"[infer] written speech heard by {cfg.asr_model}: " + " | ".join(
            f"{k} WER {v['wer']:.0f} %, content F1 {v['content_f1']:.1f} (chance {v['chance_content_f1']:.1f}), n={v['n']}"
            for k, v in out["rungs"].items()))

    # ================================================================== 01 slot attention
    def fig_slot_attention(self) -> None:
        import matplotlib.pyplot as plt

        H, K = self.hub_lang, self.cfg.hub_slots
        views = [("txt", H), ("aud", H)] + [("aud", L) for L in self.island]
        ex = self._examples(3, [v for v in views]) or self._examples(3, views[:2])
        if not ex:
            return
        rows = []
        for g in ex:
            for mod, lang in views:
                if self._view_utt(g, mod, lang) is not None:
                    rows.append((g, mod, lang))
        fig, axes = plt.subplots(len(rows), K + 2, figsize=(2.1 * (K + 2), 2.2 * len(rows)), squeeze=False)
        for r, (g, mod, lang) in enumerate(rows):
            attn, implied, _, _ = self._ground(g, mod, lang)
            img = self._image(g["img"])
            axes[r, 0].imshow(img)
            txt = self.evaluator._ref(g, lang) or ""
            axes[r, 0].set_title(f"{mod}:{lang}\n{txt[:34]}", fontsize=6.5)
            for k in range(K):
                axes[r, k + 1].imshow(img, alpha=0.45)
                axes[r, k + 1].imshow(self._attn_grid(attn[k]), cmap="inferno", alpha=0.75, extent=(0, img.size[0], img.size[1], 0),
                                      interpolation="nearest")
                axes[r, k + 1].set_title(f"slot {k} (grounder)", fontsize=6.5)
            axes[r, K + 1].imshow(img, alpha=0.45)
            axes[r, K + 1].imshow(self._attn_grid(implied), cmap="viridis", alpha=0.75,
                                  extent=(0, img.size[0], img.size[1], 0), interpolation="nearest")
            axes[r, K + 1].set_title("reader hub -> patches\n(no image used)", fontsize=6.5)
            for a in axes[r]:
                a.axis("off")
        fig.suptitle("Barycentric visual interlingua: each hub slot is a barycentre of frozen patch tokens", fontsize=10)
        fig.tight_layout()
        fig.savefig(os.path.join(self.out, "01_slot_attention.png"), dpi=120)
        plt.close(fig)
        # pointing game (SynthWorld has ground-truth object cells)
        if self.cfg.dataset == "synthworld":
            res = {}
            for mod, lang in views:
                hit = prec = tot = 0
                for g in self.groups[:80]:
                    ui = self._view_utt(g, mod, lang)
                    if ui is None:
                        continue
                    attn = self._ground(g, mod, lang)[0]
                    cells = set(self.data.utts[ui]["cells"])
                    tops = [int(a.argmax()) for a in attn]
                    hit += int(any(t in cells for t in tops))         # some slot points at a described object
                    prec += sum(t in cells for t in tops) / len(tops)  # share of slots on described objects
                    tot += 1
                if tot:
                    res[f"{mod}:{lang}"] = {"any_slot_hits": 100.0 * hit / tot, "slot_precision": 100.0 * prec / tot,
                                            "chance_precision": 100.0 * 1.35 / (self.cfg.synth_grid ** 2)}
            self.report["pointing_game"] = res

    # ================================================================== 02 flow trajectories
    @torch.no_grad()
    def fig_flow_trajectories(self) -> None:
        import matplotlib.pyplot as plt

        model, H = self.model, self.hub_lang
        fig, axes = plt.subplots(1, 4, figsize=(20, 4.6))
        stats = {}
        panels = []
        if ("txt", H) in self.store["views"]:
            panels.append(("txt", H, +1, "hub -> text (write)"))
        for L in (self.island or [H]):
            if ("aud", L) in self.store["views"]:
                panels.append(("aud", L, -1, f"speech:{L} -> hub (backward ODE)"))
        for ax, (mod, lang, direction, title) in zip(axes[:2], panels[:2]):
            rec = self.store["views"][(mod, lang)]
            n = min(24, len(rec["z"]))
            lid = torch.full((n,), model.lang2id[lang], dtype=torch.long, device=self.device)
            spoke = model.spokes[mod]
            if direction > 0:
                x = spoke.source(rec["hub"][:n].to(self.device), self.cfg.eval_temp)
            else:
                x = rec["z"][:n].to(self.device)
            _, path = spoke.solve(x, lid, direction, steps=32, return_path=True)
            P = torch.stack(path, 1).flatten(2).float().cpu()                 # [n, 33, D]
            flat = P.reshape(-1, P.shape[-1])
            mu = flat.mean(0)
            _, _, Vt = torch.pca_lowrank(flat - mu, q=2, center=False)
            xy = ((P - mu) @ Vt[:, :2]).numpy()
            for i in range(n):
                ax.plot(xy[i, :, 0], xy[i, :, 1], lw=0.9, color="#888")
                ax.scatter(xy[i, 0, 0], xy[i, 0, 1], s=14, color="#1f77b4", zorder=3)
                ax.scatter(xy[i, -1, 0], xy[i, -1, 1], s=14, color="#d62728", zorder=3)
            seg = (P[:, 1:] - P[:, :-1]).norm(dim=-1).sum(1)
            chord = (P[:, -1] - P[:, 0]).norm(dim=-1)
            stats[title] = float((chord / seg.clamp_min(1e-8)).mean())
            ax.set_title(f"{title}\nblue = start, red = end | straightness {stats[title]:.3f}", fontsize=9)
            # speed profile
            ts = torch.linspace(0, 1, 17)
            speeds = []
            for t in ts:
                tt = torch.full((n,), float(t), device=self.device)
                xt = P[:, int(round(float(t if direction > 0 else 1 - t) * 32))].reshape(n, -1, self.cfg.latent_dim).to(self.device)
                speeds.append(float(spoke.vel(xt, tt, lid).float().flatten(1).norm(dim=-1).mean()))
            axes[2].plot(ts.numpy(), speeds, label=title)
        axes[2].set_title("velocity norm along the path", fontsize=9)
        axes[2].set_xlabel("t")
        legend_if_any(axes[2], fontsize=7)
        # endpoint error of few-step solvers against a 64-step reference
        if panels:
            mod, lang, direction, title = panels[0]
            rec = self.store["views"][(mod, lang)]
            n = min(64, len(rec["z"]))
            lid = torch.full((n,), model.lang2id[lang], dtype=torch.long, device=self.device)
            x = model.spokes[mod].source(rec["hub"][:n].to(self.device), self.cfg.eval_temp)
            ref = model.spokes[mod].solve(x, lid, +1, steps=64, solver="midpoint")
            for solver in ("euler", "midpoint", "heun"):
                errs = [float(((model.spokes[mod].solve(x, lid, +1, steps=s, solver=solver) - ref) ** 2).mean()) for s in (1, 2, 4, 8, 16)]
                axes[3].semilogy([1, 2, 4, 8, 16], errs, marker="o", label=solver)
            axes[3].set_title("end-point error vs 64-step reference", fontsize=9)
            axes[3].set_xlabel("ODE steps")
            legend_if_any(axes[3], fontsize=7)
        fig.tight_layout()
        fig.savefig(os.path.join(self.out, "02_flow_trajectories.png"), dpi=120)
        plt.close(fig)
        self.report["path_straightness"] = stats

    # ================================================================== 03 hub manifold
    def fig_hub_manifold(self) -> None:
        import matplotlib.pyplot as plt

        views = self.store["views"]
        keys = [k for k in views if not (k[0] == "txt" and k[1] in self.island and not self.cfg.use_island_text)]
        X, lab, grp = [], [], []
        for i, k in enumerate(keys):
            n = min(250, len(views[k]["hub"]))
            X.append(views[k]["hub"][:n].flatten(1))
            lab += [i] * n
            grp += views[k]["g"][:n].tolist()
        if self.store["img_hub"] is not None:
            n = min(250, len(self.store["img_hub"]))
            X.append(self.store["img_hub"][:n].flatten(1))
            lab += [len(keys)] * n
            grp += list(range(n))
            keys = keys + [("img", "-")]
        X = torch.cat(X).float()
        lab, grp = np.array(lab), np.array(grp)
        Xc = X - X.mean(0)
        _, _, Vt = torch.pca_lowrank(Xc, q=2, center=False)
        pca = (Xc @ Vt[:, :2]).numpy()
        try:
            from sklearn.manifold import TSNE

            emb = TSNE(n_components=2, init="pca", perplexity=min(30, max(5, len(X) // 20)), random_state=0).fit_transform(
                F.normalize(Xc, dim=-1).numpy())
        except Exception:
            emb = pca
        fig, axes = plt.subplots(1, 3, figsize=(19, 6))
        for ax, Y, title in ((axes[0], pca, "PCA of hub codes"), (axes[1], emb, "t-SNE of hub codes")):
            for i, k in enumerate(keys):
                m = lab == i
                ax.scatter(Y[m, 0], Y[m, 1], s=9, alpha=0.6, color=LANG_COLORS[i % 6], label=f"{k[0]}:{k[1]}")
            ax.set_title(title + " (all read by backward ODE)", fontsize=10)
            legend_if_any(ax, fontsize=8, markerscale=2)
        # same caption across views: lines in t-SNE for a few groups
        for gidx in np.unique(grp)[:40]:
            pts = emb[grp == gidx]
            if len(pts) > 1:
                axes[1].plot(pts[:, 0], pts[:, 1], lw=0.4, color="k", alpha=0.35)
        # modality gap: distance between view centroids relative to within-view spread
        cent = torch.stack([X[lab == i].mean(0) for i in range(len(keys))])
        spread = torch.stack([X[lab == i].std(0).norm() for i in range(len(keys))])
        gap = torch.cdist(cent, cent) / spread.mean()
        im = axes[2].imshow(gap.numpy(), cmap="Blues")
        axes[2].set_xticks(range(len(keys)), [f"{k[0]}:{k[1]}" for k in keys], rotation=45, fontsize=8)
        axes[2].set_yticks(range(len(keys)), [f"{k[0]}:{k[1]}" for k in keys], fontsize=8)
        for i in range(len(keys)):
            for j in range(len(keys)):
                axes[2].text(j, i, f"{gap[i, j]:.2f}", ha="center", va="center", fontsize=7)
        axes[2].set_title("centroid gap between views / mean within-view spread\n(0 = views share one distribution in the hub)", fontsize=9)
        fig.colorbar(im, ax=axes[2], fraction=0.046)
        fig.tight_layout()
        fig.savefig(os.path.join(self.out, "03_hub_manifold.png"), dpi=120)
        plt.close(fig)
        self.report["hub_centroid_gap"] = {f"{keys[i][0]}:{keys[i][1]}|{keys[j][0]}:{keys[j][1]}": float(gap[i, j])
                                           for i in range(len(keys)) for j in range(i + 1, len(keys))}

    # ================================================================== 04 hub alignment of unpaired island speech
    @torch.no_grad()
    def fig_hub_alignment(self) -> None:
        import matplotlib.pyplot as plt

        views, cfg, H = self.store["views"], self.cfg, self.hub_lang
        if not self.island or ("aud", self.island[0]) not in views or ("txt", H) not in views:
            return
        a, t = views[("aud", self.island[0])], views[("txt", H)]
        pos = {g: n for n, g in enumerate(t["g"].tolist())}
        keep = [n for n, g in enumerate(a["g"].tolist()) if g in pos][:48]
        if len(keep) < 4:
            return
        src = a["hub"][keep]
        order = [pos[a["g"][n].item()] for n in keep]
        extra = [i for i in range(len(t["hub"])) if i not in set(order)][:48]
        tgt = torch.cat([t["hub"][order], t["hub"][extra]])   # first len(keep) columns are the true partners
        mu = tgt.mean(0, keepdim=True)
        fs, ft = (src - mu).flatten(1), (tgt - mu).flatten(1)
        C1, C2 = scale_free(pairwise_cos_dist(fs)), scale_free(pairwise_cos_dist(ft))
        Mx = scale_free(pairwise_cos_dist(fs, ft))
        fig, axes = plt.subplots(1, 4, figsize=(21, 5))
        res = {}
        for ax, alpha in zip(axes[:3], (0.0, 0.5, 1.0)):
            Pi = solve_fgw(C1, C2, Mx, alpha, cfg.dock_eps, max(cfg.dock_outer_iters, 20), cfg.dock_sinkhorn_iters, cfg.dock_marginal)
            acc = float((Pi.argmax(1) == torch.arange(len(keep))).float().mean()) * 100
            mass = float(Pi[torch.arange(len(keep)), torch.arange(len(keep))].sum() / Pi.sum()) * 100
            res[f"alpha={alpha}"] = {"match_acc": acc, "mass_on_true_pairs": mass, **coupling_stats(Pi)}
            ax.imshow(Pi.numpy(), cmap="magma", aspect="auto")
            ax.plot(range(len(keep)), range(len(keep)), "c.", ms=2)
            name = {0.0: "pure Gromov-Wasserstein", 0.5: "fused", 1.0: "pure Wasserstein"}[alpha]
            ax.set_title(f"alpha = {alpha} ({name})\nargmax = true partner: {acc:.0f}%  mass on true pairs: {mass:.0f}%", fontsize=9)
            ax.set_xlabel(f"hub codes of {H} text (first {len(keep)} = true partners, cyan)")
            ax.set_ylabel(f"island speech ({self.island[0]}), read with no image")
        axes[3].scatter(C1.flatten().numpy(), C2[: len(keep), : len(keep)].flatten().numpy(), s=3, alpha=0.3)
        rho = float(np.corrcoef(C1.flatten().numpy(), C2[: len(keep), : len(keep)].flatten().numpy())[0, 1])
        axes[3].set_title(f"relational geometry: island vs hub pairwise distances\n(Gromov premise) Pearson r = {rho:.3f}", fontsize=9)
        axes[3].set_xlabel("distance between two island utterances (hub coords)")
        axes[3].set_ylabel("distance between their true partners")
        fig.tight_layout()
        fig.savefig(os.path.join(self.out, "04_hub_alignment.png"), dpi=120)
        plt.close(fig)
        res["relational_isometry_r"] = rho
        self.report["hub_alignment"] = res

    # ================================================================== 05 tone
    @torch.no_grad()
    def fig_tone(self) -> None:
        import matplotlib.pyplot as plt

        model, cfg, H = self.model, self.cfg, self.hub_lang
        spoken = [l for l in model.langs if ("aud", l) in self.store["views"]]
        gates = model.audio.gate().detach().cpu().numpy()
        fig, axes = plt.subplots(1, 4, figsize=(21, 4.8))
        cols = [LANG_COLORS[i % 6] for i in range(len(spoken))]
        axes[0].bar(spoken, [gates[model.lang2id[l]] for l in spoken], color=cols)
        for i, l in enumerate(spoken):
            axes[0].text(i, gates[model.lang2id[l]] + 0.02, "tonal" if self.data.lang_info[l].get("tonal") else "non-tonal",
                         ha="center", fontsize=8)
        axes[0].set_ylim(0, 1.1)
        axes[0].set_title("learned lexical-tonality gate (label = ground truth)", fontsize=9)
        scales = [0.0, 0.25, 0.5, 1.0, 1.5, 2.0]
        res: Dict = {"gate": {l: float(gates[model.lang2id[l]]) for l in spoken}, "pitch_scaling": {}}
        for i, l in enumerate(spoken):
            utts = self.store["views"][("aud", l)]["utt"][:96].tolist()
            sub = _to(self.data._collate_audio(utts, list(range(len(utts)))), self.device)
            base = model.read("aud", model.encode("aud", sub), sub["lang"])
            ref = ((base - base.roll(1, 0)) ** 2).mean()
            disp, accs = [], []
            for s in scales:
                tone = sub["tone"].clone()
                tone[..., 0] *= s
                tone[..., 2] *= s
                hub = model.read("aud", model.encode("aud", dict(sub, tone=tone)), sub["lang"])
                disp.append(float(((hub - base) ** 2).mean() / ref.clamp_min(1e-8)))
                if cfg.dataset == "synthworld":
                    hy = self.evaluator._gen_text(model, hub.cpu(), H)
                    gs = [self.store["groups"][g] for g in self.store["views"][("aud", l)]["g"][:96].tolist()]
                    accs.append(self.evaluator._content_acc(hy, gs, H)["color_acc"])
            axes[1].plot(scales, disp, marker="o", color=cols[i], label=l)
            if accs:
                axes[2].plot(scales, accs, marker="o", color=cols[i], label=l)
            res["pitch_scaling"][l] = {"scales": scales, "hub_displacement": disp, "color_acc": accs}
        axes[1].set_title("hub displacement when the pitch contour is scaled\n(relative to the distance between two utterances)", fontsize=9)
        axes[1].set_xlabel("contour scale (1 = natural, 0 = flattened)")
        legend_if_any(axes[1], fontsize=8)
        axes[2].set_title(f"colour accuracy of speech -> {H} text vs contour scale" if cfg.dataset == "synthworld"
                          else "(content accuracy needs ground truth: SynthWorld only)", fontsize=9)
        axes[2].set_xlabel("contour scale")
        legend_if_any(axes[2], fontsize=8)
        # natural vs re-predicted contour for one island utterance
        L = (self.island or spoken)[0]
        if ("aud", L) in self.store["views"]:
            ui = int(self.store["views"][("aud", L)]["utt"][0])
            sub = _to(self.data._collate_audio([ui], [0]), self.device)
            z = model.encode("aud", sub)
            out = model.audio.decode(z, sub["lang"], sub["mask"].sum(1), model.audio.bypass_code(sub["spk"], sub["f0"]))
            nat, voiced = sub["tone"][0, :, 0].cpu().numpy(), sub["tone"][0, :, 1].cpu().numpy() > 0.5
            pred = out["pros"][0, :, 0].float().cpu().numpy()
            axes[3].plot(np.where(voiced, nat, np.nan), color="k", lw=1.6, label="natural contour")
            axes[3].plot(np.where(voiced, pred, np.nan), color="#d62728", lw=1.2, label="re-predicted from the latent")
            axes[3].set_title(f"{L}: pitch contour recovered from the hub-bound latent\n{(self.data.utts[ui].get('text') or '')[:60]}", fontsize=8)
            legend_if_any(axes[3], fontsize=8)
            v = voiced & np.isfinite(pred)
            if v.sum() > 3:
                res["contour_corr_" + L] = float(np.corrcoef(nat[v], pred[v])[0, 1])
        fig.tight_layout()
        fig.savefig(os.path.join(self.out, "05_tone.png"), dpi=120)
        plt.close(fig)
        self.report["tone"] = res

    # ================================================================== 06 / 07 attribution
    def fig_speech_attribution(self) -> None:
        import matplotlib.pyplot as plt

        model, K = self.model, self.cfg.hub_slots
        langs = [l for l in ([self.hub_lang] + self.island) if ("aud", l) in self.store["views"]]
        fig, axes = plt.subplots(3, len(langs), figsize=(8 * len(langs), 8), squeeze=False)
        res = {}
        for c, l in enumerate(langs):
            ui = int(self.store["views"][("aud", l)]["utt"][0])
            sub = _to(self.data._collate_audio([ui], [0]), self.device)
            sem = sub["sem"].clone().requires_grad_(True)
            tone = sub["tone"].clone().requires_grad_(True)
            with torch.enable_grad():
                z = model.audio.encode(sem, tone, sub["mask"], sub["lang"])
                hub = model.hub.reader(z, "aud", sub["lang"])
                sal_s, sal_t = [], []
                for k in range(K):
                    gs, gt = torch.autograd.grad((hub[0, k] ** 2).sum(), [sem, tone], retain_graph=True)
                    sal_s.append((gs * sem).abs().sum(-1)[0])
                    sal_t.append((gt * tone)[0, :, [0, 2]].abs().sum(-1))
            S, T = torch.stack(sal_s).detach().cpu().numpy(), torch.stack(sal_t).detach().cpu().numpy()
            share = float(T.sum() / (S.sum() + T.sum() + 1e-12))
            res[l] = {"tone_stream_share_of_attribution": share}
            axes[0, c].imshow(S / (S.max() + 1e-9), aspect="auto", cmap="Blues")
            axes[0, c].set_title(f"{l}: segmental-stream attribution per hub slot\n{(self.data.utts[ui].get('text') or '')[:70]}", fontsize=8)
            axes[1, c].imshow(T / (S.max() + 1e-9), aspect="auto", cmap="Reds", vmin=0, vmax=1)
            axes[1, c].set_title(f"{l}: pitch-contour attribution per hub slot (same colour scale; "
                                 f"share of total = {100 * share:.1f}%)", fontsize=8)
            f0 = sub["tone"][0, :, 0].cpu().numpy()
            v = sub["tone"][0, :, 1].cpu().numpy() > 0.5
            axes[2, c].plot(np.where(v, f0, np.nan), color="k")
            axes[2, c].set_xlim(0, len(f0) - 1)
            axes[2, c].set_title("speaker-normalised log-F0", fontsize=8)
            axes[2, c].set_xlabel("frame (25 Hz)")
            for r in (0, 1):
                axes[r, c].set_ylabel("hub slot")
        fig.tight_layout()
        fig.savefig(os.path.join(self.out, "06_speech_attribution.png"), dpi=120)
        plt.close(fig)
        self.report["speech_attribution"] = res

    def fig_text_attribution(self) -> None:
        import matplotlib.pyplot as plt

        model, K = self.model, self.cfg.hub_slots
        langs = [l for l in [self.hub_lang] + list(self.cfg.text_only_langs) if ("txt", l) in self.store["views"]]
        fig, axes = plt.subplots(len(langs), 1, figsize=(15, 2.6 * len(langs)), squeeze=False)
        for r, l in enumerate(langs):
            ui = int(self.store["views"][("txt", l)]["utt"][0])
            text = self.data.utts[ui]["text"]
            sub = self.data.unitizer.collate([text])
            sub["lang"] = torch.tensor([model.lang2id[l]])
            sub = _to(sub, self.device)
            with torch.enable_grad():
                h0 = model.text.embed_units(sub["units"]).detach().requires_grad_(True)
                z = model.text.encode_from_embeddings(h0, sub["unit_mask"])
                hub = model.hub.reader(z, "txt", sub["lang"])
                sal = torch.stack([(torch.autograd.grad((hub[0, k] ** 2).sum(), h0, retain_graph=True)[0] * h0).abs().sum(-1)[0]
                                   for k in range(K)])
            S = sal.detach().cpu().numpy()
            units = [u.decode("utf8", errors="replace") for u in self.data.unitizer.segment(text)][: S.shape[1]]
            axes[r, 0].imshow(S / (S.max() + 1e-9), aspect="auto", cmap="Purples")
            axes[r, 0].set_xticks(range(len(units)), [u if u.strip() else "_" for u in units], fontsize=7)
            axes[r, 0].set_ylabel("hub slot")
            axes[r, 0].set_title(f"{l}: grapheme-level attribution of each hub slot  ({len(units)} units for "
                                 f"{len(text.encode('utf8'))} bytes)", fontsize=9)
        fig.tight_layout()
        fig.savefig(os.path.join(self.out, "07_text_attribution.png"), dpi=120)
        plt.close(fig)

    # ================================================================== 08 retrieval
    @torch.no_grad()
    def fig_retrieval(self) -> None:
        import matplotlib.pyplot as plt

        views = self.store["views"]
        key = next((("aud", L) for L in self.island if ("aud", L) in views), ("txt", self.hub_lang))
        rec = views[key]
        n = min(40, len(rec["hub"]))
        img_ids = self.store["img_ids"]
        gal = self.store["gallery"].to(self.device)
        sc = self.model.hub.hub_to_image_scores(rec["hub"][:n].to(self.device), gal).cpu()
        tgt = torch.tensor([img_ids.index(self.store["img_of_group"][g]) for g in rec["g"][:n].tolist()])
        order = torch.argsort(tgt)
        fig = plt.figure(figsize=(18, 7))
        ax = fig.add_subplot(1, 2, 1)
        ax.imshow(sc[order][:, tgt[order]].numpy(), cmap="viridis", aspect="auto")
        ax.set_title(f"{key[0]}:{key[1]} -> image scores (hub code explained by gallery patches)\n"
                     f"diagonal = correct image | R@1 over {len(img_ids)} images: {recall_at_k(sc, tgt)['r1']:.1f}%", fontsize=9)
        ax.set_xlabel("image of query j")
        ax.set_ylabel("query i")
        for q in range(3):
            top = sc[q].topk(min(5, sc.shape[1])).indices.tolist()
            for j, gi in enumerate(top):
                a = fig.add_subplot(3, 10, q * 10 + 6 + j)
                a.imshow(self._image(img_ids[gi], 112))
                a.axis("off")
                if gi == int(tgt[q]):
                    a.set_title("correct", fontsize=7, color="green")
                if j == 0:
                    g = self.store["groups"][rec["g"][q].item()]
                    a.text(0, -18, (self.evaluator._ref(g, self.hub_lang) or "")[:60], fontsize=7)
        fig.tight_layout()
        fig.savefig(os.path.join(self.out, "08_retrieval.png"), dpi=120)
        plt.close(fig)

    # ================================================================== 09 ODE steps / the plain cycle
    @torch.no_grad()
    def fig_ode_steps(self) -> None:
        import matplotlib.pyplot as plt

        model, cfg, H = self.model, self.cfg, self.hub_lang
        views = self.store["views"]
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
        res: Dict = {"score_vs_steps": {}, "plain_cycle_vs_steps": {}}
        src = next((("aud", L) for L in self.island if ("aud", L) in views), ("aud", H) if ("aud", H) in views else None)
        if src is not None:
            utts = views[src]["utt"][:160].tolist()
            gs = [self.store["groups"][g] for g in views[src]["g"][:160].tolist()]
            sub = _to(self.data._collate_audio(utts, list(range(len(utts)))), self.device)
            z = model.encode("aud", sub)
            xs, ys = [1, 2, 4, 8, 16], []
            for s in xs:
                hub = model.read("aud", z, sub["lang"], steps=s)
                lid = torch.full((len(utts),), model.lang2id[H], dtype=torch.long, device=self.device)
                zt = model.write("txt", hub, lid, temp=cfg.eval_temp, steps=s)
                m, _ = self.evaluator._score_text(model.text.generate(zt, lid), gs, H, same_lang=(src[1] == H))
                ys.append(m.get("content_acc", m.get("cf1", 0.0)))
            axes[0].plot(xs, ys, marker="o")
            axes[0].set_xscale("log", base=2)
            axes[0].set_title(f"{src[0]}:{src[1]} -> text:{H}  ({'content accuracy' if 'content_acc' in m else 'BLEU'}) "
                              f"vs ODE steps per hop", fontsize=9)
            axes[0].set_xlabel("ODE steps (midpoint)")
            res["score_vs_steps"] = dict(zip(map(str, xs), ys))
        if ("txt", H) in views and ("aud", H) in views:
            loss = GeoBridgeLoss(cfg, model.lang2id)
            n = min(64, len(views[("txt", H)]["z"]))
            z1 = views[("txt", H)]["z"][:n].to(self.device)
            lid = torch.full((n,), model.lang2id[H], dtype=torch.long, device=self.device)
            xs = [1, 2, 4, 8, 16, 32]
            for solver in ("euler", "midpoint"):
                old = cfg.ode_solver
                cfg.ode_solver = solver
                ys = [float(loss.naive_cycle(model, "txt", z1, lid, "aud", lid, steps=s)) for s in xs]
                cfg.ode_solver = old
                axes[1].loglog(xs, ys, marker="o", label=solver)
                res["plain_cycle_vs_steps"][solver] = dict(zip(map(str, xs), ys))
            axes[1].set_title("plain ODE cycle  text -> hub -> speech -> hub -> text\n"
                              "vanishes with the step size for ANY field: it measures the solver, not alignment", fontsize=9)
            axes[1].set_xlabel("ODE steps")
            axes[1].set_ylabel("cycle MSE")
            legend_if_any(axes[1], fontsize=8)
        fig.tight_layout()
        fig.savefig(os.path.join(self.out, "09_ode_steps.png"), dpi=120)
        plt.close(fig)
        self.report["ode"] = res

    # ================================================================== 10 fertility
    def fig_fertility(self) -> None:
        import matplotlib.pyplot as plt

        cfg, data = self.cfg, self.data
        langs = [l for l in data.langs if any(u["lang"] == l and u.get("text") for u in data.utts[:5000])]
        texts = {l: [u["text"] for u in data.utts if u["lang"] == l and u.get("text")][:1500] for l in langs}
        modes = {"grapheme (ours)": TextUnitizer("grapheme", 10 ** 6, cfg.max_bytes_per_grapheme, 10 ** 6),
                 "byte": TextUnitizer("byte", 10 ** 6, 1, 10 ** 6)}
        bpe_path = os.path.join(cfg.cache_dir, f"bpe_{cfg.bpe_vocab}_{'-'.join(cfg.hub_langs)}.json")
        try:
            if not os.path.exists(bpe_path):
                tr = [u["text"] for u in data.utts if u["lang"] in cfg.hub_langs and data.img_split[u["img"]] == "train" and u.get("text")]
                TextUnitizer.train_bpe(tr, bpe_path, cfg.bpe_vocab)
            modes[f"BPE-{cfg.bpe_vocab} (hub language only)"] = TextUnitizer("bpe", 10 ** 6, 1, 10 ** 6, bpe_path)
        except Exception:
            pass
        res = {name: {l: u.fertility(texts[l]) for l in langs} for name, u in modes.items()}
        fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))
        w = 0.8 / len(modes)
        for ax, key, title in ((axes[0], "units_per_word", "encoder units per word (fertility)"),
                               (axes[1], "units_per_grapheme", "encoder units per written symbol"),
                               (axes[2], "bytes_per_grapheme", "UTF-8 bytes per written symbol (script cost)")):
            for i, name in enumerate(modes):
                vals = [res[name][l][key] for l in langs]
                ax.bar(np.arange(len(langs)) + i * w, vals, w, label=name)
                for x, v in zip(np.arange(len(langs)) + i * w, vals):
                    ax.text(x, v, f"{v:.1f}", ha="center", va="bottom", fontsize=7)
            ax.set_xticks(np.arange(len(langs)) + 0.4 - w / 2, [f"{l}\n{data.lang_info[l]['script'][:14]}" for l in langs], fontsize=8)
            ax.set_title(title, fontsize=9)
            if key != "bytes_per_grapheme":
                legend_if_any(ax, fontsize=7)
        fig.tight_layout()
        fig.savefig(os.path.join(self.out, "10_fertility.png"), dpi=120)
        plt.close(fig)
        self.report["fertility"] = res

    # ================================================================== 11 hub agreement
    def fig_hub_agreement(self) -> None:
        import matplotlib.pyplot as plt

        views, H = self.store["views"], self.hub_lang
        pairs = [(("txt", H), ("aud", H))] + [(("aud", L), ("txt", H)) for L in self.island] + [(("aud", L), ("aud", H)) for L in self.island]
        pairs = [p for p in pairs if p[0] in views and p[1] in views]
        if not pairs:
            return
        fig, axes = plt.subplots(1, len(pairs), figsize=(6.2 * len(pairs), 4.4), squeeze=False)
        res = {}
        for ax, (a, b) in zip(axes[0], pairs):
            pos = {g: n for n, g in enumerate(views[b]["g"].tolist())}
            keep = [n for n, g in enumerate(views[a]["g"].tolist()) if g in pos]
            if len(keep) < 3:
                continue
            ha = F.normalize(views[a]["hub"][keep].flatten(1), dim=-1)
            hb = F.normalize(views[b]["hub"][[pos[views[a]["g"][n].item()] for n in keep]].flatten(1), dim=-1)
            same = (ha * hb).sum(-1).numpy()
            diff = (ha * hb.roll(1, 0)).sum(-1).numpy()
            ax.hist(same, bins=30, alpha=0.7, color="#2ca02c", label="same caption")
            ax.hist(diff, bins=30, alpha=0.7, color="#888", label="different caption")
            auc = float((same[:, None] > diff[None, :]).mean())
            res[f"{a[0]}:{a[1]}|{b[0]}:{b[1]}"] = {"same_mean_cos": float(same.mean()), "diff_mean_cos": float(diff.mean()), "auc": auc}
            ax.set_title(f"hub cosine  {a[0]}:{a[1]}  vs  {b[0]}:{b[1]}\nAUC(same > different) = {auc:.3f}"
                         + ("   [no paired data between these two]" if a[1] != b[1] else ""), fontsize=9)
            legend_if_any(ax, fontsize=8)
        fig.tight_layout()
        fig.savefig(os.path.join(self.out, "11_hub_agreement.png"), dpi=120)
        plt.close(fig)
        self.report["hub_agreement"] = res


    # ================================================================== 12 language LoRA
    @torch.no_grad()
    def fig_language_lora(self) -> None:
        import matplotlib.pyplot as plt

        model = self.model
        spokes = [(n, sp) for n, sp in model.spokes.items() if n != "img"]
        if self.cfg.lora_rank <= 0 or not spokes:
            return
        fig, axes = plt.subplots(1, len(spokes), figsize=(7 * len(spokes), 4.6), squeeze=False)
        res = {}
        for ax, (name, sp) in zip(axes[0], spokes):
            rows = []
            for blk in sp.vel.blocks:
                per_lang = torch.zeros(len(model.langs))
                base = 0.0
                for mod in blk.modules():
                    if hasattr(mod, "lora_A") and getattr(mod, "rank", 0) > 0:
                        delta = torch.einsum("lor,lri->loi", mod.lora_B, mod.lora_A) * mod.scale   # [lang, out, in]
                        per_lang += delta.flatten(1).norm(dim=1).cpu() ** 2
                        base += float(mod.lin.weight.norm() ** 2)
                rows.append((per_lang.sqrt() / max(base, 1e-12) ** 0.5).detach().numpy())
            M = np.stack(rows)                                                                      # [layer, lang]
            im = ax.imshow(M, cmap="YlOrRd", aspect="auto")
            ax.set_xticks(range(len(model.langs)), model.langs)
            ax.set_ylabel("flow layer")
            ax.set_title(f"{name} spoke: ||B_l A_l|| / ||W||  per layer and language\n"
                         "(how far a language moves the shared velocity field)", fontsize=9)
            for i in range(M.shape[0]):
                for j in range(M.shape[1]):
                    ax.text(j, i, f"{M[i, j]:.3f}", ha="center", va="center", fontsize=7)
            fig.colorbar(im, ax=ax, fraction=0.046)
            res[name] = {l: float(M[:, j].mean()) for j, l in enumerate(model.langs)}
        fig.tight_layout()
        fig.savefig(os.path.join(self.out, "12_language_lora.png"), dpi=120)
        plt.close(fig)
        self.report["lora_relative_norm"] = res

    # ================================================================== 13 hub vs private
    @torch.no_grad()
    def fig_hub_vs_private(self) -> None:
        """Linear probes on three codes of the same utterances: the reader's hub code (what is used at test
        time), and the hub / private halves of the backward-ODE read-out.  The hub should carry what is said
        and not who says it (speaker labels exist on SynthWorld)."""
        import matplotlib.pyplot as plt
        from sklearn.linear_model import LogisticRegression

        model, data = self.model, self.data
        res: Dict = {}
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
        bars, labels = [], []
        for lang in [l for l in model.langs if ("aud", l) in self.store["views"]]:
            rec = self.store["views"][("aud", lang)]
            utts = rec["utt"].tolist()
            spk = [data.utts[u].get("spk", "") for u in utts]
            z = rec["z"].to(self.device)
            lid = torch.full((len(z),), model.lang2id[lang], dtype=torch.long, device=self.device)
            parts = [model.spokes["aud"].read(z[i:i + 128], lid[i:i + 128]) for i in range(0, len(z), 128)]
            codes = {"reader hub": torch.cat([model.hub.reader(z[i:i + 128], "aud", lid[i:i + 128]) for i in range(0, len(z), 128)]).flatten(1).cpu().numpy(),
                     "ODE hub": torch.cat([p[0] for p in parts]).flatten(1).cpu().numpy(),
                     "ODE private": torch.cat([p[1] for p in parts]).flatten(1).cpu().numpy()}
            targets = {}
            if len(set(spk)) > 1:
                targets["speaker"] = np.array([sorted(set(spk)).index(x) for x in spk])
            if self.cfg.dataset == "synthworld":
                targets["first colour"] = np.array([data.utts[u]["content"][0][0] for u in utts])
                targets["first shape"] = np.array([data.utts[u]["content"][0][1] for u in utts])
            n = len(utts)
            tr = np.arange(n) % 2 == 0
            for tname, y in targets.items():
                if len(set(y[tr])) < 2:
                    continue
                accs = {}
                for pname, X in codes.items():
                    mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-6
                    clf = LogisticRegression(max_iter=300, C=0.5).fit((X[tr] - mu) / sd, y[tr])
                    accs[pname] = 100.0 * float((clf.predict((X[~tr] - mu) / sd) == y[~tr]).mean())
                chance = 100.0 * float(np.bincount(y[~tr]).max() / (~tr).sum())
                res[f"{lang}/{tname}"] = {**accs, "majority_class": chance}
                bars.append((accs["reader hub"], accs["ODE hub"], accs["ODE private"], chance))
                labels.append(f"{lang}\n{tname}")
        if bars:
            x = np.arange(len(bars))
            for j, (lab, col) in enumerate((("reader hub (test-time read)", "#1f77b4"), ("ODE hub", "#9ecae1"),
                                            ("ODE private tokens", "#ff7f0e"), ("majority class", "#bbbbbb"))):
                axes[0].bar(x + (j - 1.5) * 0.2, [b[j] for b in bars], 0.2, label=lab, color=col)
            axes[0].set_xticks(x, labels, fontsize=8)
            axes[0].set_ylabel("linear-probe accuracy (%)")
            axes[0].set_title("what a speech read-out encodes (linear probes, held-out half of the test set)", fontsize=9)
            legend_if_any(axes[0], fontsize=8)
        # norm of the private read-out vs a standard Gaussian (the training-time prior of the private tokens)
        for i, lang in enumerate([l for l in model.langs if ("aud", l) in self.store["views"]]):
            rec = self.store["views"][("aud", lang)]
            z = rec["z"][:256].to(self.device)
            lid = torch.full((len(z),), model.lang2id[lang], dtype=torch.long, device=self.device)
            priv = model.spokes["aud"].read(z, lid)[1].flatten(1).float().cpu()
            axes[1].hist((priv ** 2).mean(1).sqrt().numpy(), bins=30, alpha=0.6, color=LANG_COLORS[i % 6], label=f"speech:{lang}")
            res[f"{lang}/private_rms"] = float((priv ** 2).mean().sqrt())
        axes[1].axvline(1.0, color="k", ls="--", lw=1)
        axes[1].set_title("RMS of the private code read from real speech\n(dashed = the N(0, I) prior it is written from)", fontsize=9)
        legend_if_any(axes[1], fontsize=8)
        fig.tight_layout()
        fig.savefig(os.path.join(self.out, "13_hub_vs_private.png"), dpi=120)
        plt.close(fig)
        self.report["hub_vs_private_probe"] = res

    # ================================================================== 14 codec reconstruction
    @torch.no_grad()
    def fig_codec_reconstruction(self) -> None:
        import matplotlib.pyplot as plt

        model, data = self.model, self.data
        langs = [l for l in model.langs if ("aud", l) in self.store["views"]]
        fig, axes = plt.subplots(4, len(langs), figsize=(8 * len(langs), 10), squeeze=False)
        res = {}
        for c, lang in enumerate(langs):
            ui = int(self.store["views"][("aud", lang)]["utt"][1 % len(self.store["views"][("aud", lang)]["utt"])])
            sub = _to(data._collate_audio([ui], [0]), self.device)
            z = model.encode("aud", sub)
            p = model.audio.bypass_code(sub["spk"], sub["f0"])
            out = model.audio.decode(z, sub["lang"], sub["mask"].sum(1), p)
            sem = model.audio.norm_sem(sub["sem"])[0].float().cpu().numpy()
            rec = out["sem"][0].float().cpu().numpy()
            cos = float(np.mean(np.sum(sem * rec, 1) / (np.linalg.norm(sem, axis=1) * np.linalg.norm(rec, axis=1) + 1e-8)))
            res[lang] = {"segmental_cosine": cos}
            k = min(48, sem.shape[1])
            axes[0, c].imshow(sem[:, :k].T, aspect="auto", origin="lower", cmap="viridis")
            axes[0, c].set_title(f"{lang}: segmental stream (first {k} dims)  |  {(data.utts[ui].get('text') or '')[:50]}", fontsize=8)
            axes[1, c].imshow(rec[:, :k].T, aspect="auto", origin="lower", cmap="viridis")
            axes[1, c].set_title(f"reconstructed from the {self.cfg.tokens_of('aud')}-token latent (frame cosine {cos:.3f})", fontsize=8)
            if "mel" in sub:
                tone = sub["tone"]
                tone3 = torch.stack([tone[..., 0], tone[..., 1], tone[..., 3]], -1)
                mel_hat = (model.audio.renderer(model.audio.norm_sem(sub["sem"]), tone3, p, sub["mel"].shape[1])
                           * model.audio.mel_std + model.audio.mel_mean)[0]
                axes[2, c].imshow(sub["mel"][0].float().cpu().numpy().T, aspect="auto", origin="lower", cmap="magma")
                axes[2, c].set_title("natural log-mel", fontsize=8)
                axes[3, c].imshow(mel_hat.float().cpu().numpy().T, aspect="auto", origin="lower", cmap="magma")
                axes[3, c].set_title("log-mel rendered from (segmental stream, pitch contour, bypass code)", fontsize=8)
                res[lang]["mel_l1"] = float((mel_hat - sub["mel"][0]).abs().mean())
        fig.tight_layout()
        fig.savefig(os.path.join(self.out, "14_codec_reconstruction.png"), dpi=120)
        plt.close(fig)
        self.report["codec_reconstruction"] = res


    # ================================================================== 15 generated images
    @torch.no_grad()
    def fig_generated_images(self) -> None:
        import matplotlib.pyplot as plt

        model, H = self.model, self.hub_lang
        if "img" not in model.spokes or not hasattr(model, "pixel"):
            return
        views = [("txt", H), ("aud", H)] + [("aud", L) for L in self.island]
        ex = self._examples(5, views) or self._examples(5, views[:1])
        cols = [v for v in views if all(self._view_utt(g, *v) is not None for g in ex)]
        fig, axes = plt.subplots(len(ex), 2 + 2 * len(cols), figsize=(2.0 * (2 + 2 * len(cols)), 2.2 * len(ex)), squeeze=False)
        for r, g in enumerate(ex):
            batch = _to(self.data.collate_groups([{"img": g["img"], "cap": g["cap"], "views": []}]), self.device)
            lat = model.image_latent(model.hub.project(batch["img"]))
            axes[r, 0].imshow(self._image(g["img"], 128))
            axes[r, 0].set_title((self.evaluator._ref(g, H) or "")[:30], fontsize=6.5)
            axes[r, 1].imshow(model.pixel(lat)[0].permute(1, 2, 0).float().cpu().numpy())
            axes[r, 1].set_title("decoder on the TRUE latent\n(upper bound of the preview)", fontsize=6)
            for c, (mod, lang) in enumerate(cols):
                ui = self._view_utt(g, mod, lang)
                hub = self.text_to_hub([self.data.utts[ui]["text"]], lang) if mod == "txt" else self.utterances_to_hub([ui])[0]
                for k in range(2):  # two private draws: same content, different layout
                    _, rgb = model.hub_to_image(hub)
                    axes[r, 2 + 2 * c + k].imshow(rgb[0].permute(1, 2, 0).float().cpu().numpy())
                    axes[r, 2 + 2 * c + k].set_title(f"written from {mod}:{lang}" + (" (2nd draw)" if k else ""), fontsize=6)
            for a in axes[r]:
                a.axis("off")
        fig.suptitle("Image writing: hub code -> image latent (flow) -> preview decoder.  Low-resolution preview by design.", fontsize=9)
        fig.tight_layout()
        fig.savefig(os.path.join(self.out, "15_generated_images.png"), dpi=120)
        plt.close(fig)

    # ================================================================== 16 read / write modes
    @torch.no_grad()
    def fig_read_write_modes(self) -> None:
        import matplotlib.pyplot as plt

        model, cfg, H = self.model, self.cfg, self.hub_lang
        srcs = [("aud", L) for L in self.island if ("aud", L) in self.store["views"]]
        if ("aud", H) in self.store["views"]:
            srcs.append(("aud", H))
        if not srcs:
            return
        res: Dict = {}
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
        t0s = [0.0, 0.25, 0.5, 0.75, 1.0]
        for i, (mod, lang) in enumerate(srcs):
            rec = self.store["views"][(mod, lang)]
            n = min(160, len(rec["z"]))
            gs = [self.store["groups"][g] for g in rec["g"][:n].tolist()]
            z = rec["z"][:n].to(self.device)
            lid = torch.full((n,), model.lang2id[lang], dtype=torch.long, device=self.device)
            tgt = torch.full((n,), model.lang2id[H], dtype=torch.long, device=self.device)

            def score(hub: torch.Tensor, t0: float) -> float:
                torch.manual_seed(0)
                m, _ = self.evaluator._score_text(model.text.generate(model.write("txt", hub, tgt, temp=cfg.eval_temp, t0=t0), tgt),
                                                  gs, H, same_lang=(lang == H))
                return m.get("content_acc", m.get("cf1", 0.0))

            modes = {"reader (one pass)": model.read(mod, z, lid, mode="amortized"), "backward ODE": model.read(mod, z, lid, mode="ode")}
            by_mode = {k: score(h, 0.0) for k, h in modes.items()}
            by_t0 = [score(modes["reader (one pass)"], t) for t in t0s]
            res[f"{mod}:{lang}"] = {"read_mode": by_mode, "draft_t0": dict(zip(map(str, t0s), by_t0))}
            axes[0].bar(np.arange(2) + i * 0.38, list(by_mode.values()), 0.38, label=f"{mod}:{lang} -> text:{H}")
            axes[1].plot(t0s, by_t0, marker="o", label=f"{mod}:{lang} -> text:{H}")
        axes[0].set_xticks(np.arange(2) + 0.19 * (len(srcs) - 1), ["reader (one pass)", "backward ODE"])
        axes[0].set_title("how the source is read into the hub", fontsize=9)
        legend_if_any(axes[0], fontsize=8)
        axes[1].set_title("writing from the explainer's draft: ODE started at time t0\n(0 = from the source, 1 = the draft itself, no ODE step)", fontsize=9)
        axes[1].set_xlabel("t0")
        legend_if_any(axes[1], fontsize=8)
        for ax in axes:
            ax.set_ylabel("content accuracy (%)" if cfg.dataset == "synthworld" else "BLEU")
        fig.tight_layout()
        fig.savefig(os.path.join(self.out, "16_read_write_modes.png"), dpi=120)
        plt.close(fig)
        self.report["read_write_modes"] = res


def _write_wav(path: str, wav: np.ndarray, sr: int) -> None:
    x = np.clip(np.asarray(wav, dtype=np.float32), -1, 1)
    with wave.open(path, "wb") as f:
        f.setnchannels(1)
        f.setsampwidth(2)
        f.setframerate(sr)
        f.writeframes((x * 32767).astype("<i2").tobytes())


# ======================================================================================================================
# 23. ABLATION STUDY
#     (geobridge/ablation.py)
# ======================================================================================================================
# Ablation runner: every architectural choice against its alternatives, multi-seed, resumable.
#
# Each *group* isolates one design decision.  A variant is a set of config overrides; the variant whose settings
# are the base settings is the full model and is the first row of the group's table.  For every (variant, seed)
# the runner trains (or resumes), evaluates the state selected by the run's own validation score on the test split
# and stores ``results.json``; finished cells are skipped on re-run.  Cells run cheapest first, because a session
# has a time limit.  Nothing is trained twice:
#
#   * the full model of the base seed is the main run ``<out_root>/<exp>`` itself when that run has the base
#     settings (it is trained there if it does not exist yet);
#   * variants that share the same representation phase reuse one phase-A checkpoint, matched by a hash of the
#     fields that determine phase A - the main run's when it matches - so a transport-only variant costs only
#     its phase B.
#
# Outputs in ``<out_root>/<exp>_ablation/``
#     _full/full_s<seed>/...            results of the full model (shared by every group); for other seeds than
#                                       the main run's, also the run itself
#     _phase_a/<hash>/...               representation phases that the main run does not provide
#     <group>/<variant>_s<seed>/...     complete run directories (checkpoints, curves, eval json)
#     table_<group>.md|csv|tex          mean +- std over seeds of the headline metrics
#     plot_<group>.png                  bar chart per metric with seed error bars
#     SUMMARY.md                        all tables + which variant wins each column
# ======================================================================================================================

GROUPS: Dict[str, Dict] = {
    "hub": {
        "question": "What should the hub be, and what makes it say exactly what the sentence says?",
        "variants": [
            ("barycentric hub, mutual sufficiency (ours)", {}),
            ("no counterfactual patch credit (softmax gradient only)", {"w_seek": 0.0}),
            ("counterfactual credit with a veto on moves that verifiably explain less", {"seek_verify": True}),
            ("no minimality (reader does not shape the hub)", {"w_min": 0.0}),
            ("no sufficiency (no explainer loss, hence no credit either)", {"w_expl": 0.0}),
            ("text code as a fixed target for speech, instead of mutual agreement", {"agree_mode": "teach"}),
            ("independent slots (no inhibition of return)", {"hub_inhibit": 0.0}),
            ("language-balanced losses (every language the same weight in its modality)", {"lang_balance": True}),
            ("sharp attention from the start (no anneal)", {"hub_temp_start": 1.0}),
            ("global visual pivot (scene-level)", {"hub_mode": "global"}),
            ("text anchor (image-pivoted)", {"hub_mode": "text"}),
            ("free learned anchor (FlowBind-style)", {"hub_mode": "free"}),
        ]},
    "read": {
        "question": "How should a source be read into the hub at test time?",
        "variants": [
            ("image-free reader, one pass (ours)", {}),
            ("backward ODE of the flow, trained with ODE hub-swap", {"read_mode": "ode", "exposure": "ode"}),
            ("reader, writers never trained on read hubs", {"exposure": "none", "hub_jitter": 0.0}),
            ("reader, writers trained without hub jitter", {"hub_jitter": 0.0}),
            ("reader, speech keeps its own code (no canonical text code)", {"canonical_text": False}),
            ("reader never shown written hub-language speech", {"w_reread": 0.0}),
            ("reader trained on clean speech latents only (no time-masked encodings)", {"read_aug": 0}),
        ]},
    "decoder": {
        "question": "Should the byte decoder be trained on the latents the text writer actually writes?",
        "variants": [
            ("decoder trained on written latents (write-latent exposure)", {"w_dec": 1.0}),
            ("decoder frozen after the representation phase", {"w_dec": 0.0}),
        ]},
    "island": {
        "question": "What should be done with the unpaired speech of the island language?",
        "variants": [
            ("hub-space back-translation, transport-rectified (ours)", {}),
            ("back-translation without the relay to the text writers", {"bt_relay": 0.0}),
            ("back-translation without the transport step", {"bt_rectify": 0.0}),
            ("bridge pairs only (unpaired speech unused in phase B)", {"bt_weight": 0.0}),
            ("fused Gromov-Wasserstein docking instead", {"bt_weight": 0.0, "dock_mode": "g2w"}),
            ("pure Gromov-Wasserstein docking instead (roadmap)", {"bt_weight": 0.0, "dock_mode": "gw"}),
        ]},
    "bridge": {
        "question": "How many image-speech bridge pairs does the island need, with and without back-translation?",
        "variants": [("20% bridge, back-translation (reference)", {"bridge_frac": 0.2})] + [
            (f"{int(100 * f)}% bridge, {'back-translation' if w else 'bridge pairs only'}", {"bridge_frac": f, "bt_weight": w})
            for f in (0.01, 0.05, 0.2) for w in (1.0, 0.0) if not (f == 0.2 and w == 1.0)] + [
            ("100% bridge (every island utterance paired)", {"bridge_frac": 1.0})]},
    "tone": {
        "question": "Does lexical tone need its own gated path into the hub?",
        "variants": [
            ("gated tone stream (ours)", {}),
            ("tone stream always on (no gate)", {"tone_mode": "on"}),
            ("no tone stream (single stream)", {"tone_mode": "off"}),
            ("gate without the sparsity price", {"w_gate": 0.0}),
            ("no paralinguistic bypass", {"use_bypass": False, "w_orth": 0.0}),
        ]},
    "speech": {
        "question": "What does the speech latent need in order to be said again?",
        "variants": [
            ("content tokens + a speaking path with time-ordered tokens", {"audio_anchor": "extra", "audio_private_slots": "speak"}),
            ("free queries only (no order of time)", {"audio_anchor": "off", "audio_private_slots": 0}),
            ("every token laid out in time, all read by the hub", {"audio_anchor": "all", "audio_private_slots": 0}),
        ]},
    "data": {
        "question": "What does extra data in the hub language (text, speech) buy the other languages?",
        "variants": [
            ("extra hub-language text and speech", {"extra_images": -1, "extra_speech": -1}),
            ("... the extra speech only listens (read into the hub, shaping none of it)",
             {"extra_images": -1, "extra_speech": -1, "listen_only": True}),
            ("extra hub-language text only", {"extra_images": -1, "extra_speech": 0}),
            ("extra hub-language speech only", {"extra_images": 0, "extra_speech": -1}),
            ("the benchmark's own data only", {"extra_images": 0, "extra_speech": 0}),
        ]},
    "regularise": {
        "question": "Can the encoders be regularised without losing what the credit loss buys?",
        "variants": [
            ("no encoder dropout (ours)", {"enc_dropout": 0.0, "speech_mask": 0.0}),
            ("encoder dropout 0.1 (grounder and explainer untouched)", {"enc_dropout": 0.1, "speech_mask": 0.0}),
            ("encoder dropout 0.1 + speech time masking", {"enc_dropout": 0.1, "speech_mask": 0.1}),
        ]},
    "text_unit": {
        "question": "Which text unit feeds the compressor?",
        "variants": [
            ("grapheme clusters over bytes (ours)", {}),
            ("raw bytes", {"text_unit": "byte"}),
            ("BPE learned on the hub language", {"text_unit": "bpe"}),
        ]},
    "private": {
        "question": "Do the writers need private (non-hub) degrees of freedom?",
        "variants": [
            ("hub + private Gaussian tokens (ours)", {}),
            ("private tokens fixed to zero (deterministic writers)", {"private_noise": False}),
        ]},
    "lora": {
        "question": "How much language-specific capacity do the velocity fields need?",
        "variants": [
            ("language LoRA, default rank (ours)", {}),
            ("no language LoRA (shared field + language embedding)", {"lora_rank": 0}),
            ("rank x4", {"lora_rank": "x4"}),
        ]},
    "protocol": {
        "question": "How much does an implicitly parallel corpus inflate the zero-pair numbers?",
        "variants": [
            ("strict non-parallel protocol (ours)", {}),
            ("captions visible in every language (implicitly parallel)", {"strict_nonparallel": False}),
            ("island orthography available (written upper bound)", {"use_island_text": True}),
        ]},
}

# headline columns: (pattern, short name, higher-is-better)
COLUMNS: List[Tuple[str, str, bool]] = [
    ("aud_{L}>txt_{H}/cf1", "island ST content F1", True),
    ("aud_{L}>txt_{H}/chance_cf1", "  ... chance level", False),
    ("aud_{L}>txt_{H}/bleu", "island ST BLEU", True),
    ("aud_{L}>txt_{H}/chrf", "island ST chrF++", True),
    ("aud_{L}>txt_{H}/content_acc", "island ST content acc", True),
    ("aud_{L}>txt_{H}/content2_acc", "  ... two-object sentences", True),
    ("aud_{L}>txt_{H}/color_acc", "island ST tone-word acc", True),
    ("aud_{L}>txt_{H}/flat_color_acc", "  ... with pitch removed", True),
    ("aud_{L}>img/r1", "island speech->image R@1", True),
    ("aud_{L}>img_gen/r1", "island speech->written image R@1", True),
    ("aud_{H}>txt_{H}/content_acc", "hub ASR content acc", True),
    ("aud_{H}>txt_{H}/cf1", "hub speech->text content F1", True),
    ("aud_{H}>txt_{H}/wer", "hub ASR WER", False),
    ("txt_{H}>txt_{H}/cf1", "text read+write content F1", True),
    ("img>txt_{H}/bleu_multi", "captioning BLEU (multi-ref)", True),
    ("txt_{H}>txt_{T}/content_acc", "MT hub->written content acc", True),
    ("txt_{H}>txt_{T}/bleu", "MT hub->written BLEU", True),
    ("txt_{T}>txt_{H}/bleu", "MT written->hub BLEU", True),
    ("txt_{T}>txt_{H}/cf1", "MT written->hub content F1", True),
    ("txt_{H}>img_gen/r1", "text->written image R@1", True),
    ("txt_{H}>aud_{L}/rt_bleu", "TTS into island (round-trip BLEU)", True),
    ("txt_{H}>aud_{L}/rt_content_acc", "TTS into island (round-trip content)", True),
    ("txt_{H}>txt_{H}/content2_acc", "text read+write, two-object sentences", True),
    ("ground/txt_{H}/both_covered", "both objects covered (text)", True),
    ("ground/aud_{L}/both_covered", "both objects covered (island speech)", True),
    ("ground/txt_{H}/slots_unmentioned", "slots on unmentioned objects (text)", False),
    ("tone/gate_{L}", "gate (island)", True),
    ("tone/gate_{H}", "gate (hub lang)", False),
]


class AblationRunner:
    def __init__(self, base: Config, seeds: Sequence[int] = (0,), test_items: int = 0):
        self.base, self.seeds, self.test_items = base, list(seeds), test_items
        self.root = os.path.join(base.out_root, f"{base.exp_name}_ablation")
        os.makedirs(self.root, exist_ok=True)
        self.log = get_logger("gb.ablation", os.path.join(self.root, "ablation.log"))

    def describe(self) -> None:
        for g, spec in GROUPS.items():
            print(f"\n[{g}] {spec['question']}")
            for name, over in spec["variants"]:
                print(f"    - {name:55s} {over}")

    # ------------------------------------------------------------------ one cell
    def _main_is_full(self, seed: int) -> bool:
        """The main run (``base.run_dir``) *is* the full model of the study when it has this seed and was - or
        will be - trained with exactly the base settings.  It is then never trained a second time."""

        if seed != self.base.seed:
            return False
        record = os.path.join(self.base.run_dir, "config.json")
        if not os.path.exists(record):
            return True   # not started yet: the full model is trained there and becomes the main run
        return not settings_diff(load_json(record), self.base.to_dict(), free=RESUME_FREE - {"phase_b_steps", "train_only"})

    def _resolve(self, over: Dict) -> Dict:
        """Overrides with relative values ("x4") turned into settings."""
        over = dict(over)
        if over.get("lora_rank") == "x4":
            over["lora_rank"] = self.base.lora_rank * 4
        if over.get("audio_private_slots") == "speak":   # the base's own speaking path, or one of twice the latent's length
            b = self.base
            over["audio_private_slots"] = b.audio_private_slots if b.audio_private_slots > b.private_slots else 3 * b.private_slots
        return over

    def _is_full(self, over: Dict) -> bool:
        """A variant whose settings are the base settings is the full model, wherever its group lists it."""
        return all(getattr(self.base, k) == v for k, v in self._resolve(over).items())

    def _cost(self, group: str, name: str, over: Dict, seed: int) -> int:
        """0: nothing to train (finished, or the full model); 1: one transport phase (its representation exists);
        2: a representation phase as well."""
        cfg, out_dir = self._cell(group, name, over, seed)
        if os.path.exists(os.path.join(out_dir, "results.json")) or self._is_full(over):
            return 0
        if cfg.phase_a_steps == 0 or os.path.exists(os.path.join(cfg.run_dir, "checkpoints", "last.pt")):
            return 1
        sig = cfg.signature(PHASE_A_KEYS)
        if os.path.exists(os.path.join(self.root, "_phase_a", sig, "checkpoints", "phase_a.pt")):
            return 1
        return 1 if seed == self.base.seed and sig == self.base.signature(PHASE_A_KEYS) else 2

    def _cheapest_first(self, cells: List[Tuple]) -> List[Tuple]:
        """A session has a time limit, so the order decides how many answers it gives.  Cells that need no new
        representation come first, in the order asked for; the rest follow grouped by the representation they share
        (largest group first), so that each extra representation phase is paid for once and used at once."""
        cost = [self._cost(*c) for c in cells]
        sig = [self._cell(*c)[0].signature(PHASE_A_KEYS) if k == 2 else "" for c, k in zip(cells, cost)]
        shared, first = Counter(sig), {}
        for i, sg in enumerate(sig):
            first.setdefault(sg, i)
        order = sorted(range(len(cells)), key=lambda i: (cost[i], -shared[sig[i]] if cost[i] == 2 else 0, first[sig[i]], i))
        return [cells[i] for i in order]

    def _cell(self, group: str, name: str, over: Dict, seed: int) -> Tuple[Config, str]:
        """(configuration to train, directory that holds the cell's results)."""
        over = self._resolve(over)
        if self._is_full(over):
            # the full model is the same run in every group: one run per seed, under _full/ ...
            own = self.base.clone(seed=seed, out_root=os.path.join(self.root, "_full"), exp_name=f"full_s{seed}")
            started = os.path.exists(os.path.join(own.run_dir, "checkpoints", "last.pt"))
            if not started and self._main_is_full(seed):
                return self.base.clone(), own.run_dir   # ... or the main run itself; only its results go there
            return own, own.run_dir
        slug = "".join(c if c.isalnum() else "_" for c in name).strip("_")[:48]
        cfg = self.base.clone(seed=seed, out_root=os.path.join(self.root, group), exp_name=f"{slug}_s{seed}", **over)
        return cfg, cfg.run_dir

    def _phase_a(self, cfg: Config) -> str:
        """The phase-A checkpoint shared by every configuration with this representation: an earlier one of the
        study, else the main run's when nothing that shapes phase A differs, else it is trained (once)."""
        sig = cfg.signature(PHASE_A_KEYS)
        a_cfg = cfg.clone(out_root=os.path.join(self.root, "_phase_a"), exp_name=sig, phase_b_steps=0, init_from="")
        path = os.path.join(a_cfg.run_dir, "checkpoints", "phase_a.pt")
        if os.path.exists(path):
            return path
        main, record = (os.path.join(self.base.run_dir, f) for f in ("checkpoints/phase_a.pt", "config.json"))
        if os.path.exists(main) and os.path.exists(record) and Config.load(record).signature(PHASE_A_KEYS) == sig:
            self.log.info(f"[phase A] representation {sig} taken from the main run: {main}")
            return main

        self.log.info(f"[phase A] training shared representation {sig}")
        Trainer(a_cfg).fit()
        return path

    def run_cell(self, group: str, name: str, over: Dict, seed: int) -> Dict[str, float]:

        cfg, out_dir = self._cell(group, name, over, seed)
        res_path = os.path.join(out_dir, "results.json")
        if os.path.exists(res_path):
            return load_json(res_path)
        is_main = cfg.run_dir == self.base.run_dir
        if cfg.phase_a_steps > 0 and not is_main and not os.path.exists(os.path.join(cfg.run_dir, "checkpoints", "last.pt")):
            cfg.init_from = self._phase_a(cfg)
        self.log.info(f"[cell] {group} / {name} / seed {seed}" + (f"  (the main run, {cfg.run_dir})" if is_main else ""))
        tr = Trainer(cfg)
        tr.fit()
        # every cell is scored the same way: the state its own validation score selected, on the test split
        best = tr.best["step"] if tr.best["step"] >= 0 else tr.step
        os.makedirs(out_dir, exist_ok=True)
        want = {"step": best, "split": "test", "items": self.test_items, "quick": False, **eval_settings(cfg)}
        stored = os.path.join(cfg.run_dir, "eval_test.json")   # written by the "eval" stage of the main run
        if is_main and os.path.exists(stored) and load_json(stored).get("meta") == json.loads(json.dumps(want)):
            res = load_json(stored)["metrics"]                  # the same state, split and settings: not measured twice
            self.log.info(f"[cell] test scores taken from {stored}")
        else:
            model, _ = load_model(cfg, tr.data, tr.device, os.path.join(cfg.run_dir, "checkpoints", "best.pt"))
            res = Evaluator(cfg, tr.data, tr.device).evaluate(model, "test", self.test_items, quick=False, step=best,
                                                             save_path=os.path.join(out_dir, "eval_test.json"))
        res["checkpoint_step"] = best
        res["params_total"] = tr.param_report["total"]
        res["per_language_pct"] = tr.param_report["per_language_pct_of_spokes"]
        save_json(res, res_path)
        return res

    # ------------------------------------------------------------------ whole study
    def run(self, groups: Optional[Sequence[str]] = None, only: Optional[Sequence[str]] = None) -> Dict:
        """Run every variant of the given groups (all groups by default).  ``only`` restricts the run to
        variants whose name contains one of the given substrings; the full model is always included."""
        groups = list(groups) if groups else list(GROUPS)
        cells = [(g, name, over, s) for g in groups for name, over in GROUPS[g]["variants"]
                 if not only or self._is_full(over) or any(o.lower() in name.lower() for o in only) for s in self.seeds]
        cells = self._cheapest_first(cells)
        dirs = {self._cell(*c)[1] for c in cells}
        todo = [d for d in dirs if not os.path.exists(os.path.join(d, "results.json"))]
        own_a = len({self._cell(*c)[0].signature(PHASE_A_KEYS) for c in cells if self._cost(*c) == 2})
        self.log.info(f"[ablation] {len(dirs)} runs in {len(groups)} group(s), {len(todo)} without results yet.  Order: the "
                      f"full model, then variants that reuse a representation that exists (one transport phase each), "
                      f"then the others ({own_a} more representation phase(s) to train).")
        try:
            for c in cells:
                self.run_cell(*c)
        except OutOfTime:
            self.collect(groups)   # tables of what is finished; the interrupted cell continues on the next run
            raise
        return self.collect(groups)

    def collect(self, groups: Optional[Sequence[str]] = None) -> Dict:
        """Rebuild tables from finished cells only (no training)."""
        summary: Dict = {}
        for g in (list(groups) if groups else list(GROUPS)):
            rows: Dict[str, List[Dict]] = {}
            used = set()
            # the full model first, whatever its place in the list; a second variant that is the same run under
            # these settings is left out (two rows of one run are not a comparison)
            for name, over in sorted(GROUPS[g]["variants"], key=lambda v: not self._is_full(v[1])):
                paths = tuple(os.path.join(self._cell(g, name, over, s)[1], "results.json") for s in self.seeds)
                done = [load_json(p) for p in paths if os.path.exists(p)]
                if done and paths not in used:
                    rows[name + (" (the full model)" if self._is_full(over) and "(ours)" not in name else "")] = done
                    used.add(paths)
            if len(rows) >= 2:  # a group with only the full model has nothing to compare
                summary[g] = self._report(g, GROUPS[g]["question"], rows)
        self._write_summary(summary)
        return summary

    # ------------------------------------------------------------------ reporting
    def _columns(self, rows: Dict[str, List[Dict]]) -> List[Tuple[str, str, bool]]:
        b = self.base
        H = b.hub_langs[0]
        L = b.island_langs[0] if b.island_langs else "-"
        T = next((t for t in b.text_only_langs if t != H), "-")
        keys = set(k for rs in rows.values() for r in rs for k in r)
        out = []
        for pat, short, hib in COLUMNS:
            k = pat.format(H=H, L=L, T=T)
            if k in keys:
                out.append((k, short, hib))
        return out

    def _report(self, group: str, question: str, rows: Dict[str, List[Dict]]) -> Dict:
        cols = self._columns(rows)
        stats = {name: {k: (float(np.mean([r[k] for r in rs if k in r])) if any(k in r for r in rs) else float("nan"),
                            float(np.std([r[k] for r in rs if k in r])) if sum(k in r for r in rs) > 1 else 0.0)
                        for k, _, _ in cols} for name, rs in rows.items()}
        best = {}
        for k, _, hib in cols:
            vals = {n: s[k][0] for n, s in stats.items() if not np.isnan(s[k][0])}
            if vals and max(vals.values()) - min(vals.values()) > 1e-9:  # no winner when every variant ties
                best[k] = (max if hib else min)(vals, key=vals.get)
        n_seeds = max(len(r) for r in rows.values())
        fmt = lambda n, k: (f"**{stats[n][k][0]:.2f}**" if best.get(k) == n else f"{stats[n][k][0]:.2f}") + \
                           (f" ± {stats[n][k][1]:.2f}" if n_seeds > 1 else "")
        md = [f"### {group}: {question}", "", f"_{n_seeds} seed(s); test split; mean" + (" ± std" if n_seeds > 1 else "") +
              "; bold = best in column (arrows give the direction)._", "",
              "| variant | " + " | ".join(f"{s} {'↑' if h else '↓'}" for _, s, h in cols) + " |",
              "|---|" + "---|" * len(cols)]
        for n in rows:
            md.append(f"| {n} | " + " | ".join(fmt(n, k) for k, _, _ in cols) + " |")
        with open(os.path.join(self.root, f"table_{group}.md"), "w", encoding="utf8") as f:
            f.write("\n".join(md) + "\n")
        with open(os.path.join(self.root, f"table_{group}.csv"), "w", encoding="utf8") as f:
            f.write("variant," + ",".join(f"{s} mean,{s} std" for _, s, _ in cols) + "\n")
            for n in rows:
                f.write('"' + n + '",' + ",".join(f"{stats[n][k][0]:.4f},{stats[n][k][1]:.4f}" for k, _, _ in cols) + "\n")
        tex = ["\\begin{tabular}{l" + "r" * len(cols) + "}", "\\toprule",
               "Variant & " + " & ".join(s.replace("%", "\\%").replace("_", "\\_").replace(">", "$\\to$") for _, s, _ in cols) + " \\\\",
               "\\midrule"]
        for n in rows:
            cells = []
            for k, _, _ in cols:
                v = f"{stats[n][k][0]:.2f}" + (f"$\\pm${stats[n][k][1]:.2f}" if n_seeds > 1 else "")
                cells.append(f"\\textbf{{{v}}}" if best.get(k) == n else v)
            tex.append(n.replace("%", "\\%").replace("_", "\\_").replace(">", "$\\to$") + " & " + " & ".join(cells) + " \\\\")
        tex += ["\\bottomrule", "\\end{tabular}"]
        with open(os.path.join(self.root, f"table_{group}.tex"), "w", encoding="utf8") as f:
            f.write("\n".join(tex) + "\n")
        self._plot(group, question, rows, cols, stats)
        save_json({"question": question, "stats": {n: {k: list(v) for k, v in s.items()} for n, s in stats.items()},
                   "best": best, "seeds": n_seeds}, os.path.join(self.root, f"table_{group}.json"))
        return {"md": "\n".join(md), "best": best, "cols": cols, "stats": stats}

    def _plot(self, group, question, rows, cols, stats) -> None:

        setup_matplotlib()
        import matplotlib.pyplot as plt

        cols = cols[:8]
        if not cols:
            return
        names = list(rows)
        fig, axes = plt.subplots(1, len(cols), figsize=(3.4 * len(cols), 0.42 * len(names) + 2.6), squeeze=False)
        for ax, (k, short, hib) in zip(axes[0], cols):
            means = [stats[n][k][0] for n in names]
            stds = [stats[n][k][1] for n in names]
            colors = ["#d62728" if i == 0 else "#7f9ec9" for i in range(len(names))]
            ax.barh(range(len(names)), means, xerr=stds, color=colors, capsize=2)
            ax.set_yticks(range(len(names)), names if ax is axes[0, 0] else [""] * len(names), fontsize=7)
            ax.invert_yaxis()
            ax.set_title(f"{short} {'↑' if hib else '↓'}", fontsize=8)
            ax.grid(axis="x", alpha=0.3)
        fig.suptitle(f"{group}: {question}   (red = full model)", fontsize=9)
        fig.tight_layout()
        fig.savefig(os.path.join(self.root, f"plot_{group}.png"), dpi=130)
        plt.close(fig)

    def _write_summary(self, summary: Dict) -> None:
        md = [f"# Ablation study: {self.base.exp_name}", "",
              f"Dataset `{self.base.dataset}`, seeds {self.seeds}.  Every number is measured on the test split, "
              "through the hub, with no image and no text of the island language at test time, at the state each "
              "run's own validation score selected.  The first row of every table is the full model.", ""]
        for g, rep in summary.items():
            md += [rep["md"], ""]
            ours = next(iter(rep["stats"]))
            wins = [s for k, s, _ in rep["cols"] if rep["best"].get(k) == ours]
            loses = [f"{s} (best: {rep['best'][k]})" for k, s, _ in rep["cols"] if k in rep["best"] and rep["best"][k] != ours]
            md += [f"Full model best on: {', '.join(wins) if wins else 'none'}.",
                   f"Full model not best on: {', '.join(loses) if loses else 'none'}.", "", f"![{g}](plot_{g}.png)", ""]
        with open(os.path.join(self.root, "SUMMARY.md"), "w", encoding="utf8") as f:
            f.write("\n".join(md) + "\n")
        self.log.info(f"[ablation] tables and plots in {self.root}")


# ======================================================================================================================
# 24. PIPELINE STAGES
#     (geobridge/cli.py)
# ======================================================================================================================
# Pipeline stages and the command line.
#
#     prepare   download the sources, write the split JSON, build the one-time feature cache
#     train     two-phase training (resumes from the last checkpoint by itself)
#     eval      metrics of best.pt on a split
#     infer     figures, internal views of the model, saved outputs
#     ablate    the ablation suite and its tables
#
#     python run.py prepare --preset flickr8k_yo
#     python run.py train   --preset flickr8k_yo
#     python run.py eval    --preset flickr8k_yo
#     python run.py infer   --preset flickr8k_yo
#     python run.py ablate  --preset synth --groups hub island tone
#     python run.py all     --preset synth
#
# Any config field can be overridden:  --set bridge_frac=0.05 hub_mode=global seed=1
# ======================================================================================================================

STAGES = ("prepare", "train", "eval", "infer", "ablate")


def build_config(preset: str, overrides: Optional[Dict[str, Any]] = None, exp_name: str = "") -> Config:
    cfg = Config.preset(preset).update(dict(overrides or {}))
    if exp_name:
        cfg.exp_name = exp_name
    elif cfg.exp_name == "geobridge":
        cfg.exp_name = preset
    return cfg


def release_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_stages(cfg: Config, stages: Sequence[str], split: str = "test", checkpoint: str = "", resume_from: str = "",
               groups: Optional[Sequence[str]] = None, only: Optional[Sequence[str]] = None,
               seeds: Sequence[int] = (0,), collect: bool = False) -> Dict[str, Any]:
    """Run the requested stages in pipeline order.  Every stage is restartable: finished work (downloads,
    caches, checkpoints, ablation cells) is found on disk and skipped."""
    unknown = [s for s in stages if s not in STAGES]
    if unknown:
        raise ValueError(f"unknown stage(s) {unknown}; choose from {STAGES}")
    if cfg.torch_threads:
        torch.set_num_threads(cfg.torch_threads)
    out: Dict[str, Any] = {}
    data: Optional[GeoBridgeData] = None
    if "prepare" in stages or "train" in stages:
        data = GeoBridgeData(cfg, get_logger(), resolve_device(cfg.device))
        out["data"] = data.describe()
    if "train" in stages:
        trainer = Trainer(cfg, data)
        if resume_from:
            trainer.resume_from(resume_from)
        out["valid"] = trainer.fit()
        del trainer
        release_memory()
    if "eval" in stages or "infer" in stages:
        inf = Inferencer(cfg, checkpoint=checkpoint or None, data=data)
        if "eval" in stages:
            out[split] = inf.evaluate(split)
        if "infer" in stages:
            out["report"] = inf.run_all(split)
        del inf
        release_memory()
    if "ablate" in stages:
        runner = AblationRunner(cfg, seeds=seeds)
        out["ablation"] = runner.collect(groups) if collect else runner.run(groups, only)
    return out


def main(argv: Optional[Sequence[str]] = None) -> Optional[int]:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("command", choices=[*STAGES, "all"])
    ap.add_argument("--preset", default="flickr8k_yo")
    ap.add_argument("--exp_name", default="")
    ap.add_argument("--set", nargs="*", default=[], metavar="KEY=VALUE")
    ap.add_argument("--resume_from", default="", help="explicit checkpoint (any step) instead of last.pt")
    ap.add_argument("--checkpoint", default="", help="checkpoint for eval / infer (default: best.pt)")
    ap.add_argument("--split", default="test")
    ap.add_argument("--groups", nargs="*", default=[], help="ablation groups (default: all)")
    ap.add_argument("--seeds", nargs="*", type=int, default=[0])
    ap.add_argument("--list", action="store_true", help="list ablation groups and exit")
    ap.add_argument("--only", nargs="*", default=[], help="run only variants whose name contains one of these")
    ap.add_argument("--collect", action="store_true", help="rebuild ablation tables from finished cells only")
    ap.add_argument("--hours", type=float, default=0.0,
                    help="stop training cleanly after this many hours (the step is saved; run again to continue)")
    args = ap.parse_args(argv)
    set_time_budget(args.hours)
    cfg = build_config(args.preset, dict(kv.split("=", 1) for kv in args.set), args.exp_name)
    if args.command == "ablate" and args.list:
        AblationRunner(cfg, seeds=args.seeds).describe()
        return
    stages = STAGES if args.command == "all" else (args.command,)
    try:
        out = run_stages(cfg, stages, split=args.split, checkpoint=args.checkpoint, resume_from=args.resume_from,
                         groups=args.groups or None, only=args.only or None, seeds=args.seeds, collect=args.collect)
    except OutOfTime as e:
        get_logger().info(f"[time] {e}")
        return 75   # EX_TEMPFAIL: not finished, safe to run again
    if args.command == "prepare":
        print(json.dumps(out["data"], indent=2, default=str))
    if args.command == "eval":
        print(json.dumps({k: round(v, 3) for k, v in out[args.split].items()}, indent=1))


# ======================================================================================================================
# 25. RUN
# ======================================================================================================================
def describe_runtime() -> None:
    log = get_logger()
    where = "Kaggle" if IN_KAGGLE else "Colab" if IN_COLAB else "local"
    log.info(f"[setup] {where} | torch {torch.__version__} | runs -> {WORK_DIR} | data -> {DATA_DIR}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            log.info(f"[setup] cuda:{i} {p.name} {p.total_memory / 2 ** 30:.1f} GB")
        log.info(f"[setup] {torch.cuda.device_count()} GPU(s): one batch per GPU in every training step")
    else:
        log.info("[setup] no GPU visible: this will run, slowly, on the CPU (pick a 'synth' preset to try it out)")


def show_images(paths: Sequence[str], width: int = 1100) -> None:
    """Display saved figures below the cell (no-op outside a notebook)."""
    try:
        from IPython import get_ipython
        from IPython.display import Image, display
    except ImportError:
        return
    if get_ipython() is None:
        return
    for p in paths:
        if os.path.exists(p):
            print(os.path.relpath(p, WORK_DIR))
            display(Image(filename=p, width=width))


def main_notebook() -> Dict[str, Any]:
    describe_runtime()
    log = get_logger()
    set_time_budget(TIME_BUDGET_HOURS, SESSION_START)
    if TIME_BUDGET_HOURS:
        log.info(f"[setup] session started {(time.time() - SESSION_START) / 60:.0f} min ago; training stops cleanly "
                 f"in {time_left() / 3600:.1f} h (TIME_BUDGET_HOURS = {TIME_BUDGET_HOURS})")
    results: Dict[str, Any] = {}
    paths = {"out_root": os.path.join(WORK_DIR, "runs"), "data_root": DATA_DIR}
    cfg = build_config(PRESET, {**paths, **OVERRIDES}, EXP_NAME)
    # the ablation of PRESET is the ablation of the run above: same settings, same folder name
    same = ABLATION_PRESET == PRESET
    a_cfg = cfg if "ablate" in STAGES or not ABLATION_PRESET else build_config(
        ABLATION_PRESET, {**paths, **(OVERRIDES if same else {}), **ABLATION_OVERRIDES}, EXP_NAME if same else "")
    a_root = os.path.join(a_cfg.out_root, f"{a_cfg.exp_name}_ablation")
    ablation = dict(groups=ABLATION_GROUPS or None, only=ABLATION_ONLY or None, seeds=ABLATION_SEEDS)
    try:
        if STAGES:
            results = run_stages(cfg, STAGES, split=EVAL_SPLIT, **ablation)
        if ABLATION_PRESET and "ablate" not in STAGES:
            results["ablation"] = run_stages(a_cfg, ("ablate",), **ablation)["ablation"]
    except OutOfTime as e:
        log.info(f"[time] {e}")
        log.info("[time] stopped before the session limit: run this cell again (same settings) to continue")
    if EVAL_SPLIT in results:
        keep = {k: round(v, 2) for k, v in results[EVAL_SPLIT].items()
                if k.endswith(("/cf1", "/chance_cf1", "/bleu", "/chance_bleu", "/wer", "/r1", "content_acc"))
                or k.startswith("tone/") or k == "score"}
        print(json.dumps(keep, indent=1))
    if STAGES and SHOW_FIGURES:
        show_images(sorted(glob.glob(os.path.join(cfg.run_dir, "curves", "*.png"))))
        show_images(sorted(glob.glob(os.path.join(cfg.run_dir, "inference", "*.png"))))
    if ABLATION_PRESET or "ablate" in STAGES:
        summary = os.path.join(a_root, "SUMMARY.md")
        if os.path.exists(summary):
            print(open(summary, encoding="utf8").read())
        if SHOW_FIGURES:
            show_images(sorted(glob.glob(os.path.join(a_root, "*.png"))))
    print(f"\nEverything is in {WORK_DIR}")
    return results


RESULTS = main_notebook()
